# from pathlib import Path

DATA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)

DATA_DIR.exists()


In [ ]:
for f in sorted(DATA_DIR.iterdir()):
    print(f.name)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import re
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files\n")

for file in csv_files:
    print(file.name)

In [ ]:
schema = []

for file in csv_files:
    df = pd.read_csv(
        file,
        encoding="latin1",
        nrows=5
    )
    
    for col in df.columns:
        schema.append({
            "file": file.name,
            "column": col
        })

schema_df = pd.DataFrame(schema)

schema_df

In [ ]:
for file in csv_files:
    
    df = pd.read_csv(
        file,
        encoding="latin1",
        nrows=5
    )
    
    print("\n" + "=" * 100)
    print(file.name)
    print("=" * 100)
    print("Columns:")
    print(list(df.columns))

In [ ]:
def load_csv(name):
    path = DATA_DIR / name
    
    return pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

In [ ]:
food = load_csv("food.csv")
food_attribute = load_csv("food_attribute.csv")
food_attribute_type = load_csv("food_attribute_type.csv")
food_category = load_csv("food_category.csv")

food_calorie_conversion_factor = load_csv(
    "food_calorie_conversion_factor.csv"
)

food_component = load_csv("food_component.csv")
food_nutrient = load_csv("food_nutrient.csv")
food_nutrient_conversion_factor = load_csv(
    "food_nutrient_conversion_factor.csv"
)

food_portion = load_csv("food_portion.csv")
food_protein_conversion_factor = load_csv(
    "food_protein_conversion_factor.csv"
)

foundation_food = load_csv("foundation_food.csv")

input_food = load_csv("input_food.csv")

lab_method = load_csv("lab_method.csv")
lab_method_code = load_csv("lab_method_code.csv")
lab_method_nutrient = load_csv("lab_method_nutrient.csv")

market_acquisition = load_csv("market_acquisition.csv")
measure_unit = load_csv("measure_unit.csv")
nutrient = load_csv("nutrient.csv")

sample_food = load_csv("sample_food.csv")
sub_sample_food = load_csv("sub_sample_food.csv")
sub_sample_result = load_csv("sub_sample_result.csv")

acquisition_samples = load_csv("acquisition_samples.csv")
agricultural_samples = load_csv("agricultural_samples.csv")

food_update_log_entry = load_csv(
    "food_update_log_entry.csv"
)

In [ ]:
tables = {
    "food": food,
    "food_attribute": food_attribute,
    "food_attribute_type": food_attribute_type,
    "food_category": food_category,
    "food_calorie_conversion_factor": food_calorie_conversion_factor,
    "food_component": food_component,
    "food_nutrient": food_nutrient,
    "food_nutrient_conversion_factor": food_nutrient_conversion_factor,
    "food_portion": food_portion,
    "food_protein_conversion_factor": food_protein_conversion_factor,
    "foundation_food": foundation_food,
    "input_food": input_food,
    "lab_method": lab_method,
    "lab_method_code": lab_method_code,
    "lab_method_nutrient": lab_method_nutrient,
    "market_acquisition": market_acquisition,
    "measure_unit": measure_unit,
    "nutrient": nutrient,
    "sample_food": sample_food,
    "sub_sample_food": sub_sample_food,
    "sub_sample_result": sub_sample_result,
    "acquisition_samples": acquisition_samples,
    "agricultural_samples": agricultural_samples,
    "food_update_log_entry": food_update_log_entry,
}

for name, df in tables.items():
    print(
        f"{name:45s} "
        f"rows={len(df):>10,} "
        f"columns={len(df.columns):>3}"
    )

In [ ]:
for name, df in tables.items():
    print("\n")
    print("=" * 100)
    print(name)
    print("=" * 100)
    print(list(df.columns))

In [ ]:
def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")


for name, df in tables.items():
    df.columns = [
        clean_column_name(c)
        for c in df.columns
    ]

In [ ]:
for name, df in tables.items():
    print(name)
    print(df.columns.tolist())
    print()

In [ ]:
food.head()

In [ ]:
food.info()

In [ ]:
print("Rows:", len(food))
print("Unique FDC IDs:", food["fdc_id"].nunique())
print("Missing FDC IDs:", food["fdc_id"].isna().sum())
print("Duplicate FDC IDs:", food["fdc_id"].duplicated().sum())

In [ ]:
def id_columns(df):
    return [
        c for c in df.columns
        if (
            c.endswith("_id")
            or c == "id"
            or "fdc_id" in c
        )
    ]


for name, df in tables.items():
    print(f"\n{name}")
    print(id_columns(df))

In [ ]:
food_ids = set(
    food["fdc_id"]
    .dropna()
    .astype("int64")
)

for name, df in tables.items():
    
    if "fdc_id" not in df.columns:
        continue
    
    ids = set(
        pd.to_numeric(
            df["fdc_id"],
            errors="coerce"
        )
        .dropna()
        .astype("int64")
    )
    
    missing = ids - food_ids
    
    print(
        f"{name:45s} "
        f"unique FDC IDs={len(ids):>8,} "
        f"not in food={len(missing):>8,}"
    )

In [ ]:
food_nutrient.head()

In [ ]:
nutrient.head()

In [ ]:
print(food_nutrient.columns.tolist())
print(nutrient.columns.tolist())

In [ ]:
food_nutrient["fdc_id"].nunique()
food_nutrient["nutrient_id"].nunique()

In [ ]:
food_nutrient["nutrient_id"].isna().sum()
food_nutrient["fdc_id"].isna().sum()

In [ ]:
nutrient_ids = set(
    nutrient["id"]
    .dropna()
    .astype("int64")
)

food_nutrient_ids = set(
    food_nutrient["nutrient_id"]
    .dropna()
    .astype("int64")
)

missing_nutrients = food_nutrient_ids - nutrient_ids

print(
    "Nutrient IDs not found in nutrient table:",
    len(missing_nutrients)
)

if missing_nutrients:
    print(sorted(missing_nutrients)[:20])

In [ ]:
nutrient_lookup = nutrient.copy()

nutrient_lookup = nutrient_lookup.rename(
    columns={
        c: f"nutrient_{c}"
        for c in nutrient_lookup.columns
        if c != "id"
    }
)

nutrient_lookup = nutrient_lookup.rename(
    columns={
        "nutrient_id": "nutrient_id"
    }
    if "nutrient_id" in nutrient_lookup.columns
    else {}
)

In [ ]:
nutrient_lookup.head()

In [ ]:
nutrient.columns

In [ ]:
food_nutrient_with_metadata = food_nutrient.merge(
    nutrient_lookup,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

In [ ]:
food_nutrient.columns.tolist()

In [ ]:
food_nutrient.head(20)

In [ ]:
food_nutrient.groupby(
    ["fdc_id", "nutrient_id"]
).size().value_counts().sort_index()

In [ ]:
nutrient_wide = (
    food_nutrient_with_metadata
    .pivot_table(
        index="fdc_id",
        columns="nutrient_name",
        values="amount",
        aggfunc="mean"
    )
    .reset_index()
)

In [ ]:
print(food_nutrient.columns.tolist())

In [ ]:
print(nutrient.columns.tolist())

In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 120)
    print(name)
    print("=" * 120)
    
    print("SHAPE:", df.shape)
    print("COLUMNS:")
    
    for i, col in enumerate(df.columns, 1):
        print(f"{i:3}. {col}")

In [ ]:
important_tables = [
    "food",
    "foundation_food",
    "food_nutrient",
    "nutrient",
    "food_portion",
    "measure_unit",
    "food_attribute",
    "food_attribute_type",
    "food_component",
    "sample_food",
    "sub_sample_food",
    "sub_sample_result",
    "lab_method",
    "lab_method_code",
    "lab_method_nutrient",
    "acquisition_samples",
    "market_acquisition",
    "agricultural_samples",
    "input_food",
]

for name in important_tables:
    df = tables[name]
    
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(df.head(3).to_string())


In [ ]:
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)

print("Folder exists:", DATA_DIR.exists())

for f in sorted(DATA_DIR.glob("*.csv")):
    print(f.name)


In [ ]:
def load_csv(filename):
    return pd.read_csv(
        DATA_DIR / filename,
        encoding="latin1",
        low_memory=False
    )


tables = {}

for file in DATA_DIR.glob("*.csv"):
    tables[file.stem] = load_csv(file.name)

print(f"Loaded {len(tables)} CSV files.")


In [ ]:
for name, df in tables.items():
    print(
        f"{name:45s} "
        f"rows = {len(df):>10,}   "
        f"columns = {len(df.columns):>3}"
    )


In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


In [ ]:
important_tables = [
    "food",
    "foundation_food",
    "food_nutrient",
    "nutrient",
    "food_portion",
    "measure_unit",
    "food_attribute",
    "food_attribute_type",
    "food_component",
    "sample_food",
    "sub_sample_food",
    "sub_sample_result",
    "lab_method",
    "lab_method_code",
    "lab_method_nutrient",
    "acquisition_samples",
    "market_acquisition",
    "agricultural_samples",
    "input_food",
]

for name in important_tables:
    print("\n" + "#" * 100)
    print(name)
    print("#" * 100)
    print(tables[name].head(3).to_string(index=False))


In [ ]:
foundation_ids = set(
    foundation_food["fdc_id"]
    .dropna()
    .astype("int64")
)

food_ids = set(
    food["fdc_id"]
    .dropna()
    .astype("int64")
)

print("Foundation Foods:", len(foundation_ids))
print("Food records:", len(food_ids))

print(
    "Foundation food IDs missing from food:",
    len(foundation_ids - food_ids)
)

print(
    "Foundation food IDs found in food:",
    len(foundation_ids & food_ids)
)


In [ ]:
food["data_type"].value_counts(dropna=False)


In [ ]:
food[
    food["fdc_id"].isin(foundation_ids)
]["data_type"].value_counts(dropna=False)


In [ ]:
foundation_food_nutrients = food_nutrient[
    food_nutrient["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Nutrient records belonging directly to Foundation Foods:",
    len(foundation_food_nutrients)
)

print(
    "Foundation Foods with nutrients:",
    foundation_food_nutrients["fdc_id"].nunique()
)

print(
    "Foundation Foods without nutrients:",
    len(
        foundation_ids
        - set(
            foundation_food_nutrients["fdc_id"]
            .dropna()
            .astype("int64")
        )
    )
)


In [ ]:
nutrient_duplicates = (
    foundation_food_nutrients
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

nutrient_duplicates[
    nutrient_duplicates["count"] > 1
].sort_values(
    "count",
    ascending=False
).head(20)


In [ ]:
print(
    "Unique food/nutrient combinations:",
    len(nutrient_duplicates)
)

print(
    "Food/nutrient combinations with >1 record:",
    (
        nutrient_duplicates["count"] > 1
    ).sum()
)


In [ ]:
foundation_food_nutrients.merge(
    nutrient,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    validate="many_to_one"
).head(20)


In [ ]:
nutrient_check = foundation_food_nutrients.merge(
    nutrient,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

nutrient_check["_merge"].value_counts()


In [ ]:
nutrient_check[
    [
        "fdc_id",
        "nutrient_id",
        "amount",
        "name",
        "unit_name",
        "nutrient_nbr"
    ]
].head(30)


In [ ]:
foundation_portions = food_portion[
    food_portion["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Portion records for Foundation Foods:",
    len(foundation_portions)
)

print(
    "Foundation Foods having portions:",
    foundation_portions["fdc_id"].nunique()
)


In [ ]:
portion_unit_check = foundation_portions.merge(
    measure_unit,
    left_on="measure_unit_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

portion_unit_check["_merge"].value_counts()


In [ ]:
foundation_attributes = food_attribute[
    food_attribute["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Attribute records:",
    len(foundation_attributes)
)

print(
    "Foundation Foods with attributes:",
    foundation_attributes["fdc_id"].nunique()
)


In [ ]:
attribute_check = foundation_attributes.merge(
    food_attribute_type,
    left_on="food_attribute_type_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

attribute_check["_merge"].value_counts()


In [ ]:
category_check = (
    food[
        food["fdc_id"].isin(foundation_ids)
    ]
    .merge(
        food_category,
        left_on="food_category_id",
        right_on="id",
        how="left",
        indicator=True,
        validate="many_to_one"
    )
)

category_check["_merge"].value_counts()


In [ ]:
category_check[
    [
        "fdc_id",
        "description",
        "food_category_id",
        "code",
        "food_category"
        if "food_category" in category_check.columns
        else "description_y"
    ]
].head()


In [ ]:
category_check.columns.tolist()


In [ ]:
sample_result_check = sub_sample_result.merge(
    food_nutrient[
        [
            "id",
            "fdc_id",
            "nutrient_id",
            "amount"
        ]
    ],
    left_on="food_nutrient_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

sample_result_check["_merge"].value_counts()


In [ ]:
sample_result_check.head(20)


In [ ]:
sample_ids = set(
    sample_food["fdc_id"]
    .dropna()
    .astype("int64")
)

sub_sample_parent_ids = set(
    sub_sample_food["fdc_id_of_sample_food"]
    .dropna()
    .astype("int64")
)

print(
    "Sub-sample parent IDs not found in sample_food:",
    len(
        sub_sample_parent_ids - sample_ids
    )
)


In [ ]:
sub_sample_check = sub_sample_food.merge(
    sample_food,
    left_on="fdc_id_of_sample_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_sample"),
)

sub_sample_check["_merge"].value_counts()


In [ ]:
acquisition_sample_check = acquisition_samples.merge(
    food[
        [
            "fdc_id",
            "data_type",
            "description"
        ]
    ],
    left_on="fdc_id_of_sample_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_sample"),
)

acquisition_sample_check["_merge"].value_counts()


In [ ]:
acquisition_sample_check.head(20)


In [ ]:
acquisition_food_check = acquisition_samples.merge(
    food[
        [
            "fdc_id",
            "data_type",
            "description"
        ]
    ],
    left_on="fdc_id_of_acquisition_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_acquisition"),
)

acquisition_food_check["_merge"].value_counts()


In [ ]:
foundation_type_ids = set(
    food.loc[
        food["data_type"] == "foundation_food",
        "fdc_id"
    ].dropna().astype("int64")
)

print("food.csv data_type='foundation_food':", len(foundation_type_ids))
print("foundation_food.csv records:", len(foundation_ids))

print(
    "In food.csv but NOT foundation_food.csv:",
    len(foundation_type_ids - foundation_ids)
)

print(
    "In foundation_food.csv but NOT food.csv:",
    len(foundation_ids - foundation_type_ids)
)


In [ ]:
extra_foundation_type_ids = sorted(
    foundation_type_ids - foundation_ids
)

extra_foundation_type_ids[:50]


In [ ]:
food[
    food["fdc_id"].isin(extra_foundation_type_ids)
].sort_values("fdc_id").head(100)


In [ ]:
duplicate_food_nutrients = (
    foundation_food_nutrients
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

duplicate_food_nutrients = duplicate_food_nutrients[
    duplicate_food_nutrients["count"] > 1
]

duplicate_food_nutrients


In [ ]:
duplicate_pairs = duplicate_food_nutrients[
    ["fdc_id", "nutrient_id"]
].merge(
    foundation_food_nutrients,
    on=["fdc_id", "nutrient_id"],
    how="left"
)

duplicate_pairs


In [ ]:
duplicate_pairs.merge(
    nutrient[
        ["id", "name", "unit_name", "nutrient_nbr"]
    ],
    left_on="nutrient_id",
    right_on="id",
    how="left",
    suffixes=("", "_nutrient")
)[
    [
        "fdc_id",
        "nutrient_id",
        "name",
        "unit_name",
        "amount",
        "data_points",
        "derivation_id",
        "min",
        "max",
        "median",
        "footnote",
        "min_year_acquired"
    ]
]


In [ ]:
unmatched_nutrients = nutrient_check[
    nutrient_check["_merge"] == "left_only"
].copy()

print(unmatched_nutrients.shape)

unmatched_nutrients[
    [
        "fdc_id",
        "nutrient_id",
        "amount",
        "data_points",
        "derivation_id",
        "min",
        "max",
        "median"
    ]
].head(100)


In [ ]:
print(
    "Unique unmatched nutrient IDs:",
    unmatched_nutrients["nutrient_id"].unique()
)


In [ ]:
nutrient[
    nutrient["id"].isin(
        unmatched_nutrients["nutrient_id"].dropna()
    )
]


In [ ]:
orphan_sample_parent_ids = sorted(
    sub_sample_parent_ids - sample_ids
)

print(orphan_sample_parent_ids)


In [ ]:
sub_sample_food[
    sub_sample_food["fdc_id_of_sample_food"].isin(
        orphan_sample_parent_ids
    )
]


In [ ]:
food[
    food["fdc_id"].isin(
        orphan_sample_parent_ids
    )
]


In [ ]:
unmatched_sample_results = sample_result_check[
    sample_result_check["_merge"] == "left_only"
].copy()

unmatched_sample_results


In [ ]:
unmatched_sample_results[
    [
        "food_nutrient_id",
        "adjusted_amount",
        "lab_method_id",
        "nutrient_name"
    ]
]


In [ ]:
sub_sample_result[
    sub_sample_result["food_nutrient_id"].isin(
        unmatched_sample_results["food_nutrient_id"]
    )
]


In [ ]:
foundation_components = food_component[
    food_component["fdc_id"].isin(foundation_ids)
].copy()

print("Foundation component rows:", len(foundation_components))
print(
    "Foundation foods with components:",
    foundation_components["fdc_id"].nunique()
)

print(
    "Rows with missing fdc_id:",
    food_component["fdc_id"].isna().sum()
)


In [ ]:
foundation_components.head(20)


In [ ]:
foundation_inputs = input_food[
    input_food["fdc_id"].isin(foundation_ids)
].copy()

print("Input-food rows:", len(foundation_inputs))
print(
    "Foundation foods with input foods:",
    foundation_inputs["fdc_id"].nunique()
)


In [ ]:
foundation_inputs.head(30)


In [ ]:
input_food_ids = set(
    foundation_inputs["fdc_of_input_food"]
    .dropna()
    .astype("int64")
)

print(
    "Input food IDs not found in food.csv:",
    len(input_food_ids - food_ids)
)


In [ ]:
category_check[
    [
        "fdc_id",
        "description_x",
        "food_category_id",
        "code",
        "description_y"
    ]
].head(20)


In [ ]:
food_update_log_entry.head(20)


In [ ]:
food_update_log_entry["id"].nunique(), len(food_update_log_entry)


In [ ]:
set(food_update_log_entry["id"]) == set(food["fdc_id"])


In [ ]:
foundation_ids = set(foundation_food["fdc_id"].dropna().astype(int))

food_foundation = food[
    food["data_type"].eq("foundation_food")
].copy()

food_foundation["in_foundation_table"] = (
    food_foundation["fdc_id"].astype(int).isin(foundation_ids)
)

print(
    food_foundation["in_foundation_table"]
    .value_counts()
)

extra_foundation = food_foundation[
    ~food_foundation["in_foundation_table"]
].copy()

print("Extra records:", len(extra_foundation))

display(
    extra_foundation[
        ["fdc_id", "description", "publication_date"]
    ].sort_values("fdc_id")
)


In [ ]:
nutrient[nutrient["id"] == 2066]


In [ ]:
print("Nutrient 2066 in nutrient.csv:")
display(nutrient[nutrient["id"] == 2066])

print("\nFood nutrient records:")
display(
    food_nutrient[
        food_nutrient["nutrient_id"] == 2066
    ]
)


In [ ]:
foundation_ids = set(foundation_food["fdc_id"])

nutrient_2066 = food_nutrient[
    food_nutrient["nutrient_id"] == 2066
].copy()

nutrient_2066["is_foundation"] = (
    nutrient_2066["fdc_id"].isin(foundation_ids)
)

display(nutrient_2066)
print(
    nutrient_2066["is_foundation"].value_counts()
)


In [ ]:
duplicate_pairs = (
    food_nutrient
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

duplicate_pairs = duplicate_pairs[
    duplicate_pairs["count"] > 1
]

display(duplicate_pairs)


In [ ]:
for _, row in duplicate_pairs.iterrows():

    fdc = int(row["fdc_id"])
    nutrient_id = int(row["nutrient_id"])

    print("\n" + "=" * 80)
    print("FDC ID:", fdc)
    print("Nutrient ID:", nutrient_id)

    display(
        food_nutrient[
            (food_nutrient["fdc_id"] == fdc) &
            (food_nutrient["nutrient_id"] == nutrient_id)
        ]
    )


In [ ]:
missing_parent = sub_sample_food[
    ~sub_sample_food["fdc_id_of_sample_food"]
    .isin(sample_food["fdc_id"])
]

display(missing_parent)

missing_parent_id = (
    missing_parent["fdc_id_of_sample_food"]
    .dropna()
    .unique()
)

print(missing_parent_id)


In [ ]:
display(
    food[
        food["fdc_id"].isin(missing_parent_id)
    ]
)


In [ ]:
display(
    food[
        food["fdc_id"].isin(
            missing_parent["fdc_id"]
        )
    ]
)


In [ ]:
unmatched_results = sub_sample_result[
    ~sub_sample_result["food_nutrient_id"]
    .isin(food_nutrient["id"])
].copy()

print("Unmatched:", len(unmatched_results))

display(unmatched_results)


In [ ]:
print(
    unmatched_results["nutrient_name"].value_counts()
)


In [ ]:
display(
    unmatched_results[
        [
            "food_nutrient_id",
            "adjusted_amount",
            "lab_method_id",
            "nutrient_name"
        ]
    ]
)


In [ ]:
[
    'fdc_id',
    'data_type',
    'description_x',
    'food_category_id',
    'publication_date',
    'id',
    'code',
    'description_y',
    '_merge'
]


In [ ]:
category_check["description"]


In [ ]:
category_check[
    [
        "fdc_id",
        "description_x",
        "food_category_id",
        "code",
        "description_y"
    ]
].head()


In [ ]:
category_check = category_check.rename(
    columns={
        "description_x": "food_description",
        "description_y": "category_description"
    }
)

display(
    category_check[
        [
            "fdc_id",
            "food_description",
            "food_category_id",
            "code",
            "category_description"
        ]
    ].head()
)


In [ ]:
print(food_component["fdc_id"].isna().sum())
print(food_component["fdc_id"].notna().sum())

display(
    food_component[
        food_component["fdc_id"].notna()
    ].head()
)


In [ ]:
display(food_component.head(20))


In [ ]:
print("=" * 70)
print("FINAL FOUNDATION FOOD INTEGRITY SUMMARY")
print("=" * 70)

print("Foundation foods:", len(foundation_food))
print("Foundation IDs in food:",
      foundation_food["fdc_id"].isin(food["fdc_id"]).sum())

print("Foundation foods with nutrients:",
      food_nutrient[
          food_nutrient["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with portions:",
      food_portion[
          food_portion["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with input foods:",
      input_food[
          input_food["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with attributes:",
      food_attribute[
          food_attribute["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with components:",
      food_component[
          food_component["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Unmatched nutrient IDs:",
      sorted(
          set(food_nutrient["nutrient_id"].dropna())
          - set(nutrient["id"].dropna())
      ))

print("Duplicate food/nutrient pairs:",
      len(duplicate_pairs))

print("Missing sub-sample parents:",
      len(missing_parent))

print("Unmatched sub-sample results:",
      len(unmatched_results))


In [ ]:
print(food_nutrient.columns.tolist())
print(food_nutrient.head(10).to_string())
print(nutrient.columns.tolist())
print(nutrient.head(10).to_string())


In [ ]:
print(nutrient[nutrient["id"] == 2066])
print(food_nutrient[food_nutrient["nutrient_id"] == 2066].head(20))


In [ ]:
print(components.columns.tolist())
print(components.head(20).to_string())


In [ ]:
with open("food_component.csv", "r", encoding="utf-8-sig") as f:
    print(f.readline())


In [ ]:
missing_parents = set(sub_sample["fdc_id_of_sample_food"]) - set(food["fdc_id"])

print("Missing unique parent IDs:", len(missing_parents))
print("Missing parents:", sorted(missing_parents))


In [ ]:
print(food_nutrient_lab.columns.tolist())
print(food_nutrient_lab.head())


In [ ]:
print(food_nutrient.columns.tolist())
print(food_nutrient.head(10).to_string())
print(nutrient.head(10).to_string())


In [ ]:
print(components.columns.tolist())
print(components.head(10).to_string())


In [ ]:
print(sub_sample.columns.tolist())
print(sub_sample.head(10).to_string())


In [ ]:
%whos


In [ ]:
print(nutrient[nutrient["id"] == 2066])


In [ ]:
bad_2066 = food_nutrient[food_nutrient["nutrient_id"] == 2066]

print("Rows:", len(bad_2066))
print(bad_2066.to_string())


In [ ]:
components = pd.read_csv("food_component.csv")


In [ ]:
dupes = (
    food_nutrient
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

print(dupes[dupes["count"] > 1].to_string(index=False))


In [ ]:
for _, row in dupes[dupes["count"] > 1].iterrows():
    print("\n", row["fdc_id"], row["nutrient_id"])
    print(
        food_nutrient[
            (food_nutrient["fdc_id"] == row["fdc_id"]) &
            (food_nutrient["nutrient_id"] == row["nutrient_id"])
        ].to_string(index=False)
    )


In [ ]:
# Nutrients we actually care about
wanted_nutrients = [
    "Energy",
    "Protein",
    "Total lipid (fat)",
    "Carbohydrate, by difference",
    "Fiber, total dietary",
    "Sugars, total including NLEA",
    "Calcium, Ca",
    "Iron, Fe",
    "Magnesium, Mg",
    "Phosphorus, P",
    "Potassium, K",
    "Sodium, Na",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3)",
    "Vitamin B-12",
    "Folate, total",
]

nutrient_lookup = nutrient[
    nutrient["name"].isin(wanted_nutrients)
].copy()

nutrient_lookup[
    ["id", "name", "unit_name", "nutrient_nbr", "rank"]
]


In [ ]:
# Join food nutrients to their nutrient definitions
food_nutrition = food_nutrient.merge(
    nutrient_lookup,
    left_on="nutrient_id",
    right_on="id",
    how="inner",
    suffixes=("_food_nutrient", "_nutrient")
)

# Join food descriptions
food_nutrition = food_nutrition.merge(
    food[["fdc_id", "description", "data_type"]],
    on="fdc_id",
    how="left"
)

food_nutrition[
    ["fdc_id", "description", "name", "amount", "unit_name", "data_type"]
].head(20)


In [ ]:
nutrition_wide = food_nutrition.pivot_table(
    index=["fdc_id", "description", "data_type"],
    columns="name",
    values="amount",
    aggfunc="first"
).reset_index()

nutrition_wide.columns.name = None

nutrition_wide.head()


In [ ]:
print(nutrition_wide.shape)

print(nutrition_wide.columns.tolist())

display(nutrition_wide.head())


In [ ]:
nutrition_wide[
    nutrition_wide["description"].str.contains(
        "apple", case=False, na=False
    )
].head(20)


In [ ]:
nutrition_wide[
    nutrition_wide["description"].str.contains(
        "chicken breast", case=False, na=False
    )
].head(20)


In [ ]:
print(nutrition_wide.shape)
print(nutrition_wide.columns.tolist())
display(nutrition_wide.head())


In [ ]:
# Build a searchable nutrient table directly from your existing data

search_data = (
    sub_sample_result
    .merge(
        nutrient_lookup[["id", "name", "unit_name"]],
        left_on="nutrient_id",
        right_on="id",
        how="left"
    )
    .merge(
        food[["fdc_id", "description"]].drop_duplicates("fdc_id"),
        on="fdc_id",
        how="left"
    )
)

search_data = search_data.rename(
    columns={
        "name": "nutrient_name"
    }
)

# Search for apple
apple = search_data[
    search_data["description"].str.contains(
        "apple", case=False, na=False
    )
]

print(
    apple[
        ["fdc_id", "description", "nutrient_name", "amount", "unit_name"]
    ].to_string(index=False)
)


In [ ]:
print("sub_sample_result columns:")
print(sub_sample_result.columns.tolist())

print("\nnutrient_lookup columns:")
print(nutrient_lookup.columns.tolist())

print("\nfood columns:")
print(food.columns.tolist())


In [ ]:
print("\nsub_sample_result:")
print(sub_sample_result.head().to_string(index=False))


In [ ]:
# Connect nutrient results → FDC food → food name

search_data = (
    sub_sample_result
    .merge(
        food_nutrient[["id", "fdc_id"]],
        left_on="food_nutrient_id",
        right_on="id",
        how="inner"
    )
    .merge(
        food[["fdc_id", "description"]],
        on="fdc_id",
        how="inner"
    )
)

# Search for Apple
apple = search_data[
    search_data["description"].str.contains(
        "apple", case=False, na=False
    )
].copy()

print(
    apple[
        ["fdc_id", "description", "nutrient_name", "adjusted_amount"]
    ].to_string(index=False)
)


In [ ]:
show_food("apple")


In [1]:
import pandas as pd

# 1. Load the USDA FoodData Central databases with the low_memory fix applied
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat (Total lipid), and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. Create the custom ID map for the Whole Foods
eight_whole_foods_id_map = {
    "Apple": 170279,          
    "Banana": 173944,         
    "Beef": 170208,           
    "Carrots": 170393,        
    "Chicken wings": 331897,  
    "Egg": 171287,            
    "Mushroom": 169251,       
    "Strawberries": 167762    
}

# 5. Extract just the nutrition rows for those specific foods
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(eight_whole_foods_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the FDC IDs back to the human-readable string names
reverse_food_map = {v: k for k, v in eight_whole_foods_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Leaving the variable at the bottom of the cell will render it as a clean HTML table in Jupyter
final_table

nutrient_name,"Carbohydrate, by difference",Protein,Total lipid (fat)
food,,,
Chicken wings,0.0,23.9,5.95


In [2]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Apple", "Banana", "Beef", "Carrots", "Chicken", "Egg", "Mushroom", "Strawberries"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

nutrient_name,"Carbohydrate, by difference",Protein,Total lipid (fat)
food,,,
Apple,14.8000,0.19,0.2100
Banana,20.1000,0.73,0.2200
Beef,2.8900,11.70,28.0000
Carrots,7.9200,0.81,0.4700
Chicken,0.0000,23.90,5.9500
Egg,0.9100,12.30,10.3000
Mushroom,7.5897,2.50,0.2563
Strawberries,7.6300,0.64,0.2200


In [ ]:
# Convert the pivot table to a nested dictionary for an API
nutrition_dict = final_table.to_dict(orient="index")
print(nutrition_dict)

In [ ]:
import pandas as pd

# 1. Load your USDA FoodData Central CSV files
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Get ALL Foundation Foods (Whole Foods)
foundation_foods = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. Filter and ADD .copy() to prevent the Pandas SettingWithCopyWarning
all_foundation_nutrition = food_nutrient[
    (food_nutrient["fdc_id"].isin(foundation_foods["fdc_id"])) & 
    (food_nutrient["nutrient_id"].isin(target_nutrient_ids))
].copy()

# 5. Map readable Nutrient Names
nutrient_map = target_nutrients.set_index("id")["name"].to_dict()
all_foundation_nutrition["nutrient_name"] = all_foundation_nutrition["nutrient_id"].map(nutrient_map)

# 6. Map readable Food Descriptions
food_map = foundation_foods.set_index("fdc_id")["description"].to_dict()
all_foundation_nutrition["food_description"] = all_foundation_nutrition["fdc_id"].map(food_map)

# 7. Pivot table to get ALL Foundation Foods as rows
all_foods_table = all_foundation_nutrition.pivot_table(
    index="food_description", 
    columns="nutrient_name", 
    values="amount"
).fillna(0.0)

# 8. Set Pandas options to show up to 400 rows so you can see all 361 items
pd.set_option('display.max_rows', 400)

# 9. Save to CSV
all_foods_table.to_csv("all_foundation_whole_foods.csv")

# Display the clean table
all_foods_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = [
  "Apple", "Banana", "Mango", "Orange", "Guava", "Papaya", "Pineapple",
  "Watermelon", "Muskmelon", "Pomegranate", "Grapes", "Pear", "Peach",
  "Plum", "Strawberries", "Blueberries", "Raspberries", "Cherries",
  "Coconut", "Dates", "Figs", "Raisins", "Jackfruit", "Custard Apple",
  "Sapota", "Lychee", "Dragon Fruit", "Kiwi", "Avocado",

  "Carrots", "Potatoes", "Sweet Potatoes", "Beetroot", "Radish",
  "Turnip", "Onion", "Garlic", "Ginger", "Tomato", "Cucumber",
  "Pumpkin", "Bottle Gourd", "Bitter Gourd", "Ridge Gourd",
  "Snake Gourd", "Ash Gourd", "Ivy Gourd", "Drumstick", "Okra",
  "Eggplant", "Green Peas", "Green Beans", "French Beans", "Corn",
  "Capsicum", "Green Chilli", "Cauliflower", "Broccoli", "Cabbage",
  "Spinach", "Amaranth", "Fenugreek Leaves", "Coriander", "Mint",
  "Curry Leaves", "Mushroom",

  "Lentils", "Red Lentils", "Green Lentils", "Black Lentils",
  "Yellow Lentils", "Chickpeas", "Black Chickpeas", "Kidney Beans",
  "Black Beans", "Green Gram", "Black Gram", "Pigeon Peas",
  "Cowpeas", "Soybeans", "Peanuts",

  "Rice", "Brown Rice", "White Rice", "Red Rice", "Black Rice",
  "Basmati Rice", "Oats", "Barley", "Millet", "Pearl Millet",
  "Finger Millet", "Sorghum", "Foxtail Millet", "Little Millet",
  "Barnyard Millet", "Kodo Millet", "Quinoa", "Whole Wheat",
  "Whole Wheat Flour", "Maize", "Buckwheat",

  "Almonds", "Cashews", "Walnuts", "Pistachios", "Pecans",
  "Hazelnuts", "Brazil Nuts", "Macadamia Nuts", "Pumpkin Seeds",
  "Sunflower Seeds", "Sesame Seeds", "Flax Seeds", "Chia Seeds",
  "Hemp Seeds",

  "Milk", "Curd", "Yogurt", "Paneer", "Cheese", "Eggs",
  "Chicken", "Turkey", "Beef", "Mutton", "Lamb", "Pork",
  "Fish", "Salmon", "Sardines", "Tuna", "Rohu", "Hilsa",
  "Prawns", "Crab",

  "Honey", "Fresh Coconut", "Coconut Milk"
]

dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Mango","Banana","Plantain","Guava","Papaya","Pomegranate","Grapes","Watermelon","Muskmelon","Pineapple","Coconut","Jackfruit","Custard Apple","Ramphal","Sapota","Lychee","Amla","Jamun","Bael","Wood Apple","Elephant Apple","Ber","Karonda","Kokum","Phalsa","Tamarind","Fig","Chikoo","Tadgola","Mahua","Kaitha","Kendu","Gular","Kafal","Hisalu","Kilmu","Buransh","Himalayan Raspberry","Himalayan Strawberry","Mulberry","Persimmon","Loquat","Peach","Plum","Apricot","Pear","Apple","Strawberry","Orange","Sweet Lime","Lemon","Mosambi","Mandarin","Kinnow","Pomelo","Grapefruit","Citron","Kagzi Lime","Galgal","Jamir","Narangi","Malta","Blood Orange","Dragon Fruit","Avocado","Passion Fruit","Tamarillo","Star Fruit","Bilimbi","Ambarella","Rose Apple","Wax Apple","Java Apple","Water Apple","Malay Apple","Rambutan","Mangosteen","Longan","Durian","Salak","Pulasan","Cempedak","Langsat","Duku","Santol","Breadfruit","Soursop","Star Apple","Canistel","Black Sapote","White Sapote","Mamey Sapote","Velvet Apple","Atemoya","Cherimoya","Snake Gourd Fruit","Camu Camu","Acerola","Jabuticaba","Feijoa","Lucuma","Cupuaçu","Cocona","Naranjilla","Pepino","Granadilla","Jamrul","Pilu","Jhar Ber","Jungle Jalebi","Kachnar Fruit","Chironji","Charoli","Pithraj","Pangam Fruit","Indian Persimmon","Khirni","Rasbhari","Cape Gooseberry","Ground Cherry","Lasoda","Lasora","Grewia Fruit","Gangren","Khirni","Makor","Khirni Ber","Aonla","Hog Plum","Indian Hog Plum","Amra","Ambada","Karamcha","Karamal","Karmal","Kundru Fruit","Tendu","Dhaman","Diospyros Melanoxylon Fruit","Mahua Fruit","Mahua Berry","Kusum Fruit","Palash Fruit","Buchanania Fruit","Agnimantha Fruit","Ritha Fruit","Haritaki","Bibhitaki","Harad","Baheda","Vibhitaki","Aak Fruit","Arjun Fruit","Neem Fruit","Moringa Fruit","Drumstick Fruit","Indian Gooseberry","Indian Jujube","Chinese Jujube","Wild Jujube","Desi Ber","Gokhru Fruit","Nagpur Orange","Coorg Orange","Darjeeling Orange","Khasi Mandarin","Sikkim Mandarin","Kodai Orange","Wam Orange","Sour Orange","Kumaon Lemon","Assam Lemon","Rangpur Lime","Kagzi Nimbu","Kachai Lemon","Sohiong","Sohphie","Sohshang","Sohiong Black Cherry","Sohphoh","Sohphie Plum","Sohiong Berry","Aiselu","Ainselu","Hisalu Berry","Timla","Bedu","Mehal","Kaafal","Kaphal","Kilmora","Kilmora Berry","Dadu","Daru","Daruharidra Fruit","Sea Buckthorn","Goji Berry","Wild Himalayan Cherry","Himalayan Wild Pear","Himalayan Wild Apricot","Himalayan Wild Plum","Himalayan Wild Peach","Himalayan Crab Apple","Crab Apple","Indian Crab Apple","Wild Fig","Cluster Fig","Indian Fig","Pakar Fruit","Banyan Fruit","Peepal Fruit","Indian Banyan Fig","Kadam Fruit","Jamun","Nerale Hannu","Nelli","Kundal Fruit","Narkel","Taad Fruit","Palmyra Fruit","Toddy Palm Fruit","Ice Apple","Borassus Fruit","Date Palm Fruit","Indian Date","Khejur","Taal Fruit","Phoenix Dactylifera Fruit","Areca Nut","Betel Nut","Wild Date","Doum Palm Fruit","Indian Wild Mango","Hog Plum","Wild Mango","Kokum Fruit","Kokum Berry","Myrica Fruit","Pistachio Fruit","Chironji Fruit","Cashew Apple","Cashew Fruit","Pineapple Guava","Indian Gooseberry Berry","Mango Ginger Fruit","Kokum Plum","Kundru Berry","Kharjura","Kharjur","Makhana Fruit","Lotus Fruit","Water Caltrop Fruit","Singhara","Fox Nut Fruit","Bakul Fruit","Mimusops Fruit","Nagkesar Fruit","Kadamba Fruit","Siris Fruit","Kachnar Fruit","Agnimantha Berry","Indian Laurel Fruit","Mahua Berry","Indian Laurel Cherry","Mimusops Elengi Fruit","Pangium Fruit","Wild Tamarind","Manila Tamarind","Madras Thorn","Jungle Jalebi Fruit","Kodukkapuli","Vilayati Imli","Tamarind Plum","Indian Plum","Allspice Berry","Pepper Fruit","Black Pepper Berry","Long Pepper Fruit"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Potato","Sweet Potato","Yam","Elephant Foot Yam","Greater Yam","Lesser Yam","Purple Yam","Chinese Yam","Taro","Colocasia","Cassava","Carrot","Radish","Turnip","Beetroot","Parsnip","Rutabaga","Onion","Shallot","Garlic","Ginger","Turmeric","Green Chilli","Red Chilli","Bell Pepper","Capsicum","Tomato","Brinjal","Eggplant","Okra","Drumstick","Bottle Gourd","Bitter Gourd","Ridge Gourd","Sponge Gourd","Snake Gourd","Ash Gourd","Pointed Gourd","Ivy Gourd","Round Gourd","Apple Gourd","Cucumber","Zucchini","Pumpkin","Chayote","Raw Banana","Raw Papaya","Raw Jackfruit","Green Mango","Green Peas","Snow Peas","French Beans","Cluster Beans","Broad Beans","Hyacinth Beans","Field Beans","Lima Beans","Cowpea","Black-eyed Peas","Soybean","Mung Bean","Urad Bean","Kidney Beans","Pigeon Peas","Chickpeas","Bengal Gram","Green Gram","Black Gram","Horse Gram","Lentils","Beet Greens","Amaranth Greens","Spinach","Malabar Spinach","Water Spinach","Fenugreek Leaves","Mustard Greens","Coriander Leaves","Mint Leaves","Curry Leaves","Dill Leaves","Parsley","Celery","Lettuce","Kale","Collard Greens","Swiss Chard","Bok Choy","Chinese Cabbage","Cabbage","Red Cabbage","Cauliflower","Broccoli","Brussels Sprouts","Kohlrabi","Radish Greens","Turnip Greens","Carrot Greens","Beet Greens","Mustard","Garden Cress","Watercress","Sorrel","Arugula","Leek","Spring Onion","Green Onion","Chives","Fennel Bulb","Artichoke","Asparagus","Bamboo Shoots","Lotus Root","Lotus Stem","Water Chestnut","Raw Banana Flower","Banana Stem","Banana Blossom","Drumstick Leaves","Drumstick Flowers","Agathi Leaves","Agathi Flowers","Moringa Leaves","Moringa Flowers","Neem Leaves","Neem Flowers","Sesbania Leaves","Sesbania Flowers","Roselle Leaves","Roselle Flowers","Colocasia Leaves","Colocasia Stems","Amaranth Stems","Amaranth Flowers","Pumpkin Leaves","Pumpkin Flowers","Bottle Gourd Leaves","Bottle Gourd Flowers","Ridge Gourd Leaves","Sweet Potato Leaves","Cassava Leaves","Tapioca","Tapioca Leaves","Yam Leaves","Elephant Foot Yam Leaves","Green Chickpeas","Fresh Pigeon Peas","Fresh Toor Dal","Fresh Lima Beans","Fresh Soybeans","Fresh Cowpeas","Fresh Broad Beans","Moth Beans","Matki","Field Peas","Black Gram Sprouts","Green Gram Sprouts","Bengal Gram Sprouts","Alfalfa Sprouts","Bamboo Shoot","Tender Jackfruit","Tender Tamarind","Raw Tamarind","Drumstick Pods","Mushroom","Button Mushroom","Oyster Mushroom","Shiitake Mushroom","Milky Mushroom","Paddy Straw Mushroom","Wood Ear Mushroom","Enoki Mushroom","Portobello Mushroom","Morel Mushroom","Termite Mushroom","Wild Mushroom","Corn","Baby Corn","Sweet Corn","Maize","Raw Coconut","Coconut Shoot","Avarekai","Avarakkai","Sem","Papdi","Valor Papdi","Guar","Gawar","Tendli","Tindora","Kundru","Parwal","Potol","Karela","Torai","Turai","Lauki","Dudhi","Tinda","Chichinda","Kovakkai","Peerkangai","Surakkai","Pudalangai","Poosanikai","Mathanga","Kaddu","Kumbalakai","Seemebadanekai","Chow Chow","Vazhakkai","Vazhaithandu","Vazhaipoo","Kathirikkai","Vendakkai","Murungakkai","Suran","Kachalu","Arbi","Shakarkand","Mooli","Gajar","Shalgam","Palak","Methi","Sarson","Bathua","Chaulai","Lal Saag","Poi Saag","Kolmi Saag","Nenua","Kachri","Ker","Sangri","Gunda","Kachnar Buds","Kachnar Flowers","Raw Lotus Seeds","Lotus Pods","Water Caltrop","Singhara","Makhana","Green Tamarind","Gongura","Khatta Palak","Manathakkali","Mudakathan","Pirandai","Vallarai","Thandu Keerai","Araikeerai","Sirukeerai","Pasalai Keerai","Ponnanganni Keerai","Agathi Keerai","Murungai Keerai","Keerai","Chukka Keerai","Cholai Keerai","Methi Matar","Green Garlic","Garlic Scapes","Onion Greens","Radish Pods","Mustard Pods","Drumstick Flowers","Neem Flowers","Banana Flower","Pumpkin Blossom","Squash Blossoms","Zucchini Blossoms","Artichoke Hearts","Baby Potato","New Potato","Fingerling Potato","Red Potato","Purple Potato","Raw Yam","Surti Papdi","Lilva","Tuvar Lilva","Green Chana","Matar","Matar Pods","Green Soybean","Edamame"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Rice","Brown Rice","White Rice","Red Rice","Black Rice","Purple Rice","Wild Rice","Basmati Rice","Sona Masuri Rice","Ponni Rice","Jeera Samba Rice","Seeraga Samba Rice","Kullakar Rice","Matta Rice","Kerala Red Rice","Navara Rice","Kalanamak Rice","Gobindobhog Rice","Ambemohar Rice","Indrayani Rice","Joha Rice","Katarni Rice","Bamboo Rice","Sticky Rice","Glutinous Rice","Parboiled Rice","Broken Rice","Wheat","Whole Wheat","Durum Wheat","Hard Red Wheat","Soft Wheat","Emmer Wheat","Spelt","Einkorn","Khorasan Wheat","Triticale","Rye","Barley","Pearl Barley","Hulled Barley","Naked Barley","Six-Row Barley","Two-Row Barley","Oats","Rolled Oats","Steel-Cut Oats","Millet","Pearl Millet","Finger Millet","Foxtail Millet","Little Millet","Kodo Millet","Barnyard Millet","Proso Millet","Browntop Millet","Japanese Millet","Sorghum","Jowar","Maize","Corn","Sweet Corn","Popcorn","Flint Corn","Dent Corn","Waxy Corn","White Corn","Yellow Corn","Blue Corn","Red Corn","Black Corn","Teff","Fonio","Quinoa","Amaranth","Buckwheat","Chia","Spelt","Farro","Freekeh","Einkorn","Rye","Triticale","Wild Rice","Job's Tears","Adlay","Canary Seed","Sago","Sago Pearls","Pearl Sago","Tapioca","Buckwheat Groats","Millet Groats","Oat Groats","Wheat Groats","Barley Groats","Rye Groats","Corn Grits","Wheat Germ","Wheat Bran","Rice Bran","Rice Germ","Barley Bran","Oat Bran","Sorghum Grain","Pearl Millet Grain","Finger Millet Grain","Foxtail Millet Grain","Little Millet Grain","Kodo Millet Grain","Barnyard Millet Grain","Proso Millet Grain","Browntop Millet Grain","Japanese Millet Grain","Indian Barnyard Millet","Indian Browntop Millet","Indian Kodo Millet","Indian Foxtail Millet","Indian Little Millet","Indian Proso Millet","Indian Pearl Millet","Indian Finger Millet","Indian Sorghum","Indian Maize","Indian Rice","Navara Rice","Kuthiraivali","Thinai","Samai","Varagu","Kambu","Ragi","Jowar","Bajra","Jhangora","Cheena","Kangni","Kutki","Kodra","Sama","Rajgira","Kuttu","Barley","Jau","Gehu","Cholam","Makka","Dhan","Basmati","Samba Rice","Mappillai Samba","Karunguruvai","Thooyamalli","Poongar Rice","Kichili Samba","Kattuyanam Rice","Seeraga Samba","Kichadi Samba","Arupatham Kuruvai","Illuppai Poo Samba","Ponni Rice","Kichadi Rice","Red Kavuni Rice","Black Kavuni Rice","Karuppu Kavuni","Mapillai Samba","Kattuyanam","Kullakar","Navara","Pokali Rice","Matta Rice","Wayanad Rice","Jeerakasala Rice","Gandhakasala Rice","Joha Rice","Bora Rice","Chokuwa Rice","Bao Rice","Gobindobhog","Katarni","Kalajeera Rice","Tulaipanji Rice","Radhunipagal Rice","Lal Dhan","Hansraj Rice","Ambemohar","Indrayani","Kolam Rice","Surti Kolam","HMT Rice","Sona Masuri","Ponni","Swarna Rice","MTU-1010","IR-64","IR-36","PR-14 Rice","PR-106 Rice","Sharbati Wheat","Lokwan Wheat","MP Wheat","Malwa Wheat","Durum Wheat","Emmer Wheat","Khapli Wheat","Bansi Wheat","Lok-1 Wheat","C-306 Wheat","Sujata Wheat","Kalyan Sona Wheat","HD-2967 Wheat","HD-3086 Wheat","PBW-343 Wheat","Desi Wheat","Indian Barley","Himalayan Barley","Naked Barley","Six-Row Barley","Two-Row Barley","Hulless Barley","Indian Oats","Indian Rye","Indian Sorghum","Indian Maize"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Chicken","Turkey","Duck","Goose","Quail","Pigeon","Guinea Fowl","Pheasant","Partridge","Ostrich","Emu","Rabbit","Hare","Lamb","Mutton","Goat Meat","Chevon","Beef","Veal","Buffalo Meat","Carabeef","Pork","Ham","Bacon","Venison","Deer Meat","Elk Meat","Moose Meat","Bison Meat","Yak Meat","Camel Meat","Horse Meat","Donkey Meat","Boar Meat","Wild Boar","Reindeer Meat","Antelope Meat","Gazelle Meat","Kangaroo Meat","Alligator Meat","Crocodile Meat","Frog Legs","Snail","Escargot","Liver","Chicken Liver","Beef Liver","Lamb Liver","Goat Liver","Pork Liver","Heart","Chicken Heart","Beef Heart","Lamb Heart","Goat Heart","Kidney","Chicken Kidney","Beef Kidney","Lamb Kidney","Goat Kidney","Brain","Tongue","Beef Tongue","Lamb Tongue","Sweetbreads","Tripe","Beef Tripe","Lamb Tripe","Goat Tripe","Oxtail","Beef Marrow","Bone Marrow","Chicken Gizzard","Duck Gizzard","Chicken Feet","Pork Feet","Pig Trotters","Lamb Shank","Beef Shank","Chicken Breast","Chicken Thigh","Chicken Drumstick","Chicken Wing","Turkey Breast","Duck Breast","Duck Leg","Lamb Chops","Lamb Ribs","Mutton Ribs","Goat Ribs","Pork Ribs","Pork Chops","Beef Steak","Beef Ribs","Beef Brisket","Beef Tenderloin","Beef Sirloin","Beef Chuck","Buffalo Ribs","Venison Steak","Rabbit Meat","Fish","Salmon","Atlantic Salmon","Pacific Salmon","Trout","Rainbow Trout","Brown Trout","Cod","Haddock","Pollock","Hake","Halibut","Tuna","Yellowfin Tuna","Bluefin Tuna","Skipjack Tuna","Mackerel","King Mackerel","Indian Mackerel","Sardine","Sardines","Anchovy","Herring","Shad","Hilsa","Rohu","Katla","Mrigal","Carp","Grass Carp","Silver Carp","Common Carp","Catfish","Walking Catfish","Pangasius","Tilapia","Pomfret","Silver Pomfret","Black Pomfret","Indian Butterfish","Sole","Flounder","Snapper","Red Snapper","Sea Bass","Barramundi","Grouper","Mahi Mahi","Swordfish","Marlin","Sailfish","Eel","Conger Eel","Stingray","Skate","Shark","Octopus","Squid","Cuttlefish","Prawn","Shrimp","Tiger Prawn","King Prawn","Crab","Mud Crab","Blue Crab","Snow Crab","Lobster","Rock Lobster","Spiny Lobster","Crayfish","Mussels","Clams","Oysters","Scallops","Cockles","Abalone","Whelks","Sea Urchin","Sea Cucumber","Roe","Fish Roe","Salmon Roe","Tobiko","Masago","Caviar","Eggs","Chicken Eggs","Duck Eggs","Goose Eggs","Quail Eggs","Turkey Eggs","Pigeon Eggs","Guinea Fowl Eggs","Ostrich Eggs","Fish Eggs","Milk","Cow Milk","Buffalo Milk","Goat Milk","Sheep Milk","Camel Milk","Donkey Milk","Yak Milk","Mare Milk","Full Cream Milk","Skim Milk","Buttermilk","Curd","Yogurt","Greek Yogurt","Kefir","Lassi","Chaas","Paneer","Cottage Cheese","Cheddar","Mozzarella","Parmesan","Gouda","Edam","Emmental","Swiss Cheese","Brie","Camembert","Feta","Ricotta","Mascarpone","Cream Cheese","Blue Cheese","Goat Cheese","Sheep Cheese","Buffalo Mozzarella","Processed Cheese","Cheese Curd","Milk Powder","Skimmed Milk Powder","Condensed Milk","Evaporated Milk","Cream","Heavy Cream","Whipping Cream","Sour Cream","Clotted Cream","Butter","Ghee","Clarified Butter","Whey","Whey Protein","Casein","Milk Solids","Milk Fat","Milk Skin","Khoya","Mawa","Rabri","Shrikhand","Basundi","Malai","Dahi","Mishti Doi","Chhena","Chhurpi","Khoa","Kulfi","Ice Cream","Gelato","Milkshake","Milk Tea","Milk Chocolate"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [3]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients this time)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# Display the final mega-table
final_table

nutrient_name,Ash,Beta-glucan,Biotin,"Calcium, Ca","Carbohydrate, by difference",Cholesterol,Citric acid,"Copper, Cu",Energy,Energy (Atwater General Factors),...,"Vitamin A, RAE",Vitamin B-12,Vitamin B-6,"Vitamin C, total ascorbic acid",Vitamin D (D2 + D3),"Vitamin D (D2 + D3), International Units",Vitamin D3 (cholecalciferol),Vitamin E (alpha-tocopherol),Water,"Zinc, Zn"
food,,,,,,,,,,,,,,,,,,,,,
Apple,0.1238,-,-,7.099,11.363962,-,0.0,0.003401,-,48.3763,...,-,-,0.01388,51.16,-,-,-,-,88.14,0.002125
Banana,0.3793,-,1.164,9.816,4.966175,-,-,0.061280,-,23.9398,...,-,-,0.28730,112.1,-,-,-,-,93.80,0.130400
Beef,2.7400,-,-,15.000,2.890000,-,-,0.046000,812.0,310.0000,...,3.0,0.97,0.13000,-,-,-,-,0.51,54.60,2.060000
Drumstick,0.9800,-,-,12.000,0.000000,127.0,-,0.063000,404.0,149.0000,...,7.0,0.41,0.37200,-,0.1,2.0,0.1,0.17,69.90,2.540000
Mushroom,1.0840,-,16.93,0.000,7.589700,-,-,0.177100,-,42.6655,...,-,-,0.06550,-,-,-,-,-,88.57,0.744800
Oats,1.7060,7.52,21.9,45.530,68.657550,-,-,0.427800,-,381.6260,...,-,-,0.13460,-,-,-,-,-,10.25,2.744000
Rye,1.4050,1.913,8.975,32.240,77.161800,-,-,0.338400,-,359.4000,...,-,-,0.16380,-,-,-,-,-,11.13,2.328000


In [ ]:
import pandas as pd
import json

# 1. Load the 2026 USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping foundation foods...")
for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Convert the Pandas dataframe into a raw dictionary
raw_dict = final_table.to_dict(orient="index")

# 10. Re-structure the data into organized JSON for your API
api_payload = {}

for food_name, nutrients in raw_dict.items():
    api_payload[food_name] = {
        "Macros": {
            "Energy (kcal)": nutrients.get("Energy", " - "),
            "Protein (g)": nutrients.get("Protein", " - "),
            "Carbohydrates (g)": nutrients.get("Carbohydrate, by difference", " - "),
            "Total Fat (g)": nutrients.get("Total lipid (fat)", " - "),
            "Fiber (g)": nutrients.get("Fiber, total dietary", " - ")
        },
        "Vitamins": {
            "Vitamin A (RAE)": nutrients.get("Vitamin A, RAE", " - "),
            "Vitamin C (mg)": nutrients.get("Vitamin C, total ascorbic acid", " - "),
            "Vitamin D (IU)": nutrients.get("Vitamin D (D2 + D3), International Units", " - "),
            "Vitamin B-6 (mg)": nutrients.get("Vitamin B-6", " - ")
        },
        "Minerals": {
            "Calcium (mg)": nutrients.get("Calcium, Ca", " - "),
            "Zinc (mg)": nutrients.get("Zinc, Zn", " - "),
            "Copper (mg)": nutrients.get("Copper, Cu", " - ")
        },
        # Keeps all 100+ other nutrients available in the background
        "All_Nutrients": nutrients 
    }

# 11. Print the cleanly nested JSON 
print("\n=== FINAL JSON PAYLOAD ===")
print(json.dumps(api_payload, indent=2))

In [ ]:
"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"

In [ ]:
import pandas as pd
import json

# 1. Load the USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv", low_memory=False)

# 2. Filter to "SR Legacy Foods" (This unlocks 7,000+ historical food entries)
sr_legacy_food = food[food["data_type"] == "sr_legacy_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
# I've included some standard whole foods as an example
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")
for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = sr_legacy_food[sr_legacy_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        # Grabbing the first match found in the SR Legacy dataset
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Convert the Pandas dataframe into a raw dictionary
raw_dict = final_table.to_dict(orient="index")

# 10. Re-structure the data into organized JSON for your API
api_payload = {}

for food_name, nutrients in raw_dict.items():
    api_payload[food_name] = {
        "Macros": {
            "Energy (kcal)": nutrients.get("Energy", " - "),
            "Protein (g)": nutrients.get("Protein", " - "),
            "Carbohydrates (g)": nutrients.get("Carbohydrate, by difference", " - "),
            "Total Fat (g)": nutrients.get("Total lipid (fat)", " - "),
            "Fiber (g)": nutrients.get("Fiber, total dietary", " - ")
        },
        "Vitamins": {
            "Vitamin A (RAE)": nutrients.get("Vitamin A, RAE", " - "),
            "Vitamin C (mg)": nutrients.get("Vitamin C, total ascorbic acid", " - "),
            "Vitamin D (IU)": nutrients.get("Vitamin D (D2 + D3), International Units", " - "),
            "Vitamin B-6 (mg)": nutrients.get("Vitamin B-6", " - ")
        },
        "Minerals": {
            "Calcium (mg)": nutrients.get("Calcium, Ca", " - "),
            "Zinc (mg)": nutrients.get("Zinc, Zn", " - "),
            "Copper (mg)": nutrients.get("Copper, Cu", " - ")
        },
        # Keeps all other SR Legacy nutrients available in the background
        "All_Nutrients": nutrients 
    }

# 11. Print the cleanly nested JSON 
print("\n=== FINAL SR LEGACY JSON PAYLOAD ===")
print(json.dumps(api_payload, indent=2))

In [ ]:
import pandas as pd

# ================================================================
# 1. Load the USDA SR Legacy FoodData Central databases
# ================================================================

food = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"
)

nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv"
)

food_nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. Filter to "SR Legacy Foods"
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. Foods to search
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. Dynamically find the FDC ID for each food
# ================================================================

dynamic_id_map = {}

for item in search_foods:

    # \b = strict word boundary
    # case=False = case-insensitive
    # na=False = ignore missing descriptions

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]


# ================================================================
# 5. Extract ALL nutrition rows for the selected foods
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. Map nutrient IDs to readable nutrient names
# ================================================================

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"].map(
        nutrient.set_index("id")["name"]
    )
)


# ================================================================
# 7. Map FDC IDs back to food names
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"].map(
        reverse_food_map
    )
)


# ================================================================
# 8. Pivot the table
#
# Food = rows
# Nutrients = columns
# Amount = values
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount"
)


# ================================================================
# 9. Replace missing values with " - "
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 10. Put the important nutrients first
#     Everything else remains available after them
# ================================================================

preferred_columns = [
    "Energy",
    "Protein",
    "Carbohydrate, by difference",
    "Total lipid (fat)",
    "Fiber, total dietary",

    "Vitamin A, RAE",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3), International Units",
    "Vitamin B-6",

    "Calcium, Ca",
    "Zinc, Zn",
    "Copper, Cu"
]


existing_preferred = [
    column
    for column in preferred_columns
    if column in final_table.columns
]

remaining_columns = [
    column
    for column in final_table.columns
    if column not in existing_preferred
]

final_table = final_table[
    existing_preferred + remaining_columns
]


# ================================================================
# 11. Display the final mega-table
# ================================================================

print("\n=== USDA SR LEGACY NUTRITION TABLE ===\n")

final_table


In [ ]:
import pandas as pd

# 1. Load the USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv", low_memory=False)

# 2. Filter to "SR Legacy Foods" 
sr_legacy_food = food[food["data_type"] == "sr_legacy_food"]

# 3. Dynamic search with STRICT word boundaries (\b)
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")
for item in search_foods:
    matches = sr_legacy_food[sr_legacy_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Define the primary columns you want to display in the console table
display_columns = [
    "Energy", 
    "Protein", 
    "Carbohydrate, by difference", 
    "Total lipid (fat)", 
    "Fiber, total dietary"
]

# Ensure we only try to display columns that actually exist in the result
valid_columns = [col for col in display_columns if col in final_table.columns]
display_table = final_table[valid_columns].copy()

# Rename the columns so they look much cleaner in the console
display_table.columns = ["Calories", "Protein (g)", "Carbs (g)", "Fat (g)", "Fiber (g)"]

# 10. Print the clean table to the terminal
print("\n" + "="*65)
print(" SR LEGACY MACROS (Per 100g)")
print("="*65)
print(display_table.to_string())
print("="*65)

# 11. Export the FULL 121-column dataset to Excel/CSV for your records
export_path = "sr_legacy_extracted_data.csv"
final_table.to_csv(export_path)
print(f"\n[+] Full dataset with all vitamins and minerals saved to: {export_path}")

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients this time)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# Display the final mega-table
final_table

In [ ]:
import pandas as pd
import json


# ================================================================
# 1. LOAD USDA FOODDATA CENTRAL DATABASES
# ================================================================

print("Loading USDA CSV files (this may take a moment)...")

BASE_PATH = (
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
)

food = pd.read_csv(
    BASE_PATH + r"\food.csv"
)

nutrient = pd.read_csv(
    BASE_PATH + r"\nutrient.csv"
)

food_nutrient = pd.read_csv(
    BASE_PATH + r"\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. FILTER SR LEGACY FOODS
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. FOODS TO SEARCH
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. FIND FDC IDs FOR THE FOODS
# ================================================================

dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")

for item in search_foods:

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:

        # Take the first matching food
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

    else:

        print(f"WARNING: {item} was not found.")


# ================================================================
# 5. EXTRACT NUTRIENTS FOR FOUND FOODS
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. MAP NUTRIENT ID -> NUTRIENT NAME
# ================================================================

nutrient_name_map = nutrient.set_index("id")["name"]

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"]
    .map(nutrient_name_map)
)


# ================================================================
# 7. MAP FDC ID -> FOOD NAME
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"]
    .map(reverse_food_map)
)


# ================================================================
# 8. CREATE FOOD x NUTRIENT TABLE
#
#     Food       Energy   Protein   Carbs   Fat   ...
#     Apple      752.5     2.17    44.54  11.5
#     Banana     273.0     2.76    19.74   1.7
#
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount",
    aggfunc="first"
)


# ================================================================
# 9. RENAME IMPORTANT NUTRIENTS
#
# This makes the column names shorter and easier to read.
# ================================================================

rename_columns = {
    "Energy": "Energy",
    "Protein": "Protein",
    "Carbohydrate, by difference": "Carbs",
    "Total lipid (fat)": "Fat",
    "Fiber, total dietary": "Fiber",

    "Vitamin A, RAE": "Vitamin A",
    "Vitamin C, total ascorbic acid": "Vitamin C",
    "Vitamin D (D2 + D3), International Units": "Vitamin D",
    "Vitamin B-6": "Vitamin B6",

    "Calcium, Ca": "Calcium",
    "Zinc, Zn": "Zinc",
    "Copper, Cu": "Copper"
}

final_table = final_table.rename(
    columns=rename_columns
)


# ================================================================
# 10. DEFINE THE IMPORTANT COLUMN ORDER
# ================================================================

main_columns = [
    "Energy",
    "Protein",
    "Carbs",
    "Fat",
    "Fiber",

    "Vitamin A",
    "Vitamin C",
    "Vitamin D",
    "Vitamin B6",

    "Calcium",
    "Zinc",
    "Copper"
]


# ================================================================
# 11. PUT IMPORTANT NUTRIENTS FIRST
#
# Everything else comes after them.
# ================================================================

existing_main_columns = [
    column
    for column in main_columns
    if column in final_table.columns
]

other_columns = [
    column
    for column in final_table.columns
    if column not in existing_main_columns
]

final_table = final_table[
    existing_main_columns + other_columns
]


# ================================================================
# 12. REPLACE MISSING VALUES
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 13. FORMAT NUMBERS
# ================================================================

def format_value(value):

    # Missing value
    if pd.isna(value):
        return " - "

    # Numeric value
    if isinstance(value, (int, float)):
        return f"{value:.2f}"

    # String
    return str(value)


final_table = final_table.map(format_value)


# ================================================================
# 14. PRINT FINAL TABLE
# ================================================================

print("\n")
print("=" * 200)

print("                         USDA NUTRITION TABLE")

print("=" * 200)

print(
    final_table.to_string(
        justify="right"
    )
)

print("=" * 200)


# ================================================================
# 15. OPTIONAL: SAVE TABLE TO CSV
# ================================================================

final_table.to_csv(
    "USDA_nutrition_table.csv"
)

print("\nTable saved to:")
print("USDA_nutrition_table.csv")


In [ ]:
import pandas as pd

# ================================================================
# 1. Load the USDA SR Legacy FoodData Central databases
# ================================================================

food = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"
)

nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv"
)

food_nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. Filter to "SR Legacy Foods"
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. Foods to search
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. Dynamically find the FDC ID for each food
# ================================================================

dynamic_id_map = {}

for item in search_foods:

    # \b = strict word boundary
    # case=False = case-insensitive
    # na=False = ignore missing descriptions

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]


# ================================================================
# 5. Extract ALL nutrition rows for the selected foods
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. Map nutrient IDs to readable nutrient names
# ================================================================

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"].map(
        nutrient.set_index("id")["name"]
    )
)


# ================================================================
# 7. Map FDC IDs back to food names
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"].map(
        reverse_food_map
    )
)


# ================================================================
# 8. Pivot the table
#
# Food = rows
# Nutrients = columns
# Amount = values
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount"
)


# ================================================================
# 9. Replace missing values with " - "
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 10. Put the important nutrients first
#     Everything else remains available after them
# ================================================================

preferred_columns = [
    "Energy",
    "Protein",
    "Carbohydrate, by difference",
    "Total lipid (fat)",
    "Fiber, total dietary",

    "Vitamin A, RAE",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3), International Units",
    "Vitamin B-6",

    "Calcium, Ca",
    "Zinc, Zn",
    "Copper, Cu"
]


existing_preferred = [
    column
    for column in preferred_columns
    if column in final_table.columns
]

remaining_columns = [
    column
    for column in final_table.columns
    if column not in existing_preferred
]

final_table = final_table[
    existing_preferred + remaining_columns
]


# ================================================================
# 11. Display the final mega-table
# ================================================================

print("\n=== USDA SR LEGACY NUTRITION TABLE ===\n")

final_table


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)


In [ ]:
USDA_DIR = Path(
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
)

USDA_DIR


In [ ]:
food = pd.read_csv(
    USDA_DIR / "food.csv",
    low_memory=False
)

nutrient = pd.read_csv(
    USDA_DIR / "nutrient.csv",
    low_memory=False
)

food_nutrient = pd.read_csv(
    USDA_DIR / "food_nutrient.csv",
    low_memory=False
)

food_portion = pd.read_csv(
    USDA_DIR / "food_portion.csv",
    low_memory=False
)

measure_unit = pd.read_csv(
    USDA_DIR / "measure_unit.csv",
    low_memory=False
)


In [ ]:
print("food:", food.shape)
print("nutrient:", nutrient.shape)
print("food_nutrient:", food_nutrient.shape)
print("food_portion:", food_portion.shape)
print("measure_unit:", measure_unit.shape)

In [ ]:
print("FOOD")
print(food.columns.tolist())

print("\nNUTRIENT")
print(nutrient.columns.tolist())

print("\nFOOD_NUTRIENT")
print(food_nutrient.columns.tolist())

print("\nFOOD_PORTION")
print(food_portion.columns.tolist())

print("\nMEASURE_UNIT")
print(measure_unit.columns.tolist())


In [ ]:
sr_food = food[
    food["data_type"].eq("sr_legacy_food")
].copy()

print("Total foods:", len(food))
print("SR Legacy foods:", len(sr_food))


In [ ]:
sr_food[
    ["fdc_id", "data_type", "description"]
].head(20)


In [ ]:
def search_food(query, limit=20):
    
    query = query.strip()
    
    results = sr_food[
        sr_food["description"].str.contains(
            query,
            case=False,
            na=False,
            regex=False
        )
    ].copy()
    
    return results[
        [
            "fdc_id",
            "description",
            "data_type"
        ]
    ].head(limit)


In [ ]:
search_food("apple", 20)

In [ ]:
search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]

for item in search_foods:
    
    print("\n" + "=" * 80)
    print(item)
    print("=" * 80)
    
    display(
        search_food(item, 10)
    )

In [ ]:
selected_fdc_ids = list(selected_foods.values())

selected_food_df = sr_food[
    sr_food["fdc_id"].isin(selected_fdc_ids)
].copy()

selected_food_df[
    [
        "fdc_id",
        "description",
        "data_type"
    ]
]

In [4]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. Foods you want to investigate
# ---------------------------------------------------------

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ---------------------------------------------------------
# 2. Function to find ALL Foundation Food matches
# ---------------------------------------------------------

def find_food_matches(query, foundation_food):
    """
    Search USDA Foundation Foods and return all matching foods.
    """

    # Escape the query so special regex characters don't cause problems
    escaped_query = re.escape(query)

    matches = foundation_food[
        foundation_food["description"].str.contains(
            escaped_query,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    # Keep the useful columns
    columns = ["fdc_id", "description"]

    # Only use columns that actually exist
    columns = [col for col in columns if col in matches.columns]

    return matches[columns].sort_values("description")


# ---------------------------------------------------------
# 3. Inspect every search term
# ---------------------------------------------------------

for item in search_foods:

    matches = find_food_matches(item, foundation_food)

    print("\n" + "=" * 80)
    print(f"SEARCH: {item}")
    print("=" * 80)

    if matches.empty:
        print("NO MATCHES FOUND")
    else:
        print(f"Found {len(matches)} matches:\n")
        print(matches.to_string(index=False))


SEARCH: Apple
Found 14 matches:

 fdc_id                                                       description
2003590 Apple juice, with added vitamin C, from concentrate, shelf stable
1105897                                      Apples, fuji, with skin, raw
1750340                                      Apples, fuji, with skin, raw
1105781                                      Apples, gala, with skin, raw
1750341                                      Apples, gala, with skin, raw
1105664                              Apples, granny smith, with skin, raw
1750342                              Apples, granny smith, with skin, raw
1105547                                Apples, honeycrisp, with skin, raw
1750343                                Apples, honeycrisp, with skin, raw
1105430                             Apples, red delicious, with skin, raw
1750339                             Apples, red delicious, with skin, raw
2263892                     Applesauce, unsweetened, with added vitamin C
2346

In [5]:
import pandas as pd
import re


# =========================================================
# 1. Function to search Foundation Foods
# =========================================================

def search_foundation_food(query, foundation_food):
    """
    Return all Foundation Foods whose description contains
    the search query.
    """

    query = str(query).strip()

    if not query:
        return pd.DataFrame(columns=["fdc_id", "description"])

    matches = foundation_food[
        foundation_food["description"].str.contains(
            re.escape(query),
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    return matches[["fdc_id", "description"]].sort_values("description")


# =========================================================
# 2. Function to find the best matching food
# =========================================================

def select_best_food(query, foundation_food, required_terms=None):
    """
    Find the best Foundation Food for a query.

    required_terms:
        Words/phrases that should ideally appear in the
        USDA description.

    Returns:
        selected food, confidence status, and all candidates.
    """

    matches = search_foundation_food(query, foundation_food)

    if matches.empty:
        return {
            "status": "NOT_FOUND",
            "selected": None,
            "candidates": matches
        }

    # If no extra requirements were provided,
    # return candidates for inspection.
    if not required_terms:
        return {
            "status": "REVIEW",
            "selected": None,
            "candidates": matches
        }

    # -----------------------------------------------------
    # Score every candidate
    # -----------------------------------------------------

    scored = []

    for _, row in matches.iterrows():

        description = row["description"].lower()

        score = 0

        for term in required_terms:

            term = term.lower().strip()

            if term in description:
                score += 1

        scored.append({
            "fdc_id": row["fdc_id"],
            "description": row["description"],
            "score": score
        })

    scored_df = pd.DataFrame(scored)

    scored_df = scored_df.sort_values(
        ["score", "description"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best = scored_df.iloc[0]

    # -----------------------------------------------------
    # Decide whether selection is safe
    # -----------------------------------------------------

    if best["score"] == len(required_terms):

        # Check whether another food has the same score
        top_matches = scored_df[
            scored_df["score"] == best["score"]
        ]

        if len(top_matches) == 1:
            status = "SELECTED"
            selected = best
        else:
            status = "AMBIGUOUS"
            selected = None

    else:
        status = "REVIEW"
        selected = None

    return {
        "status": status,
        "selected": selected,
        "candidates": scored_df
    }

In [8]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOODS PIPELINE
# ============================================================

import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. FILE LOCATIONS
# ============================================================
# CHANGE THESE PATHS to match your files.

DATA_FOLDER = Path(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30")

FOOD_CSV = DATA_FOLDER / "food.csv"
FOOD_NUTRIENT_CSV = DATA_FOLDER / "food_nutrient.csv"
NUTRIENT_CSV = DATA_FOLDER / "nutrient.csv"


# ============================================================
# 2. LOAD USDA DATA
# ============================================================

print("Loading USDA files...")

food = pd.read_csv(FOOD_CSV, low_memory=False)
food_nutrient = pd.read_csv(FOOD_NUTRIENT_CSV, low_memory=False)
nutrient = pd.read_csv(NUTRIENT_CSV, low_memory=False)

print("Food rows:", len(food))
print("Food nutrient rows:", len(food_nutrient))
print("Nutrient definitions:", len(nutrient))


# ============================================================
# 3. CHECK COLUMNS
# ============================================================

print("\nFood columns:")
print(food.columns.tolist())

print("\nFood nutrient columns:")
print(food_nutrient.columns.tolist())

print("\nNutrient columns:")
print(nutrient.columns.tolist())


# ============================================================
# 4. KEEP FOUNDATION FOODS ONLY
# ============================================================

foundation_food = food[
    food["data_type"]
    .astype(str)
    .str.lower()
    .eq("foundation_food")
].copy()

print("\nFoundation Foods:", len(foundation_food))


# ============================================================
# 5. CLEAN DESCRIPTION
# ============================================================

foundation_food["description"] = (
    foundation_food["description"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 6. SEARCH FUNCTION
# ============================================================

def search_foundation_food(query):
    """
    Search Foundation Foods and return all candidates.
    """

    query = str(query).strip()

    if not query:
        return pd.DataFrame(
            columns=["fdc_id", "description"]
        )

    pattern = re.escape(query)

    matches = foundation_food[
        foundation_food["description"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    return matches[
        ["fdc_id", "description"]
    ].sort_values("description")


# ============================================================
# 7. INSPECT YOUR SEARCH TERMS
# ============================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats",
    "Carrot",
    "Milk"
]

print("\n\n")
print("=" * 80)
print("FOUNDATION FOOD SEARCH")
print("=" * 80)

for item in search_foods:

    matches = search_foundation_food(item)

    print("\n" + "-" * 80)
    print("SEARCH:", item)
    print("-" * 80)

    if matches.empty:
        print("NO MATCHES FOUND")
    else:
        print("Matches:", len(matches))
        print(matches.to_string(index=False))


# ============================================================
# 8. DEFINE THE SPECIFIC FOODS WE WANT
# ============================================================
#
# These are TARGET DESCRIPTIONS.
#
# USDA's exact wording may differ in your 2026 dataset.
# Therefore, inspect the output from section 7 first and
# adjust these terms if necessary.
#

target_foods = {

    "apple": {
        "search": "Apple",
        "required_terms": ["raw"]
    },

    "banana": {
        "search": "Banana",
        "required_terms": ["raw"]
    },

    "chicken_breast": {
        "search": "Chicken",
        "required_terms": ["breast", "raw"]
    },

    "beef_ground": {
        "search": "Beef",
        "required_terms": ["ground", "raw"]
    },

    "oats": {
        "search": "Oats",
        "required_terms": ["raw"]
    },

    "carrot": {
        "search": "Carrot",
        "required_terms": ["raw"]
    },

    "milk": {
        "search": "Milk",
        "required_terms": ["whole"]
    }
}


# ============================================================
# 9. FOOD SELECTION FUNCTION
# ============================================================

def select_food(search, required_terms):
    """
    Search Foundation Foods and score candidates.

    A food is automatically selected only when:
    - all required terms are present
    - there is only one top-scoring candidate

    Otherwise it is sent for manual review.
    """

    matches = search_foundation_food(search)

    if matches.empty:

        return {
            "status": "NOT_FOUND",
            "selected": None,
            "candidates": matches
        }

    results = []

    for _, row in matches.iterrows():

        description = row["description"].lower()

        score = 0

        for term in required_terms:

            if term.lower() in description:
                score += 1

        results.append({
            "fdc_id": row["fdc_id"],
            "description": row["description"],
            "score": score
        })

    candidates = pd.DataFrame(results)

    candidates = candidates.sort_values(
        ["score", "description"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best_score = candidates.iloc[0]["score"]

    top = candidates[
        candidates["score"] == best_score
    ]

    # --------------------------------------------------------
    # Automatically select ONLY when there is one clear
    # candidate containing every required term.
    # --------------------------------------------------------

    if (
        best_score == len(required_terms)
        and len(top) == 1
    ):

        return {
            "status": "SELECTED",
            "selected": top.iloc[0],
            "candidates": candidates
        }

    elif len(top) > 1:

        return {
            "status": "AMBIGUOUS",
            "selected": None,
            "candidates": candidates
        }

    else:

        return {
            "status": "REVIEW",
            "selected": None,
            "candidates": candidates
        }


# ============================================================
# 10. SELECT FOODS
# ============================================================

selected_foods = {}
review_foods = {}

print("\n\n")
print("=" * 80)
print("SELECTING FOUNDATION FOODS")
print("=" * 80)

for food_name, config in target_foods.items():

    result = select_food(
        config["search"],
        config["required_terms"]
    )

    print("\n" + "-" * 80)
    print(food_name.upper())
    print("-" * 80)

    print("STATUS:", result["status"])

    if result["status"] == "SELECTED":

        selected = result["selected"]

        selected_foods[food_name] = {
            "fdc_id": int(selected["fdc_id"]),
            "description": selected["description"]
        }

        print("FDC ID:", selected["fdc_id"])
        print("Description:", selected["description"])

    else:

        review_foods[food_name] = result["candidates"]

        print("This food requires review.")

        print(
            result["candidates"]
            .head(20)
            .to_string(index=False)
        )


# ============================================================
# 11. SHOW SELECTED FOODS
# ============================================================

print("\n\n")
print("=" * 80)
print("SELECTED FOODS")
print("=" * 80)

for food_name, data in selected_foods.items():

    print(
        f"{food_name:20} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


# ============================================================
# 12. BUILD FDC ID MAP
# ============================================================

dynamic_id_map = {
    name: data["fdc_id"]
    for name, data in selected_foods.items()
}

print("\nFDC ID MAP:")
print(dynamic_id_map)


# ============================================================
# 13. IDENTIFY NUTRIENT COLUMNS
# ============================================================

print("\n\n")
print("=" * 80)
print("NUTRIENT TABLE INSPECTION")
print("=" * 80)

print(nutrient.head())


# ============================================================
# 14. FIND MACRONUTRIENT DEFINITIONS
# ============================================================

nutrient_name_column = None

for column in ["name", "nutrient_name"]:

    if column in nutrient.columns:
        nutrient_name_column = column
        break

if nutrient_name_column is None:

    raise ValueError(
        "Could not find nutrient name column."
    )


macro_definitions = nutrient[
    nutrient[nutrient_name_column]
    .astype(str)
    .str.contains(
        "protein|total lipid|carbohydrate",
        case=False,
        na=False,
        regex=True
    )
].copy()

print("\nPotential macronutrients:")
print(macro_definitions.to_string(index=False))


# ============================================================
# 15. FIND THE CORRECT USDA NUTRIENT IDs
# ============================================================

print("\n")
print("IMPORTANT:")
print("Use the output above to verify the nutrient IDs.")
print("Do NOT blindly assume IDs if your USDA file differs.")


# ============================================================
# 16. COMMON USDA MACRO IDS
# ============================================================
#
# USDA commonly uses:
#
# 1003 = Protein
# 1004 = Total lipid (fat)
# 1005 = Carbohydrate, by difference
#
# We verify that they exist in YOUR nutrient table.
#

COMMON_MACRO_IDS = {
    "protein_g": 1003,
    "fat_g": 1004,
    "carbohydrate_g": 1005
}


available_nutrient_ids = set(
    nutrient["id"].dropna().astype(int)
)

nutrient_ids = {}

for name, nutrient_id in COMMON_MACRO_IDS.items():

    if nutrient_id in available_nutrient_ids:

        nutrient_ids[name] = nutrient_id

    else:

        print(
            f"WARNING: nutrient ID {nutrient_id} "
            f"not found for {name}"
        )


print("\nUsing nutrient IDs:")
print(nutrient_ids)


# ============================================================
# 17. FIND THE FDC ID COLUMN
# ============================================================

if "fdc_id" not in food_nutrient.columns:

    raise ValueError(
        "food_nutrient.csv does not contain fdc_id."
    )


# ============================================================
# 18. FIND THE NUTRIENT ID COLUMN
# ============================================================

if "nutrient_id" not in food_nutrient.columns:

    raise ValueError(
        "food_nutrient.csv does not contain nutrient_id."
    )


# ============================================================
# 19. FIND THE AMOUNT COLUMN
# ============================================================

amount_column = None

for column in ["amount", "nutrient_amount"]:

    if column in food_nutrient.columns:
        amount_column = column
        break

if amount_column is None:

    raise ValueError(
        "Could not find nutrient amount column."
    )


print("\nNutrient amount column:", amount_column)


# ============================================================
# 20. EXTRACT MACROS
# ============================================================

macro_records = []

for food_name, food_data in selected_foods.items():

    fdc_id = food_data["fdc_id"]

    description = food_data["description"]

    rows = food_nutrient[
        food_nutrient["fdc_id"] == fdc_id
    ].copy()

    record = {
        "food_name": food_name,
        "fdc_id": fdc_id,
        "description": description,
        "data_type": "Foundation Food",

        # Explicit USDA reference basis
        "basis_g": 100
    }

    # --------------------------------------------------------
    # Extract each nutrient
    # --------------------------------------------------------

    for output_name, nutrient_id in nutrient_ids.items():

        nutrient_row = rows[
            rows["nutrient_id"] == nutrient_id
        ]

        if nutrient_row.empty:

            record[output_name] = None

        else:

            record[output_name] = (
                nutrient_row.iloc[0][amount_column]
            )

    macro_records.append(record)


# ============================================================
# 21. CREATE FINAL FOOD DATABASE
# ============================================================

food_database = pd.DataFrame(macro_records)


# ============================================================
# 22. CLEAN COLUMN ORDER
# ============================================================

preferred_columns = [
    "food_name",
    "fdc_id",
    "description",
    "data_type",
    "basis_g",
    "protein_g",
    "fat_g",
    "carbohydrate_g"
]

existing_columns = [
    column
    for column in preferred_columns
    if column in food_database.columns
]

food_database = food_database[existing_columns]


# ============================================================
# 23. DISPLAY FINAL DATABASE
# ============================================================

print("\n\n")
print("=" * 80)
print("FINAL FOOD DATABASE")
print("=" * 80)

print(
    food_database.to_string(index=False)
)


# ============================================================
# 24. SAVE DATABASE
# ============================================================

OUTPUT_FILE = DATA_FOLDER / "food_database.csv"

food_database.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n")
print("=" * 80)
print("DONE")
print("=" * 80)

print("Saved:", OUTPUT_FILE)


# ============================================================
# 25. SAVE FOODS THAT NEED MANUAL REVIEW
# ============================================================

if review_foods:

    review_records = []

    for food_name, candidates in review_foods.items():

        for _, row in candidates.iterrows():

            review_records.append({
                "requested_food": food_name,
                "fdc_id": row["fdc_id"],
                "description": row["description"],
                "score": row["score"]
            })

    review_df = pd.DataFrame(review_records)

    REVIEW_FILE = DATA_FOLDER / "food_selection_review.csv"

    review_df.to_csv(
        REVIEW_FILE,
        index=False
    )

    print(
        "Foods requiring review saved to:",
        REVIEW_FILE
    )

else:

    print("No food selections require manual review.")

Loading USDA files...
Food rows: 87990
Food nutrient rows: 170469
Nutrient definitions: 477

Food columns:
['fdc_id', 'data_type', 'description', 'food_category_id', 'publication_date']

Food nutrient columns:
['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id', 'min', 'max', 'median', 'footnote', 'min_year_acquired']

Nutrient columns:
['id', 'name', 'unit_name', 'nutrient_nbr', 'rank']

Foundation Foods: 469



FOUNDATION FOOD SEARCH

--------------------------------------------------------------------------------
SEARCH: Apple
--------------------------------------------------------------------------------
Matches: 14
 fdc_id                                                       description
2003590 Apple juice, with added vitamin C, from concentrate, shelf stable
1105897                                      Apples, fuji, with skin, raw
1750340                                      Apples, fuji, with skin, raw
1105781                                      Apples, g

In [15]:
# ============================================================
# USDA FOODDATA CENTRAL - FOUNDATION FOOD DATABASE
# FULL END-TO-END PIPELINE
# ============================================================
#
# INPUT FILES:
#   food.csv
#   food_nutrient.csv
#   nutrient.csv
#
# OUTPUT FILES:
#   food_database.csv
#   food_selection_review.csv
#
# ============================================================

import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Change this if your USDA CSV files are in another folder.
DATA_FOLDER = Path(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30")

FOOD_FILE = DATA_FOLDER / "food.csv"
FOOD_NUTRIENT_FILE = DATA_FOLDER / "food_nutrient.csv"
NUTRIENT_FILE = DATA_FOLDER / "nutrient.csv"

OUTPUT_DATABASE = DATA_FOLDER / "food_database.csv"
OUTPUT_REVIEW = DATA_FOLDER / "food_selection_review.csv"


# ============================================================
# 2. FOODS WE WANT
# ============================================================
#
# The search term finds candidates.
#
# required_terms are words that should appear in the USDA
# description.
#
# preferred_terms improve the ranking.
#
# IMPORTANT:
# The exact wording in your 2026 USDA dataset may differ.
# The program therefore produces a review file for anything
# it cannot confidently select.
#

TARGET_FOODS = {

    "apple": {
        "search": "apple",
        "required_terms": ["raw"],
        "preferred_terms": ["fuji", "with skin"]
    },

    "banana": {
        "search": "banana",
        "required_terms": ["raw"],
        "preferred_terms": []
    },

    "chicken_breast": {
        "search": "chicken",
        "required_terms": ["breast", "raw"],
        "preferred_terms": ["meat only", "broiler or fryers"]
    },

    "beef_ground": {
        "search": "beef",
        "required_terms": ["ground", "raw"],
        "preferred_terms": ["80% lean", "20% fat"]
    },

    "oats": {
        "search": "oats",
        "required_terms": ["raw"],
        "preferred_terms": []
    },

    "carrot": {
        "search": "carrot",
        "required_terms": ["raw"],
        "preferred_terms": []
    },

    "milk": {
        "search": "milk",
        "required_terms": ["whole"],
        "preferred_terms": ["3.25%"]
    },

    "rye": {
        "search": "rye",
        "required_terms": ["raw"],
        "preferred_terms": []
    },

    "mushroom": {
        "search": "mushroom",
        "required_terms": ["raw"],
        "preferred_terms": []
    }
}


# ============================================================
# 3. LOAD USDA FILES
# ============================================================

print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)

if not FOOD_FILE.exists():
    raise FileNotFoundError(
        f"Could not find: {FOOD_FILE}"
    )

if not FOOD_NUTRIENT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find: {FOOD_NUTRIENT_FILE}"
    )

if not NUTRIENT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find: {NUTRIENT_FILE}"
    )


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)

food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)

nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(f"food.csv rows:              {len(food):,}")
print(f"food_nutrient.csv rows:     {len(food_nutrient):,}")
print(f"nutrient.csv rows:          {len(nutrient):,}")


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_food_columns = [
    "fdc_id",
    "description",
    "data_type"
]

for column in required_food_columns:

    if column not in food.columns:
        raise ValueError(
            f"food.csv is missing required column: {column}"
        )


required_food_nutrient_columns = [
    "fdc_id",
    "nutrient_id"
]

for column in required_food_nutrient_columns:

    if column not in food_nutrient.columns:
        raise ValueError(
            f"food_nutrient.csv is missing required column: {column}"
        )


if "id" not in nutrient.columns:
    raise ValueError(
        "nutrient.csv is missing the 'id' column."
    )


if "name" not in nutrient.columns:
    raise ValueError(
        "nutrient.csv is missing the 'name' column."
    )


# ============================================================
# 5. FIND NUTRIENT AMOUNT COLUMN
# ============================================================

amount_column = None

for possible_column in [
    "amount",
    "nutrient_amount"
]:

    if possible_column in food_nutrient.columns:
        amount_column = possible_column
        break


if amount_column is None:

    raise ValueError(
        "Could not find nutrient amount column in "
        "food_nutrient.csv."
    )


print(f"Nutrient amount column:     {amount_column}")


# ============================================================
# 6. KEEP FOUNDATION FOODS ONLY
# ============================================================

foundation_food = food[
    food["data_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "foundation_food"
].copy()


foundation_food["description"] = (
    foundation_food["description"]
    .astype(str)
    .str.strip()
)


foundation_food["fdc_id"] = pd.to_numeric(
    foundation_food["fdc_id"],
    errors="coerce"
)


foundation_food = foundation_food.dropna(
    subset=["fdc_id"]
)


foundation_food["fdc_id"] = (
    foundation_food["fdc_id"]
    .astype(int)
)


print(
    f"Foundation Foods found:    "
    f"{len(foundation_food):,}"
)


if foundation_food.empty:

    raise ValueError(
        "No Foundation Foods were found. "
        "Check the USDA files and data_type column."
    )


# ============================================================
# 7. NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    """
    Normalize text for matching.
    """

    text = str(text).lower()

    # Replace punctuation with spaces
    text = re.sub(
        r"[^a-z0-9%]+",
        " ",
        text
    )

    # Collapse multiple spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


foundation_food["normalized_description"] = (
    foundation_food["description"]
    .apply(normalize_text)
)


# ============================================================
# 8. SEARCH FOUNDATION FOODS
# ============================================================

def search_foundation_food(query):
    """
    Search Foundation Foods using a word/phrase query.
    """

    query_normalized = normalize_text(query)

    if not query_normalized:
        return pd.DataFrame(
            columns=[
                "fdc_id",
                "description"
            ]
        )

    pattern = re.escape(query_normalized)

    matches = foundation_food[
        foundation_food["normalized_description"]
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    return matches[
        [
            "fdc_id",
            "description"
        ]
    ].sort_values(
        "description"
    )


# ============================================================
# 9. SCORE FOOD CANDIDATES
# ============================================================

def score_candidate(
    description,
    required_terms,
    preferred_terms
):
    """
    Score a USDA food description.

    Required terms:
        Strong requirement.

    Preferred terms:
        Increase ranking but are not mandatory.
    """

    normalized_description = normalize_text(
        description
    )

    score = 0

    missing_required = []

    # --------------------------------------------------------
    # Required terms
    # --------------------------------------------------------

    for term in required_terms:

        normalized_term = normalize_text(term)

        if normalized_term in normalized_description:

            score += 10

        else:

            missing_required.append(term)

    # --------------------------------------------------------
    # Preferred terms
    # --------------------------------------------------------

    for term in preferred_terms:

        normalized_term = normalize_text(term)

        if normalized_term in normalized_description:

            score += 3

    return score, missing_required


# ============================================================
# 10. SELECT FOOD
# ============================================================

def select_food(
    search,
    required_terms,
    preferred_terms
):
    """
    Select a Foundation Food only when there is a clear
    candidate.

    Returns:
        status
        selected food
        candidate list
    """

    matches = search_foundation_food(search)

    if matches.empty:

        return {
            "status": "NOT_FOUND",
            "selected": None,
            "candidates": matches
        }


    scored_rows = []

    for _, row in matches.iterrows():

        score, missing_required = score_candidate(
            row["description"],
            required_terms,
            preferred_terms
        )

        scored_rows.append({
            "fdc_id": int(row["fdc_id"]),
            "description": row["description"],
            "score": score,
            "missing_required": ", ".join(
                missing_required
            )
        })


    candidates = pd.DataFrame(
        scored_rows
    )


    candidates = candidates.sort_values(
        [
            "score",
            "description"
        ],
        ascending=[
            False,
            True
        ]
    ).reset_index(
        drop=True
    )


    best = candidates.iloc[0]

    required_count = len(
        required_terms
    )

    best_required_count = (
        required_count
        - len(
            [
                x for x in str(
                    best["missing_required"]
                ).split(", ")
                if x
            ]
        )
    )


    # --------------------------------------------------------
    # Candidate must contain every required term.
    # --------------------------------------------------------

    if best_required_count != required_count:

        return {
            "status": "REVIEW",
            "selected": None,
            "candidates": candidates
        }


    # --------------------------------------------------------
    # Find all candidates with same top score.
    # --------------------------------------------------------

    top_candidates = candidates[
        candidates["score"]
        == best["score"]
    ]


    # --------------------------------------------------------
    # Exactly one best candidate = automatic selection.
    # --------------------------------------------------------

    if len(top_candidates) == 1:

        return {
            "status": "SELECTED",
            "selected": best,
            "candidates": candidates
        }


    # --------------------------------------------------------
    # Multiple equally good candidates = ambiguous.
    # --------------------------------------------------------

    return {
        "status": "AMBIGUOUS",
        "selected": None,
        "candidates": candidates
    }


# ============================================================
# 11. TEST BROAD SEARCHES
# ============================================================

print("\n")
print("=" * 70)
print("FOUNDATION FOOD SEARCH RESULTS")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    matches = search_foundation_food(
        config["search"]
    )

    print("\n" + "-" * 70)
    print(
        f"{food_name.upper()} "
        f"({len(matches)} candidates)"
    )
    print("-" * 70)

    if matches.empty:

        print("NO MATCHES FOUND")

    else:

        print(
            matches
            .head(20)
            .to_string(index=False)
        )


# ============================================================
# 12. SELECT TARGET FOODS
# ============================================================

selected_foods = {}

review_records = []


print("\n")
print("=" * 70)
print("SELECTING TARGET FOUNDATION FOODS")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    result = select_food(
        search=config["search"],
        required_terms=config["required_terms"],
        preferred_terms=config["preferred_terms"]
    )


    print("\n" + "-" * 70)
    print(food_name.upper())
    print("-" * 70)

    print(
        "STATUS:",
        result["status"]
    )


    if result["status"] == "SELECTED":

        selected = result["selected"]

        selected_foods[food_name] = {
            "fdc_id": int(
                selected["fdc_id"]
            ),
            "description": selected[
                "description"
            ]
        }

        print(
            "FDC ID:",
            selected["fdc_id"]
        )

        print(
            "Description:",
            selected["description"]
        )


    else:

        candidates = result[
            "candidates"
        ]

        print(
            f"{food_name} requires review."
        )


        for _, candidate in candidates.head(20).iterrows():

            review_records.append({

                "requested_food": food_name,

                "status": result["status"],

                "fdc_id": candidate["fdc_id"],

                "description": candidate[
                    "description"
                ],

                "score": candidate["score"],

                "missing_required": candidate[
                    "missing_required"
                ]
            })


# ============================================================
# 13. SAVE REVIEW FILE
# ============================================================

if review_records:

    review_df = pd.DataFrame(
        review_records
    )

    review_df.to_csv(
        OUTPUT_REVIEW,
        index=False
    )

    print("\n")
    print(
        f"Review file saved to: "
        f"{OUTPUT_REVIEW}"
    )

else:

    print(
        "\nNo food selections require review."
    )


# ============================================================
# 14. SHOW SELECTED FOODS
# ============================================================

print("\n")
print("=" * 70)
print("SELECTED FOODS")
print("=" * 70)


if not selected_foods:

    raise ValueError(
        "No foods were selected. "
        "Check food_selection_review.csv."
    )


for food_name, data in selected_foods.items():

    print(
        f"{food_name:20} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


# ============================================================
# 15. BUILD NUTRIENT LOOKUP
# ============================================================

print("\n")
print("=" * 70)
print("FINDING NUTRIENTS")
print("=" * 70)


def find_nutrient(
    search_terms,
    preferred_unit=None
):
    """
    Find a USDA nutrient definition.

    Returns the best matching nutrient ID.
    """

    temp = nutrient.copy()

    temp["_name_normalized"] = (
        temp["name"]
        .astype(str)
        .apply(normalize_text)
    )


    # --------------------------------------------------------
    # Score nutrient names.
    # --------------------------------------------------------

    scores = []

    for _, row in temp.iterrows():

        name = row["_name_normalized"]

        score = 0

        for term in search_terms:

            normalized_term = normalize_text(
                term
            )

            if normalized_term in name:

                score += 10


        # Prefer correct unit.
        if (
            preferred_unit is not None
            and "unit_name" in row
        ):

            unit = str(
                row["unit_name"]
            ).lower()

            if preferred_unit.lower() in unit:

                score += 5


        scores.append(score)


    temp["_score"] = scores


    temp = temp[
        temp["_score"] > 0
    ].sort_values(
        "_score",
        ascending=False
    )


    if temp.empty:

        return None


    return {
        "id": int(
            temp.iloc[0]["id"]
        ),

        "name": temp.iloc[0]["name"],

        "unit": (
            temp.iloc[0]["unit_name"]
            if "unit_name" in temp.columns
            else None
        )
    }


# ============================================================
# 16. NUTRIENTS REQUIRED BY THE APP
# ============================================================

NUTRIENT_REQUESTS = {

    "calories_kcal": {
        "terms": [
            "energy"
        ],
        "unit": "kcal"
    },

    "protein_g": {
        "terms": [
            "protein"
        ],
        "unit": "g"
    },

    "fat_g": {
        "terms": [
            "total lipid"
        ],
        "unit": "g"
    },

    "carbohydrate_g": {
        "terms": [
            "carbohydrate",
            "by difference"
        ],
        "unit": "g"
    },

    "fiber_g": {
        "terms": [
            "fiber",
            "total dietary"
        ],
        "unit": "g"
    },

    "sugar_g": {
        "terms": [
            "sugars",
            "total"
        ],
        "unit": "g"
    },

    "saturated_fat_g": {
        "terms": [
            "fatty acids",
            "total saturated"
        ],
        "unit": "g"
    }
}


nutrient_lookup = {}


for output_name, config in NUTRIENT_REQUESTS.items():

    result = find_nutrient(
        config["terms"],
        config["unit"]
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result

        print(
            f"{output_name:25} "
            f"ID={result['id']:5} "
            f"UNIT={result['unit']} "
            f"NAME={result['name']}"
        )


# ============================================================
# 17. VERIFY NUTRIENT IDs
# ============================================================

if not nutrient_lookup:

    raise ValueError(
        "No nutrients could be identified."
    )


# ============================================================
# 18. EXTRACT NUTRIENTS FOR SELECTED FOODS
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING NUTRIENTS")
print("=" * 70)


nutrition_records = []


for food_name, food_data in selected_foods.items():

    fdc_id = food_data["fdc_id"]


    rows = food_nutrient[
        food_nutrient["fdc_id"]
        == fdc_id
    ].copy()


    record = {

        "food_name": food_name,

        "fdc_id": fdc_id,

        "description": food_data[
            "description"
        ],

        "data_type": "Foundation Food",

        # USDA nutrient values are stored on
        # the food's reference basis.
        #
        # For Foundation Foods, the intended
        # application basis here is 100 g.
        "basis_g": 100
    }


    for output_name, nutrient_info in nutrient_lookup.items():

        nutrient_id = nutrient_info["id"]


        nutrient_row = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]


        if nutrient_row.empty:

            record[output_name] = None

        else:

            value = nutrient_row.iloc[0][
                amount_column
            ]

            record[output_name] = (
                pd.to_numeric(
                    value,
                    errors="coerce"
                )
            )


    nutrition_records.append(
        record
    )


# ============================================================
# 19. CREATE FINAL DATABASE
# ============================================================

food_database = pd.DataFrame(
    nutrition_records
)


# ============================================================
# 20. CLEAN NUMERIC COLUMNS
# ============================================================

numeric_columns = [

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


for column in numeric_columns:

    if column in food_database.columns:

        food_database[column] = pd.to_numeric(
            food_database[column],
            errors="coerce"
        )


# ============================================================
# 21. ROUND NUTRIENTS
# ============================================================

for column in numeric_columns:

    if column in food_database.columns:

        food_database[column] = (
            food_database[column]
            .round(4)
        )


# ============================================================
# 22. ORDER COLUMNS
# ============================================================

preferred_columns = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


existing_columns = [

    column

    for column in preferred_columns

    if column in food_database.columns
]


food_database = food_database[
    existing_columns
]


# ============================================================
# 23. VALIDATE DATABASE
# ============================================================

def validate_food_database(
    database
):
    """
    Validate the final Foundation Food database.
    """

    print("\n")
    print("=" * 70)
    print("DATABASE VALIDATION")
    print("=" * 70)


    errors = []

    warnings = []


    # --------------------------------------------------------
    # Empty database
    # --------------------------------------------------------

    if database.empty:

        errors.append(
            "Database contains zero foods."
        )


    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required_columns = [

        "food_name",

        "fdc_id",

        "description",

        "data_type",

        "basis_g"
    ]


    for column in required_columns:

        if column not in database.columns:

            errors.append(
                f"Missing required column: {column}"
            )


    # --------------------------------------------------------
    # Duplicate FDC IDs
    # --------------------------------------------------------

    if "fdc_id" in database.columns:

        duplicate_ids = database[
            database["fdc_id"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_ids.empty:

            errors.append(
                "Duplicate FDC IDs found."
            )


    # --------------------------------------------------------
    # Duplicate food names
    # --------------------------------------------------------

    if "food_name" in database.columns:

        duplicate_names = database[
            database["food_name"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_names.empty:

            warnings.append(
                "Duplicate food names found."
            )


    # --------------------------------------------------------
    # Basis
    # --------------------------------------------------------

    if "basis_g" in database.columns:

        invalid_basis = database[
            database["basis_g"] <= 0
        ]


        if not invalid_basis.empty:

            errors.append(
                "Invalid basis_g value found."
            )


    # --------------------------------------------------------
    # Negative nutrients
    # --------------------------------------------------------

    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    for column in nutrient_columns:

        if column not in database.columns:
            continue


        negative = database[
            database[column].notna()
            & (database[column] < 0)
        ]


        if not negative.empty:

            errors.append(
                f"Negative values found in {column}."
            )


    # --------------------------------------------------------
    # Missing macros
    # --------------------------------------------------------

    for column in [

        "protein_g",

        "fat_g",

        "carbohydrate_g"
    ]:

        if column not in database.columns:
            continue


        missing = database[column].isna().sum()


        if missing > 0:

            warnings.append(
                f"{missing} foods have missing "
                f"{column}."
            )


    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print(
        f"Foods:                  {len(database)}"
    )


    if "fdc_id" in database.columns:

        print(
            "Unique FDC IDs:         "
            f"{database['fdc_id'].nunique()}"
        )


    if errors:

        print("\nERRORS:")

        for error in errors:

            print(
                "  ERROR:",
                error
            )


    else:

        print(
            "\nNo critical validation errors."
        )


    if warnings:

        print("\nWARNINGS:")

        for warning in warnings:

            print(
                "  WARNING:",
                warning
            )


    if errors:

        print(
            "\nSTATUS: FAIL"
        )

        return False


    print(
        "\nSTATUS: PASS"
    )

    return True


# Run validation
database_valid = validate_food_database(
    food_database
)


# ============================================================
# 24. DISPLAY FINAL DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# 25. SAVE DATABASE
# ============================================================

food_database.to_csv(
    OUTPUT_DATABASE,
    index=False
)


print("\n")
print("=" * 70)
print("DATABASE SAVED")
print("=" * 70)


print(
    f"File: {OUTPUT_DATABASE}"
)


# ============================================================
# 26. NUTRITION CALCULATOR
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):
    """
    Calculate nutrition for a quantity of food.

    Database values are normalized to 100 g.
    """

    # --------------------------------------------------------
    # Validate grams
    # --------------------------------------------------------

    grams = float(grams)


    if grams <= 0:

        raise ValueError(
            "Grams must be greater than zero."
        )


    # --------------------------------------------------------
    # Find food
    # --------------------------------------------------------

    matches = database[
        database["food_name"]
        .astype(str)
        .str.lower()
        == str(food_name)
        .lower()
    ]


    if matches.empty:

        raise ValueError(
            f"Food not found: {food_name}"
        )


    if len(matches) > 1:

        raise ValueError(
            f"Multiple foods found for: "
            f"{food_name}"
        )


    food = matches.iloc[0]


    # --------------------------------------------------------
    # Calculate multiplier
    # --------------------------------------------------------

    basis = float(
        food["basis_g"]
    )


    multiplier = (
        grams / basis
    )


    # --------------------------------------------------------
    # Calculate nutrients
    # --------------------------------------------------------

    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    result = {

        "food_name":
            food["food_name"],

        "grams":
            grams
    }


    for column in nutrient_columns:

        if column not in database.columns:

            continue


        value = food[column]


        if pd.isna(value):

            result[column] = None

        else:

            result[column] = (
                float(value)
                * multiplier
            )


    return result


# ============================================================
# 27. PRETTY PRINT NUTRITION
# ============================================================

def print_nutrition(
    result
):

    print("\n")
    print("=" * 60)

    print(
        f"{result['food_name']} "
        f"— {result['grams']:.1f} g"
    )

    print("=" * 60)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for key, label in labels.items():

        value = result.get(key)


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} "
                f"{units[key]}"
            )


# ============================================================
# 28. MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):
    """
    Calculate total nutrition for multiple foods.

    Example:

    meal_items = [
        {
            "food_name": "chicken_breast",
            "grams": 150
        },
        {
            "food_name": "banana",
            "grams": 120
        }
    ]
    """

    totals = {

        "calories_kcal": 0.0,

        "protein_g": 0.0,

        "fat_g": 0.0,

        "carbohydrate_g": 0.0,

        "fiber_g": 0.0,

        "sugar_g": 0.0,

        "saturated_fat_g": 0.0
    }


    foods = []


    for item in meal_items:

        result = calculate_nutrition(

            database,

            item["food_name"],

            item["grams"]
        )


        foods.append(
            result
        )


        for nutrient_name in totals:

            value = result.get(
                nutrient_name
            )


            if value is not None:

                totals[
                    nutrient_name
                ] += value


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# 29. PRETTY PRINT MEAL
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal["foods"]:

        print(
            f"{food['food_name']:25} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name, value in meal[
        "totals"
    ].items():

        print(
            f"{labels[nutrient_name]:20}: "
            f"{value:.2f} "
            f"{units[nutrient_name]}"
        )


# ============================================================
# 30. EXAMPLE TEST
# ============================================================
#
# Only run the example if those foods were successfully
# selected.
#

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


test_food = None


for preferred_food in [
    "chicken_breast",
    "banana",
    "apple",
    "oats"
]:

    if preferred_food in food_database[
        "food_name"
    ].values:

        test_food = preferred_food
        break


if test_food is not None:

    test_result = calculate_nutrition(
        food_database,
        test_food,
        150
    )

    print_nutrition(
        test_result
    )

else:

    print(
        "No test food was successfully selected."
    )


# ============================================================
# 31. EXAMPLE MEAL TEST
# ============================================================

available_foods = set(
    food_database[
        "food_name"
    ].values
)


example_meal = []


possible_meal = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {
        "food_name":
            "banana",

        "grams":
            120
    },

    {
        "food_name":
            "oats",

        "grams":
            80
    }
]


for item in possible_meal:

    if item["food_name"] in available_foods:

        example_meal.append(
            item
        )


if example_meal:

    meal_result = calculate_meal(
        food_database,
        example_meal
    )

    print_meal(
        meal_result
    )


# ============================================================
# 32. FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)

print(
    f"Foods selected: "
    f"{len(food_database)}"
)

print(
    f"Foods requiring review: "
    f"{len(review_records)}"
)

print(
    f"Database validation: "
    f"{'PASS' if database_valid else 'FAIL'}"
)

print(
    f"\nDatabase: "
    f"{OUTPUT_DATABASE}"
)

if review_records:

    print(
        f"Review:   "
        f"{OUTPUT_REVIEW}"
    )

print("\nDone.")

LOADING USDA FOODDATA CENTRAL FILES
food.csv rows:              87,990
food_nutrient.csv rows:     170,469
nutrient.csv rows:          477
Nutrient amount column:     amount
Foundation Foods found:    469


FOUNDATION FOOD SEARCH RESULTS

----------------------------------------------------------------------
APPLE (14 candidates)
----------------------------------------------------------------------
 fdc_id                                                       description
2003590 Apple juice, with added vitamin C, from concentrate, shelf stable
1105897                                      Apples, fuji, with skin, raw
1750340                                      Apples, fuji, with skin, raw
1105781                                      Apples, gala, with skin, raw
1750341                                      Apples, gala, with skin, raw
1105664                              Apples, granny smith, with skin, raw
1750342                              Apples, granny smith, with skin, raw
11055

In [16]:
# ============================================================
# USDA FOODDATA CENTRAL - FOUNDATION FOOD DATABASE
# FULL END-TO-END PIPELINE
# ============================================================
#
# INPUT FILES:
#   food.csv
#   food_nutrient.csv
#   nutrient.csv
#
# OUTPUT FILES:
#   food_database.csv
#   food_selection_review.csv
#
# ============================================================
# IMPORTANT
# ============================================================
#
# This version:
#
#   1. Automatically finds the USDA files under Downloads.
#   2. Uses exact USDA descriptions for target foods.
#   3. Handles duplicate FDC records.
#   4. Compares duplicate records using nutrient values.
#   5. Selects duplicates automatically when their nutrition
#      profiles are identical.
#   6. Flags duplicates when their nutrient profiles differ.
#   7. Extracts nutrition after food selection is finalized.
#
# ============================================================


import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. CONFIGURATION / FILE LOCATION
# ============================================================

print("=" * 70)
print("USDA FOODDATA CENTRAL - CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Search this folder and all subfolders.
# ------------------------------------------------------------

DOWNLOADS_FOLDER = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)


# ------------------------------------------------------------
# Find USDA files automatically.
# ------------------------------------------------------------

def find_usda_file(filename):
    """
    Search Downloads and all subfolders for a USDA CSV file.
    """

    matches = list(
        DOWNLOADS_FOLDER.rglob(filename)
    )


    if not matches:

        raise FileNotFoundError(
            f"\nCould not find '{filename}' anywhere under:\n"
            f"{DOWNLOADS_FOLDER.resolve()}\n\n"
            f"Make sure the USDA ZIP file has been extracted."
        )


    # --------------------------------------------------------
    # If multiple copies exist, show them.
    # --------------------------------------------------------

    if len(matches) > 1:

        print(
            f"\nWARNING: Multiple copies of {filename} found:"
        )

        for match in matches:

            print(
                f"  {match.resolve()}"
            )

        print(
            f"\nUsing first match:"
        )

        print(
            f"  {matches[0].resolve()}"
        )


    return matches[0]


# ------------------------------------------------------------
# Locate input files.
# ------------------------------------------------------------

FOOD_FILE = find_usda_file(
    "food.csv"
)

FOOD_NUTRIENT_FILE = find_usda_file(
    "food_nutrient.csv"
)

NUTRIENT_FILE = find_usda_file(
    "nutrient.csv"
)


# ------------------------------------------------------------
# Data folder is the folder containing food.csv.
# ------------------------------------------------------------

DATA_FOLDER = FOOD_FILE.parent


# ------------------------------------------------------------
# Output files.
# ------------------------------------------------------------

OUTPUT_DATABASE = (
    DATA_FOLDER
    / "food_database.csv"
)

OUTPUT_REVIEW = (
    DATA_FOLDER
    / "food_selection_review.csv"
)


# ------------------------------------------------------------
# Display paths.
# ------------------------------------------------------------

print(
    f"\nData folder:\n"
    f"{DATA_FOLDER.resolve()}"
)

print(
    f"\nfood.csv:\n"
    f"{FOOD_FILE.resolve()}"
)

print(
    f"\nfood_nutrient.csv:\n"
    f"{FOOD_NUTRIENT_FILE.resolve()}"
)

print(
    f"\nnutrient.csv:\n"
    f"{NUTRIENT_FILE.resolve()}"
)

print(
    f"\nOutput database:\n"
    f"{OUTPUT_DATABASE.resolve()}"
)

print(
    f"\nReview file:\n"
    f"{OUTPUT_REVIEW.resolve()}"
)


# ============================================================
# 2. VERIFY INPUT FILES
# ============================================================

for file_path in [
    FOOD_FILE,
    FOOD_NUTRIENT_FILE,
    NUTRIENT_FILE
]:

    if not file_path.exists():

        raise FileNotFoundError(
            f"Could not find:\n"
            f"{file_path.resolve()}"
        )


print("\nAll USDA input files found.")


# ============================================================
# 3. LOAD USDA FILES
# ============================================================

print("\n")
print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(
    f"food.csv rows:              "
    f"{len(food):,}"
)


print(
    f"food_nutrient.csv rows:     "
    f"{len(food_nutrient):,}"
)


print(
    f"nutrient.csv rows:          "
    f"{len(nutrient):,}"
)


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_food_columns = [

    "fdc_id",

    "description",

    "data_type"
]


for column in required_food_columns:

    if column not in food.columns:

        raise ValueError(
            f"food.csv is missing required column: "
            f"{column}"
        )


required_food_nutrient_columns = [

    "fdc_id",

    "nutrient_id"
]


for column in required_food_nutrient_columns:

    if column not in food_nutrient.columns:

        raise ValueError(
            f"food_nutrient.csv is missing required column: "
            f"{column}"
        )


if "id" not in nutrient.columns:

    raise ValueError(
        "nutrient.csv is missing the 'id' column."
    )


if "name" not in nutrient.columns:

    raise ValueError(
        "nutrient.csv is missing the 'name' column."
    )


# ============================================================
# 5. FIND NUTRIENT AMOUNT COLUMN
# ============================================================

amount_column = None


for possible_column in [

    "amount",

    "nutrient_amount"
]:

    if possible_column in food_nutrient.columns:

        amount_column = possible_column

        break


if amount_column is None:

    raise ValueError(
        "Could not find nutrient amount column in "
        "food_nutrient.csv."
    )


print(
    f"Nutrient amount column:     "
    f"{amount_column}"
)


# ============================================================
# 6. KEEP FOUNDATION FOODS ONLY
# ============================================================

foundation_food = food[
    food["data_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "foundation_food"
].copy()


foundation_food["description"] = (
    foundation_food["description"]
    .astype(str)
    .str.strip()
)


foundation_food["fdc_id"] = pd.to_numeric(
    foundation_food["fdc_id"],
    errors="coerce"
)


foundation_food = foundation_food.dropna(
    subset=["fdc_id"]
)


foundation_food["fdc_id"] = (
    foundation_food["fdc_id"]
    .astype(int)
)


print(
    f"Foundation Foods found:    "
    f"{len(foundation_food):,}"
)


if foundation_food.empty:

    raise ValueError(
        "No Foundation Foods were found."
    )


# ============================================================
# 7. NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    """
    Normalize text for matching.
    """

    text = str(text).lower()


    text = re.sub(
        r"[^a-z0-9%]+",
        " ",
        text
    )


    text = re.sub(
        r"\s+",
        " ",
        text
    )


    return text.strip()


foundation_food["normalized_description"] = (
    foundation_food["description"]
    .apply(normalize_text)
)


# ============================================================
# 8. TARGET FOODS
# ============================================================
#
# These descriptions are based directly on the USDA records
# shown in your output.
#
# ============================================================

TARGET_FOODS = {

    # --------------------------------------------------------
    # APPLE
    # --------------------------------------------------------

    "apple_fuji": {

        "exact_description":
            "Apples, fuji, with skin, raw"
    },


    # --------------------------------------------------------
    # BANANA
    # --------------------------------------------------------

    "banana": {

        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },


    # --------------------------------------------------------
    # CHICKEN BREAST
    # --------------------------------------------------------

    "chicken_breast": {

        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },


    # --------------------------------------------------------
    # GROUND BEEF 80/20
    # --------------------------------------------------------

    "beef_ground": {

        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },


    # --------------------------------------------------------
    # OATS
    # --------------------------------------------------------

    "oats_rolled": {

        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },


    # --------------------------------------------------------
    # CARROT
    # --------------------------------------------------------

    "carrot": {

        "exact_description":
            "Carrots, mature, raw"
    },


    # --------------------------------------------------------
    # WHOLE MILK
    # --------------------------------------------------------

    "milk_whole": {

        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },


    # --------------------------------------------------------
    # RYE FLOUR
    # --------------------------------------------------------

    "rye_flour": {

        "exact_description":
            "Flour, rye"
    },


    # --------------------------------------------------------
    # WHITE BUTTON MUSHROOM
    # --------------------------------------------------------

    "mushroom_white_button": {

        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# 9. SEARCH EXACT USDA DESCRIPTION
# ============================================================

def find_exact_food(
    exact_description
):
    """
    Find Foundation Foods whose normalized description
    exactly matches the requested USDA description.
    """

    target = normalize_text(
        exact_description
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target
    ].copy()


    return matches[
        [
            "fdc_id",
            "description"
        ]
    ].sort_values(
        "fdc_id"
    ).reset_index(
        drop=True
    )


# ============================================================
# 10. FIND NUTRIENTS
# ============================================================

print("\n")
print("=" * 70)
print("FINDING NUTRIENTS")
print("=" * 70)


def find_nutrient(
    search_terms,
    preferred_unit=None
):
    """
    Find a USDA nutrient definition.

    The function scores nutrient names based on the requested
    terms and preferred unit.
    """

    temp = nutrient.copy()


    temp["_name_normalized"] = (
        temp["name"]
        .astype(str)
        .apply(normalize_text)
    )


    scores = []


    for _, row in temp.iterrows():

        name = row["_name_normalized"]

        score = 0


        for term in search_terms:

            normalized_term = normalize_text(
                term
            )


            if normalized_term in name:

                score += 10


        if (
            preferred_unit is not None
            and "unit_name" in row
        ):

            unit = str(
                row["unit_name"]
            ).lower()


            if preferred_unit.lower() in unit:

                score += 5


        scores.append(
            score
        )


    temp["_score"] = scores


    temp = temp[
        temp["_score"] > 0
    ].sort_values(
        [
            "_score",
            "id"
        ],
        ascending=[
            False,
            True
        ]
    )


    if temp.empty:

        return None


    row = temp.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            (
                row["unit_name"]
                if "unit_name" in row
                else None
            )
    }


# ============================================================
# 11. NUTRIENTS REQUIRED BY THE APP
# ============================================================

NUTRIENT_REQUESTS = {

    "calories_kcal": {

        "terms": [
            "energy"
        ],

        "unit": "kcal"
    },


    "protein_g": {

        "terms": [
            "protein"
        ],

        "unit": "g"
    },


    "fat_g": {

        "terms": [
            "total lipid"
        ],

        "unit": "g"
    },


    "carbohydrate_g": {

        "terms": [
            "carbohydrate",
            "by difference"
        ],

        "unit": "g"
    },


    "fiber_g": {

        "terms": [
            "fiber",
            "total dietary"
        ],

        "unit": "g"
    },


    "sugar_g": {

        "terms": [
            "sugars",
            "total"
        ],

        "unit": "g"
    },


    "saturated_fat_g": {

        "terms": [
            "fatty acids",
            "total saturated"
        ],

        "unit": "g"
    }
}


# ============================================================
# 12. BUILD NUTRIENT LOOKUP
# ============================================================

nutrient_lookup = {}


for output_name, config in NUTRIENT_REQUESTS.items():

    result = find_nutrient(
        config["terms"],
        config["unit"]
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


        print(
            f"{output_name:25} "
            f"ID={result['id']:5} "
            f"UNIT={result['unit']} "
            f"NAME={result['name']}"
        )


if not nutrient_lookup:

    raise ValueError(
        "No nutrients could be identified."
    )


# ============================================================
# 13. SELECT EXACT TARGET FOODS
# ============================================================

print("\n")
print("=" * 70)
print("EXACT FOUNDATION FOOD SELECTION")
print("=" * 70)


selected_foods = {}

duplicate_foods = {}

not_found_foods = {}


for food_name, config in TARGET_FOODS.items():

    exact_description = config[
        "exact_description"
    ]


    matches = find_exact_food(
        exact_description
    )


    print("\n" + "-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)


    print(
        "Requested:"
    )

    print(
        exact_description
    )


    if matches.empty:

        print(
            "STATUS: NOT_FOUND"
        )


        not_found_foods[
            food_name
        ] = exact_description


        continue


    if len(matches) == 1:

        row = matches.iloc[0]


        selected_foods[
            food_name
        ] = {

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"]
        }


        print(
            "STATUS: SELECTED"
        )


        print(
            "FDC ID:",
            row["fdc_id"]
        )


        print(
            "Description:",
            row["description"]
        )


    else:

        duplicate_foods[
            food_name
        ] = matches


        print(
            "STATUS: DUPLICATE_DESCRIPTION"
        )


        print(
            "Multiple FDC records have "
            "the same description:"
        )


        print(
            matches.to_string(
                index=False
            )
        )


# ============================================================
# 14. COMPARE DUPLICATE FDC RECORDS
# ============================================================

def compare_food_nutrients(
    fdc_ids,
    food_nutrient,
    nutrient_lookup,
    amount_column
):
    """
    Compare the application nutrients for duplicate FDC IDs.
    """

    rows = food_nutrient[
        food_nutrient["fdc_id"].isin(
            fdc_ids
        )
    ].copy()


    comparison = []


    for fdc_id in fdc_ids:

        record = {

            "fdc_id":
                int(fdc_id)
        }


        food_rows = rows[
            rows["fdc_id"]
            == fdc_id
        ]


        for nutrient_name, nutrient_info in nutrient_lookup.items():

            nutrient_id = nutrient_info[
                "id"
            ]


            nutrient_row = food_rows[
                food_rows["nutrient_id"]
                == nutrient_id
            ]


            if nutrient_row.empty:

                record[
                    nutrient_name
                ] = None

            else:

                value = nutrient_row.iloc[0][
                    amount_column
                ]


                record[
                    nutrient_name
                ] = pd.to_numeric(
                    value,
                    errors="coerce"
                )


        comparison.append(
            record
        )


    return pd.DataFrame(
        comparison
    )


# ============================================================
# 15. RESOLVE DUPLICATE FOOD RECORDS
# ============================================================

print("\n")
print("=" * 70)
print("CHECKING DUPLICATE FDC RECORDS")
print("=" * 70)


duplicate_review_records = []


for food_name, candidates in duplicate_foods.items():

    fdc_ids = (
        candidates["fdc_id"]
        .astype(int)
        .tolist()
    )


    print("\n" + "-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)


    comparison = compare_food_nutrients(
        fdc_ids,
        food_nutrient,
        nutrient_lookup,
        amount_column
    )


    print(
        comparison.to_string(
            index=False
        )
    )


    nutrient_columns = [

        column

        for column in comparison.columns

        if column != "fdc_id"
    ]


    # --------------------------------------------------------
    # Determine whether all available nutrient values are
    # identical.
    # --------------------------------------------------------

    identical = True


    differences = []


    for column in nutrient_columns:

        values = comparison[
            column
        ]


        # ----------------------------------------------------
        # Compare NaN safely.
        #
        # If every value is missing, there is no evidence
        # of a difference.
        # ----------------------------------------------------

        if values.isna().all():

            continue


        # ----------------------------------------------------
        # If some are missing and others have values,
        # treat as different.
        # ----------------------------------------------------

        if values.isna().any():

            identical = False

            differences.append(
                f"{column}: missing/value difference"
            )

            continue


        # ----------------------------------------------------
        # Compare numeric values.
        # ----------------------------------------------------

        numeric_values = pd.to_numeric(
            values,
            errors="coerce"
        )


        if not numeric_values.isna().any():

            if not numeric_values.round(8).eq(
                numeric_values.iloc[0].round(8)
            ).all():

                identical = False

                differences.append(
                    f"{column}: different values"
                )


        else:

            if values.nunique(
                dropna=False
            ) > 1:

                identical = False

                differences.append(
                    f"{column}: different values"
                )


    # ========================================================
    # IDENTICAL DUPLICATES
    # ========================================================

    if identical:

        canonical_row = candidates.iloc[0]


        selected_foods[
            food_name
        ] = {

            "fdc_id":
                int(
                    canonical_row["fdc_id"]
                ),

            "description":
                canonical_row[
                    "description"
                ]
        }


        print(
            "\nRESULT: IDENTICAL DUPLICATES"
        )


        print(
            "These FDC records have identical "
            "application nutrient values."
        )


        print(
            "Using canonical FDC ID:",
            canonical_row["fdc_id"]
        )


    # ========================================================
    # DIFFERENT DUPLICATES
    # ========================================================

    else:

        print(
            "\nRESULT: DIFFERENT NUTRIENT VALUES"
        )


        print(
            "Manual review required."
        )


        if differences:

            print(
                "\nDifferences:"
            )


            for difference in differences:

                print(
                    "  -",
                    difference
                )


        for _, candidate in candidates.iterrows():

            duplicate_review_records.append({

                "requested_food":
                    food_name,

                "status":
                    "DUPLICATE_DIFFERENT_NUTRIENTS",

                "fdc_id":
                    int(
                        candidate["fdc_id"]
                    ),

                "description":
                    candidate[
                        "description"
                    ],

                "score":
                    None,

                "missing_required":
                    None
            })


# ============================================================
# 16. SELECTION SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("SELECTION SUMMARY")
print("=" * 70)


print(
    f"Target foods:            "
    f"{len(TARGET_FOODS)}"
)


print(
    f"Selected foods:          "
    f"{len(selected_foods)}"
)


print(
    f"Duplicate descriptions:  "
    f"{len(duplicate_foods)}"
)


print(
    f"Not found:               "
    f"{len(not_found_foods)}"
)


print(
    f"Duplicate review rows:   "
    f"{len(duplicate_review_records)}"
)


print("\nSelected foods:")


for food_name, data in selected_foods.items():

    print(
        f"{food_name:30} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


# ============================================================
# 17. HANDLE NOT-FOUND FOODS
# ============================================================

for food_name, description in not_found_foods.items():

    duplicate_review_records.append({

        "requested_food":
            food_name,

        "status":
            "NOT_FOUND",

        "fdc_id":
            None,

        "description":
            description,

        "score":
            None,

        "missing_required":
            None
    })


# ============================================================
# 18. EXTRACT NUTRIENTS
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING NUTRIENTS")
print("=" * 70)


nutrition_records = []


for food_name, food_data in selected_foods.items():

    fdc_id = food_data[
        "fdc_id"
    ]


    rows = food_nutrient[
        food_nutrient["fdc_id"]
        == fdc_id
    ].copy()


    record = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            food_data[
                "description"
            ],

        "data_type":
            "Foundation Food",

        "basis_g":
            100
    }


    for output_name, nutrient_info in nutrient_lookup.items():

        nutrient_id = nutrient_info[
            "id"
        ]


        nutrient_row = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]


        if nutrient_row.empty:

            record[
                output_name
            ] = None

        else:

            value = nutrient_row.iloc[0][
                amount_column
            ]


            record[
                output_name
            ] = pd.to_numeric(
                value,
                errors="coerce"
            )


    nutrition_records.append(
        record
    )


# ============================================================
# 19. CREATE FINAL DATABASE
# ============================================================

food_database = pd.DataFrame(
    nutrition_records
)


if food_database.empty:

    raise ValueError(
        "No foods were successfully selected."
    )


# ============================================================
# 20. CLEAN NUMERIC COLUMNS
# ============================================================

numeric_columns = [

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


for column in numeric_columns:

    if column in food_database.columns:

        food_database[column] = pd.to_numeric(
            food_database[column],
            errors="coerce"
        )


# ============================================================
# 21. ROUND NUTRIENTS
# ============================================================

for column in numeric_columns:

    if column in food_database.columns:

        food_database[column] = (
            food_database[column]
            .round(4)
        )


# ============================================================
# 22. ORDER COLUMNS
# ============================================================

preferred_columns = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


existing_columns = [

    column

    for column in preferred_columns

    if column in food_database.columns
]


food_database = food_database[
    existing_columns
]


# ============================================================
# 23. CREATE REVIEW FILE
# ============================================================

if duplicate_review_records:

    review_df = pd.DataFrame(
        duplicate_review_records
    )


    review_df.to_csv(
        OUTPUT_REVIEW,
        index=False
    )


    print("\n")
    print(
        f"Review file saved to:\n"
        f"{OUTPUT_REVIEW.resolve()}"
    )

else:

    # --------------------------------------------------------
    # Remove an old review file if no review is now required.
    # --------------------------------------------------------

    if OUTPUT_REVIEW.exists():

        try:

            OUTPUT_REVIEW.unlink()

        except Exception:

            pass


    print(
        "\nNo food selections require review."
    )


# ============================================================
# 24. VALIDATE DATABASE
# ============================================================

def validate_food_database(
    database
):
    """
    Validate the final Foundation Food database.
    """

    print("\n")
    print("=" * 70)
    print("DATABASE VALIDATION")
    print("=" * 70)


    errors = []

    warnings = []


    # --------------------------------------------------------
    # Empty database
    # --------------------------------------------------------

    if database.empty:

        errors.append(
            "Database contains zero foods."
        )


    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required_columns = [

        "food_name",

        "fdc_id",

        "description",

        "data_type",

        "basis_g"
    ]


    for column in required_columns:

        if column not in database.columns:

            errors.append(
                f"Missing required column: "
                f"{column}"
            )


    # --------------------------------------------------------
    # Duplicate FDC IDs
    # --------------------------------------------------------

    if "fdc_id" in database.columns:

        duplicate_ids = database[
            database["fdc_id"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_ids.empty:

            errors.append(
                "Duplicate FDC IDs found."
            )


    # --------------------------------------------------------
    # Duplicate food names
    # --------------------------------------------------------

    if "food_name" in database.columns:

        duplicate_names = database[
            database["food_name"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_names.empty:

            errors.append(
                "Duplicate food names found."
            )


    # --------------------------------------------------------
    # Basis
    # --------------------------------------------------------

    if "basis_g" in database.columns:

        invalid_basis = database[
            database["basis_g"] <= 0
        ]


        if not invalid_basis.empty:

            errors.append(
                "Invalid basis_g value found."
            )


    # --------------------------------------------------------
    # Negative nutrients
    # --------------------------------------------------------

    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    for column in nutrient_columns:

        if column not in database.columns:

            continue


        negative = database[
            database[column].notna()
            &
            (database[column] < 0)
        ]


        if not negative.empty:

            errors.append(
                f"Negative values found in "
                f"{column}."
            )


    # --------------------------------------------------------
    # Missing nutrients
    # --------------------------------------------------------

    for column in nutrient_columns:

        if column not in database.columns:

            continue


        missing = database[
            column
        ].isna().sum()


        if missing > 0:

            warnings.append(
                f"{missing} foods have missing "
                f"{column}."
            )


    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print(
        f"Foods:                  "
        f"{len(database)}"
    )


    if "fdc_id" in database.columns:

        print(
            "Unique FDC IDs:         "
            f"{database['fdc_id'].nunique()}"
        )


    if errors:

        print("\nERRORS:")


        for error in errors:

            print(
                "  ERROR:",
                error
            )

    else:

        print(
            "\nNo critical validation errors."
        )


    if warnings:

        print("\nWARNINGS:")


        for warning in warnings:

            print(
                "  WARNING:",
                warning
            )


    if errors:

        print(
            "\nSTATUS: FAIL"
        )

        return False


    print(
        "\nSTATUS: PASS"
    )

    return True


# Run validation.

database_valid = validate_food_database(
    food_database
)


# ============================================================
# 25. DISPLAY FINAL DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# 26. SAVE DATABASE
# ============================================================

food_database.to_csv(
    OUTPUT_DATABASE,
    index=False
)


print("\n")
print("=" * 70)
print("DATABASE SAVED")
print("=" * 70)


print(
    f"File:\n"
    f"{OUTPUT_DATABASE.resolve()}"
)


# ============================================================
# 27. NUTRITION CALCULATOR
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):
    """
    Calculate nutrition for a quantity of food.

    Database values are normalized to 100 g.
    """

    grams = float(
        grams
    )


    if grams <= 0:

        raise ValueError(
            "Grams must be greater than zero."
        )


    matches = database[
        database["food_name"]
        .astype(str)
        .str.lower()
        ==
        str(food_name).lower()
    ]


    if matches.empty:

        raise ValueError(
            f"Food not found: "
            f"{food_name}"
        )


    if len(matches) > 1:

        raise ValueError(
            f"Multiple foods found for: "
            f"{food_name}"
        )


    food = matches.iloc[0]


    basis = float(
        food["basis_g"]
    )


    multiplier = (
        grams / basis
    )


    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    result = {

        "food_name":
            food["food_name"],

        "grams":
            grams
    }


    for column in nutrient_columns:

        if column not in database.columns:

            continue


        value = food[column]


        if pd.isna(value):

            result[column] = None

        else:

            result[column] = (
                float(value)
                * multiplier
            )


    return result


# ============================================================
# 28. PRETTY PRINT NUTRITION
# ============================================================

def print_nutrition(
    result
):

    print("\n")
    print("=" * 60)


    print(
        f"{result['food_name']} "
        f"— {result['grams']:.1f} g"
    )


    print("=" * 60)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for key, label in labels.items():

        value = result.get(
            key
        )


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} "
                f"{units[key]}"
            )


# ============================================================
# 29. MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):
    """
    Calculate total nutrition for multiple foods.

    Example:

    meal_items = [
        {
            "food_name": "chicken_breast",
            "grams": 150
        },
        {
            "food_name": "banana",
            "grams": 120
        }
    ]
    """

    totals = {

        "calories_kcal": 0.0,

        "protein_g": 0.0,

        "fat_g": 0.0,

        "carbohydrate_g": 0.0,

        "fiber_g": 0.0,

        "sugar_g": 0.0,

        "saturated_fat_g": 0.0
    }


    foods = []


    for item in meal_items:

        result = calculate_nutrition(

            database,

            item["food_name"],

            item["grams"]
        )


        foods.append(
            result
        )


        for nutrient_name in totals:

            value = result.get(
                nutrient_name
            )


            if value is not None:

                totals[
                    nutrient_name
                ] += value


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# 30. PRETTY PRINT MEAL
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal["foods"]:

        print(
            f"{food['food_name']:30} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name, value in meal[
        "totals"
    ].items():

        print(
            f"{labels[nutrient_name]:20}: "
            f"{value:.2f} "
            f"{units[nutrient_name]}"
        )


# ============================================================
# 31. EXAMPLE NUTRITION TEST
# ============================================================

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


test_food = None


for preferred_food in [

    "chicken_breast",

    "banana",

    "apple_fuji",

    "oats_rolled"
]:

    if preferred_food in food_database[
        "food_name"
    ].values:

        test_food = preferred_food

        break


if test_food is not None:

    test_result = calculate_nutrition(

        food_database,

        test_food,

        150
    )


    print_nutrition(
        test_result
    )

else:

    print(
        "No test food was successfully selected."
    )


# ============================================================
# 32. EXAMPLE MEAL TEST
# ============================================================

available_foods = set(
    food_database[
        "food_name"
    ].values
)


example_meal = []


possible_meal = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },


    {
        "food_name":
            "banana",

        "grams":
            120
    },


    {
        "food_name":
            "oats_rolled",

        "grams":
            80
    }
]


for item in possible_meal:

    if item["food_name"] in available_foods:

        example_meal.append(
            item
        )


if example_meal:

    meal_result = calculate_meal(

        food_database,

        example_meal
    )


    print_meal(
        meal_result
    )


# ============================================================
# 33. FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)


print(
    f"Target foods requested:     "
    f"{len(TARGET_FOODS)}"
)


print(
    f"Foods selected:             "
    f"{len(food_database)}"
)


print(
    f"Foods requiring review:     "
    f"{len(duplicate_review_records)}"
)


print(
    f"Database validation:        "
    f"{'PASS' if database_valid else 'FAIL'}"
)


print(
    f"\nDatabase:"
)


print(
    OUTPUT_DATABASE.resolve()
)


if duplicate_review_records:

    print(
        f"\nReview:"
    )


    print(
        OUTPUT_REVIEW.resolve()
    )


print("\nDone.")


USDA FOODDATA CENTRAL - CONFIGURATION

Data folder:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30

food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv

food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv

nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv

Output database:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_database.csv

Review file:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_selection_review.csv

All USDA input files found.


LOADING USDA FOODDATA C

In [17]:
# ============================================================
# USDA FOODDATA CENTRAL - FOUNDATION FOOD DATABASE
# COMPLETE END-TO-END PIPELINE
# ============================================================
#
# INPUT FILES:
#   food.csv
#   food_nutrient.csv
#   nutrient.csv
#
# OUTPUT FILES:
#   food_database.csv
#   food_selection_review.csv
#   food_source_mapping.csv
#
# ============================================================
# IMPORTANT
# ============================================================
#
# This version uses EXACT USDA food descriptions rather than
# broad keyword searches.
#
# It also handles duplicate USDA records correctly.
#
# Example:
#
#   Apples, fuji, with skin, raw
#
# may have:
#
#   FDC 1105897
#   FDC 1750340
#
# Both records are preserved in food_source_mapping.csv.
#
# The application uses one preferred/canonical FDC record
# for food_database.csv.
#
# ============================================================

import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# YOUR USDA DATA FOLDER
# ------------------------------------------------------------
#
# IMPORTANT:
# Change ONLY this path if your files are somewhere else.
#
# Your successful run showed that this is the folder containing:
#
#   food.csv
#   food_nutrient.csv
#   nutrient.csv
#
# ------------------------------------------------------------

DATA_FOLDER = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)


# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

FOOD_FILE = DATA_FOLDER / "food.csv"

FOOD_NUTRIENT_FILE = (
    DATA_FOLDER / "food_nutrient.csv"
)

NUTRIENT_FILE = (
    DATA_FOLDER / "nutrient.csv"
)


# ------------------------------------------------------------
# OUTPUT FILES
# ------------------------------------------------------------

OUTPUT_DATABASE = (
    DATA_FOLDER / "food_database.csv"
)

OUTPUT_REVIEW = (
    DATA_FOLDER / "food_selection_review.csv"
)

OUTPUT_SOURCE_MAPPING = (
    DATA_FOLDER / "food_source_mapping.csv"
)


# ============================================================
# 2. TARGET FOODS
# ============================================================
#
# We intentionally use exact USDA descriptions.
#
# These descriptions come from the actual USDA data you
# showed in your previous output.
#
# ============================================================

TARGET_FOODS = {

    # --------------------------------------------------------
    # APPLE
    # --------------------------------------------------------

    "apple_fuji": {

        "exact_description":
            "Apples, fuji, with skin, raw"
    },


    # --------------------------------------------------------
    # BANANA
    # --------------------------------------------------------

    "banana": {

        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },


    # --------------------------------------------------------
    # CHICKEN BREAST
    # --------------------------------------------------------

    "chicken_breast": {

        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },


    # --------------------------------------------------------
    # GROUND BEEF 80/20
    # --------------------------------------------------------

    "beef_ground": {

        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },


    # --------------------------------------------------------
    # OATS
    # --------------------------------------------------------

    "oats_rolled": {

        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },


    # --------------------------------------------------------
    # CARROT
    # --------------------------------------------------------

    "carrot": {

        "exact_description":
            "Carrots, mature, raw"
    },


    # --------------------------------------------------------
    # WHOLE MILK
    # --------------------------------------------------------

    "milk_whole": {

        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },


    # --------------------------------------------------------
    # RYE FLOUR
    # --------------------------------------------------------

    "rye_flour": {

        "exact_description":
            "Flour, rye"
    },


    # --------------------------------------------------------
    # WHITE BUTTON MUSHROOM
    # --------------------------------------------------------

    "mushroom_white_button": {

        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# 3. START
# ============================================================

print("\n")
print("=" * 70)
print("USDA FOODDATA CENTRAL FOUNDATION FOOD PIPELINE")
print("=" * 70)

print(
    "\nData folder:"
)

print(
    DATA_FOLDER
)


# ============================================================
# 4. VERIFY INPUT FILES
# ============================================================

print("\n")
print("=" * 70)
print("CHECKING INPUT FILES")
print("=" * 70)


input_files = {

    "food.csv":
        FOOD_FILE,

    "food_nutrient.csv":
        FOOD_NUTRIENT_FILE,

    "nutrient.csv":
        NUTRIENT_FILE
}


missing_files = []


for name, path in input_files.items():

    if path.exists():

        print(
            f"FOUND:     {path}"
        )

    else:

        print(
            f"MISSING:   {path}"
        )

        missing_files.append(
            str(path)
        )


if missing_files:

    raise FileNotFoundError(
        "\nCould not find the following USDA files:\n"
        + "\n".join(
            missing_files
        )
        + "\n\nCheck DATA_FOLDER at the top of the script."
    )


# ============================================================
# 5. LOAD USDA FILES
# ============================================================

print("\n")
print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(
    f"food.csv rows:              {len(food):,}"
)


print(
    f"food_nutrient.csv rows:     "
    f"{len(food_nutrient):,}"
)


print(
    f"nutrient.csv rows:          "
    f"{len(nutrient):,}"
)


# ============================================================
# 6. CHECK REQUIRED COLUMNS
# ============================================================

print("\n")
print("=" * 70)
print("CHECKING REQUIRED COLUMNS")
print("=" * 70)


required_food_columns = [

    "fdc_id",

    "description",

    "data_type"
]


for column in required_food_columns:

    if column not in food.columns:

        raise ValueError(
            f"food.csv is missing required column: "
            f"{column}"
        )


required_food_nutrient_columns = [

    "fdc_id",

    "nutrient_id"
]


for column in required_food_nutrient_columns:

    if column not in food_nutrient.columns:

        raise ValueError(
            "food_nutrient.csv is missing required "
            f"column: {column}"
        )


if "id" not in nutrient.columns:

    raise ValueError(
        "nutrient.csv is missing the 'id' column."
    )


if "name" not in nutrient.columns:

    raise ValueError(
        "nutrient.csv is missing the 'name' column."
    )


print(
    "Required columns: PASS"
)


# ============================================================
# 7. FIND NUTRIENT AMOUNT COLUMN
# ============================================================

amount_column = None


for possible_column in [

    "amount",

    "nutrient_amount"
]:

    if possible_column in food_nutrient.columns:

        amount_column = (
            possible_column
        )

        break


if amount_column is None:

    raise ValueError(
        "Could not find nutrient amount column "
        "in food_nutrient.csv."
    )


print(
    f"Nutrient amount column:     "
    f"{amount_column}"
)


# ============================================================
# 8. NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    """
    Normalize text for reliable USDA description matching.
    """

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9%]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# 9. KEEP FOUNDATION FOODS ONLY
# ============================================================

print("\n")
print("=" * 70)
print("FILTERING FOUNDATION FOODS")
print("=" * 70)


foundation_food = food[
    food["data_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "foundation_food"
].copy()


foundation_food["description"] = (
    foundation_food["description"]
    .astype(str)
    .str.strip()
)


foundation_food["fdc_id"] = pd.to_numeric(
    foundation_food["fdc_id"],
    errors="coerce"
)


foundation_food = foundation_food.dropna(
    subset=["fdc_id"]
)


foundation_food["fdc_id"] = (
    foundation_food["fdc_id"]
    .astype(int)
)


if foundation_food.empty:

    raise ValueError(
        "No Foundation Foods were found."
    )


print(
    f"Foundation Foods found:    "
    f"{len(foundation_food):,}"
)


# ============================================================
# 10. NORMALIZED DESCRIPTION
# ============================================================

foundation_food[
    "normalized_description"
] = (

    foundation_food["description"]

    .apply(
        normalize_text
    )
)


# ============================================================
# 11. EXACT FOOD SELECTOR
# ============================================================

def select_exact_food(
    exact_description
):
    """
    Find Foundation Foods using an exact normalized USDA
    description.

    Possible statuses:

        SELECTED
        DUPLICATE_DESCRIPTION
        NOT_FOUND
    """

    target = normalize_text(
        exact_description
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target
    ].copy()


    # --------------------------------------------------------
    # No match
    # --------------------------------------------------------

    if matches.empty:

        return {

            "status":
                "NOT_FOUND",

            "selected":
                None,

            "candidates":
                matches
        }


    # --------------------------------------------------------
    # One exact match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]

        return {

            "status":
                "SELECTED",

            "selected": {

                "fdc_id":
                    int(row["fdc_id"]),

                "description":
                    row["description"]
            },

            "candidates":
                matches
        }


    # --------------------------------------------------------
    # Multiple FDC records with same description
    # --------------------------------------------------------

    return {

        "status":
            "DUPLICATE_DESCRIPTION",

        "selected":
            None,

        "candidates":
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
    }


# ============================================================
# 12. INITIAL FOOD SELECTION
# ============================================================

selected_foods = {}

duplicate_foods = {}

not_found_foods = {}


print("\n")
print("=" * 70)
print("EXACT FOUNDATION FOOD SELECTION")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    result = select_exact_food(
        config["exact_description"]
    )


    print("\n" + "-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)


    print(
        "Requested:"
    )

    print(
        config["exact_description"]
    )


    print(
        "STATUS:",
        result["status"]
    )


    if result["status"] == "SELECTED":

        selected_foods[food_name] = (
            result["selected"]
        )


        print(
            "FDC ID:",
            result["selected"]["fdc_id"]
        )


        print(
            "Description:",
            result["selected"]["description"]
        )


    elif (
        result["status"]
        == "DUPLICATE_DESCRIPTION"
    ):

        duplicate_foods[food_name] = (
            result["candidates"]
        )


        print(
            "Multiple FDC IDs have the same "
            "description:"
        )


        print(
            result["candidates"]
            .to_string(
                index=False
            )
        )


    else:

        not_found_foods[food_name] = (
            config["exact_description"]
        )


        print(
            "NOT FOUND:"
        )


        print(
            config["exact_description"]
        )


# ============================================================
# 13. BUILD FOUNDATION NUTRIENT SET
# ============================================================
#
# This is important.
#
# We do NOT simply look at nutrient.csv and assume that a
# nutrient definition is actually present in Foundation Foods.
#
# Instead we first determine which nutrient IDs are actually
# used by Foundation Food records.
#
# ============================================================

foundation_ids = set(

    foundation_food[
        "fdc_id"
    ]
    .astype(int)
)


foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(
            foundation_ids
        )
    ]
    .copy()
)


foundation_nutrient_ids = set(

    foundation_food_nutrients[
        "nutrient_id"
    ]
    .dropna()
    .astype(int)
)


nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce"
)


nutrient_available = nutrient[
    nutrient["id"]
    .isin(
        foundation_nutrient_ids
    )
].copy()


print("\n")
print("=" * 70)
print("NUTRIENTS ACTUALLY AVAILABLE IN FOUNDATION FOODS")
print("=" * 70)


print(
    f"Foundation nutrient IDs: "
    f"{len(foundation_nutrient_ids):,}"
)


# ============================================================
# 14. DISPLAY ENERGY CANDIDATES
# ============================================================

energy_candidates = nutrient_available[
    nutrient_available["name"]
    .astype(str)
    .str.contains(
        "energy",
        case=False,
        na=False
    )
].copy()


if energy_candidates.empty:

    print(
        "\nWARNING: No energy nutrient definitions "
        "were found in Foundation Foods."
    )

else:

    print(
        "\nEnergy nutrient candidates:"
    )


    energy_display_columns = [

        "id",

        "name"
    ]


    if "unit_name" in energy_candidates.columns:

        energy_display_columns.append(
            "unit_name"
        )


    print(
        energy_candidates[
            energy_display_columns
        ].to_string(
            index=False
        )
    )


# ============================================================
# 15. FIND NUTRIENT CANDIDATES
# ============================================================

def get_nutrient_candidates(
    search_terms,
    preferred_unit=None,
    selected_fdc_ids=None
):
    """
    Find nutrient definitions that:

    1. Actually exist in Foundation Foods.
    2. Match the requested nutrient name.
    3. Prefer the requested unit.
    4. Prefer nutrients present in the selected foods.
    """

    temp = nutrient_available.copy()


    temp["_name_normalized"] = (
        temp["name"]
        .astype(str)
        .apply(
            normalize_text
        )
    )


    # --------------------------------------------------------
    # Count coverage in selected food records.
    # --------------------------------------------------------

    coverage = {}


    if selected_fdc_ids:

        selected_rows = (
            food_nutrient[
                food_nutrient["fdc_id"]
                .isin(
                    selected_fdc_ids
                )
            ]
        )


        for nutrient_id in (
            selected_rows[
                "nutrient_id"
            ]
            .dropna()
            .astype(int)
            .unique()
        ):

            count = (
                selected_rows[
                    selected_rows[
                        "nutrient_id"
                    ]
                    == nutrient_id
                ]["fdc_id"]
                .nunique()
            )


            coverage[
                int(nutrient_id)
            ] = count


    scores = []


    for _, row in temp.iterrows():

        name = row[
            "_name_normalized"
        ]


        score = 0


        # ----------------------------------------------------
        # Name matching
        # ----------------------------------------------------

        for term in search_terms:

            normalized_term = (
                normalize_text(term)
            )


            if normalized_term in name:

                score += 10


        # ----------------------------------------------------
        # Unit preference
        # ----------------------------------------------------

        if preferred_unit is not None:

            unit = str(
                row.get(
                    "unit_name",
                    ""
                )
            ).lower()


            if (
                preferred_unit.lower()
                in unit
            ):

                score += 25


        # ----------------------------------------------------
        # Selected food coverage
        # ----------------------------------------------------

        nutrient_id = int(
            row["id"]
        )


        nutrient_coverage = coverage.get(
            nutrient_id,
            0
        )


        score += (
            nutrient_coverage
            * 5
        )


        scores.append(
            score
        )


    temp["_score"] = scores


    temp["_coverage"] = (
        temp["id"]
        .astype(int)
        .map(
            coverage
        )
        .fillna(0)
        .astype(int)
    )


    temp = temp[
        temp["_score"] > 0
    ].sort_values(
        [
            "_coverage",
            "_score"
        ],
        ascending=[
            False,
            False
        ]
    )


    return temp


# ============================================================
# 16. IMPROVED NUTRIENT FINDER
# ============================================================

def find_nutrient_in_foundation(
    search_terms,
    preferred_unit=None,
    selected_fdc_ids=None
):
    """
    Find the best nutrient definition that actually occurs
    in Foundation Food nutrient records.
    """

    candidates = get_nutrient_candidates(

        search_terms,

        preferred_unit,

        selected_fdc_ids
    )


    if candidates.empty:

        return None


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row.get(
                "unit_name",
                None
            ),

        "coverage":
            int(
                row["_coverage"]
            )
    }


# ============================================================
# 17. NUTRIENTS REQUIRED BY APPLICATION
# ============================================================

NUTRIENT_REQUESTS = {

    "calories_kcal": {

        "terms": [
            "energy"
        ],

        "unit":
            "kcal"
    },


    "protein_g": {

        "terms": [
            "protein"
        ],

        "unit":
            "g"
    },


    "fat_g": {

        "terms": [
            "total lipid"
        ],

        "unit":
            "g"
    },


    "carbohydrate_g": {

        "terms": [
            "carbohydrate",
            "by difference"
        ],

        "unit":
            "g"
    },


    "fiber_g": {

        "terms": [
            "fiber",
            "total dietary"
        ],

        "unit":
            "g"
    },


    "sugar_g": {

        "terms": [
            "sugars",
            "total"
        ],

        "unit":
            "g"
    },


    "saturated_fat_g": {

        "terms": [
            "fatty acids",
            "total saturated"
        ],

        "unit":
            "g"
    }
}


# ============================================================
# 18. NUTRIENT LOOKUP
# ============================================================

selected_fdc_ids = [

    int(data["fdc_id"])

    for data in selected_foods.values()
]


# Add duplicate FDC IDs too because we may inspect them.

for candidates in duplicate_foods.values():

    selected_fdc_ids.extend(

        candidates[
            "fdc_id"
        ]
        .astype(int)
        .tolist()
    )


selected_fdc_ids = list(
    set(
        selected_fdc_ids
    )
)


nutrient_lookup = {}


print("\n")
print("=" * 70)
print("SELECTING NUTRIENTS")
print("=" * 70)


for output_name, config in (
    NUTRIENT_REQUESTS.items()
):

    result = find_nutrient_in_foundation(

        config["terms"],

        config["unit"],

        selected_fdc_ids
    )


    if result is None:

        print(
            f"WARNING: {output_name} "
            "not available in Foundation Foods."
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


        print(

            f"{output_name:25} "

            f"ID={result['id']:5} "

            f"UNIT={str(result['unit']):5} "

            f"COVERAGE="
            f"{result['coverage']:2} "

            f"NAME={result['name']}"
        )


# ============================================================
# 19. REQUIRE IMPORTANT NUTRIENTS
# ============================================================
#
# Calories, protein, fat and carbohydrate are considered
# core application nutrients.
#
# Fiber, sugar and saturated fat may legitimately be missing
# from some USDA records.
#
# ============================================================

required_application_nutrients = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g"
]


missing_required_nutrients = [

    name

    for name in required_application_nutrients

    if name not in nutrient_lookup
]


if missing_required_nutrients:

    raise ValueError(

        "The following required nutrients could not be "
        "identified in Foundation Foods:\n"

        + "\n".join(
            missing_required_nutrients
        )

        + "\n\n"
        "The pipeline has been stopped because the "
        "nutrition database would be incomplete."
    )


# ============================================================
# 20. COMPARE DUPLICATE FDC RECORDS
# ============================================================

def compare_food_nutrients(

    fdc_ids,

    food_nutrient,

    nutrient_lookup,

    amount_column
):
    """
    Compare requested nutrient values for duplicate USDA
    records.
    """

    rows = food_nutrient[
        food_nutrient[
            "fdc_id"
        ].isin(
            fdc_ids
        )
    ].copy()


    comparison = []


    for fdc_id in fdc_ids:

        record = {

            "fdc_id":
                int(fdc_id)
        }


        food_rows = rows[
            rows["fdc_id"]
            == fdc_id
        ]


        for nutrient_name, nutrient_info in (
            nutrient_lookup.items()
        ):

            nutrient_id = (
                nutrient_info["id"]
            )


            nutrient_row = food_rows[
                food_rows[
                    "nutrient_id"
                ]
                == nutrient_id
            ]


            if nutrient_row.empty:

                record[
                    nutrient_name
                ] = None

            else:

                value = nutrient_row.iloc[0][
                    amount_column
                ]


                record[
                    nutrient_name
                ] = pd.to_numeric(
                    value,
                    errors="coerce"
                )


        comparison.append(
            record
        )


    return pd.DataFrame(
        comparison
    )


# ============================================================
# 21. DUPLICATE FOOD ANALYSIS
# ============================================================

print("\n")
print("=" * 70)
print("CHECKING DUPLICATE FDC RECORDS")
print("=" * 70)


duplicate_comparisons = {}


for food_name, candidates in (
    duplicate_foods.items()
):

    fdc_ids = (

        candidates[
            "fdc_id"
        ]
        .astype(int)
        .tolist()
    )


    print("\n" + "-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)


    comparison = compare_food_nutrients(

        fdc_ids,

        food_nutrient,

        nutrient_lookup,

        amount_column
    )


    duplicate_comparisons[
        food_name
    ] = comparison


    print(
        comparison.to_string(
            index=False
        )
    )


# ============================================================
# 22. SELECT CANONICAL RECORD FOR DUPLICATES
# ============================================================
#
# We do NOT say duplicate records are identical.
#
# Instead:
#
#   - preserve every USDA FDC ID
#   - select one preferred record for the application
#
# Preference:
#
#   1. Most complete core nutrition data
#   2. Most complete overall requested nutrition data
#   3. Lowest FDC ID as deterministic tie-breaker
#
# ============================================================

for food_name, candidates in (
    duplicate_foods.items()
):

    comparison = duplicate_comparisons[
        food_name
    ].copy()


    nutrient_columns = [

        column

        for column in comparison.columns

        if column != "fdc_id"
    ]


    # --------------------------------------------------------
    # Calculate completeness.
    # --------------------------------------------------------

    comparison[
        "_core_complete"
    ] = comparison[
        [
            "calories_kcal",
            "protein_g",
            "fat_g",
            "carbohydrate_g"
        ]
    ].notna().sum(
        axis=1
    )


    comparison[
        "_total_complete"
    ] = comparison[
        nutrient_columns
    ].notna().sum(
        axis=1
    )


    comparison = comparison.sort_values(

        [

            "_core_complete",

            "_total_complete",

            "fdc_id"
        ],

        ascending=[

            False,

            False,

            True
        ]
    ).reset_index(
        drop=True
    )


    preferred_fdc_id = int(
        comparison.iloc[0]["fdc_id"]
    )


    # Find description from original candidate list.

    description_row = candidates[
        candidates["fdc_id"]
        .astype(int)
        == preferred_fdc_id
    ].iloc[0]


    selected_foods[
        food_name
    ] = {

        "fdc_id":
            preferred_fdc_id,

        "description":
            description_row[
                "description"
            ]
    }


    print("\n")

    print(
        f"{food_name}: selected preferred "
        f"FDC ID {preferred_fdc_id}"
    )


    print(
        "Reason: highest available nutrition "
        "completeness."
    )


# ============================================================
# 23. BUILD SOURCE MAPPING
# ============================================================
#
# This preserves every USDA record associated with the
# application food.
#
# Example:
#
# apple_fuji
#    1105897
#    1750340
#
# One is marked preferred.
#
# ============================================================

source_mapping_records = []


for food_name, config in (
    TARGET_FOODS.items()
):

    requested_description = (
        config["exact_description"]
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == normalize_text(
            requested_description
        )
    ].copy()


    preferred_id = None


    if food_name in selected_foods:

        preferred_id = int(
            selected_foods[
                food_name
            ]["fdc_id"]
        )


    for _, row in matches.iterrows():

        fdc_id = int(
            row["fdc_id"]
        )


        source_mapping_records.append({

            "app_food_id":
                food_name,

            "fdc_id":
                fdc_id,

            "description":
                row["description"],

            "source":
                "USDA FoodData Central Foundation Food",

            "is_preferred":
                fdc_id == preferred_id
        })


food_source_mapping = pd.DataFrame(
    source_mapping_records
)


# ============================================================
# 24. SAVE SOURCE MAPPING
# ============================================================

food_source_mapping.to_csv(

    OUTPUT_SOURCE_MAPPING,

    index=False
)


print("\n")
print(
    f"Source mapping saved to:"
)

print(
    OUTPUT_SOURCE_MAPPING
)


# ============================================================
# 25. REVIEW FILE
# ============================================================

review_records = []


# ------------------------------------------------------------
# Not found foods
# ------------------------------------------------------------

for food_name, description in (
    not_found_foods.items()
):

    review_records.append({

        "app_food_id":
            food_name,

        "status":
            "NOT_FOUND",

        "fdc_id":
            None,

        "description":
            description,

        "score":
            None,

        "reason":
            "Exact USDA description was not found."
    })


# ------------------------------------------------------------
# Duplicate foods
# ------------------------------------------------------------

for food_name, candidates in (
    duplicate_foods.items()
):

    preferred_id = int(
        selected_foods[
            food_name
        ]["fdc_id"]
    )


    comparison = duplicate_comparisons[
        food_name
    ]


    for _, row in candidates.iterrows():

        fdc_id = int(
            row["fdc_id"]
        )


        preferred = (
            fdc_id
            == preferred_id
        )


        review_records.append({

            "app_food_id":
                food_name,

            "status":
                (
                    "PREFERRED_DUPLICATE"
                    if preferred
                    else
                    "DUPLICATE_SOURCE"
                ),

            "fdc_id":
                fdc_id,

            "description":
                row["description"],

            "score":
                None,

            "reason":
                (
                    "Preferred canonical USDA "
                    "record selected."
                    if preferred
                    else
                    "Duplicate USDA record preserved "
                    "in source mapping."
                )
        })


review_df = pd.DataFrame(
    review_records
)


if not review_df.empty:

    review_df.to_csv(

        OUTPUT_REVIEW,

        index=False
    )


    print("\n")

    print(
        f"Review file saved to:"
    )

    print(
        OUTPUT_REVIEW
    )

else:

    print(
        "\nNo food selection review required."
    )


# ============================================================
# 26. SHOW SELECTED FOODS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL SELECTED FOODS")
print("=" * 70)


for food_name, data in (
    selected_foods.items()
):

    print(

        f"{food_name:25} "

        f"{data['fdc_id']:10} "

        f"{data['description']}"
    )


print("\n")

print(
    f"Foods selected: "
    f"{len(selected_foods)}"
)


if len(selected_foods) != len(
    TARGET_FOODS
):

    print(
        "WARNING: Not every requested food "
        "was selected."
    )


# ============================================================
# 27. EXTRACT NUTRIENTS
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING NUTRIENTS")
print("=" * 70)


nutrition_records = []


for food_name, food_data in (
    selected_foods.items()
):

    fdc_id = int(
        food_data["fdc_id"]
    )


    rows = food_nutrient[
        food_nutrient[
            "fdc_id"
        ]
        == fdc_id
    ].copy()


    record = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            food_data[
                "description"
            ],

        "data_type":
            "Foundation Food",

        "basis_g":
            100
    }


    for output_name, nutrient_info in (
        nutrient_lookup.items()
    ):

        nutrient_id = int(
            nutrient_info["id"]
        )


        nutrient_rows = rows[
            rows[
                "nutrient_id"
            ]
            == nutrient_id
        ]


        if nutrient_rows.empty:

            record[
                output_name
            ] = None

        else:

            value = nutrient_rows.iloc[0][
                amount_column
            ]


            record[
                output_name
            ] = pd.to_numeric(
                value,
                errors="coerce"
            )


    nutrition_records.append(
        record
    )


# ============================================================
# 28. CREATE DATABASE
# ============================================================

food_database = pd.DataFrame(
    nutrition_records
)


# ============================================================
# 29. NUMERIC CLEANUP
# ============================================================

numeric_columns = [

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


for column in numeric_columns:

    if column in food_database.columns:

        food_database[
            column
        ] = pd.to_numeric(

            food_database[
                column
            ],

            errors="coerce"
        )


# ============================================================
# 30. ROUND NUTRIENTS
# ============================================================

for column in numeric_columns:

    if column in food_database.columns:

        food_database[
            column
        ] = (

            food_database[
                column
            ]

            .round(4)
        )


# ============================================================
# 31. ORDER COLUMNS
# ============================================================

preferred_columns = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


existing_columns = [

    column

    for column in preferred_columns

    if column in food_database.columns
]


food_database = food_database[
    existing_columns
]


# ============================================================
# 32. DISPLAY DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# 33. VALIDATION
# ============================================================

def validate_food_database(
    database
):
    """
    Validate the final Foundation Food database.
    """

    print("\n")
    print("=" * 70)
    print("DATABASE VALIDATION")
    print("=" * 70)


    errors = []

    warnings = []


    # --------------------------------------------------------
    # Empty database
    # --------------------------------------------------------

    if database.empty:

        errors.append(
            "Database contains zero foods."
        )


    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required_columns = [

        "food_name",

        "fdc_id",

        "description",

        "data_type",

        "basis_g"
    ]


    for column in required_columns:

        if column not in database.columns:

            errors.append(
                f"Missing required column: {column}"
            )


    # --------------------------------------------------------
    # Duplicate FDC IDs
    # --------------------------------------------------------

    if "fdc_id" in database.columns:

        duplicate_ids = database[
            database["fdc_id"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_ids.empty:

            errors.append(
                "Duplicate FDC IDs found."
            )


    # --------------------------------------------------------
    # Duplicate application food names
    # --------------------------------------------------------

    if "food_name" in database.columns:

        duplicate_names = database[
            database["food_name"]
            .duplicated(
                keep=False
            )
        ]


        if not duplicate_names.empty:

            errors.append(
                "Duplicate application food names found."
            )


    # --------------------------------------------------------
    # Basis
    # --------------------------------------------------------

    if "basis_g" in database.columns:

        invalid_basis = database[
            database["basis_g"] <= 0
        ]


        if not invalid_basis.empty:

            errors.append(
                "Invalid basis_g value found."
            )


        non_100_basis = database[
            database["basis_g"] != 100
        ]


        if not non_100_basis.empty:

            warnings.append(
                "Some foods do not have a 100 g basis."
            )


    # --------------------------------------------------------
    # Negative nutrients
    # --------------------------------------------------------

    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    for column in nutrient_columns:

        if column not in database.columns:

            continue


        negative = database[
            database[column].notna()
            &
            (
                database[column]
                < 0
            )
        ]


        if not negative.empty:

            errors.append(
                f"Negative values found in {column}."
            )


    # --------------------------------------------------------
    # Missing nutrients
    # --------------------------------------------------------

    for column in nutrient_columns:

        if column not in database.columns:

            continue


        missing = int(
            database[column]
            .isna()
            .sum()
        )


        if missing > 0:

            warnings.append(

                f"{missing} foods have missing "
                f"{column}."
            )


    # --------------------------------------------------------
    # Required nutrients
    # --------------------------------------------------------

    for column in [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g"
    ]:

        if column not in database.columns:

            errors.append(
                f"Required nutrient column missing: "
                f"{column}"
            )

            continue


        missing = database[
            column
        ].isna()


        if missing.any():

            missing_foods = database.loc[
                missing,
                "food_name"
            ].tolist()


            errors.append(

                f"Required nutrient {column} "
                f"is missing for: "
                f"{', '.join(missing_foods)}"
            )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(
        f"Foods:                  "
        f"{len(database)}"
    )


    if "fdc_id" in database.columns:

        print(

            "Unique FDC IDs:         "

            f"{database['fdc_id'].nunique()}"
        )


    if errors:

        print("\nERRORS:")


        for error in errors:

            print(
                "  ERROR:",
                error
            )

    else:

        print(
            "\nNo critical validation errors."
        )


    if warnings:

        print("\nWARNINGS:")


        for warning in warnings:

            print(
                "  WARNING:",
                warning
            )


    if errors:

        print(
            "\nSTATUS: FAIL"
        )

        return False


    print(
        "\nSTATUS: PASS"
    )

    return True


# Run validation.

database_valid = (
    validate_food_database(
        food_database
    )
)


# ============================================================
# 34. STOP IF DATABASE INVALID
# ============================================================

if not database_valid:

    print("\n")
    print("=" * 70)
    print("PIPELINE STOPPED")
    print("=" * 70)

    print(
        "The database was not saved as a valid "
        "application database."
    )

    raise ValueError(
        "Food database validation failed."
    )


# ============================================================
# 35. SAVE DATABASE
# ============================================================

food_database.to_csv(

    OUTPUT_DATABASE,

    index=False
)


print("\n")
print("=" * 70)
print("DATABASE SAVED")
print("=" * 70)


print(
    f"File:"
)

print(
    OUTPUT_DATABASE
)


# ============================================================
# 36. NUTRITION CALCULATOR
# ============================================================

def calculate_nutrition(

    database,

    food_name,

    grams
):
    """
    Calculate nutrition for a quantity of food.

    USDA database values are represented per 100 g.
    """

    # --------------------------------------------------------
    # Validate grams
    # --------------------------------------------------------

    try:

        grams = float(
            grams
        )

    except Exception:

        raise ValueError(
            "Grams must be numeric."
        )


    if grams <= 0:

        raise ValueError(
            "Grams must be greater than zero."
        )


    # --------------------------------------------------------
    # Find food
    # --------------------------------------------------------

    matches = database[

        database[
            "food_name"
        ]
        .astype(str)
        .str.lower()

        ==

        str(food_name)
        .lower()
    ]


    if matches.empty:

        raise ValueError(
            f"Food not found: {food_name}"
        )


    if len(matches) > 1:

        raise ValueError(
            f"Multiple foods found for: "
            f"{food_name}"
        )


    food = matches.iloc[0]


    # --------------------------------------------------------
    # Basis
    # --------------------------------------------------------

    basis = float(
        food["basis_g"]
    )


    multiplier = (
        grams / basis
    )


    # --------------------------------------------------------
    # Nutrients
    # --------------------------------------------------------

    nutrient_columns = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    result = {

        "food_name":
            food["food_name"],

        "grams":
            grams
    }


    for column in nutrient_columns:

        if column not in database.columns:

            continue


        value = food[column]


        if pd.isna(value):

            result[column] = None

        else:

            result[column] = (

                float(value)
                * multiplier
            )


    return result


# ============================================================
# 37. PRETTY PRINT NUTRITION
# ============================================================

def print_nutrition(
    result
):

    print("\n")
    print("=" * 60)


    print(

        f"{result['food_name']} "
        f"— {result['grams']:.1f} g"
    )


    print("=" * 60)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for key, label in labels.items():

        value = result.get(
            key
        )


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(

                f"{label:20}: "

                f"{value:.2f} "

                f"{units[key]}"
            )


# ============================================================
# 38. MEAL CALCULATOR
# ============================================================

def calculate_meal(

    database,

    meal_items
):
    """
    Calculate nutrition for multiple foods.

    IMPORTANT:
    Missing != zero.

    If ANY food in the meal is missing a particular nutrient,
    the meal total for that nutrient is reported as None.

    Example:

        Chicken fiber = missing
        Banana fiber = 2.6 g

    Meal fiber = unknown

    We do NOT report 2.6 g because the chicken contribution
    is unknown.
    """

    nutrient_names = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    totals = {

        nutrient:
            0.0

        for nutrient in nutrient_names
    }


    nutrient_complete = {

        nutrient:
            True

        for nutrient in nutrient_names
    }


    foods = []


    # --------------------------------------------------------
    # Calculate each food
    # --------------------------------------------------------

    for item in meal_items:

        if "food_name" not in item:

            raise ValueError(
                "Meal item is missing food_name."
            )


        if "grams" not in item:

            raise ValueError(
                "Meal item is missing grams."
            )


        result = calculate_nutrition(

            database,

            item["food_name"],

            item["grams"]
        )


        foods.append(
            result
        )


        # ----------------------------------------------------
        # Add nutrients.
        # ----------------------------------------------------

        for nutrient in nutrient_names:

            value = result.get(
                nutrient
            )


            if value is None:

                nutrient_complete[
                    nutrient
                ] = False

            else:

                totals[
                    nutrient
                ] += float(
                    value
                )


    # --------------------------------------------------------
    # Missing nutrient protection.
    #
    # If even ONE food is missing a nutrient, the meal total
    # is unknown.
    # --------------------------------------------------------

    for nutrient in nutrient_names:

        if not nutrient_complete[
            nutrient
        ]:

            totals[
                nutrient
            ] = None


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# 39. PRETTY PRINT MEAL
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal["foods"]:

        print(

            f"{food['food_name']:25} "

            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name, value in (
        meal["totals"].items()
    ):

        label = labels[
            nutrient_name
        ]


        unit = units[
            nutrient_name
        ]


        if value is None:

            print(

                f"{label:20}: -"
            )

        else:

            print(

                f"{label:20}: "

                f"{value:.2f} "

                f"{unit}"
            )


# ============================================================
# 40. EXAMPLE FOOD TEST
# ============================================================

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


test_food = None


for preferred_food in [

    "chicken_breast",

    "banana",

    "apple_fuji",

    "oats_rolled",

    "beef_ground"
]:

    if preferred_food in set(
        food_database[
            "food_name"
        ]
    ):

        test_food = preferred_food

        break


if test_food is not None:

    test_result = calculate_nutrition(

        food_database,

        test_food,

        150
    )


    print_nutrition(
        test_result
    )

else:

    print(
        "No test food was successfully selected."
    )


# ============================================================
# 41. EXAMPLE MEAL TEST
# ============================================================

available_foods = set(

    food_database[
        "food_name"
    ]
)


possible_meal = [

    {

        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {

        "food_name":
            "banana",

        "grams":
            120
    },

    {

        "food_name":
            "oats_rolled",

        "grams":
            80
    }
]


example_meal = [

    item

    for item in possible_meal

    if item["food_name"]
    in available_foods
]


if example_meal:

    meal_result = calculate_meal(

        food_database,

        example_meal
    )


    print_meal(
        meal_result
    )

else:

    print(
        "No example meal foods were available."
    )


# ============================================================
# 42. FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print(

    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)


print(

    f"Foods requested:            "
    f"{len(TARGET_FOODS)}"
)


print(

    f"Foods selected:             "
    f"{len(food_database)}"
)


print(

    f"Duplicate food descriptions:"
    f" {len(duplicate_foods)}"
)


print(

    f"Foods not found:             "
    f"{len(not_found_foods)}"
)


print(

    f"Database validation:         "
    f"{'PASS' if database_valid else 'FAIL'}"
)


print("\n")
print(
    "OUTPUT FILES"
)
print("-" * 70)


print(
    f"Database:"
)

print(
    OUTPUT_DATABASE
)


print(
    f"\nSource mapping:"
)

print(
    OUTPUT_SOURCE_MAPPING
)


if not review_df.empty:

    print(
        f"\nReview:"
    )

    print(
        OUTPUT_REVIEW
    )


print("\n")
print("=" * 70)
print("DONE")
print("=" * 70)




USDA FOODDATA CENTRAL FOUNDATION FOOD PIPELINE

Data folder:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


CHECKING INPUT FILES
FOUND:     C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
FOUND:     C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
FOUND:     C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


LOADING USDA FOODDATA CENTRAL FILES
food.csv rows:              87,990
food_nutrient.csv rows:     170,469
nutrient.csv rows:          477


CHECKING REQUIRED COLUMNS
Required columns: PASS
Nutrient amount column:     amount


FILTERING FOUNDATION FOODS
Foundation Foods found:    469


EXACT FOUNDATION FOOD SELECTION

-------------------------------

In [21]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD PIPELINE
# ============================================================
#
# PURPOSE
# -------
# Build a clean application nutrition database from the USDA
# FoodData Central Foundation Foods CSV files.
#
# INPUTS
# ------
# food.csv
# food_nutrient.csv
# nutrient.csv
#
# OUTPUTS
# -------
# food_database.csv
# food_source_mapping.csv
# food_selection_review.csv
#
# IMPORTANT DESIGN RULES
# ----------------------
# 1. Foundation Foods are detected robustly.
# 2. Foods are selected using exact USDA descriptions.
# 3. Duplicate FDC records are NOT blindly merged.
# 4. Nutrients are selected using exact USDA nutrient names.
# 5. Nutrient IDs are never selected using fuzzy substring matching.
# 6. Missing nutrient values remain NaN.
# 7. Calories come from the USDA energy nutrient.
# 8. All nutrition values are stored per 100 g.
# ============================================================


from pathlib import Path
import re
import math
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

SEARCH_ROOT = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)


OUTPUT_DIRECTORY = None
# None = save next to the USDA CSV files


# ============================================================
# TARGET FOODS
# ============================================================
#
# These descriptions are based on the USDA descriptions from
# the dataset you showed.
#
# Do NOT add "raw" unless USDA actually has it in the
# description.
# ============================================================

TARGET_FOODS = {

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# GENERAL TEXT NORMALIZATION
# ============================================================

def normalize_text(value):
    """
    Normalize ordinary text for exact comparisons.

    Example:
        'Apples, Fuji, With Skin, Raw'
        ->
        'apples fuji with skin raw'
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ============================================================
# ROBUST DATA TYPE NORMALIZATION
# ============================================================

def normalize_data_type(value):
    """
    Normalize USDA data_type values.

    This deliberately removes spaces, underscores, hyphens,
    punctuation, etc.

    Therefore all of these become:

        foundationfood

    foundation_food
    foundation food
    Foundation Food
    FOUNDATION_FOOD
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        value
    )


# ============================================================
# FIND USDA FILES
# ============================================================

print("\n")
print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)

print("\nSearch directory:")
print(SEARCH_ROOT)


if not SEARCH_ROOT.exists():

    raise FileNotFoundError(
        f"Search directory does not exist:\n"
        f"{SEARCH_ROOT}"
    )


def find_usda_file(filename, search_root):

    matches = list(
        search_root.rglob(filename)
    )

    if not matches:

        return None

    # Prefer files inside directories containing
    # "foundation_food" if multiple copies exist.
    foundation_matches = [
        p for p in matches
        if "foundation_food" in str(p).lower()
    ]

    if foundation_matches:
        return foundation_matches[0]

    return matches[0]


FOOD_FILE = find_usda_file(
    "food.csv",
    SEARCH_ROOT
)

FOOD_NUTRIENT_FILE = find_usda_file(
    "food_nutrient.csv",
    SEARCH_ROOT
)

NUTRIENT_FILE = find_usda_file(
    "nutrient.csv",
    SEARCH_ROOT
)


if FOOD_FILE is None:

    raise FileNotFoundError(
        "Could not find food.csv"
    )


if FOOD_NUTRIENT_FILE is None:

    raise FileNotFoundError(
        "Could not find food_nutrient.csv"
    )


if NUTRIENT_FILE is None:

    raise FileNotFoundError(
        "Could not find nutrient.csv"
    )


print("\n")
print("=" * 70)
print("SELECTED USDA FILES")
print("=" * 70)

print("\nfood.csv:")
print(FOOD_FILE)

print("\nfood_nutrient.csv:")
print(FOOD_NUTRIENT_FILE)

print("\nnutrient.csv:")
print(NUTRIENT_FILE)


# ============================================================
# LOAD CSV FILES
# ============================================================

print("\n")
print("=" * 70)
print("READING USDA CSV FILES")
print("=" * 70)


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(
    f"\nfood.csv rows:          {len(food):,}"
)

print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# REQUIRED COLUMN VALIDATION
# ============================================================

required_food_columns = {
    "fdc_id",
    "description",
    "data_type"
}


required_food_nutrient_columns = {
    "fdc_id",
    "nutrient_id",
    "amount"
}


required_nutrient_columns = {
    "id",
    "name",
    "unit_name"
}


missing = (
    required_food_columns
    - set(food.columns)
)

if missing:

    raise RuntimeError(
        "food.csv is missing columns: "
        + ", ".join(sorted(missing))
    )


missing = (
    required_food_nutrient_columns
    - set(food_nutrient.columns)
)

if missing:

    raise RuntimeError(
        "food_nutrient.csv is missing columns: "
        + ", ".join(sorted(missing))
    )


missing = (
    required_nutrient_columns
    - set(nutrient.columns)
)

if missing:

    raise RuntimeError(
        "nutrient.csv is missing columns: "
        + ", ".join(sorted(missing))
    )


# ============================================================
# NORMALIZE KEY COLUMNS
# ============================================================

food["fdc_id"] = pd.to_numeric(
    food["fdc_id"],
    errors="coerce"
)


food_nutrient["fdc_id"] = pd.to_numeric(
    food_nutrient["fdc_id"],
    errors="coerce"
)


food_nutrient["nutrient_id"] = pd.to_numeric(
    food_nutrient["nutrient_id"],
    errors="coerce"
)


nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce"
)


# Remove invalid IDs.

food = food[
    food["fdc_id"].notna()
].copy()


food_nutrient = food_nutrient[
    food_nutrient["fdc_id"].notna()
    &
    food_nutrient["nutrient_id"].notna()
].copy()


nutrient = nutrient[
    nutrient["id"].notna()
].copy()


food["fdc_id"] = (
    food["fdc_id"]
    .astype(int)
)


food_nutrient["fdc_id"] = (
    food_nutrient["fdc_id"]
    .astype(int)
)


food_nutrient["nutrient_id"] = (
    food_nutrient["nutrient_id"]
    .astype(int)
)


nutrient["id"] = (
    nutrient["id"]
    .astype(int)
)


# ============================================================
# USDA DATA TYPE DIAGNOSTICS
# ============================================================

print("\n")
print("=" * 70)
print("USDA DATA TYPES")
print("=" * 70)

print(
    food["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# ROBUST FOUNDATION FOOD DETECTION
# ============================================================
#
# THIS IS THE IMPORTANT FIX FOR YOUR CURRENT ERROR.
#
# Do NOT compare:
#
#     normalized_data_type == "foundation food"
#
# because your normalization may produce:
#
#     foundationfood
#
# instead.
#
# We normalize both sides through the same function.
# ============================================================

food["normalized_data_type"] = (
    food["data_type"]
    .apply(normalize_data_type)
)


print("\n")
print("=" * 70)
print("NORMALIZED DATA TYPES")
print("=" * 70)

print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


FOUNDATION_DATA_TYPE = normalize_data_type(
    "foundation_food"
)


foundation_food = food[
    food["normalized_data_type"]
    == FOUNDATION_DATA_TYPE
].copy()


if foundation_food.empty:

    print("\n")
    print("=" * 70)
    print("FOUNDATION FOOD DETECTION FAILED")
    print("=" * 70)

    print(
        "\nExpected normalized value:"
    )

    print(
        FOUNDATION_DATA_TYPE
    )

    print(
        "\nActual normalized values:"
    )

    print(
        food["normalized_data_type"]
        .value_counts(dropna=False)
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found. "
        "The data_type normalization is not "
        "matching the USDA Foundation Food records."
    )


# ============================================================
# NORMALIZE FOOD DESCRIPTIONS
# ============================================================

foundation_food[
    "normalized_description"
] = (
    foundation_food[
        "description"
    ]
    .apply(normalize_text)
)


print("\n")
print("=" * 70)
print("FOUNDATION FOOD SUMMARY")
print("=" * 70)

print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)


print(
    f"Unique Foundation FDC IDs: "
    f"{foundation_food['fdc_id'].nunique():,}"
)


# ============================================================
# SHOW TARGET SEARCH RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FOUNDATION FOOD TARGET SEARCH")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    target_description = (
        config["exact_description"]
    )

    target_normalized = normalize_text(
        target_description
    )

    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target_normalized
    ].copy()


    print("\n")
    print("-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)

    print(
        "Requested:"
    )

    print(
        target_description
    )

    print(
        f"Matches: {len(matches)}"
    )


    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
            .to_string(index=False)
        )


# ============================================================
# EXACT FOOD SELECTOR
# ============================================================

def select_exact_food(
    exact_description
):
    """
    Select Foundation Food records matching an exact
    normalized USDA description.

    Returns:

        SELECTED
        DUPLICATE_DESCRIPTION
        NOT_FOUND
    """

    target = normalize_text(
        exact_description
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target
    ].copy()


    if matches.empty:

        return {
            "status":
                "NOT_FOUND",

            "selected":
                None,

            "candidates":
                matches
        }


    if len(matches) == 1:

        row = matches.iloc[0]

        return {

            "status":
                "SELECTED",

            "selected": {

                "fdc_id":
                    int(row["fdc_id"]),

                "description":
                    row["description"]
            },

            "candidates":
                matches
        }


    return {

        "status":
            "DUPLICATE_DESCRIPTION",

        "selected":
            None,

        "candidates":
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
    }


# ============================================================
# INITIAL FOOD SELECTION
# ============================================================

selected_foods = {}

duplicate_foods = {}

not_found_foods = {}


print("\n")
print("=" * 70)
print("SELECTING TARGET FOUNDATION FOODS")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    result = select_exact_food(
        config["exact_description"]
    )


    print("\n")
    print("-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)

    print(
        "STATUS:",
        result["status"]
    )


    if result["status"] == "SELECTED":

        selected_foods[
            food_name
        ] = result["selected"]


        print(
            "FDC ID:",
            result["selected"]["fdc_id"]
        )

        print(
            "Description:",
            result["selected"]["description"]
        )


    elif result["status"] == "DUPLICATE_DESCRIPTION":

        duplicate_foods[
            food_name
        ] = result["candidates"]


        print(
            "Duplicate USDA descriptions:"
        )

        print(
            result["candidates"]
            .to_string(index=False)
        )


    else:

        not_found_foods[
            food_name
        ] = config[
            "exact_description"
        ]


        print(
            "NOT FOUND"
        )


# ============================================================
# FOUNDATION NUTRIENT IDs
# ============================================================
#
# Only nutrient IDs actually attached to Foundation Foods
# are considered.
# ============================================================

foundation_ids = set(
    foundation_food[
        "fdc_id"
    ]
    .astype(int)
)


foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_ids)
    ]
    .copy()
)


print("\n")
print("=" * 70)
print("FOUNDATION NUTRIENT COVERAGE")
print("=" * 70)

print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)


foundation_nutrient_ids = set(
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .dropna()
    .astype(int)
)


nutrient_available = nutrient[
    nutrient["id"]
    .isin(foundation_nutrient_ids)
].copy()


print(
    f"Nutrient IDs available in "
    f"Foundation Foods: "
    f"{len(nutrient_available):,}"
)


# ============================================================
# NUTRIENT NORMALIZATION
# ============================================================

nutrient[
    "normalized_name"
] = (
    nutrient["name"]
    .apply(normalize_text)
)


nutrient_available[
    "normalized_name"
] = (
    nutrient_available["name"]
    .apply(normalize_text)
)


# ============================================================
# NUTRIENT COVERAGE FUNCTION
# ============================================================

def nutrient_coverage(
    nutrient_id
):
    """
    Number of Foundation Food records that contain a
    non-null amount for the nutrient.
    """

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "nutrient_id"
        ]
        == int(nutrient_id)
    ]


    if rows.empty:
        return 0


    return int(
        rows["amount"]
        .notna()
        .sum()
    )


# ============================================================
# SHOW ENERGY CANDIDATES
# ============================================================

print("\n")
print("=" * 70)
print("ENERGY NUTRIENTS ACTUALLY AVAILABLE")
print("=" * 70)


ENERGY_NAMES = {
    normalize_text("Energy"),
    normalize_text(
        "Energy (Atwater General Factors)"
    ),
    normalize_text(
        "Energy (Atwater Specific Factors)"
    )
}


energy_candidates = nutrient_available[
    nutrient_available[
        "normalized_name"
    ]
    .isin(ENERGY_NAMES)
].copy()


energy_candidates = energy_candidates[
    energy_candidates[
        "unit_name"
    ]
    .astype(str)
    .str.upper()
    == "KCAL"
].copy()


if energy_candidates.empty:

    raise RuntimeError(
        "No KCAL Energy nutrient exists "
        "in the Foundation Foods dataset."
    )


energy_candidates[
    "coverage"
] = (
    energy_candidates[
        "id"
    ]
    .apply(nutrient_coverage)
)


print(
    energy_candidates[
        [
            "id",
            "name",
            "unit_name",
            "coverage"
        ]
    ]
    .sort_values(
        [
            "coverage",
            "id"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(index=False)
)


# ============================================================
# ENERGY PRIORITY
# ============================================================

ENERGY_PRIORITY = {

    2047: 3,

    2048: 2,

    1008: 1
}


energy_candidates[
    "priority"
] = (
    energy_candidates[
        "id"
    ]
    .map(
        ENERGY_PRIORITY
    )
    .fillna(0)
)


energy_candidates = (
    energy_candidates
    .sort_values(
        [
            "coverage",
            "priority"
        ],
        ascending=[
            False,
            False
        ]
    )
)


energy_row = (
    energy_candidates
    .iloc[0]
)


# ============================================================
# EXACT NUTRIENT FINDER
# ============================================================

def find_exact_nutrient(
    exact_name,
    allowed_units=None
):
    """
    Find a nutrient by EXACT normalized USDA nutrient name.

    No fuzzy matching.
    """

    target = normalize_text(
        exact_name
    )


    candidates = nutrient_available[
        nutrient_available[
            "normalized_name"
        ]
        == target
    ].copy()


    if allowed_units is not None:

        allowed_units = {
            str(x).upper()
            for x in allowed_units
        }


        candidates = candidates[
            candidates[
                "unit_name"
            ]
            .astype(str)
            .str.upper()
            .isin(allowed_units)
        ]


    if candidates.empty:

        return None


    candidates[
        "coverage"
    ] = (
        candidates[
            "id"
        ]
        .apply(nutrient_coverage)
    )


    candidates = candidates.sort_values(
        [
            "coverage",
            "id"
        ],
        ascending=[
            False,
            True
        ]
    )


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row["unit_name"],

        "coverage":
            int(row["coverage"])
    }


# ============================================================
# FINAL NUTRIENT MAP
# ============================================================

nutrient_lookup = {}


# ------------------------------------------------------------
# Calories
# ------------------------------------------------------------

nutrient_lookup[
    "calories_kcal"
] = {

    "id":
        int(energy_row["id"]),

    "name":
        energy_row["name"],

    "unit":
        energy_row["unit_name"],

    "coverage":
        int(energy_row["coverage"])
}


# ------------------------------------------------------------
# Exact USDA nutrients
# ------------------------------------------------------------

EXACT_NUTRIENTS = {

    "protein_g": (
        "Protein",
        {"G"}
    ),

    "fat_g": (
        "Total lipid (fat)",
        {"G"}
    ),

    "carbohydrate_g": (
        "Carbohydrate, by difference",
        {"G"}
    ),

    "fiber_g": (
        "Fiber, total dietary",
        {"G"}
    ),

    "sugar_g": (
        "Sugars, Total",
        {"G"}
    ),

    "saturated_fat_g": (
        "Fatty acids, total saturated",
        {"G"}
    )
}


for output_name, (
    exact_name,
    allowed_units
) in EXACT_NUTRIENTS.items():

    result = find_exact_nutrient(
        exact_name,
        allowed_units
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}: "
            f"{exact_name}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


# ============================================================
# FINAL NUTRIENT MAP DISPLAY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOUNDATION NUTRIENT MAP")
print("=" * 70)


for output_name, info in nutrient_lookup.items():

    print(
        f"{output_name:25} "
        f"ID={info['id']:5} "
        f"UNIT={str(info['unit']):5} "
        f"COVERAGE={info['coverage']:4} "
        f"NAME={info['name']}"
    )


# ============================================================
# CRITICAL NUTRIENT SAFETY VALIDATION
# ============================================================

EXPECTED_NUTRIENT_NAMES = {

    "calories_kcal": {

        normalize_text(
            "Energy"
        ),

        normalize_text(
            "Energy (Atwater General Factors)"
        ),

        normalize_text(
            "Energy (Atwater Specific Factors)"
        )
    },

    "protein_g": {

        normalize_text(
            "Protein"
        )
    },

    "fat_g": {

        normalize_text(
            "Total lipid (fat)"
        )
    },

    "carbohydrate_g": {

        normalize_text(
            "Carbohydrate, by difference"
        )
    },

    "fiber_g": {

        normalize_text(
            "Fiber, total dietary"
        )
    },

    "sugar_g": {

        normalize_text(
            "Sugars, Total"
        )
    },

    "saturated_fat_g": {

        normalize_text(
            "Fatty acids, total saturated"
        )
    }
}


for output_name, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    if output_name not in nutrient_lookup:

        raise RuntimeError(
            f"CRITICAL: Missing nutrient mapping: "
            f"{output_name}"
        )


    actual_name = normalize_text(
        nutrient_lookup[
            output_name
        ]["name"]
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"CRITICAL NUTRIENT MAPPING ERROR:\n"
            f"{output_name} -> "
            f"{nutrient_lookup[output_name]['name']}"
        )


print("\n")
print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# CHECK FOR DUPLICATE FOOD RECORDS
# ============================================================

def compare_food_nutrients(
    fdc_ids
):
    """
    Compare selected nutrients across duplicate FDC records.
    """

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "fdc_id"
        ]
        .isin(fdc_ids)
    ].copy()


    comparison = []


    for fdc_id in fdc_ids:

        record = {

            "fdc_id":
                int(fdc_id)
        }


        food_rows = rows[
            rows["fdc_id"]
            == int(fdc_id)
        ]


        for nutrient_name, info in (
            nutrient_lookup.items()
        ):

            nutrient_id = info["id"]


            nutrient_rows = food_rows[
                food_rows[
                    "nutrient_id"
                ]
                == nutrient_id
            ]


            if nutrient_rows.empty:

                record[
                    nutrient_name
                ] = np.nan

            else:

                record[
                    nutrient_name
                ] = pd.to_numeric(
                    nutrient_rows.iloc[0][
                        "amount"
                    ],
                    errors="coerce"
                )


        comparison.append(
            record
        )


    return pd.DataFrame(
        comparison
    )


# ============================================================
# DUPLICATE REVIEW
# ============================================================

print("\n")
print("=" * 70)
print("CHECKING DUPLICATE FDC RECORDS")
print("=" * 70)


duplicate_review_rows = []


for food_name, candidates in (
    duplicate_foods.items()
):

    fdc_ids = (
        candidates[
            "fdc_id"
        ]
        .astype(int)
        .tolist()
    )


    print("\n")
    print("-" * 70)

    print(
        food_name.upper()
    )

    print("-" * 70)


    comparison = compare_food_nutrients(
        fdc_ids
    )


    print(
        comparison.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Determine whether the available nutrient values are
    # identical.
    #
    # NaN vs NaN is considered equal.
    # --------------------------------------------------------

    nutrient_columns = [
        column
        for column in comparison.columns
        if column != "fdc_id"
    ]


    identical = True


    for column in nutrient_columns:

        values = comparison[
            column
        ]


        if len(values) <= 1:
            continue


        first = values.iloc[0]


        for value in values.iloc[1:]:

            if pd.isna(first) and pd.isna(value):

                continue


            if pd.isna(first) != pd.isna(value):

                identical = False

                break


            if not np.isclose(
                float(first),
                float(value),
                rtol=0,
                atol=1e-12
            ):

                identical = False

                break


        if not identical:
            break


    if identical:

        print(
            "\nRESULT: Nutrient profiles identical."
        )

        print(
            "A canonical FDC ID will be used."
        )

    else:

        print(
            "\nRESULT: Nutrient profiles differ."
        )

        print(
            "Keeping the first USDA record as "
            "the application-preferred record."
        )


    # --------------------------------------------------------
    # Use first record as preferred record.
    #
    # IMPORTANT:
    # All duplicate FDC IDs are preserved in the source
    # mapping later.
    # --------------------------------------------------------

    canonical_fdc_id = fdc_ids[0]


    canonical_description = (
        candidates.iloc[0]["description"]
    )


    selected_foods[
        food_name
    ] = {

        "fdc_id":
            int(canonical_fdc_id),

        "description":
            canonical_description
    }


    duplicate_review_rows.append({

        "food_name":
            food_name,

        "fdc_ids":
            ",".join(
                str(x)
                for x in fdc_ids
            ),

        "canonical_fdc_id":
            int(canonical_fdc_id),

        "nutrients_identical":
            identical
    })


# ============================================================
# CHECK NOT FOUND FOODS
# ============================================================

if not_found_foods:

    print("\n")
    print("=" * 70)
    print("FOODS NOT FOUND")
    print("=" * 70)


    for food_name, description in (
        not_found_foods.items()
    ):

        print(
            f"{food_name}: {description}"
        )


# ============================================================
# SELECTION SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD SELECTION SUMMARY")
print("=" * 70)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)


print(
    f"Foods selected: "
    f"{len(selected_foods)}"
)


print(
    f"Foods requiring duplicate review: "
    f"{len(duplicate_foods)}"
)


print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)


print("\nSelected foods:")


for food_name, info in (
    selected_foods.items()
):

    print(
        f"{food_name:25} "
        f"{info['fdc_id']:10} "
        f"{info['description']}"
    )


# ============================================================
# FAIL IF TARGET FOODS ARE MISSING
# ============================================================

if not_found_foods:

    raise RuntimeError(
        "One or more required target foods "
        "were not found. "
        "See the output above."
    )


# ============================================================
# EXTRACT NUTRIENTS FOR ONE FOOD
# ============================================================

def extract_food_nutrients(
    fdc_id
):
    """
    Extract selected nutrients for one USDA food.

    All values are per 100 g.

    Missing USDA nutrients remain NaN.
    """

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "fdc_id"
        ]
        == int(fdc_id)
    ]


    result = {}


    for output_name, info in (
        nutrient_lookup.items()
    ):

        nutrient_id = info["id"]


        nutrient_rows = rows[
            rows[
                "nutrient_id"
            ]
            == nutrient_id
        ]


        if nutrient_rows.empty:

            result[
                output_name
            ] = np.nan

        else:

            # In the normal Foundation Food structure,
            # there should be one row for each nutrient.
            #
            # If multiple rows somehow exist, use the first
            # non-null amount.

            amounts = pd.to_numeric(
                nutrient_rows[
                    "amount"
                ],
                errors="coerce"
            ).dropna()


            if amounts.empty:

                result[
                    output_name
                ] = np.nan

            else:

                result[
                    output_name
                ] = float(
                    amounts.iloc[0]
                )


    return result


# ============================================================
# BUILD FOOD DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("BUILDING FINAL FOOD DATABASE")
print("=" * 70)


database_rows = []


for food_name, info in (
    selected_foods.items()
):

    fdc_id = int(
        info["fdc_id"]
    )


    nutrient_values = (
        extract_food_nutrients(
            fdc_id
        )
    )


    row = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            info["description"],

        "data_type":
            "Foundation Food",

        "basis_g":
            100
    }


    row.update(
        nutrient_values
    )


    database_rows.append(
        row
    )


food_database = pd.DataFrame(
    database_rows
)


# ============================================================
# ORDER COLUMNS
# ============================================================

DATABASE_COLUMNS = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


food_database = food_database[
    DATABASE_COLUMNS
]


# ============================================================
# NUMERIC COLUMN CLEANUP
# ============================================================

numeric_columns = [

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


for column in numeric_columns:

    food_database[
        column
    ] = pd.to_numeric(
        food_database[column],
        errors="coerce"
    )


# ============================================================
# DATABASE DISPLAY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# FINAL NUTRIENT SANITY CHECK
# ============================================================

print("\n")
print("=" * 70)
print("NUTRIENT DATABASE SANITY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# Check that each selected nutrient maps to the correct USDA
# nutrient name.
# ------------------------------------------------------------

for column, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    actual_name = normalize_text(
        nutrient_lookup[
            column
        ]["name"]
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"INVALID NUTRIENT MAPPING:\n"
            f"{column} -> "
            f"{nutrient_lookup[column]['name']}"
        )


    print(
        f"PASS: {column:25} "
        f"-> {nutrient_lookup[column]['name']}"
    )


# ------------------------------------------------------------
# Check nutrient IDs are not accidentally reused for
# unrelated output columns.
# ------------------------------------------------------------

nutrient_id_by_output = {

    name:
        info["id"]

    for name, info in (
        nutrient_lookup.items()
    )
}


reverse_ids = {}


for output_name, nutrient_id in (
    nutrient_id_by_output.items()
):

    reverse_ids.setdefault(
        nutrient_id,
        []
    ).append(
        output_name
    )


for nutrient_id, outputs in (
    reverse_ids.items()
):

    if len(outputs) > 1:

        # The same ID being used by multiple outputs is almost
        # certainly an error for this database.

        raise RuntimeError(
            "CRITICAL: Same nutrient ID is mapped "
            f"to multiple outputs: "
            f"ID {nutrient_id} -> "
            f"{outputs}"
        )


print(
    "\nNo nutrient IDs are incorrectly reused."
)


# ============================================================
# CHECK CALORIES
# ============================================================

calorie_values = food_database[
    "calories_kcal"
]


if calorie_values.notna().sum() == 0:

    raise RuntimeError(
        "CRITICAL: No food has a calorie value."
    )


print(
    f"\nFoods with calories: "
    f"{calorie_values.notna().sum()}"
    f"/{len(food_database)}"
)


# ============================================================
# CHECK MACROS
# ============================================================

for column in [
    "protein_g",
    "fat_g",
    "carbohydrate_g"
]:

    count = (
        food_database[
            column
        ]
        .notna()
        .sum()
    )


    print(
        f"Foods with {column}: "
        f"{count}/{len(food_database)}"
    )


# ============================================================
# CHECK FOR IMPOSSIBLE NEGATIVE VALUES
# ============================================================

nonnegative_columns = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


for column in nonnegative_columns:

    negative = food_database[
        food_database[column]
        .notna()
        &
        (
            food_database[column]
            < 0
        )
    ]


    if not negative.empty:

        raise RuntimeError(
            f"Negative values found in "
            f"{column}."
        )


print(
    "\nNo negative nutrient values detected."
)


# ============================================================
# CHECK MACRO SANITY
# ============================================================
#
# This is NOT used to calculate calories.
#
# It is only a diagnostic check.
#
# Approximate macro calories:
#
# protein x 4
# carbohydrate x 4
# fat x 9
#
# USDA calorie value remains authoritative.
# ============================================================

food_database[
    "macro_calorie_check"
] = (
    food_database[
        "protein_g"
    ] * 4
    +
    food_database[
        "carbohydrate_g"
    ] * 4
    +
    food_database[
        "fat_g"
    ] * 9
)


food_database[
    "macro_calorie_difference"
] = (
    food_database[
        "calories_kcal"
    ]
    -
    food_database[
        "macro_calorie_check"
    ]
)


print("\n")
print("=" * 70)
print("CALORIE SANITY CHECK")
print("=" * 70)


print(
    food_database[
        [
            "food_name",
            "calories_kcal",
            "macro_calorie_check",
            "macro_calorie_difference"
        ]
    ]
    .to_string(index=False)
)


# ============================================================
# REMOVE DIAGNOSTIC COLUMNS FROM FINAL DATABASE
# ============================================================

food_database = food_database[
    DATABASE_COLUMNS
]


# ============================================================
# SOURCE MAPPING TABLE
# ============================================================
#
# This preserves duplicate USDA FDC records instead of
# throwing them away.
#
# Example:
#
# apple_fuji
#     1105897
#     1750340
#
# The application still has one preferred FDC ID, but the
# underlying USDA records remain visible.
# ============================================================

source_mapping_rows = []


for food_name, config in (
    TARGET_FOODS.items()
):

    target_normalized = normalize_text(
        config["exact_description"]
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target_normalized
    ]


    preferred_fdc_id = int(
        selected_foods[
            food_name
        ]["fdc_id"]
    )


    for _, row in matches.iterrows():

        source_mapping_rows.append({

            "food_name":
                food_name,

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "source":
                "USDA FoodData Central Foundation Food",

            "is_preferred":
                int(
                    row["fdc_id"]
                    == preferred_fdc_id
                )
        })


food_source_mapping = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# SELECTION REVIEW FILE
# ============================================================

review_rows = []


for food_name, config in (
    TARGET_FOODS.items()
):

    target_normalized = normalize_text(
        config["exact_description"]
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target_normalized
    ]


    selected_id = selected_foods[
        food_name
    ]["fdc_id"]


    review_rows.append({

        "food_name":
            food_name,

        "requested_description":
            config["exact_description"],

        "match_count":
            len(matches),

        "selected_fdc_id":
            selected_id,

        "status":
            (
                "SELECTED"
                if len(matches) >= 1
                else "NOT_FOUND"
            ),

        "all_matching_fdc_ids":
            ",".join(
                str(int(x))
                for x in matches[
                    "fdc_id"
                ].tolist()
            )
    })


food_selection_review = pd.DataFrame(
    review_rows
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

if OUTPUT_DIRECTORY is None:

    OUTPUT_DIRECTORY = (
        FOOD_FILE.parent
    )

else:

    OUTPUT_DIRECTORY = Path(
        OUTPUT_DIRECTORY
    )


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SAVE DATABASE
# ============================================================

DATABASE_FILE = (
    OUTPUT_DIRECTORY
    /
    "food_database.csv"
)


SOURCE_MAPPING_FILE = (
    OUTPUT_DIRECTORY
    /
    "food_source_mapping.csv"
)


REVIEW_FILE = (
    OUTPUT_DIRECTORY
    /
    "food_selection_review.csv"
)


food_database.to_csv(
    DATABASE_FILE,
    index=False
)


food_source_mapping.to_csv(
    SOURCE_MAPPING_FILE,
    index=False
)


food_selection_review.to_csv(
    REVIEW_FILE,
    index=False
)


# ============================================================
# DATABASE VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DATABASE VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Number of foods
# ------------------------------------------------------------

if len(food_database) != len(TARGET_FOODS):

    raise RuntimeError(
        "Unexpected number of foods in final database."
    )


print(
    f"Foods: "
    f"{len(food_database)}"
)


# ------------------------------------------------------------
# Unique FDC IDs
# ------------------------------------------------------------

unique_fdc_ids = (
    food_database[
        "fdc_id"
    ]
    .nunique()
)


print(
    f"Unique FDC IDs: "
    f"{unique_fdc_ids}"
)


if unique_fdc_ids != len(food_database):

    raise RuntimeError(
        "Duplicate FDC IDs found in final database."
    )


# ------------------------------------------------------------
# Basis
# ------------------------------------------------------------

if not (
    food_database[
        "basis_g"
    ]
    == 100
).all():

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


print(
    "Basis: 100 g for all foods"
)


# ------------------------------------------------------------
# Data type
# ------------------------------------------------------------

if not (
    food_database[
        "data_type"
    ]
    == "Foundation Food"
).all():

    raise RuntimeError(
        "Non-Foundation food found in final database."
    )


print(
    "Data type: Foundation Food"
)


# ------------------------------------------------------------
# Critical macro columns
# ------------------------------------------------------------

for column in [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g"
]:

    if (
        food_database[
            column
        ]
        .notna()
        .sum()
        == 0
    ):

        raise RuntimeError(
            f"Critical nutrient column "
            f"{column} is entirely missing."
        )


print(
    "Critical nutrient coverage: PASS"
)


print(
    "\nSTATUS: PASS"
)


# ============================================================
# NUTRITION CALCULATION FUNCTION
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):
    """
    Calculate nutrition for an arbitrary food weight.

    Database values are per 100 g.

    Missing nutrients remain None.
    """

    if grams is None:

        raise ValueError(
            "grams cannot be None"
        )


    grams = float(
        grams
    )


    if not math.isfinite(
        grams
    ):

        raise ValueError(
            "grams must be finite"
        )


    if grams < 0:

        raise ValueError(
            "grams cannot be negative"
        )


    matches = database[
        database[
            "food_name"
        ]
        == food_name
    ]


    if matches.empty:

        raise KeyError(
            f"Food not found: "
            f"{food_name}"
        )


    row = matches.iloc[0]


    factor = (
        grams
        /
        float(row["basis_g"])
    )


    result = {

        "food_name":
            food_name,

        "grams":
            grams,

        "fdc_id":
            int(row["fdc_id"])
    }


    for nutrient_name in [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"

    ]:

        value = row[
            nutrient_name
        ]


        if pd.isna(value):

            result[
                nutrient_name
            ] = None

        else:

            result[
                nutrient_name
            ] = float(value) * factor


    return result


# ============================================================
# MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):
    """
    meal_items example:

    [
        {
            "food_name": "chicken_breast",
            "grams": 150
        },
        {
            "food_name": "oats_rolled",
            "grams": 50
        }
    ]

    Missing nutrients remain missing.

    Missing != zero.
    """

    nutrient_names = [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"
    ]


    totals = {

        nutrient:
            0.0

        for nutrient in nutrient_names
    }


    has_value = {

        nutrient:
            False

        for nutrient in nutrient_names
    }


    foods = []


    for item in meal_items:

        result = calculate_nutrition(

            database,

            item[
                "food_name"
            ],

            item[
                "grams"
            ]
        )


        foods.append(
            result
        )


        for nutrient in nutrient_names:

            value = result.get(
                nutrient
            )


            if value is not None:

                totals[
                    nutrient
                ] += float(
                    value
                )

                has_value[
                    nutrient
                ] = True


    # --------------------------------------------------------
    # If no food in the meal has a nutrient value, preserve
    # the nutrient as missing.
    #
    # If at least one food has a value, total the known values.
    #
    # This avoids displaying:
    #
    #     Fiber: 0.00 g
    #
    # when USDA actually reported no fiber information.
    # --------------------------------------------------------

    for nutrient in nutrient_names:

        if not has_value[
            nutrient
        ]:

            totals[
                nutrient
            ] = None


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# MEAL PRINTER
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal[
        "foods"
    ]:

        print(
            f"{food['food_name']:25} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name, value in (
        meal[
            "totals"
        ].items()
    ):

        label = labels[
            nutrient_name
        ]

        unit = units[
            nutrient_name
        ]


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} "
                f"{unit}"
            )


# ============================================================
# EXAMPLE NUTRITION TEST
# ============================================================

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


test_food = "chicken_breast"


if test_food in set(
    food_database[
        "food_name"
    ]
):

    chicken_150g = calculate_nutrition(
        food_database,
        test_food,
        150
    )


    print(
        f"\n{test_food} — 150 g"
    )


    for nutrient_name in [

        "calories_kcal",

        "protein_g",

        "fat_g",

        "carbohydrate_g",

        "fiber_g",

        "sugar_g",

        "saturated_fat_g"

    ]:

        value = chicken_150g[
            nutrient_name
        ]


        if value is None:

            print(
                f"{nutrient_name:25}: -"
            )

        else:

            print(
                f"{nutrient_name:25}: "
                f"{value:.2f}"
            )


# ============================================================
# EXAMPLE MEAL TEST
# ============================================================

example_meal = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {
        "food_name":
            "oats_rolled",

        "grams":
            50
    },

    {
        "food_name":
            "apple_fuji",

        "grams":
            150
    }
]


meal = calculate_meal(
    food_database,
    example_meal
)


print_meal(
    meal
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food)}"
)


print(
    f"Foods selected: "
    f"{len(food_database)}"
)


print(
    f"Foods requiring review: "
    f"{len(duplicate_foods)}"
)


print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)


print(
    "\nDatabase validation: PASS"
)


print(
    "\nDatabase:"
)

print(
    DATABASE_FILE
)


print(
    "\nSource mapping:"
)

print(
    SOURCE_MAPPING_FILE
)


print(
    "\nReview:"
)

print(
    REVIEW_FILE
)


print(
    "\nDone."
)




LOADING USDA FOODDATA CENTRAL FILES

Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


SELECTED USDA FILES

food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv

food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv

nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


READING USDA CSV FILES

food.csv rows:          87,990
food_nutrient.csv rows: 170,469
nutrient.csv rows:      477


USDA DATA TYPES
data_type
sub_sample_food             75055
market_acquisition           7577
sample_food                  4079
agricultural_acquisition      810
foundation_food               469


NORMALIZED DATA TYPES
normaliz

In [23]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD DATABASE BUILDER
# ============================================================
#
# This pipeline:
#
# 1. Finds USDA FoodData Central CSV files
# 2. Loads food.csv
# 3. Loads food_nutrient.csv
# 4. Loads nutrient.csv
# 5. Correctly identifies Foundation Foods
# 6. Selects exact target foods
# 7. Handles duplicate USDA descriptions
# 8. Chooses the duplicate with the best nutrient completeness
# 9. Uses exact USDA nutrient names
# 10. Prevents incorrect nutrient-ID reuse
# 11. Builds a 100 g nutrition database
# 12. Validates the database
# 13. Runs a nutrition calculation test
# 14. Runs a meal calculation test
#
# IMPORTANT:
# Missing nutrient != zero.
#
# NaN values are preserved in the database.
# Meal totals only report zero when an actual zero is present
# or when a nutrient value is explicitly available as zero.
# ============================================================


from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# CONFIGURATION
# ============================================================

SEARCH_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)


OUTPUT_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodDta_Central_foundation_food_csv_2026-04-30"
)


FOOD_OUTPUT = (
    OUTPUT_DIR /
    "food_database.csv"
)


SOURCE_MAPPING_OUTPUT = (
    OUTPUT_DIR /
    "food_source_mapping.csv"
)


REVIEW_OUTPUT = (
    OUTPUT_DIR /
    "food_selection_review.csv"
)


BASIS_G = 100.0


# ============================================================
# TARGET FOODS
# ============================================================
#
# Exact USDA descriptions are intentionally used.
#
# Do NOT use broad fuzzy food matching here.
# ============================================================

TARGET_FOODS = {

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# REQUIRED NUTRIENTS
# ============================================================

NUTRIENT_COLUMNS = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


# ============================================================
# NORMALIZATION FUNCTIONS
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = (
        value
        .strip()
        .lower()
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def normalize_nutrient_name(value):

    return normalize_text(value)


# ============================================================
# FIND USDA FILES
# ============================================================

print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)

print("\nSearch directory:")
print(SEARCH_DIR)


def find_file(filename):

    matches = list(
        SEARCH_DIR.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"Could not find {filename} "
            f"under {SEARCH_DIR}"
        )

    # Prefer the file inside the requested
    # Foundation Food download directory.
    preferred = [
        p for p in matches
        if "FoodData_Central_foundation_food_csv"
        in str(p)
    ]

    if preferred:
        return preferred[0]

    return matches[0]


FOOD_FILE = find_file(
    "food.csv"
)

FOOD_NUTRIENT_FILE = find_file(
    "food_nutrient.csv"
)

NUTRIENT_FILE = find_file(
    "nutrient.csv"
)


print("\n")
print("=" * 70)
print("SELECTED USDA FILES")
print("=" * 70)

print("\nfood.csv:")
print(FOOD_FILE)

print("\nfood_nutrient.csv:")
print(FOOD_NUTRIENT_FILE)

print("\nnutrient.csv:")
print(NUTRIENT_FILE)


# ============================================================
# READ CSV FILES
# ============================================================

print("\n")
print("=" * 70)
print("READING USDA CSV FILES")
print("=" * 70)


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(
    f"\nfood.csv rows:          "
    f"{len(food):,}"
)

print(
    f"food_nutrient.csv rows: "
    f"{len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      "
    f"{len(nutrient):,}"
)


# ============================================================
# BASIC COLUMN VALIDATION
# ============================================================

required_food_columns = {
    "fdc_id",
    "description",
    "data_type"
}


required_food_nutrient_columns = {
    "fdc_id",
    "nutrient_id",
    "amount"
}


required_nutrient_columns = {
    "id",
    "name",
    "unit_name"
}


if not required_food_columns.issubset(
    food.columns
):

    missing = (
        required_food_columns
        - set(food.columns)
    )

    raise RuntimeError(
        f"food.csv missing columns: {missing}"
    )


if not required_food_nutrient_columns.issubset(
    food_nutrient.columns
):

    missing = (
        required_food_nutrient_columns
        - set(food_nutrient.columns)
    )

    raise RuntimeError(
        f"food_nutrient.csv missing columns: {missing}"
    )


if not required_nutrient_columns.issubset(
    nutrient.columns
):

    missing = (
        required_nutrient_columns
        - set(nutrient.columns)
    )

    raise RuntimeError(
        f"nutrient.csv missing columns: {missing}"
    )


# ============================================================
# DATA TYPE NORMALIZATION
# ============================================================

food["fdc_id"] = (
    pd.to_numeric(
        food["fdc_id"],
        errors="coerce"
    )
    .astype("Int64")
)


food_nutrient["fdc_id"] = (
    pd.to_numeric(
        food_nutrient["fdc_id"],
        errors="coerce"
    )
    .astype("Int64")
)


food_nutrient["nutrient_id"] = (
    pd.to_numeric(
        food_nutrient["nutrient_id"],
        errors="coerce"
    )
    .astype("Int64")
)


nutrient["id"] = (
    pd.to_numeric(
        nutrient["id"],
        errors="coerce"
    )
    .astype("Int64")
)


# ============================================================
# USDA DATA TYPES
# ============================================================

print("\n")
print("=" * 70)
print("USDA DATA TYPES")
print("=" * 70)

print(
    food["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# NORMALIZE DATA TYPES
# ============================================================

food["normalized_data_type"] = (
    food["data_type"]
    .astype(str)
    .apply(normalize_text)
)


print("\n")
print("=" * 70)
print("NORMALIZED DATA TYPES")
print("=" * 70)

print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# FIND FOUNDATION FOODS
# ============================================================
#
# IMPORTANT:
#
# normalized value is:
#
#     foundationfood
#
# because spaces are removed by the USDA normalization logic.
#
# Do not accidentally compare against:
#
#     "foundation food"
#
# ============================================================

foundation_food = food[
    food["normalized_data_type"]
    == "foundationfood"
].copy()


if foundation_food.empty:

    print("\n")
    print(
        "FOUNDATION FOOD DETECTION FAILED"
    )

    print(
        food["normalized_data_type"]
        .value_counts(dropna=False)
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

foundation_food[
    "normalized_description"
] = (
    foundation_food[
        "description"
    ]
    .apply(normalize_text)
)


foundation_ids = set(
    foundation_food[
        "fdc_id"
    ]
    .dropna()
    .astype(int)
)


print("\n")
print("=" * 70)
print("FOUNDATION FOOD SUMMARY")
print("=" * 70)

print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)

print(
    f"Unique Foundation FDC IDs: "
    f"{len(foundation_ids):,}"
)


if len(foundation_ids) != len(
    foundation_food
):

    raise RuntimeError(
        "Duplicate FDC IDs found inside "
        "Foundation Foods."
    )


# ============================================================
# FOUNDATION FOOD TARGET SEARCH
# ============================================================

print("\n")
print("=" * 70)
print("FOUNDATION FOOD TARGET SEARCH")
print("=" * 70)


target_matches = {}


for food_name, config in TARGET_FOODS.items():

    requested = config[
        "exact_description"
    ]

    normalized_target = normalize_text(
        requested
    )

    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == normalized_target
    ].copy()


    target_matches[food_name] = matches


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)

    print("Requested:")
    print(requested)

    print(
        f"Matches: {len(matches)}"
    )

    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
            .sort_values("fdc_id")
            .to_string(index=False)
        )


# ============================================================
# FOUNDATION FOOD NUTRIENT ROWS
# ============================================================

foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_ids)
    ]
    .copy()
)


foundation_food_nutrients[
    "fdc_id"
] = (
    foundation_food_nutrients[
        "fdc_id"
    ]
    .astype(int)
)


foundation_food_nutrients[
    "nutrient_id"
] = (
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .astype(int)
)


print("\n")
print("=" * 70)
print("FOUNDATION NUTRIENT COVERAGE")
print("=" * 70)

print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)


foundation_nutrient_ids = set(
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .dropna()
    .astype(int)
)


print(
    f"Nutrient IDs available in "
    f"Foundation Foods: "
    f"{len(foundation_nutrient_ids):,}"
)


# ============================================================
# NORMALIZE NUTRIENT NAMES
# ============================================================

nutrient[
    "normalized_name"
] = (
    nutrient[
        "name"
    ]
    .apply(normalize_nutrient_name)
)


# ============================================================
# NUTRIENT COVERAGE FUNCTION
# ============================================================

def nutrient_coverage(
    nutrient_id
):

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "nutrient_id"
        ]
        == int(nutrient_id)
    ]


    if rows.empty:

        return 0


    return int(
        rows["amount"]
        .notna()
        .sum()
    )


# ============================================================
# EXACT NUTRIENT FINDER
# ============================================================

def find_exact_nutrient(
    exact_name,
    allowed_units=None
):

    target = normalize_nutrient_name(
        exact_name
    )


    candidates = nutrient[
        nutrient[
            "normalized_name"
        ]
        == target
    ].copy()


    # Only nutrient IDs that actually occur
    # in Foundation Foods.
    candidates = candidates[
        candidates["id"]
        .astype(int)
        .isin(
            foundation_nutrient_ids
        )
    ]


    if allowed_units is not None:

        allowed_units = {
            str(x).upper()
            for x in allowed_units
        }

        candidates = candidates[
            candidates[
                "unit_name"
            ]
            .astype(str)
            .str.upper()
            .isin(allowed_units)
        ]


    if candidates.empty:

        return None


    candidates[
        "coverage"
    ] = (
        candidates["id"]
        .astype(int)
        .apply(nutrient_coverage)
    )


    candidates = candidates.sort_values(
        [
            "coverage",
            "id"
        ],
        ascending=[
            False,
            True
        ]
    )


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row["unit_name"],

        "coverage":
            int(row["coverage"])
    }


# ============================================================
# ENERGY NUTRIENT SELECTION
# ============================================================

energy_candidates = nutrient[
    nutrient[
        "normalized_name"
    ].isin(
        {
            "energy",
            "energy (atwater general factors)",
            "energy (atwater specific factors)"
        }
    )
].copy()


energy_candidates = energy_candidates[
    energy_candidates[
        "id"
    ]
    .astype(int)
    .isin(
        foundation_nutrient_ids
    )
]


energy_candidates = energy_candidates[
    energy_candidates[
        "unit_name"
    ]
    .astype(str)
    .str.upper()
    == "KCAL"
]


energy_candidates[
    "coverage"
] = (
    energy_candidates[
        "id"
    ]
    .astype(int)
    .apply(nutrient_coverage)
)


ENERGY_PRIORITY = {

    2047: 3,

    2048: 2,

    1008: 1
}


energy_candidates[
    "priority"
] = (
    energy_candidates[
        "id"
    ]
    .astype(int)
    .map(
        ENERGY_PRIORITY
    )
    .fillna(0)
)


energy_candidates = (
    energy_candidates
    .sort_values(
        [
            "coverage",
            "priority"
        ],
        ascending=[
            False,
            False
        ]
    )
)


print("\n")
print("=" * 70)
print("ENERGY NUTRIENTS ACTUALLY AVAILABLE")
print("=" * 70)


if not energy_candidates.empty:

    print(
        energy_candidates[
            [
                "id",
                "name",
                "unit_name",
                "coverage"
            ]
        ]
        .to_string(index=False)
    )


if energy_candidates.empty:

    raise RuntimeError(
        "No KCAL energy nutrient exists "
        "in Foundation Foods."
    )


energy_row = energy_candidates.iloc[0]


nutrient_lookup = {}


nutrient_lookup[
    "calories_kcal"
] = {

    "id":
        int(
            energy_row["id"]
        ),

    "name":
        energy_row["name"],

    "unit":
        energy_row["unit_name"],

    "coverage":
        int(
            energy_row["coverage"]
        )
}


# ============================================================
# EXACT MACRO/MICRO NUTRIENTS
# ============================================================

EXACT_NUTRIENTS = {

    "protein_g": (
        "Protein",
        {"G"}
    ),

    "fat_g": (
        "Total lipid (fat)",
        {"G"}
    ),

    "carbohydrate_g": (
        "Carbohydrate, by difference",
        {"G"}
    ),

    "fiber_g": (
        "Fiber, total dietary",
        {"G"}
    ),

    "sugar_g": (
        "Sugars, Total",
        {"G"}
    ),

    "saturated_fat_g": (
        "Fatty acids, total saturated",
        {"G"}
    )
}


for output_name, (
    exact_name,
    allowed_units
) in EXACT_NUTRIENTS.items():

    result = find_exact_nutrient(
        exact_name,
        allowed_units
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}: "
            f"{exact_name}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


# ============================================================
# FINAL NUTRIENT MAP
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOUNDATION NUTRIENT MAP")
print("=" * 70)


for output_name, info in (
    nutrient_lookup.items()
):

    print(
        f"{output_name:25} "
        f"ID={info['id']:5} "
        f"UNIT={info['unit']:<5} "
        f"COVERAGE={info['coverage']:4} "
        f"NAME={info['name']}"
    )


# ============================================================
# NUTRIENT SAFETY VALIDATION
# ============================================================

EXPECTED_NUTRIENT_NAMES = {

    "calories_kcal": {
        "energy",
        "energy (atwater general factors)",
        "energy (atwater specific factors)"
    },

    "protein_g": {
        "protein"
    },

    "fat_g": {
        "total lipid (fat)"
    },

    "carbohydrate_g": {
        "carbohydrate, by difference"
    },

    "fiber_g": {
        "fiber, total dietary"
    },

    "sugar_g": {
        "sugars, total"
    },

    "saturated_fat_g": {
        "fatty acids, total saturated"
    }
}


for output_name, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    if output_name not in nutrient_lookup:

        raise RuntimeError(
            f"CRITICAL: Missing nutrient "
            f"mapping: {output_name}"
        )


    actual_name = (
        normalize_nutrient_name(
            nutrient_lookup[
                output_name
            ]["name"]
        )
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"CRITICAL NUTRIENT MAPPING ERROR: "
            f"{output_name} -> "
            f"{nutrient_lookup[output_name]['name']}"
        )


print("\n")
print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# EXTRACT NUTRIENT VALUE
# ============================================================

def get_nutrient_value(
    fdc_id,
    nutrient_id
):

    rows = foundation_food_nutrients[
        (
            foundation_food_nutrients[
                "fdc_id"
            ]
            == int(fdc_id)
        )
        &
        (
            foundation_food_nutrients[
                "nutrient_id"
            ]
            == int(nutrient_id)
        )
    ]


    if rows.empty:

        return np.nan


    values = pd.to_numeric(
        rows["amount"],
        errors="coerce"
    )


    values = values[
        values.notna()
    ]


    if values.empty:

        return np.nan


    return float(
        values.iloc[0]
    )


# ============================================================
# COMPARE DUPLICATE RECORDS
# ============================================================

def compare_food_nutrients(
    fdc_ids
):

    comparison = []


    for fdc_id in fdc_ids:

        record = {

            "fdc_id":
                int(fdc_id)
        }


        for nutrient_name, info in (
            nutrient_lookup.items()
        ):

            record[
                nutrient_name
            ] = get_nutrient_value(
                fdc_id,
                info["id"]
            )


        comparison.append(
            record
        )


    return pd.DataFrame(
        comparison
    )


# ============================================================
# DUPLICATE SELECTION SCORING
# ============================================================
#
# IMPORTANT CHANGE:
#
# We no longer simply choose the first FDC ID.
#
# We score each duplicate according to nutrient completeness.
#
# Critical nutrients receive higher weights.
#
# Calories are especially important because this is a
# nutrition-tracking application.
# ============================================================

NUTRIENT_COMPLETENESS_WEIGHTS = {

    "calories_kcal":
        10,

    "protein_g":
        5,

    "fat_g":
        5,

    "carbohydrate_g":
        5,

    "fiber_g":
        2,

    "sugar_g":
        2,

    "saturated_fat_g":
        2
}


def score_duplicate_record(
    row
):

    score = 0.0


    for nutrient_name, weight in (
        NUTRIENT_COMPLETENESS_WEIGHTS.items()
    ):

        value = row[
            nutrient_name
        ]


        if pd.notna(value):

            score += weight


    return score


# ============================================================
# SELECT BEST USDA RECORD
# ============================================================

selected_foods = {}

duplicate_reviews = []

not_found_foods = []


print("\n")
print("=" * 70)
print("SELECTING TARGET FOUNDATION FOODS")
print("=" * 70)


for food_name, config in (
    TARGET_FOODS.items()
):

    matches = target_matches[
        food_name
    ]


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)


    if matches.empty:

        print(
            "STATUS: NOT_FOUND"
        )

        not_found_foods.append(
            food_name
        )

        continue


    # --------------------------------------------------------
    # One exact USDA record
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]


        selected_foods[
            food_name
        ] = {

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "data_type":
                row["data_type"],

            "selection_reason":
                "unique exact USDA description"
        }


        print(
            "STATUS: SELECTED"
        )

        print(
            "FDC ID:",
            int(row["fdc_id"])
        )

        print(
            "Description:",
            row["description"]
        )

        continue


    # --------------------------------------------------------
    # Duplicate USDA descriptions
    # --------------------------------------------------------

    fdc_ids = (
        matches[
            "fdc_id"
        ]
        .astype(int)
        .tolist()
    )


    comparison = compare_food_nutrients(
        fdc_ids
    )


    comparison[
        "completeness_score"
    ] = (
        comparison
        .apply(
            score_duplicate_record,
            axis=1
        )
    )


    comparison = comparison.sort_values(
        [
            "completeness_score",
            "calories_kcal",
            "protein_g",
            "fat_g",
            "carbohydrate_g"
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False
        ],
        na_position="last"
    )


    best_fdc_id = int(
        comparison.iloc[0]["fdc_id"]
    )


    best_match = matches[
        matches["fdc_id"].astype(int)
        == best_fdc_id
    ].iloc[0]


    selected_foods[
        food_name
    ] = {

        "fdc_id":
            best_fdc_id,

        "description":
            best_match["description"],

        "data_type":
            best_match["data_type"],

        "selection_reason":
            "duplicate resolved by nutrient completeness"
    }


    duplicate_reviews.append({

        "food_name":
            food_name,

        "selected_fdc_id":
            best_fdc_id,

        "reason":
            "Selected duplicate with highest "
            "critical nutrient completeness"
    })


    print(
        "STATUS: DUPLICATE_RESOLVED"
    )

    print(
        "Selected FDC ID:",
        best_fdc_id
    )

    print(
        "Description:",
        best_match["description"]
    )


    print("\nDuplicate comparison:")

    print(
        comparison.to_string(
            index=False
        )
    )


# ============================================================
# SHOW FINAL SELECTIONS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD SELECTION SUMMARY")
print("=" * 70)

print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)

print(
    f"Foods selected: "
    f"{len(selected_foods)}"
)

print(
    f"Duplicate groups resolved: "
    f"{len(duplicate_reviews)}"
)

print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)


print("\nSelected foods:")


for food_name, data in (
    selected_foods.items()
):

    print(
        f"{food_name:28} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


if not_found_foods:

    raise RuntimeError(
        "Some target foods were not found: "
        + ", ".join(not_found_foods)
    )


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("BUILDING FINAL FOOD DATABASE")
print("=" * 70)


database_rows = []


for food_name, selection in (
    selected_foods.items()
):

    fdc_id = selection[
        "fdc_id"
    ]


    source_row = foundation_food[
        foundation_food[
            "fdc_id"
        ].astype(int)
        == int(fdc_id)
    ]


    if source_row.empty:

        raise RuntimeError(
            f"Selected FDC ID {fdc_id} "
            f"not found in Foundation Foods."
        )


    source_row = source_row.iloc[0]


    record = {

        "food_name":
            food_name,

        "fdc_id":
            int(fdc_id),

        "description":
            source_row["description"],

        "data_type":
            source_row["data_type"],

        "basis_g":
            BASIS_G
    }


    for nutrient_name, info in (
        nutrient_lookup.items()
    ):

        record[
            nutrient_name
        ] = get_nutrient_value(
            fdc_id,
            info["id"]
        )


    database_rows.append(
        record
    )


food_database = pd.DataFrame(
    database_rows
)


# ============================================================
# DATABASE COLUMN ORDER
# ============================================================

food_database = food_database[
    [
        "food_name",
        "fdc_id",
        "description",
        "data_type",
        "basis_g",

        "calories_kcal",
        "protein_g",
        "fat_g",
        "carbohydrate_g",
        "fiber_g",
        "sugar_g",
        "saturated_fat_g"
    ]
]


# ============================================================
# FINAL FOOD DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)

print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# CREATE SOURCE MAPPING
# ============================================================

source_mapping_rows = []


for food_name, selection in (
    selected_foods.items()
):

    source_mapping_rows.append({

        "food_name":
            food_name,

        "fdc_id":
            selection["fdc_id"],

        "description":
            selection["description"],

        "source":
            "USDA FoodData Central Foundation Foods",

        "is_preferred":
            True,

        "selection_reason":
            selection[
                "selection_reason"
            ]
    })


source_mapping = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# CREATE REVIEW FILE
# ============================================================

review_rows = []


for food_name, matches in (
    target_matches.items()
):

    if len(matches) <= 1:
        continue


    selected_id = selected_foods[
        food_name
    ]["fdc_id"]


    comparison = compare_food_nutrients(
        matches[
            "fdc_id"
        ]
        .astype(int)
        .tolist()
    )


    comparison[
        "food_name"
    ] = food_name


    comparison[
        "selected"
    ] = (
        comparison["fdc_id"]
        == selected_id
    )


    review_rows.append(
        comparison
    )


if review_rows:

    review_df = pd.concat(
        review_rows,
        ignore_index=True
    )

else:

    review_df = pd.DataFrame()


# ============================================================
# DATABASE VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("NUTRIENT DATABASE SANITY CHECK")
print("=" * 70)


EXPECTED_DATABASE_NAMES = {

    "calories_kcal": {
        "energy",
        "energy (atwater general factors)",
        "energy (atwater specific factors)"
    },

    "protein_g": {
        "protein"
    },

    "fat_g": {
        "total lipid (fat)"
    },

    "carbohydrate_g": {
        "carbohydrate, by difference"
    },

    "fiber_g": {
        "fiber, total dietary"
    },

    "sugar_g": {
        "sugars, total"
    },

    "saturated_fat_g": {
        "fatty acids, total saturated"
    }
}


for column, allowed_names in (
    EXPECTED_DATABASE_NAMES.items()
):

    actual = normalize_nutrient_name(
        nutrient_lookup[
            column
        ]["name"]
    )


    if actual not in allowed_names:

        raise RuntimeError(
            f"INVALID NUTRIENT MAPPING: "
            f"{column} -> {actual}"
        )


    print(
        f"PASS: {column:25} "
        f"-> "
        f"{nutrient_lookup[column]['name']}"
    )


# ============================================================
# CHECK DUPLICATE FDC IDs
# ============================================================

if (
    food_database["fdc_id"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Duplicate FDC IDs exist in the "
        "final application database."
    )


# ============================================================
# CHECK BASIS
# ============================================================

if not (
    food_database[
        "basis_g"
    ]
    == 100
).all():

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


# ============================================================
# CHECK DATA TYPE
# ============================================================

invalid_types = food_database[
    food_database[
        "data_type"
    ]
    .astype(str)
    .apply(normalize_text)
    != "foundationfood"
]


if not invalid_types.empty:

    raise RuntimeError(
        "Non-Foundation foods detected."
    )


# ============================================================
# CHECK NEGATIVE NUTRIENTS
# ============================================================

for column in NUTRIENT_COLUMNS:

    values = pd.to_numeric(
        food_database[column],
        errors="coerce"
    )


    negative = values[
        values < 0
    ]


    if not negative.empty:

        raise RuntimeError(
            f"Negative values found in "
            f"{column}"
        )


print("\n")
print(
    "No negative nutrient values detected."
)


# ============================================================
# CRITICAL NUTRIENT COVERAGE
# ============================================================

critical_columns = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g"
]


print("\n")


for column in critical_columns:

    count = int(
        food_database[
            column
        ]
        .notna()
        .sum()
    )


    total = len(
        food_database
    )


    print(
        f"Foods with {column}: "
        f"{count}/{total}"
    )


# Calories SHOULD now be available for all
# selected foods if duplicate resolution worked
# correctly.

missing_calories = food_database[
    food_database[
        "calories_kcal"
    ]
    .isna()
]


if not missing_calories.empty:

    print("\n")
    print(
        "WARNING: Some foods still have "
        "missing calories:"
    )

    print(
        missing_calories[
            [
                "food_name",
                "fdc_id",
                "description"
            ]
        ]
        .to_string(index=False)
    )


# ============================================================
# CALORIE SANITY CHECK
# ============================================================
#
# This is NOT used to calculate calories.
#
# USDA energy remains the authoritative calorie value.
#
# This is only a diagnostic comparison:
#
# 4 kcal/g protein
# 9 kcal/g fat
# 4 kcal/g carbohydrate
# ============================================================

print("\n")
print("=" * 70)
print("CALORIE SANITY CHECK")
print("=" * 70)


calorie_check_rows = []


for _, row in (
    food_database.iterrows()
):

    protein = row[
        "protein_g"
    ]

    fat = row[
        "fat_g"
    ]

    carbs = row[
        "carbohydrate_g"
    ]

    calories = row[
        "calories_kcal"
    ]


    if (
        pd.notna(protein)
        and pd.notna(fat)
        and pd.notna(carbs)
    ):

        macro_calories = (
            protein * 4
            +
            fat * 9
            +
            carbs * 4
        )

    else:

        macro_calories = np.nan


    if (
        pd.notna(calories)
        and pd.notna(macro_calories)
    ):

        difference = (
            calories
            - macro_calories
        )

    else:

        difference = np.nan


    calorie_check_rows.append({

        "food_name":
            row["food_name"],

        "calories_kcal":
            calories,

        "macro_calorie_check":
            macro_calories,

        "macro_calorie_difference":
            difference
    })


calorie_check = pd.DataFrame(
    calorie_check_rows
)


print(
    calorie_check.to_string(
        index=False
    )
)


# ============================================================
# NUTRITION CALCULATION
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):

    if grams is None:

        raise ValueError(
            "grams cannot be None."
        )


    grams = float(
        grams
    )


    if grams < 0:

        raise ValueError(
            "grams cannot be negative."
        )


    matches = database[
        database[
            "food_name"
        ]
        == food_name
    ]


    if matches.empty:

        raise KeyError(
            f"Food not found: "
            f"{food_name}"
        )


    row = matches.iloc[0]


    factor = (
        grams
        /
        float(row["basis_g"])
    )


    result = {

        "food_name":
            food_name,

        "grams":
            grams
    }


    for nutrient_name in (
        NUTRIENT_COLUMNS
    ):

        value = row[
            nutrient_name
        ]


        if pd.isna(value):

            result[
                nutrient_name
            ] = None

        else:

            result[
                nutrient_name
            ] = float(value) * factor


    return result


# ============================================================
# MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):

    totals = {

        nutrient:
            0.0

        for nutrient in NUTRIENT_COLUMNS
    }


    has_value = {

        nutrient:
            False

        for nutrient in NUTRIENT_COLUMNS
    }


    foods = []


    for item in meal_items:

        if (
            "food_name"
            not in item
        ):

            raise ValueError(
                "Meal item missing "
                "'food_name'."
            )


        if (
            "grams"
            not in item
        ):

            raise ValueError(
                "Meal item missing "
                "'grams'."
            )


        result = calculate_nutrition(

            database,

            item[
                "food_name"
            ],

            item[
                "grams"
            ]
        )


        foods.append(
            result
        )


        for nutrient in (
            NUTRIENT_COLUMNS
        ):

            value = result.get(
                nutrient
            )


            if value is not None:

                totals[
                    nutrient
                ] += float(value)

                has_value[
                    nutrient
                ] = True


    # --------------------------------------------------------
    # Missing != zero.
    #
    # If no food in the meal provides a nutrient,
    # leave the total as None.
    # --------------------------------------------------------

    for nutrient in (
        NUTRIENT_COLUMNS
    ):

        if not has_value[
            nutrient
        ]:

            totals[
                nutrient
            ] = None


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# MEAL PRINTER
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal[
        "foods"
    ]:

        print(
            f"{food['food_name']:25} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name, value in (
        meal["totals"].items()
    ):

        label = labels[
            nutrient_name
        ]

        unit = units[
            nutrient_name
        ]


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} {unit}"
            )


# ============================================================
# EXAMPLE NUTRITION TEST
# ============================================================

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


chicken_test = calculate_nutrition(
    food_database,
    "chicken_breast",
    150
)


print(
    "\nchicken_breast — 150 g"
)


for nutrient_name in (
    NUTRIENT_COLUMNS
):

    value = chicken_test[
        nutrient_name
    ]


    if value is None:

        print(
            f"{nutrient_name:25}: -"
        )

    else:

        print(
            f"{nutrient_name:25}: "
            f"{value:.2f}"
        )


# ============================================================
# EXAMPLE MEAL TEST
# ============================================================

example_meal = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {
        "food_name":
            "oats_rolled",

        "grams":
            50
    },

    {
        "food_name":
            "apple_fuji",

        "grams":
            150
    }
]


meal = calculate_meal(
    food_database,
    example_meal
)


print_meal(
    meal
)


# ============================================================
# SAVE DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("SAVING DATABASE FILES")
print("=" * 70)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


food_database.to_csv(
    FOOD_OUTPUT,
    index=False
)


source_mapping.to_csv(
    SOURCE_MAPPING_OUTPUT,
    index=False
)


if not review_df.empty:

    review_df.to_csv(
        REVIEW_OUTPUT,
        index=False
    )

else:

    pd.DataFrame(
        columns=[
            "food_name",
            "fdc_id"
        ]
    ).to_csv(
        REVIEW_OUTPUT,
        index=False
    )


print("\nDatabase:")
print(FOOD_OUTPUT)

print("\nSource mapping:")
print(SOURCE_MAPPING_OUTPUT)

print("\nReview:")
print(REVIEW_OUTPUT)


# ============================================================
# FINAL DATABASE VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DATABASE VALIDATION")
print("=" * 70)


print(
    f"Foods: "
    f"{len(food_database)}"
)


print(
    f"Unique FDC IDs: "
    f"{food_database['fdc_id'].nunique()}"
)


print(
    "Basis: "
    "100 g for all foods"
)


print(
    "Data type: "
    "Foundation Food"
)


critical_coverage_pass = True


for column in critical_columns:

    if food_database[
        column
    ].isna().any():

        critical_coverage_pass = False


if not critical_coverage_pass:

    print(
        "WARNING: Critical nutrient "
        "coverage is incomplete."
    )

else:

    print(
        "Critical nutrient coverage: PASS"
    )


if (
    len(food_database)
    != food_database[
        "fdc_id"
    ].nunique()
):

    raise RuntimeError(
        "Final database contains "
        "duplicate FDC IDs."
    )


if (
    len(food_database)
    != len(TARGET_FOODS)
):

    raise RuntimeError(
        "Final database does not contain "
        "all target foods."
    )


if not critical_coverage_pass:

    raise RuntimeError(
        "Critical nutrient coverage "
        "validation failed."
    )


print("\n")
print(
    "STATUS: PASS"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

print(
    f"Foundation Foods available: "
    f"{len(foundation_food)}"
)

print(
    f"Foods selected: "
    f"{len(food_database)}"
)

print(
    f"Duplicate groups resolved: "
    f"{len(duplicate_reviews)}"
)

print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)

print(
    "Database validation: PASS"
)

print("\nDatabase:")
print(FOOD_OUTPUT)

print("\nSource mapping:")
print(SOURCE_MAPPING_OUTPUT)

print("\nReview:")
print(REVIEW_OUTPUT)

print("\nDone.")


LOADING USDA FOODDATA CENTRAL FILES

Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


SELECTED USDA FILES

food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv

food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv

nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


READING USDA CSV FILES

food.csv rows:          87,990
food_nutrient.csv rows: 170,469
nutrient.csv rows:      477


USDA DATA TYPES
data_type
sub_sample_food             75055
market_acquisition           7577
sample_food                  4079
agricultural_acquisition      810
foundation_food               469


NORMALIZED DATA TYPES
normalized

RuntimeError: No Foundation Foods were found.

In [24]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD PIPELINE
# ============================================================
#
# Purpose:
#   1. Load USDA FoodData Central CSV files
#   2. Correctly identify Foundation Foods
#   3. Select specific target foods
#   4. Handle duplicate USDA descriptions
#   5. Select nutrient IDs using exact USDA nutrient names
#   6. Prefer duplicate records with better nutrient coverage
#   7. Build a 100 g nutrition database
#   8. Validate nutrient mappings
#   9. Validate calories against macros
#  10. Calculate example meals
#
# IMPORTANT:
#   - Missing nutrient != zero
#   - Exact nutrient-name matching is used
#   - No fuzzy nutrient matching
#   - All database values are normalized to 100 g
#
# ============================================================


import os
import re
import math
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

SEARCH_DIRECTORY = (
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
)


FOOD_CSV = os.path.join(
    SEARCH_DIRECTORY,
    "food.csv"
)


FOOD_NUTRIENT_CSV = os.path.join(
    SEARCH_DIRECTORY,
    "food_nutrient.csv"
)


NUTRIENT_CSV = os.path.join(
    SEARCH_DIRECTORY,
    "nutrient.csv"
)


DATABASE_OUTPUT = os.path.join(
    SEARCH_DIRECTORY,
    "food_database.csv"
)


SOURCE_MAPPING_OUTPUT = os.path.join(
    SEARCH_DIRECTORY,
    "food_source_mapping.csv"
)


REVIEW_OUTPUT = os.path.join(
    SEARCH_DIRECTORY,
    "food_selection_review.csv"
)


# ============================================================
# DISPLAY HELPERS
# ============================================================

def print_header(title):

    print("\n")
    print("=" * 70)
    print(title)
    print("=" * 70)


def print_section(title):

    print("\n")
    print("-" * 70)
    print(title)
    print("-" * 70)


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = value.strip().lower()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def normalize_data_type(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
    )


def normalize_nutrient_name(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = value.strip().lower()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


# ============================================================
# LOAD USDA FILES
# ============================================================

print_header(
    "LOADING USDA FOODDATA CENTRAL FILES"
)

print(
    "Search directory:"
)

print(
    SEARCH_DIRECTORY
)


# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

required_files = {
    "food.csv": FOOD_CSV,
    "food_nutrient.csv": FOOD_NUTRIENT_CSV,
    "nutrient.csv": NUTRIENT_CSV
}


for name, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"Required USDA file not found:\n{path}"
        )


print_section(
    "SELECTED USDA FILES"
)


for name, path in required_files.items():

    print(
        f"{name}:"
    )

    print(
        path
    )


# ============================================================
# READ CSV FILES
# ============================================================

print_section(
    "READING USDA CSV FILES"
)


food = pd.read_csv(
    FOOD_CSV,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_CSV,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_CSV,
    low_memory=False
)


print(
    f"food.csv rows:          {len(food):,}"
)


print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)


print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# REQUIRED COLUMN VALIDATION
# ============================================================

required_food_columns = {
    "fdc_id",
    "data_type",
    "description"
}


required_food_nutrient_columns = {
    "fdc_id",
    "nutrient_id",
    "amount"
}


required_nutrient_columns = {
    "id",
    "name",
    "unit_name"
}


missing = (
    required_food_columns
    - set(food.columns)
)


if missing:

    raise RuntimeError(
        "food.csv is missing required columns: "
        + str(sorted(missing))
    )


missing = (
    required_food_nutrient_columns
    - set(food_nutrient.columns)
)


if missing:

    raise RuntimeError(
        "food_nutrient.csv is missing required columns: "
        + str(sorted(missing))
    )


missing = (
    required_nutrient_columns
    - set(nutrient.columns)
)


if missing:

    raise RuntimeError(
        "nutrient.csv is missing required columns: "
        + str(sorted(missing))
    )


# ============================================================
# CLEAN ID COLUMNS
# ============================================================

food["fdc_id"] = pd.to_numeric(
    food["fdc_id"],
    errors="coerce"
)


food_nutrient["fdc_id"] = pd.to_numeric(
    food_nutrient["fdc_id"],
    errors="coerce"
)


food_nutrient["nutrient_id"] = pd.to_numeric(
    food_nutrient["nutrient_id"],
    errors="coerce"
)


nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce"
)


# Remove invalid IDs

food = food[
    food["fdc_id"].notna()
].copy()


food_nutrient = food_nutrient[
    food_nutrient["fdc_id"].notna()
    &
    food_nutrient["nutrient_id"].notna()
].copy()


nutrient = nutrient[
    nutrient["id"].notna()
].copy()


food["fdc_id"] = (
    food["fdc_id"]
    .astype(int)
)


food_nutrient["fdc_id"] = (
    food_nutrient["fdc_id"]
    .astype(int)
)


food_nutrient["nutrient_id"] = (
    food_nutrient["nutrient_id"]
    .astype(int)
)


nutrient["id"] = (
    nutrient["id"]
    .astype(int)
)


# ============================================================
# USDA DATA TYPES
# ============================================================

print_section(
    "USDA DATA TYPES"
)


print(
    food["data_type"]
    .astype(str)
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# CORRECT FOUNDATION FOOD DETECTION
# ============================================================
#
# IMPORTANT:
#
# Do NOT use:
#
#     normalize_data_type(data_type)
#     == "foundationfood"
#
# because the actual USDA value is:
#
#     foundation_food
#
# We deliberately preserve the underscore.
#
# ============================================================

food["normalized_data_type"] = (
    food["data_type"]
    .apply(normalize_data_type)
)


print_section(
    "NORMALIZED DATA TYPES"
)


print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# Robust Foundation Food filter
# ------------------------------------------------------------

foundation_food = food[
    food["normalized_data_type"]
    == "foundation_food"
].copy()


# ------------------------------------------------------------
# Safety fallback
# ------------------------------------------------------------
#
# If USDA ever changes capitalization or whitespace,
# this second check still identifies the correct records.
# ------------------------------------------------------------

if foundation_food.empty:

    foundation_food = food[
        food["data_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("foundation_food")
    ].copy()


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if foundation_food.empty:

    print_section(
        "FOUNDATION FOOD DETECTION FAILED"
    )

    print(
        "Actual data_type values:"
    )

    print(
        food["data_type"]
        .astype(str)
        .value_counts(
            dropna=False
        )
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found. "
        "Expected USDA data_type='foundation_food'."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

print_section(
    "FOUNDATION FOOD SUMMARY"
)


print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)


print(
    f"Unique Foundation FDC IDs: "
    f"{foundation_food['fdc_id'].nunique():,}"
)


if (
    foundation_food["fdc_id"].nunique()
    != len(foundation_food)
):

    raise RuntimeError(
        "Foundation Food FDC IDs are not unique."
    )


# ============================================================
# NORMALIZE FOOD DESCRIPTIONS
# ============================================================

foundation_food[
    "normalized_description"
] = (
    foundation_food["description"]
    .apply(normalize_text)
)


# ============================================================
# TARGET FOODS
# ============================================================

TARGET_FOODS = {

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# TARGET SEARCH
# ============================================================

print_section(
    "FOUNDATION FOOD TARGET SEARCH"
)


target_candidates = {}


for food_name, config in TARGET_FOODS.items():

    target = normalize_text(
        config["exact_description"]
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target
    ].copy()


    target_candidates[
        food_name
    ] = matches


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)


    print(
        "Requested:"
    )


    print(
        config["exact_description"]
    )


    print(
        f"Matches: {len(matches)}"
    )


    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
            .sort_values("fdc_id")
            .to_string(
                index=False
            )
        )


# ============================================================
# FOUNDATION NUTRIENT COVERAGE
# ============================================================

foundation_ids = set(
    foundation_food["fdc_id"]
    .astype(int)
)


foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_ids)
    ]
    .copy()
)


print_section(
    "FOUNDATION NUTRIENT COVERAGE"
)


print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)


foundation_nutrient_ids = set(
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .dropna()
    .astype(int)
)


print(
    f"Nutrient IDs available in Foundation Foods: "
    f"{len(foundation_nutrient_ids):,}"
)


# ============================================================
# NUTRIENT NORMALIZATION
# ============================================================

nutrient[
    "normalized_name"
] = (
    nutrient["name"]
    .apply(normalize_nutrient_name)
)


# ============================================================
# NUTRIENT COVERAGE FUNCTION
# ============================================================

def nutrient_coverage(nutrient_id):

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "nutrient_id"
        ]
        == int(nutrient_id)
    ]


    if rows.empty:

        return 0


    return int(
        rows["amount"]
        .notna()
        .sum()
    )


# ============================================================
# EXACT NUTRIENT FINDER
# ============================================================

def find_exact_nutrient(
    exact_name,
    allowed_units=None
):

    target = normalize_nutrient_name(
        exact_name
    )


    candidates = nutrient[
        nutrient["normalized_name"]
        == target
    ].copy()


    if allowed_units is not None:

        allowed_units = {
            str(unit).strip().upper()
            for unit in allowed_units
        }


        candidates = candidates[
            candidates["unit_name"]
            .astype(str)
            .str.strip()
            .str.upper()
            .isin(allowed_units)
        ]


    if candidates.empty:

        return None


    candidates["coverage"] = (
        candidates["id"]
        .apply(nutrient_coverage)
    )


    candidates = candidates.sort_values(
        [
            "coverage",
            "id"
        ],
        ascending=[
            False,
            True
        ]
    )


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row["unit_name"],

        "coverage":
            int(row["coverage"])
    }


# ============================================================
# ENERGY NUTRIENT SELECTION
# ============================================================

print_section(
    "ENERGY NUTRIENTS ACTUALLY AVAILABLE"
)


ENERGY_NAMES = {
    "energy",
    "energy (atwater general factors)",
    "energy (atwater specific factors)"
}


energy_candidates = nutrient[
    nutrient["normalized_name"]
    .isin(ENERGY_NAMES)
].copy()


energy_candidates = energy_candidates[
    energy_candidates["unit_name"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "KCAL"
]


energy_candidates["coverage"] = (
    energy_candidates["id"]
    .apply(nutrient_coverage)
)


ENERGY_PRIORITY = {

    2047: 3,

    2048: 2,

    1008: 1
}


energy_candidates["priority"] = (
    energy_candidates["id"]
    .map(ENERGY_PRIORITY)
    .fillna(0)
)


if energy_candidates.empty:

    raise RuntimeError(
        "No KCAL energy nutrient exists "
        "in Foundation Foods."
    )


print(
    energy_candidates[
        [
            "id",
            "name",
            "unit_name",
            "coverage"
        ]
    ]
    .sort_values(
        [
            "coverage",
            "priority"
        ],
        ascending=[
            False,
            False
        ]
    )
    .to_string(
        index=False
    )
)


energy_candidates = (
    energy_candidates
    .sort_values(
        [
            "coverage",
            "priority"
        ],
        ascending=[
            False,
            False
        ]
    )
)


energy_row = (
    energy_candidates.iloc[0]
)


nutrient_lookup = {}


nutrient_lookup[
    "calories_kcal"
] = {

    "id":
        int(energy_row["id"]),

    "name":
        energy_row["name"],

    "unit":
        energy_row["unit_name"],

    "coverage":
        int(energy_row["coverage"])
}


# ============================================================
# EXACT MACRO/MICRO NUTRIENTS
# ============================================================

EXACT_NUTRIENTS = {

    "protein_g": (
        "Protein",
        {"G"}
    ),

    "fat_g": (
        "Total lipid (fat)",
        {"G"}
    ),

    "carbohydrate_g": (
        "Carbohydrate, by difference",
        {"G"}
    ),

    "fiber_g": (
        "Fiber, total dietary",
        {"G"}
    ),

    "sugar_g": (
        "Sugars, Total",
        {"G"}
    ),

    "saturated_fat_g": (
        "Fatty acids, total saturated",
        {"G"}
    )
}


for output_name, (
    exact_name,
    allowed_units
) in EXACT_NUTRIENTS.items():

    result = find_exact_nutrient(
        exact_name,
        allowed_units
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}: {exact_name}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


# ============================================================
# FINAL NUTRIENT MAP
# ============================================================

print_section(
    "FINAL FOUNDATION NUTRIENT MAP"
)


for output_name, info in nutrient_lookup.items():

    print(
        f"{output_name:25} "
        f"ID={info['id']:5} "
        f"UNIT={info['unit']} "
        f"COVERAGE={info['coverage']:4} "
        f"NAME={info['name']}"
    )


# ============================================================
# NUTRIENT MAPPING SAFETY VALIDATION
# ============================================================

EXPECTED_NUTRIENT_NAMES = {

    "calories_kcal": {
        "energy",
        "energy (atwater general factors)",
        "energy (atwater specific factors)"
    },

    "protein_g": {
        "protein"
    },

    "fat_g": {
        "total lipid (fat)"
    },

    "carbohydrate_g": {
        "carbohydrate, by difference"
    },

    "fiber_g": {
        "fiber, total dietary"
    },

    "sugar_g": {
        "sugars, total"
    },

    "saturated_fat_g": {
        "fatty acids, total saturated"
    }
}


for output_name, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    if output_name not in nutrient_lookup:

        raise RuntimeError(
            f"CRITICAL: Missing nutrient mapping: "
            f"{output_name}"
        )


    actual_name = normalize_nutrient_name(
        nutrient_lookup[
            output_name
        ]["name"]
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"CRITICAL NUTRIENT MAPPING ERROR: "
            f"{output_name} mapped to "
            f"'{nutrient_lookup[output_name]['name']}'"
        )


print("\n")
print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# NUTRIENT LOOKUP TABLE
# ============================================================

nutrient_ids_needed = {
    info["id"]
    for info in nutrient_lookup.values()
}


# ============================================================
# BUILD NUTRIENT MATRIX
# ============================================================

nutrient_rows = foundation_food_nutrients[
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .isin(nutrient_ids_needed)
].copy()


nutrient_rows = nutrient_rows[
    [
        "fdc_id",
        "nutrient_id",
        "amount"
    ]
]


# ============================================================
# GET FOOD NUTRIENT VALUE
# ============================================================

def get_food_nutrient_value(
    fdc_id,
    output_name
):

    nutrient_id = nutrient_lookup[
        output_name
    ]["id"]


    rows = nutrient_rows[
        (
            nutrient_rows["fdc_id"]
            == int(fdc_id)
        )
        &
        (
            nutrient_rows["nutrient_id"]
            == int(nutrient_id)
        )
    ]


    if rows.empty:

        return np.nan


    values = pd.to_numeric(
        rows["amount"],
        errors="coerce"
    ).dropna()


    if values.empty:

        return np.nan


    # Normally one record exists.
    # If multiple exist, use the first valid value.
    return float(
        values.iloc[0]
    )


# ============================================================
# DUPLICATE PROFILE COMPARISON
# ============================================================

NUTRIENT_OUTPUT_NAMES = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


def build_food_profile(fdc_id):

    result = {
        "fdc_id": int(fdc_id)
    }


    for nutrient_name in NUTRIENT_OUTPUT_NAMES:

        result[nutrient_name] = (
            get_food_nutrient_value(
                fdc_id,
                nutrient_name
            )
        )


    return result


# ============================================================
# DUPLICATE RECORD SELECTION
# ============================================================
#
# We do NOT simply choose the first USDA record.
#
# Preference:
#
#   1. Has calories
#   2. Has protein
#   3. Has fat
#   4. Has carbohydrate
#   5. Has fiber
#   6. Has sugar
#   7. Has saturated fat
#   8. Higher overall critical nutrient coverage
#   9. Lower FDC ID as final deterministic tie-breaker
#
# Missing values remain missing.
#
# ============================================================

CRITICAL_NUTRIENTS = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g"
]


OPTIONAL_NUTRIENTS = [

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


def duplicate_score(profile):

    score = 0


    # Critical nutrients are weighted more heavily.

    weights = {

        "calories_kcal": 100,

        "protein_g": 50,

        "fat_g": 50,

        "carbohydrate_g": 50,

        "fiber_g": 10,

        "sugar_g": 10,

        "saturated_fat_g": 10
    }


    for nutrient_name, weight in weights.items():

        value = profile.get(
            nutrient_name
        )


        if pd.notna(value):

            score += weight


    return score


selected_foods = {}

duplicate_reviews = []

not_found_foods = []


print_section(
    "SELECTING TARGET FOUNDATION FOODS"
)


for food_name, config in TARGET_FOODS.items():

    matches = target_candidates[
        food_name
    ].copy()


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)


    print(
        "Requested:"
    )


    print(
        config["exact_description"]
    )


    if matches.empty:

        print(
            "STATUS: NOT_FOUND"
        )


        not_found_foods.append(
            food_name
        )


        continue


    # --------------------------------------------------------
    # Single exact match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]


        selected_foods[
            food_name
        ] = {

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "data_type":
                row["data_type"]
        }


        print(
            "STATUS: SELECTED"
        )


        print(
            f"FDC ID: {int(row['fdc_id'])}"
        )


        print(
            f"Description: {row['description']}"
        )


        continue


    # --------------------------------------------------------
    # Duplicate descriptions
    # --------------------------------------------------------

    print(
        "STATUS: DUPLICATE_DESCRIPTION"
    )


    profiles = []


    for fdc_id in (
        matches["fdc_id"]
        .astype(int)
        .tolist()
    ):

        profile = build_food_profile(
            fdc_id
        )


        profile["score"] = (
            duplicate_score(
                profile
            )
        )


        profiles.append(
            profile
        )


    comparison = pd.DataFrame(
        profiles
    )


    display_columns = [
        "fdc_id"
    ] + NUTRIENT_OUTPUT_NAMES + [
        "score"
    ]


    print(
        comparison[
            display_columns
        ]
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Sort by nutrient completeness
    # --------------------------------------------------------

    comparison = comparison.sort_values(
        [
            "score",
            "fdc_id"
        ],
        ascending=[
            False,
            True
        ]
    )


    chosen_fdc_id = int(
        comparison.iloc[0]["fdc_id"]
    )


    chosen_row = matches[
        matches["fdc_id"]
        == chosen_fdc_id
    ].iloc[0]


    selected_foods[
        food_name
    ] = {

        "fdc_id":
            chosen_fdc_id,

        "description":
            chosen_row["description"],

        "data_type":
            chosen_row["data_type"]
    }


    # --------------------------------------------------------
    # Review information
    # --------------------------------------------------------

    duplicate_reviews.append({

        "food_name":
            food_name,

        "description":
            config["exact_description"],

        "candidate_count":
            len(matches),

        "selected_fdc_id":
            chosen_fdc_id,

        "candidate_fdc_ids":
            ",".join(
                str(x)
                for x in matches[
                    "fdc_id"
                ]
                .astype(int)
                .tolist()
            ),

        "selection_reason":
            "Preferred record with highest "
            "critical nutrient completeness"
    })


    print(
        f"RESULT: Selected FDC ID "
        f"{chosen_fdc_id}"
    )


# ============================================================
# FINAL SELECTION SUMMARY
# ============================================================

print_section(
    "FINAL FOOD SELECTION SUMMARY"
)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)


print(
    f"Foods selected: "
    f"{len(selected_foods)}"
)


print(
    f"Foods requiring duplicate review: "
    f"{len(duplicate_reviews)}"
)


print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)


if not_found_foods:

    print(
        "\nNOT FOUND:"
    )

    for name in not_found_foods:

        print(
            f"  - {name}"
        )


print(
    "\nSelected foods:"
)


for name, data in selected_foods.items():

    print(
        f"{name:25} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print_section(
    "BUILDING FINAL FOOD DATABASE"
)


food_database_rows = []


for food_name, selected in selected_foods.items():

    fdc_id = int(
        selected["fdc_id"]
    )


    source_rows = foundation_food[
        foundation_food["fdc_id"]
        == fdc_id
    ]


    if source_rows.empty:

        raise RuntimeError(
            f"Selected FDC ID {fdc_id} "
            f"was not found in Foundation Foods."
        )


    source = source_rows.iloc[0]


    row = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            source["description"],

        "data_type":
            "Foundation Food",

        "basis_g":
            100.0
    }


    for nutrient_name in NUTRIENT_OUTPUT_NAMES:

        value = get_food_nutrient_value(
            fdc_id,
            nutrient_name
        )


        row[nutrient_name] = value


    food_database_rows.append(
        row
    )


food_database = pd.DataFrame(
    food_database_rows
)


# ============================================================
# ORDER COLUMNS
# ============================================================

DATABASE_COLUMNS = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


food_database = food_database[
    DATABASE_COLUMNS
]


# ============================================================
# PRINT DATABASE
# ============================================================

print_section(
    "FINAL FOOD DATABASE"
)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# SAVE SOURCE MAPPING
# ============================================================

source_mapping_rows = []


for food_name, selected in selected_foods.items():

    fdc_id = int(
        selected["fdc_id"]
    )


    source_mapping_rows.append({

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            selected["description"],

        "source":
            "USDA FoodData Central Foundation Foods",

        "is_preferred":
            True
    })


    # Add all alternate duplicate records

    matches = target_candidates[
        food_name
    ]


    for alternate_id in (
        matches["fdc_id"]
        .astype(int)
        .tolist()
    ):

        if alternate_id == fdc_id:

            continue


        alternate_rows = foundation_food[
            foundation_food["fdc_id"]
            == alternate_id
        ]


        if alternate_rows.empty:

            continue


        alternate = alternate_rows.iloc[0]


        source_mapping_rows.append({

            "food_name":
                food_name,

            "fdc_id":
                alternate_id,

            "description":
                alternate["description"],

            "source":
                "USDA FoodData Central Foundation Foods",

            "is_preferred":
                False
        })


source_mapping = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# SAVE REVIEW FILE
# ============================================================

review = pd.DataFrame(
    duplicate_reviews
)


# ============================================================
# DATABASE VALIDATION
# ============================================================

print_section(
    "NUTRIENT DATABASE SANITY CHECK"
)


expected_names = {

    "calories_kcal": [
        "Energy",
        "Energy (Atwater General Factors)",
        "Energy (Atwater Specific Factors)"
    ],

    "protein_g": [
        "Protein"
    ],

    "fat_g": [
        "Total lipid (fat)"
    ],

    "carbohydrate_g": [
        "Carbohydrate, by difference"
    ],

    "fiber_g": [
        "Fiber, total dietary"
    ],

    "sugar_g": [
        "Sugars, Total"
    ],

    "saturated_fat_g": [
        "Fatty acids, total saturated"
    ]
}


for column, valid_names in expected_names.items():

    actual = nutrient_lookup[
        column
    ]["name"]


    if actual not in valid_names:

        raise RuntimeError(
            f"INVALID MAPPING: "
            f"{column} -> {actual}"
        )


    print(
        f"PASS: {column:25} "
        f"-> {actual}"
    )


# ------------------------------------------------------------
# Check database uniqueness
# ------------------------------------------------------------

if (
    food_database["food_name"]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate application food names found."
    )


if (
    food_database["fdc_id"]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate preferred FDC IDs found."
    )


# ------------------------------------------------------------
# Check basis
# ------------------------------------------------------------

if not (
    food_database["basis_g"]
    == 100
).all():

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


# ------------------------------------------------------------
# Check data type
# ------------------------------------------------------------

if not (
    food_database["data_type"]
    == "Foundation Food"
).all():

    raise RuntimeError(
        "Database contains non-Foundation foods."
    )


# ============================================================
# CHECK NEGATIVE NUTRIENTS
# ============================================================

numeric_nutrients = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


negative_values = []


for column in numeric_nutrients:

    bad_rows = food_database[
        food_database[column]
        .notna()
        &
        (
            food_database[column]
            < 0
        )
    ]


    if not bad_rows.empty:

        for _, row in bad_rows.iterrows():

            negative_values.append({

                "food_name":
                    row["food_name"],

                "nutrient":
                    column,

                "value":
                    row[column]
            })


if negative_values:

    print(
        "\nWARNING: Negative nutrient values detected:"
    )


    print(
        pd.DataFrame(
            negative_values
        )
        .to_string(
            index=False
        )
    )


    raise RuntimeError(
        "Negative nutrient values found."
    )


print(
    "\nNo negative nutrient values detected."
)


# ============================================================
# CRITICAL NUTRIENT COVERAGE
# ============================================================

print_section(
    "CRITICAL NUTRIENT COVERAGE"
)


critical_coverage = {

    column:
        int(
            food_database[column]
            .notna()
            .sum()
        )

    for column in [
        "calories_kcal",
        "protein_g",
        "fat_g",
        "carbohydrate_g"
    ]
}


for column, count in critical_coverage.items():

    print(
        f"Foods with {column}: "
        f"{count}/{len(food_database)}"
    )


# ------------------------------------------------------------
# Protein/fat/carbohydrate must exist for every target food.
# Calories should also exist because this is a macro database.
# ------------------------------------------------------------

for column in [
    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "calories_kcal"
]:

    if (
        food_database[column]
        .isna()
        .any()
    ):

        missing_foods = (
            food_database[
                food_database[column]
                .isna()
            ]["food_name"]
            .tolist()
        )


        raise RuntimeError(
            f"Critical nutrient '{column}' "
            f"is missing for: "
            f"{missing_foods}"
        )


print(
    "\nCritical nutrient coverage: PASS"
)


# ============================================================
# CALORIE SANITY CHECK
# ============================================================
#
# This is a VALIDATION check only.
#
# USDA Energy remains the primary calorie value.
#
# We do NOT replace USDA calories with macro calories.
#
# ============================================================

print_section(
    "CALORIE SANITY CHECK"
)


calorie_checks = []


for _, row in food_database.iterrows():

    protein = row["protein_g"]

    fat = row["fat_g"]

    carbs = row["carbohydrate_g"]

    calories = row["calories_kcal"]


    if (
        pd.isna(protein)
        or
        pd.isna(fat)
        or
        pd.isna(carbs)
        or
        pd.isna(calories)
    ):

        macro_calories = np.nan

        difference = np.nan

    else:

        macro_calories = (
            protein * 4
            +
            carbs * 4
            +
            fat * 9
        )


        difference = (
            calories
            - macro_calories
        )


    calorie_checks.append({

        "food_name":
            row["food_name"],

        "calories_kcal":
            calories,

        "macro_calorie_check":
            macro_calories,

        "macro_calorie_difference":
            difference
    })


calorie_check_df = pd.DataFrame(
    calorie_checks
)


print(
    calorie_check_df.to_string(
        index=False
    )
)


# ============================================================
# DUPLICATE REVIEW SUMMARY
# ============================================================

if duplicate_reviews:

    print_section(
        "DUPLICATE USDA RECORD REVIEW"
    )


    for review_row in duplicate_reviews:

        print(
            f"{review_row['food_name']}: "
            f"selected FDC "
            f"{review_row['selected_fdc_id']} "
            f"because it has the best "
            f"critical nutrient completeness."
        )


# ============================================================
# SAVE DATABASE FILES
# ============================================================

print_section(
    "SAVING DATABASE FILES"
)


food_database.to_csv(
    DATABASE_OUTPUT,
    index=False
)


source_mapping.to_csv(
    SOURCE_MAPPING_OUTPUT,
    index=False
)


review.to_csv(
    REVIEW_OUTPUT,
    index=False
)


print(
    "Database:"
)


print(
    DATABASE_OUTPUT
)


print(
    "\nSource mapping:"
)


print(
    SOURCE_MAPPING_OUTPUT
)


print(
    "\nReview:"
)


print(
    REVIEW_OUTPUT
)


# ============================================================
# NUTRITION CALCULATION
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):

    if grams is None:

        raise ValueError(
            "grams cannot be None."
        )


    grams = float(
        grams
    )


    if grams <= 0:

        raise ValueError(
            "grams must be greater than zero."
        )


    matches = database[
        database["food_name"]
        == food_name
    ]


    if matches.empty:

        raise KeyError(
            f"Food not found: {food_name}"
        )


    row = matches.iloc[0]


    multiplier = (
        grams
        /
        float(row["basis_g"])
    )


    result = {

        "food_name":
            food_name,

        "grams":
            grams
    }


    for nutrient_name in NUTRIENT_OUTPUT_NAMES:

        value = row[
            nutrient_name
        ]


        if pd.isna(value):

            result[
                nutrient_name
            ] = None

        else:

            result[
                nutrient_name
            ] = float(value) * multiplier


    return result


# ============================================================
# MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):

    totals = {

        nutrient_name:
            0.0

        for nutrient_name
        in NUTRIENT_OUTPUT_NAMES
    }


    has_value = {

        nutrient_name:
            False

        for nutrient_name
        in NUTRIENT_OUTPUT_NAMES
    }


    foods = []


    for item in meal_items:

        if "food_name" not in item:

            raise ValueError(
                "Meal item missing food_name."
            )


        if "grams" not in item:

            raise ValueError(
                "Meal item missing grams."
            )


        result = calculate_nutrition(

            database,

            item["food_name"],

            item["grams"]
        )


        foods.append(
            result
        )


        for nutrient_name in (
            NUTRIENT_OUTPUT_NAMES
        ):

            value = result.get(
                nutrient_name
            )


            if value is not None:

                totals[
                    nutrient_name
                ] += float(value)


                has_value[
                    nutrient_name
                ] = True


    # --------------------------------------------------------
    # Missing != zero
    #
    # If no food in the meal provides a nutrient,
    # leave the meal nutrient as None.
    # --------------------------------------------------------

    for nutrient_name in (
        NUTRIENT_OUTPUT_NAMES
    ):

        if not has_value[
            nutrient_name
        ]:

            totals[
                nutrient_name
            ] = None


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# MEAL PRINTER
# ============================================================

def print_meal(
    meal
):

    print_section(
        "MEAL"
    )


    for food in meal["foods"]:

        print(
            f"{food['food_name']:25} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print(
        "TOTAL NUTRITION"
    )
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name in (
        NUTRIENT_OUTPUT_NAMES
    ):

        value = meal[
            "totals"
        ][nutrient_name]


        label = labels[
            nutrient_name
        ]


        unit = units[
            nutrient_name
        ]


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} {unit}"
            )


# ============================================================
# EXAMPLE NUTRITION TEST
# ============================================================

print_section(
    "EXAMPLE NUTRITION TEST"
)


chicken_test = calculate_nutrition(

    food_database,

    "chicken_breast",

    150
)


print(
    "\nchicken_breast — 150 g"
)


for nutrient_name in NUTRIENT_OUTPUT_NAMES:

    value = chicken_test[
        nutrient_name
    ]


    if value is None:

        print(
            f"{nutrient_name:25}: -"
        )

    else:

        unit = (
            "kcal"
            if nutrient_name
            == "calories_kcal"
            else "g"
        )


        print(
            f"{nutrient_name:25}: "
            f"{value:.2f} {unit}"
        )


# ============================================================
# EXAMPLE MEAL
# ============================================================

example_meal_items = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {
        "food_name":
            "oats_rolled",

        "grams":
            50
    },

    {
        "food_name":
            "apple_fuji",

        "grams":
            150
    }
]


example_meal = calculate_meal(

    food_database,

    example_meal_items
)


print_meal(
    example_meal
)


# ============================================================
# FINAL DATABASE VALIDATION
# ============================================================

print_section(
    "DATABASE VALIDATION"
)


print(
    f"Foods: "
    f"{len(food_database)}"
)


print(
    f"Unique FDC IDs: "
    f"{food_database['fdc_id'].nunique()}"
)


print(
    "Basis: 100 g for all foods"
)


print(
    "Data type: Foundation Food"
)


print(
    "Critical nutrient coverage: PASS"
)


print(
    "\nSTATUS: PASS"
)


# ============================================================
# PIPELINE COMPLETE
# ============================================================

print_section(
    "PIPELINE COMPLETE"
)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food)}"
)


print(
    f"Foods selected: "
    f"{len(selected_foods)}"
)


print(
    f"Foods requiring review: "
    f"{len(duplicate_reviews)}"
)


print(
    f"Foods not found: "
    f"{len(not_found_foods)}"
)


print(
    "\nDatabase validation: PASS"
)


print(
    "\nDatabase:"
)


print(
    DATABASE_OUTPUT
)


print(
    "\nSource mapping:"
)


print(
    SOURCE_MAPPING_OUTPUT
)


print(
    "\nReview:"
)


print(
    REVIEW_OUTPUT
)


print(
    "\nDone."
)




LOADING USDA FOODDATA CENTRAL FILES
Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


----------------------------------------------------------------------
SELECTED USDA FILES
----------------------------------------------------------------------
food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


----------------------------------------------------------------------
READING USDA CSV FILES
----------------------------------------------------------------------
food.csv rows:          87,990
food_nutrie

KeyError: 'priority'

In [25]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD DATABASE BUILDER
# ============================================================
#
# This script:
#
#   1. Loads USDA FoodData Central CSV files
#   2. Detects Foundation Foods correctly
#   3. Matches exact target food descriptions
#   4. Handles duplicate USDA descriptions
#   5. Selects the best duplicate using nutrient completeness
#   6. Uses exact USDA nutrient names
#   7. Selects Energy only from actual Foundation nutrients
#   8. Builds a 100 g nutrition database
#   9. Validates nutrient mappings
#  10. Validates calorie/macro consistency
#  11. Saves the final database
#  12. Provides food-weight and meal calculations
#
# IMPORTANT:
# Do NOT fuzzy-match nutrient names.
# ============================================================


import os
import re
import math
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = (
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
)


FOOD_FILE = os.path.join(
    BASE_DIR,
    "food.csv"
)

FOOD_NUTRIENT_FILE = os.path.join(
    BASE_DIR,
    "food_nutrient.csv"
)

NUTRIENT_FILE = os.path.join(
    BASE_DIR,
    "nutrient.csv"
)


DATABASE_FILE = os.path.join(
    BASE_DIR,
    "food_database.csv"
)

SOURCE_MAPPING_FILE = os.path.join(
    BASE_DIR,
    "food_source_mapping.csv"
)

REVIEW_FILE = os.path.join(
    BASE_DIR,
    "food_selection_review.csv"
)


# ============================================================
# TARGET FOODS
# ============================================================

TARGET_FOODS = {

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    }
}


# ============================================================
# REQUIRED NUTRIENTS
# ============================================================

OUTPUT_NUTRIENTS = [

    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g"
]


# ============================================================
# EXACT USDA NUTRIENT DEFINITIONS
# ============================================================
#
# Never use a fuzzy nutrient matcher here.
# ============================================================

EXACT_NUTRIENTS = {

    "protein_g": {
        "name": "Protein",
        "unit": "G"
    },

    "fat_g": {
        "name": "Total lipid (fat)",
        "unit": "G"
    },

    "carbohydrate_g": {
        "name": "Carbohydrate, by difference",
        "unit": "G"
    },

    "fiber_g": {
        "name": "Fiber, total dietary",
        "unit": "G"
    },

    "sugar_g": {
        "name": "Sugars, Total",
        "unit": "G"
    },

    "saturated_fat_g": {
        "name": "Fatty acids, total saturated",
        "unit": "G"
    }
}


# ============================================================
# ENERGY PRIORITY
# ============================================================
#
# 2047 = Energy (Atwater General Factors)
# 2048 = Energy (Atwater Specific Factors)
# 1008 = Energy
#
# We prefer 2047 when coverage is equal.
# ============================================================

ENERGY_PRIORITY = {

    2047: 3,
    2048: 2,
    1008: 1
}


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def normalize_text(value):
    """
    Normalize general USDA text for exact comparison.
    """

    if pd.isna(value):
        return ""

    value = str(value)

    value = value.strip().lower()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def normalize_data_type(value):
    """
    Normalize USDA data_type values.

    Example:
        foundation_food
        Foundation Food
        FOUNDATION FOOD

    all become:
        foundationfood
    """

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    value = re.sub(
        r"[\s_\-]+",
        "",
        value
    )

    return value


def safe_int(value):
    """
    Safely convert a value to int.
    """

    if pd.isna(value):
        return None

    try:
        return int(float(value))
    except Exception:
        return None


# ============================================================
# START
# ============================================================

print("\n")
print("=" * 70)
print("LOADING USDA FOODDATA CENTRAL FILES")
print("=" * 70)

print("\nSearch directory:")
print(BASE_DIR)


# ============================================================
# VERIFY FILES
# ============================================================

required_files = {

    "food.csv": FOOD_FILE,
    "food_nutrient.csv": FOOD_NUTRIENT_FILE,
    "nutrient.csv": NUTRIENT_FILE
}


missing_files = []

for name, path in required_files.items():

    if not os.path.isfile(path):

        missing_files.append(path)


if missing_files:

    print("\nMissing files:")

    for path in missing_files:
        print(path)

    raise FileNotFoundError(
        "One or more USDA CSV files could not be found."
    )


print("\n")
print("-" * 70)
print("SELECTED USDA FILES")
print("-" * 70)

print("food.csv:")
print(FOOD_FILE)

print("food_nutrient.csv:")
print(FOOD_NUTRIENT_FILE)

print("nutrient.csv:")
print(NUTRIENT_FILE)


# ============================================================
# READ CSV FILES
# ============================================================

print("\n")
print("-" * 70)
print("READING USDA CSV FILES")
print("-" * 70)


food = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_FILE,
    low_memory=False
)


nutrient = pd.read_csv(
    NUTRIENT_FILE,
    low_memory=False
)


print(
    f"food.csv rows:          {len(food):,}"
)

print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# REQUIRED COLUMN CHECK
# ============================================================

required_food_columns = {
    "fdc_id",
    "description",
    "data_type"
}


required_food_nutrient_columns = {
    "fdc_id",
    "nutrient_id",
    "amount"
}


required_nutrient_columns = {
    "id",
    "name",
    "unit_name"
}


if not required_food_columns.issubset(
    food.columns
):

    raise RuntimeError(
        "food.csv is missing required columns: "
        + str(
            required_food_columns
            - set(food.columns)
        )
    )


if not required_food_nutrient_columns.issubset(
    food_nutrient.columns
):

    raise RuntimeError(
        "food_nutrient.csv is missing required columns: "
        + str(
            required_food_nutrient_columns
            - set(food_nutrient.columns)
        )
    )


if not required_nutrient_columns.issubset(
    nutrient.columns
):

    raise RuntimeError(
        "nutrient.csv is missing required columns: "
        + str(
            required_nutrient_columns
            - set(nutrient.columns)
        )
    )


# ============================================================
# NORMALIZE IDS
# ============================================================

food["fdc_id"] = pd.to_numeric(
    food["fdc_id"],
    errors="coerce"
)


food_nutrient["fdc_id"] = pd.to_numeric(
    food_nutrient["fdc_id"],
    errors="coerce"
)


food_nutrient["nutrient_id"] = pd.to_numeric(
    food_nutrient["nutrient_id"],
    errors="coerce"
)


nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce"
)


food = food.dropna(
    subset=["fdc_id"]
).copy()


food_nutrient = food_nutrient.dropna(
    subset=[
        "fdc_id",
        "nutrient_id"
    ]
).copy()


nutrient = nutrient.dropna(
    subset=["id"]
).copy()


food["fdc_id"] = (
    food["fdc_id"]
    .astype(int)
)


food_nutrient["fdc_id"] = (
    food_nutrient["fdc_id"]
    .astype(int)
)


food_nutrient["nutrient_id"] = (
    food_nutrient["nutrient_id"]
    .astype(int)
)


nutrient["id"] = (
    nutrient["id"]
    .astype(int)
)


# ============================================================
# USDA DATA TYPES
# ============================================================

print("\n")
print("=" * 70)
print("USDA DATA TYPES")
print("=" * 70)

print(
    food["data_type"]
    .astype(str)
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# NORMALIZE DATA TYPES
# ============================================================

food["normalized_data_type"] = (
    food["data_type"]
    .apply(normalize_data_type)
)


print("\n")
print("=" * 70)
print("NORMALIZED DATA TYPES")
print("=" * 70)

print(
    food["normalized_data_type"]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# FIND FOUNDATION FOODS
# ============================================================
#
# IMPORTANT:
#
# normalized_data_type is:
#
#     foundationfood
#
# NOT:
#
#     foundation food
#
# ============================================================

foundation_food = food[
    food["normalized_data_type"]
    == "foundationfood"
].copy()


if foundation_food.empty:

    print("\n")
    print("=" * 70)
    print("FOUNDATION FOOD DETECTION FAILED")
    print("=" * 70)

    print(
        food["normalized_data_type"]
        .value_counts(
            dropna=False
        )
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

foundation_food[
    "normalized_description"
] = (
    foundation_food["description"]
    .apply(normalize_text)
)


print("\n")
print("=" * 70)
print("FOUNDATION FOOD SUMMARY")
print("=" * 70)

print(
    "Foundation Foods found:",
    len(foundation_food)
)

print(
    "Unique Foundation FDC IDs:",
    foundation_food["fdc_id"]
    .nunique()
)


if (
    foundation_food["fdc_id"]
    .nunique()
    != len(foundation_food)
):

    raise RuntimeError(
        "Foundation Foods contain duplicate FDC IDs."
    )


# ============================================================
# TARGET FOOD SEARCH
# ============================================================

print("\n")
print("=" * 70)
print("FOUNDATION FOOD TARGET SEARCH")
print("=" * 70)


target_matches = {}


for food_name, config in TARGET_FOODS.items():

    requested_description = (
        config["exact_description"]
    )

    target = normalize_text(
        requested_description
    )

    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == target
    ].copy()


    target_matches[food_name] = matches


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)

    print(
        "Requested:"
    )

    print(
        requested_description
    )

    print(
        "Matches:",
        len(matches)
    )


    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description"
                ]
            ]
            .sort_values(
                "fdc_id"
            )
            .to_string(
                index=False
            )
        )

    else:

        print(
            "NO MATCH FOUND"
        )


# ============================================================
# FOUNDATION NUTRIENT COVERAGE
# ============================================================

foundation_ids = set(
    foundation_food["fdc_id"]
    .astype(int)
)


foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_ids)
    ]
    .copy()
)


print("\n")
print("=" * 70)
print("FOUNDATION NUTRIENT COVERAGE")
print("=" * 70)

print(
    "Foundation nutrient rows:",
    f"{len(foundation_food_nutrients):,}"
)


foundation_nutrient_ids = set(
    foundation_food_nutrients[
        "nutrient_id"
    ]
    .dropna()
    .astype(int)
)


print(
    "Nutrient IDs available in Foundation Foods:",
    len(foundation_nutrient_ids)
)


# ============================================================
# NORMALIZE NUTRIENT NAMES
# ============================================================

nutrient[
    "normalized_name"
] = (
    nutrient["name"]
    .apply(normalize_text)
)


# ============================================================
# NUTRIENT COVERAGE FUNCTION
# ============================================================

def nutrient_coverage(
    nutrient_id
):
    """
    Number of Foundation Food records
    that contain a non-missing value for
    this nutrient.
    """

    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "nutrient_id"
        ]
        == int(nutrient_id)
    ]


    if rows.empty:

        return 0


    return int(
        pd.to_numeric(
            rows["amount"],
            errors="coerce"
        )
        .notna()
        .sum()
    )


# ============================================================
# ENERGY NUTRIENTS
# ============================================================

print("\n")
print("=" * 70)
print("ENERGY NUTRIENTS ACTUALLY AVAILABLE")
print("=" * 70)


energy_names = {

    "energy",

    "energy (atwater general factors)",

    "energy (atwater specific factors)"
}


energy_candidates = nutrient[
    nutrient["normalized_name"]
    .isin(energy_names)
].copy()


energy_candidates = energy_candidates[
    energy_candidates["unit_name"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "KCAL"
].copy()


# ------------------------------------------------------------
# GUARANTEE COVERAGE COLUMN EXISTS
# ------------------------------------------------------------

if not energy_candidates.empty:

    energy_candidates[
        "coverage"
    ] = (
        energy_candidates["id"]
        .apply(nutrient_coverage)
    )

else:

    energy_candidates[
        "coverage"
    ] = pd.Series(
        dtype="int64"
    )


# ------------------------------------------------------------
# GUARANTEE PRIORITY COLUMN EXISTS
# ------------------------------------------------------------

energy_candidates[
    "priority"
] = (
    energy_candidates["id"]
    .map(ENERGY_PRIORITY)
    .fillna(0)
    .astype(int)
)


if energy_candidates.empty:

    raise RuntimeError(
        "No KCAL energy nutrient exists "
        "in the Foundation Foods dataset."
    )


print(
    energy_candidates[
        [
            "id",
            "name",
            "unit_name",
            "coverage",
            "priority"
        ]
    ]
    .sort_values(
        [
            "coverage",
            "priority",
            "id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .to_string(
        index=False
    )
)


# ============================================================
# SELECT ENERGY
# ============================================================
#
# Highest Foundation coverage wins.
#
# If coverage ties:
#     2047 > 2048 > 1008
# ============================================================

energy_candidates = (
    energy_candidates
    .sort_values(
        [
            "coverage",
            "priority",
            "id"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .copy()
)


energy_row = (
    energy_candidates
    .iloc[0]
)


nutrient_lookup = {}


nutrient_lookup[
    "calories_kcal"
] = {

    "id":
        int(energy_row["id"]),

    "name":
        energy_row["name"],

    "unit":
        energy_row["unit_name"],

    "coverage":
        int(energy_row["coverage"])
}


# ============================================================
# EXACT NUTRIENT FINDER
# ============================================================

def find_exact_nutrient(
    exact_name,
    allowed_units=None
):
    """
    Find a nutrient by EXACT normalized USDA name.

    No fuzzy matching.
    """

    target = normalize_text(
        exact_name
    )


    candidates = nutrient[
        nutrient["normalized_name"]
        == target
    ].copy()


    if allowed_units is not None:

        allowed_units = {
            str(unit)
            .strip()
            .upper()
            for unit in allowed_units
        }


        candidates = candidates[
            candidates["unit_name"]
            .astype(str)
            .str.strip()
            .str.upper()
            .isin(allowed_units)
        ].copy()


    if candidates.empty:

        return None


    candidates[
        "coverage"
    ] = (
        candidates["id"]
        .apply(nutrient_coverage)
    )


    candidates = (
        candidates
        .sort_values(
            [
                "coverage",
                "id"
            ],
            ascending=[
                False,
                True
            ]
        )
        .copy()
    )


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row["unit_name"],

        "coverage":
            int(row["coverage"])
    }


# ============================================================
# EXACT MACRO / MICRO NUTRIENTS
# ============================================================

for output_name, config in EXACT_NUTRIENTS.items():

    result = find_exact_nutrient(

        config["name"],

        {
            config["unit"]
        }
    )


    if result is None:

        print(
            f"WARNING: Could not find "
            f"{output_name}: "
            f"{config['name']}"
        )

    else:

        nutrient_lookup[
            output_name
        ] = result


# ============================================================
# FINAL NUTRIENT MAP
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOUNDATION NUTRIENT MAP")
print("=" * 70)


for output_name in OUTPUT_NUTRIENTS:

    if output_name not in nutrient_lookup:

        print(
            f"{output_name:25} NOT FOUND"
        )

        continue


    info = nutrient_lookup[
        output_name
    ]


    print(
        f"{output_name:25} "
        f"ID={info['id']:5} "
        f"UNIT={info['unit']} "
        f"COVERAGE={info['coverage']:4} "
        f"NAME={info['name']}"
    )


# ============================================================
# NUTRIENT MAPPING SAFETY CHECK
# ============================================================

EXPECTED_NUTRIENT_NAMES = {

    "calories_kcal": {
        "energy",
        "energy (atwater general factors)",
        "energy (atwater specific factors)"
    },

    "protein_g": {
        "protein"
    },

    "fat_g": {
        "total lipid (fat)"
    },

    "carbohydrate_g": {
        "carbohydrate, by difference"
    },

    "fiber_g": {
        "fiber, total dietary"
    },

    "sugar_g": {
        "sugars, total"
    },

    "saturated_fat_g": {
        "fatty acids, total saturated"
    }
}


for output_name, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    if output_name not in nutrient_lookup:

        raise RuntimeError(
            f"CRITICAL: Missing nutrient mapping: "
            f"{output_name}"
        )


    actual_name = normalize_text(
        nutrient_lookup[
            output_name
        ]["name"]
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"CRITICAL NUTRIENT MAPPING ERROR: "
            f"{output_name} mapped to "
            f"'{nutrient_lookup[output_name]['name']}'"
        )


print("\n")
print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# NUTRIENT LOOKUP TABLE
# ============================================================

nutrient_id_to_output = {

    info["id"]: output_name

    for output_name, info
    in nutrient_lookup.items()
}


# ============================================================
# FUNCTION TO GET FOOD NUTRIENTS
# ============================================================

def get_food_nutrients(
    fdc_id
):
    """
    Return all requested nutrient values
    for one FDC food.

    Values are per 100 g for Foundation Foods.
    """

    fdc_id = int(fdc_id)


    rows = foundation_food_nutrients[
        foundation_food_nutrients[
            "fdc_id"
        ]
        == fdc_id
    ].copy()


    result = {

        nutrient_name: np.nan

        for nutrient_name
        in OUTPUT_NUTRIENTS
    }


    if rows.empty:

        return result


    # --------------------------------------------------------
    # Convert amount to numeric
    # --------------------------------------------------------

    rows[
        "amount_numeric"
    ] = pd.to_numeric(
        rows["amount"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # Extract exact nutrient IDs
    # --------------------------------------------------------

    for output_name, info in nutrient_lookup.items():

        nutrient_id = int(
            info["id"]
        )


        matches = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]


        if matches.empty:

            continue


        values = pd.to_numeric(
            matches["amount"],
            errors="coerce"
        ).dropna()


        if values.empty:

            continue


        # USDA normally has one relevant row.
        # If duplicates exist, use the first.
        result[
            output_name
        ] = float(
            values.iloc[0]
        )


    return result


# ============================================================
# DUPLICATE FOOD COMPLETENESS SCORE
# ============================================================
#
# We do NOT automatically choose the first FDC ID.
#
# We prefer the USDA record with the most complete
# critical nutrient profile.
#
# Calories gets extra importance because a food without
# calories is less useful for a calorie-tracking application.
# ============================================================

DUPLICATE_SCORE_WEIGHTS = {

    "calories_kcal": 3,

    "protein_g": 2,

    "fat_g": 2,

    "carbohydrate_g": 2,

    "fiber_g": 1,

    "sugar_g": 1,

    "saturated_fat_g": 1
}


def score_food_record(
    fdc_id
):
    """
    Score one USDA food record based on
    nutrient completeness.
    """

    values = get_food_nutrients(
        fdc_id
    )


    score = 0

    available_count = 0


    for nutrient_name, weight in (
        DUPLICATE_SCORE_WEIGHTS.items()
    ):

        value = values.get(
            nutrient_name,
            np.nan
        )


        if pd.notna(value):

            score += weight

            available_count += 1


    return {

        "score": score,

        "available_count":
            available_count,

        "values":
            values
    }


# ============================================================
# SELECT BEST DUPLICATE RECORD
# ============================================================

selected_foods = {}

duplicate_review_rows = []


print("\n")
print("=" * 70)
print("SELECTING TARGET FOUNDATION FOODS")
print("=" * 70)


for food_name, config in TARGET_FOODS.items():

    matches = target_matches[
        food_name
    ].copy()


    print("\n")
    print("-" * 70)
    print(food_name.upper())
    print("-" * 70)


    if matches.empty:

        print(
            "STATUS: NOT_FOUND"
        )

        continue


    # --------------------------------------------------------
    # One exact match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]


        selected_foods[
            food_name
        ] = {

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "data_type":
                row["data_type"],

            "duplicate":
                False
        }


        print(
            "STATUS: SELECTED"
        )

        print(
            "FDC ID:",
            int(row["fdc_id"])
        )

        print(
            "Description:",
            row["description"]
        )

        continue


    # --------------------------------------------------------
    # Duplicate description
    # --------------------------------------------------------

    print(
        "STATUS: DUPLICATE_DESCRIPTION"
    )


    candidate_rows = []


    for _, row in matches.iterrows():

        fdc_id = int(
            row["fdc_id"]
        )


        score_info = score_food_record(
            fdc_id
        )


        candidate_rows.append({

            "fdc_id":
                fdc_id,

            "description":
                row["description"],

            "score":
                score_info["score"],

            "available_count":
                score_info[
                    "available_count"
                ]
        })


    candidate_df = pd.DataFrame(
        candidate_rows
    )


    # --------------------------------------------------------
    # Sort:
    #
    # 1. Highest completeness score
    # 2. Highest nutrient count
    # 3. Lowest FDC ID as deterministic tie breaker
    # --------------------------------------------------------

    candidate_df = (
        candidate_df
        .sort_values(
            [
                "score",
                "available_count",
                "fdc_id"
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


    print(
        candidate_df.to_string(
            index=False
        )
    )


    best_fdc_id = int(
        candidate_df.iloc[0]["fdc_id"]
    )


    best_row = matches[
        matches["fdc_id"]
        == best_fdc_id
    ].iloc[0]


    selected_foods[
        food_name
    ] = {

        "fdc_id":
            best_fdc_id,

        "description":
            best_row["description"],

        "data_type":
            best_row["data_type"],

        "duplicate":
            True
    }


    # --------------------------------------------------------
    # Save duplicate review information
    # --------------------------------------------------------

    duplicate_review_rows.append({

        "food_name":
            food_name,

        "requested_description":
            config["exact_description"],

        "selected_fdc_id":
            best_fdc_id,

        "candidate_count":
            len(candidate_df),

        "candidate_fdc_ids":
            ",".join(
                candidate_df[
                    "fdc_id"
                ]
                .astype(str)
                .tolist()
            ),

        "candidate_scores":
            ",".join(
                candidate_df[
                    "score"
                ]
                .astype(str)
                .tolist()
            )
    })


    print(
        "\nSelected preferred FDC ID:",
        best_fdc_id
    )

    print(
        "Reason: highest critical nutrient completeness."
    )


# ============================================================
# BUILD SOURCE MAPPING
# ============================================================
#
# Keep ALL USDA duplicate records internally.
#
# The application database uses one preferred FDC ID,
# but source mapping preserves all matching USDA records.
# ============================================================

source_mapping_rows = []


for food_name, config in TARGET_FOODS.items():

    matches = target_matches[
        food_name
    ].copy()


    if matches.empty:

        continue


    preferred_fdc_id = selected_foods[
        food_name
    ]["fdc_id"]


    for _, row in matches.iterrows():

        source_mapping_rows.append({

            "food_name":
                food_name,

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "source":
                "USDA FoodData Central Foundation Food",

            "is_preferred":
                int(
                    int(row["fdc_id"])
                    == int(preferred_fdc_id)
                )
        })


source_mapping = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print("\n")
print("=" * 70)
print("BUILDING FINAL FOOD DATABASE")
print("=" * 70)


database_rows = []


for food_name, selected in (
    selected_foods.items()
):

    fdc_id = int(
        selected["fdc_id"]
    )


    food_row = foundation_food[
        foundation_food["fdc_id"]
        == fdc_id
    ]


    if food_row.empty:

        raise RuntimeError(
            f"Selected FDC ID {fdc_id} "
            f"not found in Foundation Foods."
        )


    food_row = food_row.iloc[0]


    nutrients_for_food = (
        get_food_nutrients(
            fdc_id
        )
    )


    row = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            food_row["description"],

        "data_type":
            "Foundation Food",

        "basis_g":
            100
    }


    for nutrient_name in OUTPUT_NUTRIENTS:

        row[
            nutrient_name
        ] = nutrients_for_food[
            nutrient_name
        ]


    database_rows.append(
        row
    )


food_database = pd.DataFrame(
    database_rows
)


# ============================================================
# COLUMN ORDER
# ============================================================

database_columns = [

    "food_name",

    "fdc_id",

    "description",

    "data_type",

    "basis_g",

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g"
]


food_database = food_database[
    database_columns
]


# ============================================================
# FINAL FOOD DATABASE DISPLAY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL FOOD DATABASE")
print("=" * 70)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# NUTRIENT DATABASE SANITY CHECK
# ============================================================

print("\n")
print("=" * 70)
print("NUTRIENT DATABASE SANITY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# Verify nutrient mappings one more time
# ------------------------------------------------------------

for output_name, allowed_names in (
    EXPECTED_NUTRIENT_NAMES.items()
):

    actual = normalize_text(
        nutrient_lookup[
            output_name
        ]["name"]
    )


    if actual not in allowed_names:

        raise RuntimeError(
            f"INVALID MAPPING: "
            f"{output_name} -> {actual}"
        )


    print(
        f"PASS: {output_name:25} "
        f"-> "
        f"{nutrient_lookup[output_name]['name']}"
    )


# ------------------------------------------------------------
# Verify no nutrient IDs are reused
# ------------------------------------------------------------

mapped_ids = [
    info["id"]
    for info in nutrient_lookup.values()
]


if len(mapped_ids) != len(
    set(mapped_ids)
):

    raise RuntimeError(
        "CRITICAL: Multiple application nutrients "
        "are using the same USDA nutrient ID."
    )


print("\n")
print(
    "No nutrient IDs are incorrectly reused."
)


# ============================================================
# COVERAGE REPORT
# ============================================================

print("\n")


for nutrient_name in OUTPUT_NUTRIENTS:

    count = int(
        food_database[
            nutrient_name
        ]
        .notna()
        .sum()
    )


    print(
        f"Foods with {nutrient_name}: "
        f"{count}/{len(food_database)}"
    )


# ============================================================
# NEGATIVE VALUE CHECK
# ============================================================

numeric_columns = [
    column
    for column in OUTPUT_NUTRIENTS
]


negative_values = []


for column in numeric_columns:

    mask = (
        food_database[column]
        .notna()
        &
        (
            food_database[column]
            < 0
        )
    )


    if mask.any():

        for _, row in (
            food_database[mask]
            .iterrows()
        ):

            negative_values.append({

                "food_name":
                    row["food_name"],

                "nutrient":
                    column,

                "value":
                    row[column]
            })


if negative_values:

    print("\n")
    print(
        "WARNING: Negative nutrient values detected:"
    )

    print(
        pd.DataFrame(
            negative_values
        )
        .to_string(
            index=False
        )
    )

else:

    print("\n")
    print(
        "No negative nutrient values detected."
    )


# ============================================================
# CALORIE SANITY CHECK
# ============================================================
#
# This does NOT replace USDA calories.
#
# It only compares USDA Energy with:
#
#     protein * 4
#   + carbohydrate * 4
#   + fat * 9
#
# Missing macros are handled as missing.
# ============================================================

print("\n")
print("=" * 70)
print("CALORIE SANITY CHECK")
print("=" * 70)


macro_check_rows = []


for _, row in food_database.iterrows():

    protein = row["protein_g"]

    carbs = row["carbohydrate_g"]

    fat = row["fat_g"]

    calories = row["calories_kcal"]


    if (
        pd.notna(protein)
        and
        pd.notna(carbs)
        and
        pd.notna(fat)
    ):

        macro_calories = (
            float(protein) * 4
            +
            float(carbs) * 4
            +
            float(fat) * 9
        )

    else:

        macro_calories = np.nan


    if (
        pd.notna(calories)
        and
        pd.notna(macro_calories)
    ):

        difference = (
            float(calories)
            -
            float(macro_calories)
        )

    else:

        difference = np.nan


    macro_check_rows.append({

        "food_name":
            row["food_name"],

        "calories_kcal":
            calories,

        "macro_calorie_check":
            macro_calories,

        "macro_calorie_difference":
            difference
    })


calorie_check = pd.DataFrame(
    macro_check_rows
)


print(
    calorie_check.to_string(
        index=False
    )
)


# ============================================================
# FINAL DATABASE VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DATABASE VALIDATION")
print("=" * 70)


print(
    "Foods:",
    len(food_database)
)


print(
    "Unique FDC IDs:",
    food_database["fdc_id"].nunique()
)


print(
    "Basis values:",
    sorted(
        food_database["basis_g"]
        .dropna()
        .unique()
        .tolist()
    )
)


print(
    "Data types:",
    food_database["data_type"]
    .unique()
    .tolist()
)


# ------------------------------------------------------------
# Must contain exactly all target foods
# ------------------------------------------------------------

expected_food_names = set(
    TARGET_FOODS.keys()
)


actual_food_names = set(
    food_database[
        "food_name"
    ]
)


missing_food_names = (
    expected_food_names
    -
    actual_food_names
)


if missing_food_names:

    raise RuntimeError(
        "Missing target foods: "
        +
        str(
            sorted(
                missing_food_names
            )
        )
    )


# ------------------------------------------------------------
# Unique IDs
# ------------------------------------------------------------

if (
    food_database["fdc_id"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Final database contains duplicate FDC IDs."
    )


# ------------------------------------------------------------
# Basis
# ------------------------------------------------------------

if not (
    food_database["basis_g"]
    == 100
).all():

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


# ------------------------------------------------------------
# Data type
# ------------------------------------------------------------

if not (
    food_database["data_type"]
    == "Foundation Food"
).all():

    raise RuntimeError(
        "Final database contains non-Foundation foods."
    )


# ------------------------------------------------------------
# Critical nutrient coverage
# ------------------------------------------------------------

critical_nutrients = [

    "protein_g",

    "fat_g",

    "carbohydrate_g"
]


for nutrient_name in critical_nutrients:

    if (
        food_database[
            nutrient_name
        ]
        .isna()
        .any()
    ):

        missing_foods = (
            food_database[
                food_database[
                    nutrient_name
                ].isna()
            ]["food_name"]
            .tolist()
        )


        raise RuntimeError(
            f"Critical nutrient "
            f"{nutrient_name} is missing for: "
            f"{missing_foods}"
        )


print(
    "Critical nutrient coverage: PASS"
)


print("\n")
print(
    "STATUS: PASS"
)


# ============================================================
# SAVE FINAL DATABASE
# ============================================================

food_database.to_csv(
    DATABASE_FILE,
    index=False
)


source_mapping.to_csv(
    SOURCE_MAPPING_FILE,
    index=False
)


review_df = pd.DataFrame(
    duplicate_review_rows
)


review_df.to_csv(
    REVIEW_FILE,
    index=False
)


# ============================================================
# SAVE CONFIRMATION
# ============================================================

print("\n")
print("=" * 70)
print("FILES SAVED")
print("=" * 70)


print(
    "Database:"
)

print(
    DATABASE_FILE
)


print("\nSource mapping:")

print(
    SOURCE_MAPPING_FILE
)


print("\nDuplicate review:")

print(
    REVIEW_FILE
)


# ============================================================
# APPLICATION NUTRITION CALCULATOR
# ============================================================

def calculate_nutrition(
    database,
    food_name,
    grams
):
    """
    Calculate nutrition for a food weight.

    USDA database values are per 100 g.
    """

    if grams is None:

        raise ValueError(
            "grams cannot be None."
        )


    grams = float(
        grams
    )


    if grams < 0:

        raise ValueError(
            "grams cannot be negative."
        )


    matches = database[
        database["food_name"]
        == food_name
    ]


    if matches.empty:

        raise KeyError(
            f"Food not found: {food_name}"
        )


    row = matches.iloc[0]


    multiplier = (
        grams / 100.0
    )


    result = {

        "food_name":
            food_name,

        "grams":
            grams
    }


    for nutrient_name in OUTPUT_NUTRIENTS:

        value = row[
            nutrient_name
        ]


        if pd.isna(value):

            result[
                nutrient_name
            ] = None

        else:

            result[
                nutrient_name
            ] = float(value) * multiplier


    return result


# ============================================================
# MEAL CALCULATOR
# ============================================================

def calculate_meal(
    database,
    meal_items
):
    """
    Calculate the total nutrition of a meal.

    Example:

        meal_items = [
            {
                "food_name": "chicken_breast",
                "grams": 150
            },
            {
                "food_name": "oats_rolled",
                "grams": 50
            }
        ]

    IMPORTANT:
    Missing nutrients are NOT converted to zero.

    If no food in the meal provides a value for a nutrient,
    the meal total remains None.
    """

    totals = {

        nutrient_name: 0.0

        for nutrient_name
        in OUTPUT_NUTRIENTS
    }


    has_value = {

        nutrient_name: False

        for nutrient_name
        in OUTPUT_NUTRIENTS
    }


    foods = []


    for item in meal_items:

        if "food_name" not in item:

            raise ValueError(
                "Meal item is missing food_name."
            )


        if "grams" not in item:

            raise ValueError(
                "Meal item is missing grams."
            )


        result = calculate_nutrition(

            database,

            item["food_name"],

            item["grams"]
        )


        foods.append(
            result
        )


        for nutrient_name in OUTPUT_NUTRIENTS:

            value = result.get(
                nutrient_name
            )


            if value is None:

                continue


            totals[
                nutrient_name
            ] += float(value)


            has_value[
                nutrient_name
            ] = True


    # --------------------------------------------------------
    # Missing != zero
    # --------------------------------------------------------

    for nutrient_name in OUTPUT_NUTRIENTS:

        if not has_value[
            nutrient_name
        ]:

            totals[
                nutrient_name
            ] = None


    return {

        "foods":
            foods,

        "totals":
            totals
    }


# ============================================================
# MEAL PRINTER
# ============================================================

def print_meal(
    meal
):

    print("\n")
    print("=" * 70)
    print("MEAL")
    print("=" * 70)


    for food in meal["foods"]:

        print(
            f"{food['food_name']:25} "
            f"{food['grams']:.1f} g"
        )


    print("\n")
    print("-" * 70)
    print("TOTAL NUTRITION")
    print("-" * 70)


    labels = {

        "calories_kcal":
            "Calories",

        "protein_g":
            "Protein",

        "fat_g":
            "Fat",

        "carbohydrate_g":
            "Carbohydrate",

        "fiber_g":
            "Fiber",

        "sugar_g":
            "Sugar",

        "saturated_fat_g":
            "Saturated fat"
    }


    units = {

        "calories_kcal":
            "kcal",

        "protein_g":
            "g",

        "fat_g":
            "g",

        "carbohydrate_g":
            "g",

        "fiber_g":
            "g",

        "sugar_g":
            "g",

        "saturated_fat_g":
            "g"
    }


    for nutrient_name in OUTPUT_NUTRIENTS:

        label = labels[
            nutrient_name
        ]

        unit = units[
            nutrient_name
        ]


        value = meal[
            "totals"
        ][
            nutrient_name
        ]


        if value is None:

            print(
                f"{label:20}: -"
            )

        else:

            print(
                f"{label:20}: "
                f"{value:.2f} "
                f"{unit}"
            )


# ============================================================
# EXAMPLE FOOD TEST
# ============================================================

print("\n")
print("=" * 70)
print("EXAMPLE NUTRITION TEST")
print("=" * 70)


chicken_test = calculate_nutrition(

    food_database,

    "chicken_breast",

    150
)


print(
    "\nchicken_breast — 150 g"
)


for nutrient_name in OUTPUT_NUTRIENTS:

    value = chicken_test[
        nutrient_name
    ]


    if value is None:

        print(
            f"{nutrient_name:25}: -"
        )

    else:

        print(
            f"{nutrient_name:25}: "
            f"{value:.2f}"
        )


# ============================================================
# EXAMPLE MEAL TEST
# ============================================================

meal_items = [

    {
        "food_name":
            "chicken_breast",

        "grams":
            150
    },

    {
        "food_name":
            "oats_rolled",

        "grams":
            50
    },

    {
        "food_name":
            "apple_fuji",

        "grams":
            150
    }
]


meal = calculate_meal(
    food_database,
    meal_items
)


print_meal(
    meal
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print(
    "Foundation Foods available:",
    len(foundation_food)
)


print(
    "Foods selected:",
    len(selected_foods)
)


print(
    "Foods requiring duplicate review:",
    len(duplicate_review_rows)
)


print(
    "Foods not found:",
    sum(
        1
        for matches
        in target_matches.values()
        if matches.empty
    )
)


print(
    "\nDatabase validation: PASS"
)


print("\nDatabase:")

print(
    DATABASE_FILE
)


print("\nSource mapping:")

print(
    SOURCE_MAPPING_FILE
)


print("\nReview:")

print(
    REVIEW_FILE
)


print("\nDone.")




LOADING USDA FOODDATA CENTRAL FILES

Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


----------------------------------------------------------------------
SELECTED USDA FILES
----------------------------------------------------------------------
food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


----------------------------------------------------------------------
READING USDA CSV FILES
----------------------------------------------------------------------
food.csv rows:          87,990
food_nutri

In [26]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD DATABASE BUILDER
# ============================================================
#
# PURPOSE
# -------
# Build a small, auditable nutrition database from USDA
# FoodData Central Foundation Foods.
#
# IMPORTANT POLICIES
# ------------------
#
# 1. Only Foundation Foods are used.
#
# 2. Food matching uses exact normalized USDA descriptions.
#    No fuzzy food matching is used.
#
# 3. Nutrient matching uses exact USDA nutrient names.
#    No fuzzy nutrient matching is used.
#
# 4. Nutrient coverage is based on UNIQUE FDC FOOD IDs.
#
# 5. Duplicate FDC ID + nutrient ID rows are explicitly
#    detected. They are never silently resolved using
#    "first row wins".
#
# 6. If duplicate nutrient rows have:
#       - identical usable amounts -> safely collapse them
#       - one usable amount + missing values -> use usable amount
#       - conflicting usable amounts -> STOP
#
# 7. Duplicate food descriptions are resolved using:
#
#       highest requested nutrient completeness
#       THEN lowest FDC ID as deterministic tie-breaker
#
#    The lower FDC ID does NOT mean the USDA record is
#    nutritionally superior.
#
# 8. USDA Energy is used as the primary calorie value.
#    Calories are NOT calculated from 4/4/9.
#
# 9. Macro-derived calories are only a diagnostic comparison.
#
# 10. Missing nutrient values remain NaN.
#     Missing != zero.
#
# 11. Foundation nutrient values are stored on a 100 g basis.
#
# 12. The final nutrient mapping is saved separately for
#     auditability.
#
# ============================================================


import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
)


FOOD_CSV = BASE_DIR / "food.csv"
FOOD_NUTRIENT_CSV = BASE_DIR / "food_nutrient.csv"
NUTRIENT_CSV = BASE_DIR / "nutrient.csv"


OUTPUT_DATABASE = BASE_DIR / "food_database.csv"
OUTPUT_SOURCE_MAPPING = BASE_DIR / "food_source_mapping.csv"
OUTPUT_SELECTION_REVIEW = BASE_DIR / "food_selection_review.csv"
OUTPUT_NUTRIENT_MAPPING = BASE_DIR / "nutrient_mapping.csv"
OUTPUT_DUPLICATE_NUTRIENTS = BASE_DIR / "duplicate_food_nutrient_review.csv"


# ============================================================
# TARGET FOODS
# ============================================================
#
# These are the exact USDA descriptions we expect.
#
# IMPORTANT:
# Do not change these descriptions casually.
# They are part of the food-selection validation.
# ============================================================

TARGET_FOODS = {

    "CHICKEN_BREAST": {
        "food_name": "chicken_breast",
        "exact_description":
            "Chicken, breast, boneless, skinless, raw",
    },

    "BEEF_GROUND": {
        "food_name": "beef_ground",
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw",
    },

    "OATS_ROLLED": {
        "food_name": "oats_rolled",
        "exact_description":
            "Oats, whole grain, rolled, old fashioned",
    },

    "CARROT": {
        "food_name": "carrot",
        "exact_description":
            "Carrots, mature, raw",
    },

    "RYE_FLOUR": {
        "food_name": "rye_flour",
        "exact_description":
            "Flour, rye",
    },

    "APPLE_FUJI": {
        "food_name": "apple_fuji",
        "exact_description":
            "Apples, fuji, with skin, raw",
    },

    "BANANA": {
        "food_name": "banana",
        "exact_description":
            "Bananas, ripe and slightly ripe, raw",
    },

    "MILK_WHOLE": {
        "food_name": "milk_whole",
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D",
    },

    "MUSHROOM_WHITE_BUTTON": {
        "food_name": "mushroom_white_button",
        "exact_description":
            "Mushrooms, white button",
    },
}


# ============================================================
# APPLICATION NUTRIENTS
# ============================================================
#
# Exact USDA nutrient names.
#
# NEVER replace this with fuzzy matching.
# ============================================================

EXACT_NUTRIENTS = {

    "protein_g": (
        "Protein",
        {"G"},
    ),

    "fat_g": (
        "Total lipid (fat)",
        {"G"},
    ),

    "carbohydrate_g": (
        "Carbohydrate, by difference",
        {"G"},
    ),

    "fiber_g": (
        "Fiber, total dietary",
        {"G"},
    ),

    "sugar_g": (
        "Sugars, Total",
        {"G"},
    ),

    "saturated_fat_g": (
        "Fatty acids, total saturated",
        {"G"},
    ),
}


# ============================================================
# ENERGY POLICY
# ============================================================
#
# Select the KCAL energy nutrient with the highest number
# of UNIQUE Foundation Food records containing a usable
# value.
#
# If coverage ties:
#
#     2047 > 2048 > 1008
#
# 2047 = Energy (Atwater General Factors)
# 2048 = Energy (Atwater Specific Factors)
# 1008 = Energy
#
# USDA Energy remains the primary calorie value.
# ============================================================

ENERGY_PRIORITY = {

    2047: 3,
    2048: 2,
    1008: 1,
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_nutrient_name(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("  ", " ")
    )


def normalize_data_type(value):

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
    )


def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = value.strip().lower()

    # Normalize whitespace.
    value = re.sub(r"\s+", " ", value)

    return value


def safe_int(value):

    if pd.isna(value):
        return None

    return int(float(value))


def safe_float(value):

    if pd.isna(value):
        return np.nan

    try:
        return float(value)
    except Exception:
        return np.nan


def print_section(title, char="="):

    print("\n")
    print(char * 70)
    print(title)
    print(char * 70)


# ============================================================
# FILE EXISTENCE CHECK
# ============================================================

print_section(
    "LOADING USDA FOODDATA CENTRAL FILES"
)

print(
    "Search directory:"
)
print(
    BASE_DIR
)


required_files = {

    "food.csv": FOOD_CSV,

    "food_nutrient.csv":
        FOOD_NUTRIENT_CSV,

    "nutrient.csv":
        NUTRIENT_CSV,
}


missing_files = []

for name, path in required_files.items():

    if not path.exists():

        missing_files.append(
            f"{name}: {path}"
        )


if missing_files:

    print("\nMissing USDA files:")

    for item in missing_files:
        print(item)

    raise FileNotFoundError(
        "One or more required USDA CSV files "
        "were not found."
    )


print_section(
    "SELECTED USDA FILES",
    "-"
)

print("food.csv:")
print(FOOD_CSV)

print("food_nutrient.csv:")
print(FOOD_NUTRIENT_CSV)

print("nutrient.csv:")
print(NUTRIENT_CSV)


# ============================================================
# READ USDA CSV FILES
# ============================================================

print_section(
    "READING USDA CSV FILES",
    "-"
)


food = pd.read_csv(
    FOOD_CSV,
    low_memory=False,
)


food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_CSV,
    low_memory=False,
)


nutrient = pd.read_csv(
    NUTRIENT_CSV,
    low_memory=False,
)


print(
    f"food.csv rows:          {len(food):,}"
)

print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# REQUIRED COLUMN CHECK
# ============================================================

print_section(
    "USDA COLUMN VALIDATION",
    "-"
)


required_food_columns = {
    "fdc_id",
    "description",
    "data_type",
}


required_food_nutrient_columns = {
    "fdc_id",
    "nutrient_id",
    "amount",
}


required_nutrient_columns = {
    "id",
    "name",
    "unit_name",
}


missing_food_columns = (
    required_food_columns
    - set(food.columns)
)


missing_food_nutrient_columns = (
    required_food_nutrient_columns
    - set(food_nutrient.columns)
)


missing_nutrient_columns = (
    required_nutrient_columns
    - set(nutrient.columns)
)


if missing_food_columns:

    raise RuntimeError(
        "food.csv is missing required columns: "
        + str(sorted(missing_food_columns))
    )


if missing_food_nutrient_columns:

    raise RuntimeError(
        "food_nutrient.csv is missing required columns: "
        + str(sorted(missing_food_nutrient_columns))
    )


if missing_nutrient_columns:

    raise RuntimeError(
        "nutrient.csv is missing required columns: "
        + str(sorted(missing_nutrient_columns))
    )


print("food.csv columns: PASS")
print("food_nutrient.csv columns: PASS")
print("nutrient.csv columns: PASS")


# ============================================================
# NORMALIZE PRIMARY DATA TYPES
# ============================================================

food["fdc_id"] = pd.to_numeric(
    food["fdc_id"],
    errors="coerce",
)


food_nutrient["fdc_id"] = pd.to_numeric(
    food_nutrient["fdc_id"],
    errors="coerce",
)


food_nutrient["nutrient_id"] = pd.to_numeric(
    food_nutrient["nutrient_id"],
    errors="coerce",
)


nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce",
)


if food["fdc_id"].isna().any():

    raise RuntimeError(
        "food.csv contains invalid FDC IDs."
    )


if food_nutrient["fdc_id"].isna().any():

    raise RuntimeError(
        "food_nutrient.csv contains invalid FDC IDs."
    )


if food_nutrient["nutrient_id"].isna().any():

    raise RuntimeError(
        "food_nutrient.csv contains invalid nutrient IDs."
    )


if nutrient["id"].isna().any():

    raise RuntimeError(
        "nutrient.csv contains invalid nutrient IDs."
    )


food["fdc_id"] = (
    food["fdc_id"]
    .astype(int)
)


food_nutrient["fdc_id"] = (
    food_nutrient["fdc_id"]
    .astype(int)
)


food_nutrient["nutrient_id"] = (
    food_nutrient["nutrient_id"]
    .astype(int)
)


nutrient["id"] = (
    nutrient["id"]
    .astype(int)
)


food_nutrient["amount_numeric"] = (
    pd.to_numeric(
        food_nutrient["amount"],
        errors="coerce",
    )
)


# ============================================================
# USDA DATA TYPES
# ============================================================

print_section(
    "USDA DATA TYPES",
    "-"
)


print(
    food["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# NORMALIZED DATA TYPES
# ============================================================

food["normalized_data_type"] = (
    food["data_type"]
    .apply(normalize_data_type)
)


print_section(
    "NORMALIZED DATA TYPES",
    "-"
)


print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# FOUNDATION FOOD DETECTION
# ============================================================
#
# This intentionally uses:
#
#     foundationfood
#
# after normalization.
#
# This handles both:
#
#     foundation_food
#     Foundation Food
#
# without depending on exact formatting.
# ============================================================

foundation_food = food[
    food["normalized_data_type"]
    == "foundationfood"
].copy()


if foundation_food.empty:

    print(
        "FOUNDATION FOOD DETECTION FAILED"
    )

    print(
        food["data_type"]
        .value_counts(dropna=False)
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

foundation_food["normalized_description"] = (
    foundation_food["description"]
    .apply(normalize_text)
)


print_section(
    "FOUNDATION FOOD SUMMARY",
    "-"
)


print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)


print(
    f"Unique Foundation FDC IDs: "
    f"{foundation_food['fdc_id'].nunique():,}"
)


if (
    foundation_food["fdc_id"].nunique()
    != len(foundation_food)
):

    raise RuntimeError(
        "Foundation Food table contains duplicate FDC IDs."
    )


foundation_ids = set(
    foundation_food["fdc_id"]
    .astype(int)
)


# ============================================================
# FOUNDATION FOOD NUTRIENT RECORDS
# ============================================================

foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_ids)
    ]
    .copy()
)


foundation_food_nutrients["fdc_id"] = (
    foundation_food_nutrients["fdc_id"]
    .astype(int)
)


foundation_food_nutrients["nutrient_id"] = (
    foundation_food_nutrients["nutrient_id"]
    .astype(int)
)


print_section(
    "FOUNDATION NUTRIENT COVERAGE",
    "-"
)


print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)


print(
    f"Nutrient IDs available in Foundation Foods: "
    f"{foundation_food_nutrients['nutrient_id'].nunique():,}"
)


# ============================================================
# DUPLICATE FOOD/NUTRIENT ROW CHECK
# ============================================================
#
# IMPORTANT:
#
# We do NOT silently use:
#
#     values.iloc[0]
#
# when duplicates exist.
#
# Instead we inspect every FDC ID + nutrient ID combination.
# ============================================================

print_section(
    "FOOD/NUTRIENT DUPLICATE CHECK",
    "-"
)


duplicate_nutrient_rows = (
    foundation_food_nutrients
    .groupby(
        [
            "fdc_id",
            "nutrient_id",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
)


duplicate_nutrient_rows = (
    duplicate_nutrient_rows[
        duplicate_nutrient_rows["count"] > 1
    ]
    .copy()
)


if duplicate_nutrient_rows.empty:

    print(
        "PASS: No duplicate FDC ID + nutrient ID combinations."
    )

else:

    print(
        "WARNING:",
        len(duplicate_nutrient_rows),
        "duplicate food/nutrient combinations found."
    )

    print(
        duplicate_nutrient_rows
        .to_string(index=False)
    )


# ============================================================
# RESOLVE DUPLICATE FOOD/NUTRIENT ROWS
# ============================================================
#
# Resolution policy:
#
# Case A:
#     one usable amount
#     remaining duplicates missing
#
#     -> use usable amount
#
# Case B:
#     multiple usable amounts but all identical
#
#     -> use that common amount
#
# Case C:
#     multiple different usable amounts
#
#     -> STOP
#
# This is deliberately strict.
# ============================================================

print_section(
    "RESOLVING FOOD/NUTRIENT RECORDS",
    "-"
)


def resolve_duplicate_nutrient_group(group):

    values = pd.to_numeric(
        group["amount"],
        errors="coerce",
    )


    valid_values = (
        values.dropna()
        .astype(float)
    )


    if valid_values.empty:

        return np.nan


    unique_values = (
        valid_values
        .drop_duplicates()
        .tolist()
    )


    # --------------------------------------------------------
    # All usable duplicate values agree.
    # --------------------------------------------------------

    if len(unique_values) == 1:

        return float(
            unique_values[0]
        )


    # --------------------------------------------------------
    # Conflicting values.
    # --------------------------------------------------------

    fdc_id = int(
        group["fdc_id"].iloc[0]
    )

    nutrient_id = int(
        group["nutrient_id"].iloc[0]
    )


    nutrient_name_match = nutrient[
        nutrient["id"]
        == nutrient_id
    ]


    if nutrient_name_match.empty:

        nutrient_name = (
            "UNKNOWN"
        )

    else:

        nutrient_name = (
            nutrient_name_match
            .iloc[0]["name"]
        )


    raise RuntimeError(
        "\n"
        "CONFLICTING DUPLICATE NUTRIENT VALUES\n"
        f"FDC ID: {fdc_id}\n"
        f"Nutrient ID: {nutrient_id}\n"
        f"Nutrient: {nutrient_name}\n"
        f"Values: {unique_values}\n"
        "\n"
        "The database builder refuses to choose "
        "one value automatically."
    )


# ------------------------------------------------------------
# Build one canonical nutrient row per:
#
#     fdc_id + nutrient_id
# ------------------------------------------------------------

canonical_food_nutrients = (
    foundation_food_nutrients
    .groupby(
        [
            "fdc_id",
            "nutrient_id",
        ],
        as_index=False,
        sort=False,
    )
    .apply(
        lambda group: pd.Series(
            {
                "amount": resolve_duplicate_nutrient_group(
                    group
                )
            }
        ),
        include_groups=False,
    )
    .reset_index()
)


# Depending on pandas version, groupby/apply can produce
# different index column names. Normalize the result.

if "level_0" in canonical_food_nutrients.columns:

    canonical_food_nutrients = (
        canonical_food_nutrients
        .drop(columns=["level_0"])
    )


if "level_1" in canonical_food_nutrients.columns:

    canonical_food_nutrients = (
        canonical_food_nutrients
        .drop(columns=["level_1"])
    )


# Ensure expected columns exist.

required_canonical_columns = {
    "fdc_id",
    "nutrient_id",
    "amount",
}


if not required_canonical_columns.issubset(
    set(canonical_food_nutrients.columns)
):

    # Safer fallback implementation.
    canonical_rows = []


    for (
        (fdc_id, nutrient_id),
        group,
    ) in foundation_food_nutrients.groupby(
        [
            "fdc_id",
            "nutrient_id",
        ],
        sort=False,
    ):

        resolved_amount = (
            resolve_duplicate_nutrient_group(
                group
            )
        )


        canonical_rows.append(
            {
                "fdc_id":
                    int(fdc_id),

                "nutrient_id":
                    int(nutrient_id),

                "amount":
                    resolved_amount,
            }
        )


    canonical_food_nutrients = (
        pd.DataFrame(
            canonical_rows
        )
    )


canonical_food_nutrients["fdc_id"] = (
    canonical_food_nutrients["fdc_id"]
    .astype(int)
)


canonical_food_nutrients["nutrient_id"] = (
    canonical_food_nutrients["nutrient_id"]
    .astype(int)
)


canonical_food_nutrients["amount"] = (
    pd.to_numeric(
        canonical_food_nutrients["amount"],
        errors="coerce",
    )
)


# ------------------------------------------------------------
# Verify canonical uniqueness
# ------------------------------------------------------------

canonical_duplicates = (
    canonical_food_nutrients
    .duplicated(
        subset=[
            "fdc_id",
            "nutrient_id",
        ],
        keep=False,
    )
)


if canonical_duplicates.any():

    raise RuntimeError(
        "Canonical nutrient table still contains "
        "duplicate FDC ID + nutrient ID combinations."
    )


print(
    "Canonical nutrient table created:"
)

print(
    f"Rows: {len(canonical_food_nutrients):,}"
)

print(
    "Unique FDC ID + nutrient ID combinations: PASS"
)


# ============================================================
# NUTRIENT COVERAGE
# ============================================================
#
# IMPORTANT:
#
# Coverage means:
#
# "How many UNIQUE Foundation Foods have a usable
#  value for this nutrient?"
#
# NOT:
#
# "How many rows exist for this nutrient?"
# ============================================================

def nutrient_coverage(
    nutrient_id
):

    rows = canonical_food_nutrients[
        canonical_food_nutrients["nutrient_id"]
        == int(nutrient_id)
    ]


    if rows.empty:

        return 0


    valid_foods = rows[
        pd.to_numeric(
            rows["amount"],
            errors="coerce",
        ).notna()
    ]


    return int(
        valid_foods["fdc_id"]
        .nunique()
    )


# ============================================================
# ENERGY NUTRIENTS ACTUALLY AVAILABLE
# ============================================================

energy_candidates = nutrient[
    nutrient["name"]
    .apply(normalize_nutrient_name)
    .isin(
        {
            "energy",
            "energy (atwater general factors)",
            "energy (atwater specific factors)",
        }
    )
].copy()


energy_candidates = (
    energy_candidates[
        energy_candidates["unit_name"]
        .astype(str)
        .str.upper()
        == "KCAL"
    ]
    .copy()
)


energy_candidates["coverage"] = (
    energy_candidates["id"]
    .apply(nutrient_coverage)
)


energy_candidates["priority"] = (
    energy_candidates["id"]
    .map(ENERGY_PRIORITY)
    .fillna(0)
    .astype(int)
)


energy_candidates = (
    energy_candidates
    .sort_values(
        [
            "coverage",
            "priority",
            "id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
)


print_section(
    "ENERGY NUTRIENTS ACTUALLY AVAILABLE",
    "-"
)


if energy_candidates.empty:

    raise RuntimeError(
        "No KCAL Energy nutrient exists "
        "in the Foundation Foods dataset."
    )


print(
    energy_candidates[
        [
            "id",
            "name",
            "unit_name",
            "coverage",
            "priority",
        ]
    ]
    .to_string(index=False)
)


# ============================================================
# SELECT ENERGY NUTRIENT
# ============================================================

energy_row = (
    energy_candidates
    .iloc[0]
)


nutrient_lookup = {}


nutrient_lookup[
    "calories_kcal"
] = {

    "id":
        int(energy_row["id"]),

    "name":
        energy_row["name"],

    "unit":
        energy_row["unit_name"],

    "coverage":
        int(energy_row["coverage"]),
}


# ============================================================
# EXACT NUTRIENT SELECTION
# ============================================================

def find_exact_nutrient(
    exact_name,
    allowed_units=None,
):

    target = normalize_nutrient_name(
        exact_name
    )


    candidates = nutrient[
        nutrient["name"]
        .apply(normalize_nutrient_name)
        == target
    ].copy()


    if allowed_units is not None:

        allowed_units = {
            str(x).upper()
            for x in allowed_units
        }


        candidates = candidates[
            candidates["unit_name"]
            .astype(str)
            .str.upper()
            .isin(allowed_units)
        ]


    if candidates.empty:

        return None


    candidates["coverage"] = (
        candidates["id"]
        .apply(nutrient_coverage)
    )


    candidates = (
        candidates
        .sort_values(
            [
                "coverage",
                "id",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )


    row = candidates.iloc[0]


    return {

        "id":
            int(row["id"]),

        "name":
            row["name"],

        "unit":
            row["unit_name"],

        "coverage":
            int(row["coverage"]),
    }


for (
    output_name,
    (
        exact_name,
        allowed_units,
    ),
) in EXACT_NUTRIENTS.items():

    result = find_exact_nutrient(
        exact_name,
        allowed_units,
    )


    if result is None:

        raise RuntimeError(
            f"CRITICAL: Could not find exact "
            f"USDA nutrient: {exact_name}"
        )


    nutrient_lookup[
        output_name
    ] = result


# ============================================================
# FINAL FOUNDATION NUTRIENT MAP
# ============================================================

print_section(
    "FINAL FOUNDATION NUTRIENT MAP"
)


for output_name, info in nutrient_lookup.items():

    print(
        f"{output_name:25} "
        f"ID={info['id']:5} "
        f"UNIT={str(info['unit']).upper():4} "
        f"COVERAGE={info['coverage']:4} "
        f"NAME={info['name']}"
    )


# ============================================================
# NUTRIENT MAPPING VALIDATION
# ============================================================

EXPECTED_NUTRIENT_NAMES = {

    "calories_kcal": {
        "energy",
        "energy (atwater general factors)",
        "energy (atwater specific factors)",
    },

    "protein_g": {
        "protein",
    },

    "fat_g": {
        "total lipid (fat)",
    },

    "carbohydrate_g": {
        "carbohydrate, by difference",
    },

    "fiber_g": {
        "fiber, total dietary",
    },

    "sugar_g": {
        "sugars, total",
    },

    "saturated_fat_g": {
        "fatty acids, total saturated",
    },
}


for (
    output_name,
    allowed_names,
) in EXPECTED_NUTRIENT_NAMES.items():

    if output_name not in nutrient_lookup:

        raise RuntimeError(
            f"CRITICAL: Missing nutrient mapping: "
            f"{output_name}"
        )


    actual_name = (
        normalize_nutrient_name(
            nutrient_lookup[
                output_name
            ]["name"]
        )
    )


    if actual_name not in allowed_names:

        raise RuntimeError(
            f"CRITICAL NUTRIENT MAPPING ERROR: "
            f"{output_name} mapped to "
            f"'{nutrient_lookup[output_name]['name']}'"
        )


print_section(
    "NUTRIENT MAPPING VALIDATION"
)


print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# NUTRIENT MAPPING DATAFRAME
# ============================================================

nutrient_mapping_rows = []


for (
    application_name,
    info,
) in nutrient_lookup.items():

    nutrient_mapping_rows.append(
        {

            "application_name":
                application_name,

            "usda_nutrient_id":
                int(info["id"]),

            "usda_name":
                info["name"],

            "unit":
                info["unit"],

            "coverage_unique_foods":
                int(info["coverage"]),
        }
    )


nutrient_mapping_df = pd.DataFrame(
    nutrient_mapping_rows
)


# ============================================================
# SAVE NUTRIENT MAPPING
# ============================================================

nutrient_mapping_df.to_csv(
    OUTPUT_NUTRIENT_MAPPING,
    index=False,
)


# ============================================================
# TARGET FOOD SEARCH
# ============================================================

print_section(
    "FOUNDATION FOOD TARGET SEARCH"
)


target_matches = {}


for (
    target_key,
    config,
) in TARGET_FOODS.items():

    expected_description = (
        config["exact_description"]
    )


    normalized_expected = normalize_text(
        expected_description
    )


    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == normalized_expected
    ].copy()


    target_matches[
        target_key
    ] = matches


    print("\n")
    print("-" * 70)
    print(target_key)
    print("-" * 70)

    print(
        "Requested:"
    )

    print(
        expected_description
    )

    print(
        f"Matches: {len(matches)}"
    )


    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description",
                ]
            ]
            .sort_values("fdc_id")
            .to_string(index=False)
        )


# ============================================================
# GET CANONICAL FOOD NUTRIENTS
# ============================================================

nutrient_id_to_output = {

    info["id"]:
        output_name

    for (
        output_name,
        info,
    ) in nutrient_lookup.items()
}


def get_food_nutrients(
    fdc_id
):

    rows = canonical_food_nutrients[
        canonical_food_nutrients["fdc_id"]
        == int(fdc_id)
    ]


    result = {}


    for (
        output_name,
        info,
    ) in nutrient_lookup.items():

        nutrient_id = int(
            info["id"]
        )


        match = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]


        if match.empty:

            result[
                output_name
            ] = np.nan

        else:

            # Canonical table guarantees one row.
            value = match.iloc[0]["amount"]

            result[
                output_name
            ] = safe_float(value)


    return result


# ============================================================
# DUPLICATE FOOD SELECTION
# ============================================================
#
# Policy:
#
# 1. Exact description match only.
#
# 2. If exactly one record:
#       select it.
#
# 3. If duplicates:
#       calculate requested nutrient completeness.
#
# 4. Select highest completeness.
#
# 5. If tied:
#       lowest FDC ID.
#
# The tie-breaker is deterministic.
# It does NOT imply the lower FDC ID is nutritionally better.
# ============================================================

print_section(
    "SELECTING TARGET FOUNDATION FOODS"
)


selected_foods = {}

selection_review_rows = []


for (
    target_key,
    config,
) in TARGET_FOODS.items():

    matches = (
        target_matches[
            target_key
        ]
        .copy()
    )


    print("\n")
    print("-" * 70)
    print(target_key)
    print("-" * 70)


    # --------------------------------------------------------
    # No match
    # --------------------------------------------------------

    if matches.empty:

        print(
            "STATUS: NOT_FOUND"
        )


        selection_review_rows.append(
            {

                "food_name":
                    config["food_name"],

                "target_key":
                    target_key,

                "expected_description":
                    config["exact_description"],

                "fdc_id":
                    np.nan,

                "description":
                    np.nan,

                "status":
                    "NOT_FOUND",

                "requested_nutrient_completeness":
                    0,

                "selection_reason":
                    "No exact Foundation Food description match.",
            }
        )


        continue


    # --------------------------------------------------------
    # Single match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]


        fdc_id = int(
            row["fdc_id"]
        )


        selected_foods[
            config["food_name"]
        ] = row.to_dict()


        print(
            "STATUS: SELECTED"
        )

        print(
            f"FDC ID: {fdc_id}"
        )

        print(
            f"Description: {row['description']}"
        )


        selection_review_rows.append(
            {

                "food_name":
                    config["food_name"],

                "target_key":
                    target_key,

                "expected_description":
                    config["exact_description"],

                "fdc_id":
                    fdc_id,

                "description":
                    row["description"],

                "status":
                    "SELECTED",

                "requested_nutrient_completeness":
                    np.nan,

                "selection_reason":
                    "Unique exact Foundation Food description match.",
            }
        )


        continue


    # --------------------------------------------------------
    # Duplicate descriptions
    # --------------------------------------------------------

    candidate_rows = []


    for _, row in matches.iterrows():

        fdc_id = int(
            row["fdc_id"]
        )


        nutrient_values = (
            get_food_nutrients(
                fdc_id
            )
        )


        completeness = sum(
            pd.notna(
                nutrient_values[
                    output_name
                ]
            )
            for output_name
            in nutrient_lookup.keys()
        )


        candidate_rows.append(
            {

                "fdc_id":
                    fdc_id,

                "description":
                    row["description"],

                "requested_nutrient_completeness":
                    int(completeness),

                "available_count":
                    int(completeness),
            }
        )


    candidates_df = pd.DataFrame(
        candidate_rows
    )


    # --------------------------------------------------------
    # Deterministic selection:
    #
    # highest completeness
    # then lowest FDC ID
    # --------------------------------------------------------

    candidates_df = (
        candidates_df
        .sort_values(
            [
                "requested_nutrient_completeness",
                "fdc_id",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


    selected_fdc_id = int(
        candidates_df.iloc[0]["fdc_id"]
    )


    selected_row = matches[
        matches["fdc_id"]
        == selected_fdc_id
    ].iloc[0]


    selected_foods[
        config["food_name"]
    ] = selected_row.to_dict()


    print(
        "STATUS: DUPLICATE_DESCRIPTION"
    )


    print(
        candidates_df[
            [
                "fdc_id",
                "description",
                "requested_nutrient_completeness",
            ]
        ]
        .to_string(index=False)
    )


    print(
        f"\nSelected preferred FDC ID: "
        f"{selected_fdc_id}"
    )


    print(
        "Reason: highest requested nutrient "
        "completeness; FDC ID used as deterministic "
        "tie-breaker."
    )


    # --------------------------------------------------------
    # Save review rows for every duplicate candidate.
    # --------------------------------------------------------

    for _, candidate in candidates_df.iterrows():

        candidate_fdc_id = int(
            candidate["fdc_id"]
        )


        is_selected = (
            candidate_fdc_id
            == selected_fdc_id
        )


        selection_review_rows.append(
            {

                "food_name":
                    config["food_name"],

                "target_key":
                    target_key,

                "expected_description":
                    config["exact_description"],

                "fdc_id":
                    candidate_fdc_id,

                "description":
                    candidate["description"],

                "status":
                    "SELECTED"
                    if is_selected
                    else "DUPLICATE_CANDIDATE",

                "requested_nutrient_completeness":
                    int(
                        candidate[
                            "requested_nutrient_completeness"
                        ]
                    ),

                "selection_reason":
                    (
                        "Highest requested nutrient completeness; "
                        "FDC ID used as deterministic tie-breaker."
                        if is_selected
                        else
                        "Duplicate exact description candidate."
                    ),
            }
        )


# ============================================================
# DESCRIPTION VALIDATION
# ============================================================

print_section(
    "SELECTED FOOD DESCRIPTION VALIDATION"
)


for (
    target_key,
    config,
) in TARGET_FOODS.items():

    food_name = config["food_name"]


    if food_name not in selected_foods:

        raise RuntimeError(
            f"Selected food missing: {food_name}"
        )


    selected = (
        selected_foods[
            food_name
        ]
    )


    actual = normalize_text(
        selected["description"]
    )


    expected = normalize_text(
        config["exact_description"]
    )


    if actual != expected:

        raise RuntimeError(
            f"DESCRIPTION VALIDATION FAILED: "
            f"{food_name}: "
            f"expected '{expected}', "
            f"got '{selected['description']}'"
        )


    print(
        f"PASS: {food_name:28} "
        f"-> {selected['description']}"
    )


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print_section(
    "BUILDING FINAL FOOD DATABASE"
)


database_rows = []


for (
    target_key,
    config,
) in TARGET_FOODS.items():

    food_name = config["food_name"]


    if food_name not in selected_foods:

        continue


    selected = (
        selected_foods[
            food_name
        ]
    )


    fdc_id = int(
        selected["fdc_id"]
    )


    nutrients = (
        get_food_nutrients(
            fdc_id
        )
    )


    database_rows.append(
        {

            "food_name":
                food_name,

            "fdc_id":
                fdc_id,

            "description":
                selected["description"],

            "data_type":
                "Foundation Food",

            "basis_g":
                100,

            "calories_kcal":
                nutrients[
                    "calories_kcal"
                ],

            "protein_g":
                nutrients[
                    "protein_g"
                ],

            "fat_g":
                nutrients[
                    "fat_g"
                ],

            "carbohydrate_g":
                nutrients[
                    "carbohydrate_g"
                ],

            "fiber_g":
                nutrients[
                    "fiber_g"
                ],

            "sugar_g":
                nutrients[
                    "sugar_g"
                ],

            "saturated_fat_g":
                nutrients[
                    "saturated_fat_g"
                ],
        }
    )


food_database = pd.DataFrame(
    database_rows
)


# ============================================================
# SORT DATABASE IN TARGET ORDER
# ============================================================

food_order = [
    config["food_name"]
    for config
    in TARGET_FOODS.values()
]


food_database["_sort_order"] = (
    food_database["food_name"]
    .map(
        {
            name: index
            for index, name
            in enumerate(food_order)
        }
    )
)


food_database = (
    food_database
    .sort_values("_sort_order")
    .drop(columns="_sort_order")
    .reset_index(drop=True)
)


# ============================================================
# DISPLAY FINAL FOOD DATABASE
# ============================================================

print_section(
    "FINAL FOOD DATABASE"
)


print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# NUTRIENT DATABASE SANITY CHECK
# ============================================================

print_section(
    "NUTRIENT DATABASE SANITY CHECK"
)


expected_columns = [

    "calories_kcal",

    "protein_g",

    "fat_g",

    "carbohydrate_g",

    "fiber_g",

    "sugar_g",

    "saturated_fat_g",
]


for (
    column,
    valid_names,
) in EXPECTED_NUTRIENT_NAMES.items():

    actual_name = normalize_nutrient_name(
        nutrient_lookup[
            column
        ]["name"]
    )


    if actual_name not in valid_names:

        raise RuntimeError(
            f"INVALID MAPPING: "
            f"{column} -> {actual_name}"
        )


    print(
        f"PASS: {column:25} "
        f"-> {nutrient_lookup[column]['name']}"
    )


print(
    "\nNo nutrient IDs are incorrectly reused."
)


# ------------------------------------------------------------
# Verify nutrient IDs are unique.
# ------------------------------------------------------------

mapped_ids = [
    info["id"]
    for info
    in nutrient_lookup.values()
]


if len(mapped_ids) != len(set(mapped_ids)):

    raise RuntimeError(
        "CRITICAL: Multiple application nutrients "
        "reuse the same USDA nutrient ID."
    )


# ------------------------------------------------------------
# Verify units.
# ------------------------------------------------------------

if (
    str(
        nutrient_lookup[
            "calories_kcal"
        ]["unit"]
    ).upper()
    != "KCAL"
):

    raise RuntimeError(
        "CRITICAL: calories_kcal is not KCAL."
    )


for column in [
    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g",
]:

    if (
        str(
            nutrient_lookup[
                column
            ]["unit"]
        ).upper()
        != "G"
    ):

        raise RuntimeError(
            f"CRITICAL: {column} is not measured in grams."
        )


# ============================================================
# COVERAGE REPORT
# ============================================================

for column in expected_columns:

    count = int(
        food_database[column]
        .notna()
        .sum()
    )


    print(
        f"Foods with {column}: "
        f"{count}/{len(food_database)}"
    )


# ============================================================
# NEGATIVE NUTRIENT VALIDATION
# ============================================================

negative_values = []


for column in expected_columns:

    numeric_values = pd.to_numeric(
        food_database[column],
        errors="coerce",
    )


    negative_mask = (
        numeric_values < 0
    )


    if negative_mask.any():

        for _, row in (
            food_database[
                negative_mask
            ]
            .iterrows()
        ):

            negative_values.append(
                {

                    "food_name":
                        row["food_name"],

                    "nutrient":
                        column,

                    "value":
                        row[column],
                }
            )


if negative_values:

    negative_df = pd.DataFrame(
        negative_values
    )


    print(
        negative_df.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "Negative nutrient values detected."
    )


print(
    "\nNo negative nutrient values detected."
)


# ============================================================
# 100-G BASIS VALIDATION
# ============================================================

if not (
    food_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "CRITICAL: Not every food has a 100 g basis."
    )


print(
    "100 g basis validation: PASS"
)


# ============================================================
# DATA TYPE VALIDATION
# ============================================================

if not (
    food_database["data_type"]
    .eq("Foundation Food")
    .all()
):

    raise RuntimeError(
        "CRITICAL: Non-Foundation Food record detected."
    )


print(
    "Foundation Food data type validation: PASS"
)


# ============================================================
# FOOD COUNT / FDC ID VALIDATION
# ============================================================

expected_food_names = {
    config["food_name"]
    for config
    in TARGET_FOODS.values()
}


actual_food_names = set(
    food_database["food_name"]
)


missing_food_names = (
    expected_food_names
    -
    actual_food_names
)


if missing_food_names:

    raise RuntimeError(
        "Missing selected foods: "
        + str(sorted(missing_food_names))
    )


if (
    food_database["fdc_id"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "CRITICAL: Duplicate FDC IDs exist "
        "in final food database."
    )


if (
    food_database["food_name"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "CRITICAL: Duplicate food names exist "
        "in final food database."
    )


# ============================================================
# CRITICAL NUTRIENT COVERAGE
# ============================================================
#
# Protein, fat, and carbohydrate are treated as critical
# structural macros for this Foundation layer.
#
# Other nutrients are allowed to be missing because USDA
# does not provide every nutrient for every Foundation Food.
# ============================================================

critical_database_columns = [

    "protein_g",

    "fat_g",

    "carbohydrate_g",
]


for column in critical_database_columns:

    if (
        food_database[column]
        .isna()
        .any()
    ):

        missing_foods = (
            food_database[
                food_database[column].isna()
            ]["food_name"]
            .tolist()
        )


        raise RuntimeError(
            f"CRITICAL nutrient coverage failure: "
            f"{column} missing for {missing_foods}"
        )


print(
    "Critical nutrient coverage: PASS"
)


# ============================================================
# MACRO ENERGY COMPARISON
# ============================================================
#
# IMPORTANT:
#
# This is a diagnostic comparison only.
#
# It does NOT replace USDA Energy.
#
# USDA calories remain the primary calorie value.
# ============================================================

print_section(
    "MACRO ENERGY COMPARISON"
)


food_database["macro_calorie_check"] = (

    food_database["protein_g"] * 4

    +

    food_database["carbohydrate_g"] * 4

    +

    food_database["fat_g"] * 9
)


food_database[
    "macro_calorie_difference"
] = (

    food_database["calories_kcal"]

    -

    food_database["macro_calorie_check"]
)


print(
    food_database[
        [
            "food_name",
            "calories_kcal",
            "macro_calorie_check",
            "macro_calorie_difference",
        ]
    ]
    .to_string(index=False)
)


# ============================================================
# FINAL DATABASE VALIDATION
# ============================================================

print_section(
    "DATABASE VALIDATION"
)


print(
    f"Foods: {len(food_database)}"
)


print(
    f"Unique FDC IDs: "
    f"{food_database['fdc_id'].nunique()}"
)


print(
    f"Basis values: "
    f"{food_database['basis_g'].unique().tolist()}"
)


print(
    f"Data types: "
    f"{food_database['data_type'].unique().tolist()}"
)


print(
    "Critical nutrient coverage: PASS"
)


if len(food_database) != len(
    TARGET_FOODS
):

    raise RuntimeError(
        f"Expected {len(TARGET_FOODS)} foods "
        f"but built {len(food_database)}."
    )


print(
    "\nSTATUS: PASS"
)


# ============================================================
# BUILD SOURCE MAPPING
# ============================================================

source_mapping_rows = []


for (
    target_key,
    config,
) in TARGET_FOODS.items():

    food_name = config["food_name"]


    if food_name not in selected_foods:

        continue


    selected = (
        selected_foods[
            food_name
        ]
    )


    fdc_id = int(
        selected["fdc_id"]
    )


    source_mapping_rows.append(
        {

            "food_name":
                food_name,

            "target_key":
                target_key,

            "fdc_id":
                fdc_id,

            "description":
                selected["description"],

            "data_type":
                selected.get(
                    "data_type",
                    "Foundation Food",
                ),

            "expected_description":
                config["exact_description"],

            "source_file":
                str(FOOD_CSV),

            "nutrient_source_file":
                str(FOOD_NUTRIENT_CSV),

            "nutrient_mapping_file":
                str(NUTRIENT_CSV),

            "basis_g":
                100,

            "food_selection_policy":
                (
                    "Exact normalized USDA description; "
                    "duplicate descriptions resolved by highest "
                    "requested nutrient completeness, then lowest "
                    "FDC ID as deterministic tie-breaker."
                ),

            "nutrient_selection_policy":
                (
                    "Exact USDA nutrient name and allowed unit; "
                    "coverage based on unique Foundation Food IDs."
                ),

            "energy_policy":
                (
                    "Highest unique-food coverage among KCAL Energy "
                    "nutrients; tie priority 2047 > 2048 > 1008."
                ),
        }
    )


source_mapping = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# SAVE SELECTION REVIEW
# ============================================================

selection_review = pd.DataFrame(
    selection_review_rows
)


# ============================================================
# SAVE DUPLICATE NUTRIENT REVIEW
# ============================================================

if duplicate_nutrient_rows.empty:

    duplicate_review = pd.DataFrame(
        columns=[
            "fdc_id",
            "nutrient_id",
            "count",
        ]
    )

else:

    duplicate_review = (
        duplicate_nutrient_rows
        .copy()
    )


# ============================================================
# SAVE FILES
# ============================================================

print_section(
    "SAVING DATABASE FILES"
)


food_database_to_save = (
    food_database[
        [
            "food_name",
            "fdc_id",
            "description",
            "data_type",
            "basis_g",
            "calories_kcal",
            "protein_g",
            "fat_g",
            "carbohydrate_g",
            "fiber_g",
            "sugar_g",
            "saturated_fat_g",
        ]
    ]
    .copy()
)


food_database_to_save.to_csv(
    OUTPUT_DATABASE,
    index=False,
)


source_mapping.to_csv(
    OUTPUT_SOURCE_MAPPING,
    index=False,
)


selection_review.to_csv(
    OUTPUT_SELECTION_REVIEW,
    index=False,
)


nutrient_mapping_df.to_csv(
    OUTPUT_NUTRIENT_MAPPING,
    index=False,
)


duplicate_review.to_csv(
    OUTPUT_DUPLICATE_NUTRIENTS,
    index=False,
)


print(
    "Database:"
)

print(
    OUTPUT_DATABASE
)


print(
    "\nSource mapping:"
)

print(
    OUTPUT_SOURCE_MAPPING
)


print(
    "\nDuplicate food review:"
)

print(
    OUTPUT_SELECTION_REVIEW
)


print(
    "\nNutrient mapping:"
)

print(
    OUTPUT_NUTRIENT_MAPPING
)


print(
    "\nDuplicate food/nutrient review:"
)

print(
    OUTPUT_DUPLICATE_NUTRIENTS
)


# ============================================================
# RELOAD SAVED DATABASE
# ============================================================
#
# This verifies that what was written to disk can be read back
# without changing the structural guarantees.
# ============================================================

print_section(
    "RELOADING SAVED DATABASE"
)


reloaded_database = pd.read_csv(
    OUTPUT_DATABASE
)


if len(reloaded_database) != len(
    food_database_to_save
):

    raise RuntimeError(
        "Saved database row count does not match "
        "in-memory database."
    )


if (
    reloaded_database["fdc_id"]
    .nunique()
    != len(reloaded_database)
):

    raise RuntimeError(
        "Reloaded database contains duplicate FDC IDs."
    )


if not (
    reloaded_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "Reloaded database contains non-100-g records."
    )


print(
    "Saved database reload validation: PASS"
)


# ============================================================
# EXAMPLE NUTRITION CALCULATOR
# ============================================================

def get_food_row(
    food_name
):

    matches = food_database[
        food_database["food_name"]
        == food_name
    ]


    if matches.empty:

        raise KeyError(
            f"Food not found: {food_name}"
        )


    return matches.iloc[0]


def calculate_food_nutrition(
    food_name,
    weight_g,
):

    if weight_g < 0:

        raise ValueError(
            "Food weight cannot be negative."
        )


    row = get_food_row(
        food_name
    )


    multiplier = (
        float(weight_g)
        /
        float(row["basis_g"])
    )


    result = {

        "food_name":
            food_name,

        "weight_g":
            float(weight_g),
    }


    for column in expected_columns:

        value = row[column]


        if pd.isna(value):

            result[column] = np.nan

        else:

            result[column] = (
                float(value)
                * multiplier
            )


    return result


# ============================================================
# FORMAT NUTRITION VALUE
# ============================================================

def format_nutrition_value(
    value,
    decimals=2,
):

    if pd.isna(value):

        return "-"


    return f"{float(value):.{decimals}f}"


# ============================================================
# CHICKEN TEST
# ============================================================

print_section(
    "EXAMPLE NUTRITION TEST"
)


chicken_test = (
    calculate_food_nutrition(
        "chicken_breast",
        150,
    )
)


print(
    "\nchicken_breast — 150 g"
)


for column in expected_columns:

    print(
        f"{column:25}: "
        f"{format_nutrition_value(chicken_test[column])}"
    )


# ============================================================
# MEAL CALCULATOR
# ============================================================

def calculate_meal(
    foods_and_weights
):

    totals = {

        "calories_kcal":
            0.0,

        "protein_g":
            0.0,

        "fat_g":
            0.0,

        "carbohydrate_g":
            0.0,

        "fiber_g":
            0.0,

        "sugar_g":
            0.0,

        "saturated_fat_g":
            0.0,
    }


    present_flags = {

        column:
            False

        for column
        in expected_columns
    }


    for (
        food_name,
        weight_g,
    ) in foods_and_weights:

        nutrition = (
            calculate_food_nutrition(
                food_name,
                weight_g,
            )
        )


        for column in expected_columns:

            value = nutrition[column]


            if pd.isna(value):

                # Missing != zero.
                continue


            totals[column] += (
                float(value)
            )


            present_flags[
                column
            ] = True


    # --------------------------------------------------------
    # Restore NaN where every component food was missing
    # the nutrient.
    # --------------------------------------------------------

    for column in expected_columns:

        if not present_flags[column]:

            totals[column] = np.nan


    return totals


# ============================================================
# EXAMPLE MEAL
# ============================================================

print_section(
    "MEAL"
)


meal = [

    (
        "chicken_breast",
        150.0,
    ),

    (
        "oats_rolled",
        50.0,
    ),

    (
        "apple_fuji",
        150.0,
    ),
]


for (
    food_name,
    weight_g,
) in meal:

    print(
        f"{food_name:25} "
        f"{weight_g:6.1f} g"
    )


meal_totals = calculate_meal(
    meal
)


print_section(
    "TOTAL NUTRITION",
    "-"
)


meal_labels = {

    "calories_kcal":
        "Calories",

    "protein_g":
        "Protein",

    "fat_g":
        "Fat",

    "carbohydrate_g":
        "Carbohydrate",

    "fiber_g":
        "Fiber",

    "sugar_g":
        "Sugar",

    "saturated_fat_g":
        "Saturated fat",
}


meal_units = {

    "calories_kcal":
        "kcal",

    "protein_g":
        "g",

    "fat_g":
        "g",

    "carbohydrate_g":
        "g",

    "fiber_g":
        "g",

    "sugar_g":
        "g",

    "saturated_fat_g":
        "g",
}


for column in expected_columns:

    label = meal_labels[
        column
    ]


    unit = meal_units[
        column
    ]


    value = meal_totals[
        column
    ]


    print(
        f"{label:20} : "
        f"{format_nutrition_value(value)} "
        f"{unit if not pd.isna(value) else ''}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print_section(
    "PIPELINE COMPLETE"
)


foods_selected = len(
    food_database
)


foods_not_found = (
    len(TARGET_FOODS)
    -
    foods_selected
)


duplicate_food_count = sum(
    len(matches) > 1
    for matches
    in target_matches.values()
)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)


print(
    f"Foods selected: "
    f"{foods_selected}"
)


print(
    f"Foods requiring duplicate review: "
    f"{duplicate_food_count}"
)


print(
    f"Foods not found: "
    f"{foods_not_found}"
)


print(
    "\nDatabase validation: PASS"
)


print(
    "\nDatabase:"
)

print(
    OUTPUT_DATABASE
)


print(
    "\nSource mapping:"
)

print(
    OUTPUT_SOURCE_MAPPING
)


print(
    "\nReview:"
)

print(
    OUTPUT_SELECTION_REVIEW
)


print(
    "\nNutrient mapping:"
)

print(
    OUTPUT_NUTRIENT_MAPPING
)


print(
    "\nDuplicate nutrient review:"
)

print(
    OUTPUT_DUPLICATE_NUTRIENTS
)


print(
    "\nDone."
)




LOADING USDA FOODDATA CENTRAL FILES
Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30


----------------------------------------------------------------------
SELECTED USDA FILES
----------------------------------------------------------------------
food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv


----------------------------------------------------------------------
READING USDA CSV FILES
----------------------------------------------------------------------
food.csv rows:          87,990
food_nutrie

In [27]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOOD DATABASE BUILDER
# ============================================================
#
# Purpose:
#   Build a small, auditable nutrition database from USDA
#   FoodData Central Foundation Foods CSV files.
#
# Features:
#   - Validates USDA CSV files and required columns
#   - Detects Foundation Foods robustly
#   - Normalizes USDA data_type values
#   - Detects duplicate FDC ID + nutrient ID rows
#   - Safely resolves duplicate nutrient rows
#   - Uses unique-FDC-ID nutrient coverage
#   - Explicitly selects the energy nutrient
#   - Performs exact normalized food-description matching
#   - Handles duplicate food descriptions deterministically
#   - Validates selected descriptions
#   - Preserves missing nutrients as NaN
#   - Builds a 100-g nutrition database
#   - Performs macro-energy comparison
#   - Saves complete audit files
#   - Reloads the saved database and validates it
#   - Runs a sample nutrition calculation
#
# IMPORTANT:
#   Missing nutrient != zero.
#
#   NaN means USDA did not provide a usable value for the
#   selected nutrient in that food record.
#
# ============================================================

import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

USDA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30"
    r"\FoodData_Central_foundation_food_csv_2026-04-30"
)

FOOD_CSV = USDA_DIR / "food.csv"
FOOD_NUTRIENT_CSV = USDA_DIR / "food_nutrient.csv"
NUTRIENT_CSV = USDA_DIR / "nutrient.csv"


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

DATABASE_CSV = USDA_DIR / "food_database.csv"
SOURCE_MAPPING_CSV = USDA_DIR / "food_source_mapping.csv"
FOOD_SELECTION_REVIEW_CSV = USDA_DIR / "food_selection_review.csv"
NUTRIENT_MAPPING_CSV = USDA_DIR / "nutrient_mapping.csv"
DUPLICATE_NUTRIENT_REVIEW_CSV = (
    USDA_DIR / "duplicate_food_nutrient_review.csv"
)


# ============================================================
# TARGET FOODS
# ============================================================
#
# The exact USDA description is deliberately used.
#
# DO NOT replace these with fuzzy matching.
#
# If USDA changes the description in a future data release,
# the pipeline should fail rather than silently select another
# food.
# ============================================================

TARGET_FOODS = {

    "chicken_breast": {
        "exact_description":
            "Chicken, breast, boneless, skinless, raw"
    },

    "beef_ground": {
        "exact_description":
            "Beef, ground, 80% lean meat / 20% fat, raw"
    },

    "oats_rolled": {
        "exact_description":
            "Oats, whole grain, rolled, old fashioned"
    },

    "carrot": {
        "exact_description":
            "Carrots, mature, raw"
    },

    "rye_flour": {
        "exact_description":
            "Flour, rye"
    },

    "apple_fuji": {
        "exact_description":
            "Apples, fuji, with skin, raw"
    },

    "banana": {
        "exact_description":
            "Bananas, ripe and slightly ripe, raw"
    },

    "milk_whole": {
        "exact_description":
            "Milk, whole, 3.25% milkfat, with added vitamin D"
    },

    "mushroom_white_button": {
        "exact_description":
            "Mushrooms, white button"
    },
}


# ============================================================
# APPLICATION NUTRIENT POLICY
# ============================================================
#
# These are USDA nutrient IDs, but the application names are
# our stable internal names.
#
# Energy is selected separately because multiple KCAL
# nutrients exist in Foundation Foods.
# ============================================================

CRITICAL_NUTRIENTS = {
    "protein_g": {
        "nutrient_id": 1003,
        "expected_unit": "G",
        "expected_name": "Protein",
    },

    "fat_g": {
        "nutrient_id": 1004,
        "expected_unit": "G",
        "expected_name": "Total lipid (fat)",
    },

    "carbohydrate_g": {
        "nutrient_id": 1005,
        "expected_unit": "G",
        "expected_name": "Carbohydrate, by difference",
    },

    "fiber_g": {
        "nutrient_id": 1079,
        "expected_unit": "G",
        "expected_name": "Fiber, total dietary",
    },

    "sugar_g": {
        "nutrient_id": 1063,
        "expected_unit": "G",
        "expected_name": "Sugars, Total",
    },

    "saturated_fat_g": {
        "nutrient_id": 1258,
        "expected_unit": "G",
        "expected_name": "Fatty acids, total saturated",
    },
}


# ------------------------------------------------------------
# Energy policy
# ------------------------------------------------------------
#
# Select the KCAL nutrient with the highest number of UNIQUE
# Foundation Food records containing a valid value.
#
# If coverage ties:
#
#   2047 > 2048 > 1008
#
# This is an explicit application policy.
# ============================================================

ENERGY_PRIORITY = {
    2047: 3,   # Energy (Atwater General Factors)
    2048: 2,   # Energy (Atwater Specific Factors)
    1008: 1,   # Energy
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def print_section(title):
    """Print a consistent section heading."""

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def print_subsection(title):
    """Print a consistent subsection heading."""

    print()
    print("-" * 70)
    print(title)
    print("-" * 70)


def normalize_text(value):
    """
    Normalize text for exact comparison.

    This is NOT fuzzy matching.

    It:
      - converts NaN to empty string
      - Unicode normalizes
      - lowercases
      - converts punctuation/separators to spaces
      - collapses whitespace
    """

    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.lower().strip()

    # Convert punctuation/separators to spaces.
    value = re.sub(
        r"[^a-z0-9%]+",
        " ",
        value
    )

    # Collapse whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def require_columns(df, required_columns, filename):
    """Fail if required columns are missing."""

    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{filename} is missing required columns: "
            f"{missing}"
        )


def to_numeric_series(series):
    """Safely convert a pandas Series to numeric."""

    return pd.to_numeric(
        series,
        errors="coerce"
    )


# ============================================================
# START
# ============================================================

print_section(
    "LOADING USDA FOODDATA CENTRAL FILES"
)

print("Search directory:")
print(USDA_DIR)


# ============================================================
# FILE EXISTENCE CHECK
# ============================================================

print_subsection(
    "CHECKING USDA FILES"
)

required_files = [
    FOOD_CSV,
    FOOD_NUTRIENT_CSV,
    NUTRIENT_CSV,
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required USDA file not found:\n{path}"
        )

    print(
        f"PASS: {path.name}"
    )


# ============================================================
# SELECTED FILES
# ============================================================

print_subsection(
    "SELECTED USDA FILES"
)

print("food.csv:")
print(FOOD_CSV)

print("food_nutrient.csv:")
print(FOOD_NUTRIENT_CSV)

print("nutrient.csv:")
print(NUTRIENT_CSV)


# ============================================================
# READ CSV FILES
# ============================================================

print_section(
    "READING USDA CSV FILES"
)

food = pd.read_csv(
    FOOD_CSV,
    low_memory=False
)

food_nutrient = pd.read_csv(
    FOOD_NUTRIENT_CSV,
    low_memory=False
)

nutrient = pd.read_csv(
    NUTRIENT_CSV,
    low_memory=False
)

print(
    f"food.csv rows:          {len(food):,}"
)

print(
    f"food_nutrient.csv rows: {len(food_nutrient):,}"
)

print(
    f"nutrient.csv rows:      {len(nutrient):,}"
)


# ============================================================
# USDA COLUMN VALIDATION
# ============================================================

print_section(
    "USDA COLUMN VALIDATION"
)

require_columns(
    food,
    [
        "fdc_id",
        "description",
        "data_type",
    ],
    "food.csv"
)

require_columns(
    food_nutrient,
    [
        "fdc_id",
        "nutrient_id",
        "amount",
    ],
    "food_nutrient.csv"
)

require_columns(
    nutrient,
    [
        "id",
        "name",
        "unit_name",
    ],
    "nutrient.csv"
)

print(
    "food.csv columns: PASS"
)

print(
    "food_nutrient.csv columns: PASS"
)

print(
    "nutrient.csv columns: PASS"
)


# ============================================================
# NORMALIZE DATA TYPES
# ============================================================

print_section(
    "USDA DATA TYPES"
)

print(
    food["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


food["normalized_data_type"] = (
    food["data_type"]
    .astype("string")
    .str.lower()
    .str.replace(
        r"[^a-z0-9]+",
        "",
        regex=True
    )
)

print_subsection(
    "NORMALIZED DATA TYPES"
)

print(
    food["normalized_data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# ============================================================
# FOUNDATION FOOD DETECTION
# ============================================================

foundation_food = food[
    food["normalized_data_type"]
    == "foundationfood"
].copy()


if foundation_food.empty:

    print()
    print(
        "FOUNDATION FOOD DETECTION FAILED"
    )

    print(
        food["normalized_data_type"]
        .value_counts(dropna=False)
        .to_string()
    )

    raise RuntimeError(
        "No Foundation Foods were found."
    )


# ============================================================
# FOUNDATION FOOD SUMMARY
# ============================================================

print_section(
    "FOUNDATION FOOD SUMMARY"
)

print(
    f"Foundation Foods found: "
    f"{len(foundation_food):,}"
)

foundation_food_ids = set(
    foundation_food["fdc_id"]
    .astype(int)
)

print(
    f"Unique Foundation FDC IDs: "
    f"{len(foundation_food_ids):,}"
)

if len(foundation_food_ids) != len(foundation_food):

    duplicate_food_ids = (
        foundation_food[
            foundation_food["fdc_id"]
            .duplicated(keep=False)
        ]
        .sort_values("fdc_id")
    )

    print(
        duplicate_food_ids[
            [
                "fdc_id",
                "description",
                "data_type",
            ]
        ]
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate Foundation Food FDC IDs detected."
    )


# ============================================================
# PREPARE FOUNDATION NUTRIENTS
# ============================================================

foundation_food_nutrients = (
    food_nutrient[
        food_nutrient["fdc_id"]
        .isin(foundation_food_ids)
    ]
    .copy()
)


print_section(
    "FOUNDATION NUTRIENT COVERAGE"
)

print(
    f"Foundation nutrient rows: "
    f"{len(foundation_food_nutrients):,}"
)

print(
    f"Nutrient IDs available in Foundation Foods: "
    f"{foundation_food_nutrients['nutrient_id'].nunique():,}"
)


# ============================================================
# DUPLICATE FOOD/NUTRIENT CHECK
# ============================================================

print_section(
    "FOOD/NUTRIENT DUPLICATE CHECK"
)

duplicate_nutrient_rows = (
    foundation_food_nutrients
    .groupby(
        [
            "fdc_id",
            "nutrient_id",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

duplicate_nutrient_rows = (
    duplicate_nutrient_rows[
        duplicate_nutrient_rows["count"] > 1
    ]
    .copy()
)

if duplicate_nutrient_rows.empty:

    print(
        "PASS: No duplicate FDC ID + nutrient ID combinations."
    )

else:

    print(
        "WARNING:",
        len(duplicate_nutrient_rows),
        "duplicate food/nutrient combinations found."
    )

    print(
        duplicate_nutrient_rows
        .to_string(index=False)
    )


# ============================================================
# RESOLVE DUPLICATE FOOD/NUTRIENT RECORDS
# ============================================================
#
# Safe policy:
#
#   1. No duplicate:
#        keep row.
#
#   2. Duplicate rows with identical numeric amount:
#        collapse to one row.
#
#   3. Duplicate rows with conflicting amounts:
#        FAIL.
#
# We do NOT blindly use values.iloc[0].
# ============================================================

print_section(
    "RESOLVING FOOD/NUTRIENT RECORDS"
)


def resolve_duplicate_food_nutrients(df):
    """
    Canonicalize duplicate fdc_id + nutrient_id rows.

    Identical nutrient amounts are considered safe duplicates.

    Conflicting nutrient amounts cause the pipeline to fail,
    because silently choosing one would be unsafe.
    """

    key_columns = [
        "fdc_id",
        "nutrient_id",
    ]

    amount_numeric = (
        pd.to_numeric(
            df["amount"],
            errors="coerce"
        )
    )

    working = df.copy()

    working["_amount_numeric"] = amount_numeric

    conflicting_records = []

    canonical_rows = []

    for (
        (fdc_id, nutrient_id),
        group
    ) in working.groupby(
        key_columns,
        sort=False,
        dropna=False
    ):

        numeric_values = (
            group["_amount_numeric"]
            .dropna()
            .unique()
        )

        # ----------------------------------------------------
        # No duplicate
        # ----------------------------------------------------

        if len(group) == 1:

            row = group.iloc[0].copy()

            canonical_rows.append(
                row
            )

            continue

        # ----------------------------------------------------
        # All amounts missing
        # ----------------------------------------------------

        if len(numeric_values) == 0:

            row = group.iloc[0].copy()

            canonical_rows.append(
                row
            )

            continue

        # ----------------------------------------------------
        # Multiple different numeric values
        # ----------------------------------------------------

        if len(numeric_values) > 1:

            conflict = group.copy()

            conflict[
                "conflict_fdc_id"
            ] = fdc_id

            conflict[
                "conflict_nutrient_id"
            ] = nutrient_id

            conflicting_records.append(
                conflict
            )

            continue

        # ----------------------------------------------------
        # Duplicate rows but same numeric amount
        # ----------------------------------------------------

        row = group.iloc[0].copy()

        canonical_rows.append(
            row
        )

    # --------------------------------------------------------
    # Fail on conflicts
    # --------------------------------------------------------

    if conflicting_records:

        conflicts = pd.concat(
            conflicting_records,
            ignore_index=True
        )

        review_columns = [
            col
            for col in [
                "fdc_id",
                "nutrient_id",
                "amount",
                "data_points",
                "derivation_id",
                "min",
                "max",
                "median",
                "footnote",
            ]
            if col in conflicts.columns
        ]

        conflicts[
            review_columns
        ].to_csv(
            DUPLICATE_NUTRIENT_REVIEW_CSV,
            index=False
        )

        print()
        print(
            "ERROR: Conflicting duplicate nutrient values detected."
        )

        print(
            conflicts[
                review_columns
            ]
            .to_string(index=False)
        )

        print()
        print(
            "Conflict review saved to:"
        )

        print(
            DUPLICATE_NUTRIENT_REVIEW_CSV
        )

        raise RuntimeError(
            "Conflicting duplicate FDC ID + nutrient ID "
            "records detected. Pipeline stopped."
        )

    # --------------------------------------------------------
    # Build canonical table
    # --------------------------------------------------------

    canonical = pd.DataFrame(
        canonical_rows
    )

    canonical = (
        canonical
        .drop(
            columns=[
                "_amount_numeric"
            ],
            errors="ignore"
        )
        .reset_index(drop=True)
    )

    return canonical


foundation_food_nutrients_canonical = (
    resolve_duplicate_food_nutrients(
        foundation_food_nutrients
    )
)


# ============================================================
# CANONICAL TABLE VALIDATION
# ============================================================

duplicate_after_resolution = (
    foundation_food_nutrients_canonical
    .duplicated(
        subset=[
            "fdc_id",
            "nutrient_id",
        ]
    )
    .any()
)


if duplicate_after_resolution:

    raise RuntimeError(
        "Canonical nutrient table still contains duplicate "
        "FDC ID + nutrient ID combinations."
    )


print(
    "Canonical nutrient table created:"
)

print(
    f"Rows: "
    f"{len(foundation_food_nutrients_canonical):,}"
)

print(
    "Unique FDC ID + nutrient ID combinations: PASS"
)


# ============================================================
# NUTRIENT LOOKUP
# ============================================================

nutrient_lookup = (
    nutrient[
        [
            "id",
            "name",
            "unit_name",
        ]
    ]
    .copy()
)

nutrient_lookup["id"] = (
    pd.to_numeric(
        nutrient_lookup["id"],
        errors="coerce"
    )
)

nutrient_lookup = (
    nutrient_lookup
    .dropna(subset=["id"])
    .copy()
)

nutrient_lookup["id"] = (
    nutrient_lookup["id"]
    .astype(int)
)


# ============================================================
# MERGE NUTRIENT METADATA
# ============================================================

foundation_food_nutrients_canonical = (
    foundation_food_nutrients_canonical
    .merge(
        nutrient_lookup,
        left_on="nutrient_id",
        right_on="id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_nutrient")
    )
)


# ============================================================
# UNIQUE-FDC-ID NUTRIENT COVERAGE
# ============================================================

def nutrient_coverage(nutrient_id):
    """
    Return the number of UNIQUE Foundation Foods with a
    usable numeric value for the specified nutrient.
    """

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "nutrient_id"
            ]
            == int(nutrient_id)
        ]
    )

    if rows.empty:
        return 0

    valid_foods = rows[
        pd.to_numeric(
            rows["amount"],
            errors="coerce"
        ).notna()
    ]

    return int(
        valid_foods["fdc_id"].nunique()
    )


# ============================================================
# ENERGY NUTRIENTS
# ============================================================

print_section(
    "ENERGY NUTRIENTS ACTUALLY AVAILABLE"
)

energy_candidates = []

for nutrient_id, priority in ENERGY_PRIORITY.items():

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:
        continue

    nutrient_row = (
        matching.iloc[0]
    )

    coverage = nutrient_coverage(
        nutrient_id
    )

    energy_candidates.append(
        {
            "id": nutrient_id,
            "name": nutrient_row["name"],
            "unit_name": nutrient_row["unit_name"],
            "coverage": coverage,
            "priority": priority,
        }
    )


energy_candidates_df = pd.DataFrame(
    energy_candidates
)


if energy_candidates_df.empty:

    raise RuntimeError(
        "No supported KCAL energy nutrients were found."
    )


energy_candidates_df = (
    energy_candidates_df
    .sort_values(
        [
            "coverage",
            "priority",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


print(
    energy_candidates_df
    .to_string(index=False)
)


# ============================================================
# SELECT ENERGY NUTRIENT
# ============================================================

selected_energy = (
    energy_candidates_df.iloc[0]
)

ENERGY_NUTRIENT_ID = int(
    selected_energy["id"]
)

ENERGY_NUTRIENT_NAME = (
    selected_energy["name"]
)

ENERGY_NUTRIENT_UNIT = (
    selected_energy["unit_name"]
)

ENERGY_COVERAGE = int(
    selected_energy["coverage"]
)


if ENERGY_NUTRIENT_UNIT.upper() != "KCAL":

    raise RuntimeError(
        "Selected energy nutrient is not KCAL."
    )


# ============================================================
# BUILD FINAL NUTRIENT MAP
# ============================================================

NUTRIENT_MAP = {

    "calories_kcal": {
        "nutrient_id": ENERGY_NUTRIENT_ID,
        "expected_unit": "KCAL",
        "expected_name": ENERGY_NUTRIENT_NAME,
    },
}


for application_name, config in CRITICAL_NUTRIENTS.items():

    NUTRIENT_MAP[
        application_name
    ] = config


# ============================================================
# FINAL NUTRIENT MAP DISPLAY
# ============================================================

print_section(
    "FINAL FOUNDATION NUTRIENT MAP"
)

nutrient_mapping_rows = []

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Required nutrient ID not found: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    actual_name = str(
        nutrient_row["name"]
    )

    actual_unit = str(
        nutrient_row["unit_name"]
    )

    coverage = nutrient_coverage(
        nutrient_id
    )

    print(
        f"{application_name:<25} "
        f"ID={nutrient_id:>5} "
        f"UNIT={actual_unit:<5} "
        f"COVERAGE={coverage:>4} "
        f"NAME={actual_name}"
    )

    nutrient_mapping_rows.append(
        {
            "application_name":
                application_name,

            "USDA_nutrient_id":
                nutrient_id,

            "USDA_name":
                actual_name,

            "unit":
                actual_unit,

            "coverage":
                coverage,
        }
    )


nutrient_mapping_df = pd.DataFrame(
    nutrient_mapping_rows
)


# ============================================================
# NUTRIENT MAPPING VALIDATION
# ============================================================

print_section(
    "NUTRIENT MAPPING VALIDATION"
)

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Missing nutrient ID: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    actual_name = str(
        nutrient_row["name"]
    )

    actual_unit = str(
        nutrient_row["unit_name"]
    )

    expected_name = str(
        config["expected_name"]
    )

    expected_unit = str(
        config["expected_unit"]
    )

    if actual_name != expected_name:

        raise RuntimeError(
            f"NUTRIENT NAME VALIDATION FAILED: "
            f"{application_name}: "
            f"expected '{expected_name}', "
            f"got '{actual_name}'"
        )

    if actual_unit.upper() != expected_unit.upper():

        raise RuntimeError(
            f"NUTRIENT UNIT VALIDATION FAILED: "
            f"{application_name}: "
            f"expected '{expected_unit}', "
            f"got '{actual_unit}'"
        )


# Ensure no two application fields accidentally use
# the same nutrient ID.

used_nutrient_ids = [
    int(config["nutrient_id"])
    for config in NUTRIENT_MAP.values()
]

if len(used_nutrient_ids) != len(
    set(used_nutrient_ids)
):

    raise RuntimeError(
        "Two or more application nutrient fields "
        "use the same USDA nutrient ID."
    )


print(
    "NUTRIENT MAPPING VALIDATION: PASS"
)


# ============================================================
# FOUNDATION FOOD TARGET SEARCH
# ============================================================

print_section(
    "FOUNDATION FOOD TARGET SEARCH"
)

foundation_food = foundation_food.copy()

foundation_food[
    "normalized_description"
] = (
    foundation_food["description"]
    .apply(normalize_text)
)


target_matches = {}

for food_name, config in TARGET_FOODS.items():

    expected_description = (
        config["exact_description"]
    )

    normalized_expected = (
        normalize_text(
            expected_description
        )
    )

    matches = foundation_food[
        foundation_food[
            "normalized_description"
        ]
        == normalized_expected
    ].copy()

    target_matches[
        food_name
    ] = matches

    print_subsection(
        food_name.upper()
    )

    print(
        "Requested:"
    )

    print(
        expected_description
    )

    print(
        f"Matches: {len(matches)}"
    )

    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description",
                ]
            ]
            .sort_values("fdc_id")
            .to_string(index=False)
        )

    if matches.empty:

        raise RuntimeError(
            f"Food not found: "
            f"{food_name}: "
            f"{expected_description}"
        )


# ============================================================
# SELECT TARGET FOUNDATION FOODS
# ============================================================

print_section(
    "SELECTING TARGET FOUNDATION FOODS"
)


def requested_nutrient_completeness(
    fdc_id
):
    """
    Count how many requested application nutrients have
    non-null values for this food.

    This includes:
      calories_kcal
      protein_g
      fat_g
      carbohydrate_g
      fiber_g
      sugar_g
      saturated_fat_g
    """

    selected_nutrient_ids = [
        int(config["nutrient_id"])
        for config in NUTRIENT_MAP.values()
    ]

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "fdc_id"
            ]
            == int(fdc_id)
        ]
    )

    if rows.empty:
        return 0

    rows = rows[
        rows["nutrient_id"]
        .isin(selected_nutrient_ids)
    ]

    valid_count = (
        pd.to_numeric(
            rows["amount"],
            errors="coerce"
        )
        .notna()
        .sum()
    )

    return int(
        valid_count
    )


selected_foods = {}
food_selection_review_rows = []


for food_name, config in TARGET_FOODS.items():

    matches = (
        target_matches[
            food_name
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Single exact match
    # --------------------------------------------------------

    if len(matches) == 1:

        row = matches.iloc[0]

        fdc_id = int(
            row["fdc_id"]
        )

        selected_foods[
            food_name
        ] = row

        print_subsection(
            food_name.upper()
        )

        print(
            "STATUS: SELECTED"
        )

        print(
            f"FDC ID: {fdc_id}"
        )

        print(
            f"Description: "
            f"{row['description']}"
        )

        food_selection_review_rows.append(
            {
                "food_name":
                    food_name,

                "fdc_id":
                    fdc_id,

                "description":
                    row["description"],

                "match_count":
                    len(matches),

                "requested_nutrient_completeness":
                    requested_nutrient_completeness(
                        fdc_id
                    ),

                "selection_reason":
                    "Unique exact normalized description match",
            }
        )

        continue

    # --------------------------------------------------------
    # Duplicate exact description
    # --------------------------------------------------------

    matches[
        "requested_nutrient_completeness"
    ] = (
        matches["fdc_id"]
        .apply(
            requested_nutrient_completeness
        )
    )

    # Highest completeness first.
    #
    # FDC ID ascending is ONLY a deterministic tie-breaker.
    matches = (
        matches
        .sort_values(
            [
                "requested_nutrient_completeness",
                "fdc_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )

    selected_row = (
        matches.iloc[0]
    )

    selected_fdc_id = int(
        selected_row["fdc_id"]
    )

    selected_foods[
        food_name
    ] = selected_row

    print_subsection(
        food_name.upper()
    )

    print(
        "STATUS: DUPLICATE_DESCRIPTION"
    )

    print(
        matches[
            [
                "fdc_id",
                "description",
                "requested_nutrient_completeness",
            ]
        ]
        .to_string(index=False)
    )

    print()
    print(
        f"Selected preferred FDC ID: "
        f"{selected_fdc_id}"
    )

    print(
        "Reason: highest requested nutrient "
        "completeness; FDC ID used as "
        "deterministic tie-breaker."
    )

    for _, review_row in matches.iterrows():

        food_selection_review_rows.append(
            {
                "food_name":
                    food_name,

                "fdc_id":
                    int(review_row["fdc_id"]),

                "description":
                    review_row["description"],

                "match_count":
                    len(matches),

                "requested_nutrient_completeness":
                    int(
                        review_row[
                            "requested_nutrient_completeness"
                        ]
                    ),

                "selection_reason":
                    (
                        "Highest requested nutrient "
                        "completeness; FDC ID used "
                        "as deterministic tie-breaker"
                    ),
            }
        )


food_selection_review_df = pd.DataFrame(
    food_selection_review_rows
)


# ============================================================
# SELECTED FOOD DESCRIPTION VALIDATION
# ============================================================

print_section(
    "SELECTED FOOD DESCRIPTION VALIDATION"
)

for food_name, config in TARGET_FOODS.items():

    if food_name not in selected_foods:

        raise RuntimeError(
            f"Selected food missing: "
            f"{food_name}"
        )

    selected = (
        selected_foods[
            food_name
        ]
    )

    actual = normalize_text(
        selected["description"]
    )

    expected = normalize_text(
        config["exact_description"]
    )

    if actual != expected:

        raise RuntimeError(
            f"DESCRIPTION VALIDATION FAILED: "
            f"{food_name}: "
            f"expected '{expected}', "
            f"got '{actual}'"
        )

    print(
        f"PASS: {food_name:<30} "
        f"-> {selected['description']}"
    )


# ============================================================
# BUILD FINAL FOOD DATABASE
# ============================================================

print_section(
    "BUILDING FINAL FOOD DATABASE"
)


def get_food_nutrients(
    fdc_id
):
    """
    Return application nutrient values for one FDC ID.

    Missing nutrients remain NaN.

    This function operates only on the canonical nutrient table,
    so duplicate nutrient rows cannot silently overwrite values.
    """

    rows = (
        foundation_food_nutrients_canonical[
            foundation_food_nutrients_canonical[
                "fdc_id"
            ]
            == int(fdc_id)
        ]
    )

    result = {}

    for application_name, config in NUTRIENT_MAP.items():

        nutrient_id = int(
            config["nutrient_id"]
        )

        matching = rows[
            rows["nutrient_id"]
            == nutrient_id
        ]

        if matching.empty:

            result[
                application_name
            ] = np.nan

            continue

        if len(matching) != 1:

            raise RuntimeError(
                f"Canonical nutrient table has "
                f"unexpected duplicate rows: "
                f"FDC {fdc_id}, "
                f"nutrient {nutrient_id}"
            )

        value = pd.to_numeric(
            matching.iloc[0]["amount"],
            errors="coerce"
        )

        if pd.isna(value):

            result[
                application_name
            ] = np.nan

        else:

            result[
                application_name
            ] = float(value)

    return result


database_rows = []

source_mapping_rows = []


for food_name, config in TARGET_FOODS.items():

    selected = (
        selected_foods[
            food_name
        ]
    )

    fdc_id = int(
        selected["fdc_id"]
    )

    description = str(
        selected["description"]
    )

    nutrient_values = (
        get_food_nutrients(
            fdc_id
        )
    )

    row = {

        "food_name":
            food_name,

        "fdc_id":
            fdc_id,

        "description":
            description,

        "data_type":
            "Foundation Food",

        "basis_g":
            100,
    }

    row.update(
        nutrient_values
    )

    database_rows.append(
        row
    )

    source_mapping_rows.append(
        {
            "food_name":
                food_name,

            "fdc_id":
                fdc_id,

            "description":
                description,

            "data_type":
                "Foundation Food",

            "basis_g":
                100,

            "source_file":
                str(FOOD_CSV),

            "food_nutrient_source_file":
                str(FOOD_NUTRIENT_CSV),

            "nutrient_source_file":
                str(NUTRIENT_CSV),

            "energy_nutrient_id":
                ENERGY_NUTRIENT_ID,

            "energy_nutrient_name":
                ENERGY_NUTRIENT_NAME,
        }
    )


food_database = pd.DataFrame(
    database_rows
)


source_mapping_df = pd.DataFrame(
    source_mapping_rows
)


# ============================================================
# COLUMN ORDER
# ============================================================

database_columns = [

    "food_name",
    "fdc_id",
    "description",
    "data_type",
    "basis_g",

    "calories_kcal",

    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g",
]


food_database = (
    food_database[
        database_columns
    ]
)


# ============================================================
# FINAL FOOD DATABASE DISPLAY
# ============================================================

print_section(
    "FINAL FOOD DATABASE"
)

print(
    food_database.to_string(
        index=False
    )
)


# ============================================================
# NUTRIENT DATABASE SANITY CHECK
# ============================================================

print_section(
    "NUTRIENT DATABASE SANITY CHECK"
)

for application_name, config in NUTRIENT_MAP.items():

    nutrient_id = int(
        config["nutrient_id"]
    )

    matching = nutrient_lookup[
        nutrient_lookup["id"]
        == nutrient_id
    ]

    if matching.empty:

        raise RuntimeError(
            f"Missing nutrient ID: "
            f"{nutrient_id}"
        )

    nutrient_row = (
        matching.iloc[0]
    )

    print(
        f"PASS: {application_name:<25} "
        f"-> {nutrient_row['name']}"
    )


print()
print(
    "No nutrient IDs are incorrectly reused."
)


# ============================================================
# NUTRIENT COVERAGE COUNTS
# ============================================================

nutrient_columns = list(
    NUTRIENT_MAP.keys()
)

for column in nutrient_columns:

    coverage_count = (
        food_database[column]
        .notna()
        .sum()
    )

    print(
        f"Foods with {column}: "
        f"{coverage_count}/"
        f"{len(food_database)}"
    )


# ============================================================
# NEGATIVE NUTRIENT CHECK
# ============================================================

negative_records = []

for column in nutrient_columns:

    numeric_values = pd.to_numeric(
        food_database[column],
        errors="coerce"
    )

    mask = (
        numeric_values < 0
    )

    if mask.any():

        negative_rows = (
            food_database.loc[
                mask,
                [
                    "food_name",
                    "fdc_id",
                    column,
                ]
            ]
            .copy()
        )

        negative_records.append(
            negative_rows
        )


if negative_records:

    negative_df = pd.concat(
        negative_records,
        ignore_index=True
    )

    print()
    print(
        negative_df
        .to_string(index=False)
    )

    raise RuntimeError(
        "Negative nutrient values detected."
    )

else:

    print()
    print(
        "No negative nutrient values detected."
    )


# ============================================================
# 100-G BASIS VALIDATION
# ============================================================

if not (
    food_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "100 g basis validation failed."
    )

print(
    "100 g basis validation: PASS"
)


# ============================================================
# FOUNDATION FOOD DATA TYPE VALIDATION
# ============================================================

if not (
    food_database["data_type"]
    .eq("Foundation Food")
    .all()
):

    raise RuntimeError(
        "Foundation Food data type validation failed."
    )

print(
    "Foundation Food data type validation: PASS"
)


# ============================================================
# CRITICAL NUTRIENT COVERAGE
# ============================================================

required_for_database = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g",
]

critical_missing = []

for column in required_for_database:

    if (
        food_database[column]
        .isna()
        .any()
    ):

        missing_foods = (
            food_database.loc[
                food_database[column].isna(),
                "food_name"
            ]
            .tolist()
        )

        critical_missing.append(
            (
                column,
                missing_foods
            )
        )


if critical_missing:

    for column, foods in critical_missing:

        print(
            f"WARNING: {column} missing for "
            f"{foods}"
        )

    raise RuntimeError(
        "Critical nutrient coverage failed."
    )

else:

    print(
        "Critical nutrient coverage: PASS"
    )


# ============================================================
# MACRO ENERGY COMPARISON
# ============================================================
#
# This is diagnostic only.
#
# It does NOT replace USDA calorie values.
#
# Simple calculation:
#
#   protein * 4
# + carbohydrate * 4
# + fat * 9
#
# USDA energy can differ because the actual USDA energy
# methodology is more specific than this simple calculation.
# ============================================================

print_section(
    "MACRO ENERGY COMPARISON"
)

macro_rows = []


for _, row in food_database.iterrows():

    protein = row["protein_g"]
    carbs = row["carbohydrate_g"]
    fat = row["fat_g"]
    calories = row["calories_kcal"]

    if any(
        pd.isna(value)
        for value in [
            protein,
            carbs,
            fat,
            calories,
        ]
    ):

        macro_calories = np.nan
        difference = np.nan

    else:

        macro_calories = (
            protein * 4
            + carbs * 4
            + fat * 9
        )

        difference = (
            calories
            - macro_calories
        )

    macro_rows.append(
        {
            "food_name":
                row["food_name"],

            "calories_kcal":
                calories,

            "macro_calorie_check":
                macro_calories,

            "macro_calorie_difference":
                difference,
        }
    )


macro_comparison_df = pd.DataFrame(
    macro_rows
)


print(
    macro_comparison_df
    .to_string(index=False)
)


# ============================================================
# DATABASE VALIDATION
# ============================================================

print_section(
    "DATABASE VALIDATION"
)

expected_food_names = set(
    TARGET_FOODS.keys()
)

actual_food_names = set(
    food_database["food_name"]
)

missing_food_names = (
    expected_food_names
    -
    actual_food_names
)

unexpected_food_names = (
    actual_food_names
    -
    expected_food_names
)


if missing_food_names:

    raise RuntimeError(
        f"Missing expected foods: "
        f"{sorted(missing_food_names)}"
    )


if unexpected_food_names:

    raise RuntimeError(
        f"Unexpected foods in database: "
        f"{sorted(unexpected_food_names)}"
    )


if (
    len(food_database)
    != len(TARGET_FOODS)
):

    raise RuntimeError(
        "Final food count does not match TARGET_FOODS."
    )


if (
    food_database["fdc_id"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Duplicate FDC IDs exist in final database."
    )


if (
    food_database["food_name"]
    .nunique()
    != len(food_database)
):

    raise RuntimeError(
        "Duplicate food names exist in final database."
    )


if not (
    food_database["basis_g"]
    .eq(100)
    .all()
):

    raise RuntimeError(
        "Not all foods use a 100 g basis."
    )


if not (
    food_database["data_type"]
    .eq("Foundation Food")
    .all()
):

    raise RuntimeError(
        "Not all records are Foundation Foods."
    )


# Validate every final description again.

for _, row in food_database.iterrows():

    food_name = row["food_name"]

    expected = normalize_text(
        TARGET_FOODS[
            food_name
        ]["exact_description"]
    )

    actual = normalize_text(
        row["description"]
    )

    if expected != actual:

        raise RuntimeError(
            f"Final description mismatch: "
            f"{food_name}"
        )


print(
    f"Foods: {len(food_database)}"
)

print(
    f"Unique FDC IDs: "
    f"{food_database['fdc_id'].nunique()}"
)

print(
    "Basis values:",
    food_database["basis_g"]
    .unique()
    .tolist()
)

print(
    "Data types:",
    food_database["data_type"]
    .unique()
    .tolist()
)

print(
    "Critical nutrient coverage: PASS"
)

print()
print(
    "STATUS: PASS"
)


# ============================================================
# SAVE DATABASE FILES
# ============================================================

print_section(
    "SAVING DATABASE FILES"
)


# ------------------------------------------------------------
# Database
# ------------------------------------------------------------

food_database.to_csv(
    DATABASE_CSV,
    index=False
)

print(
    "Database:"
)

print(
    DATABASE_CSV
)


# ------------------------------------------------------------
# Source mapping
# ------------------------------------------------------------

source_mapping_df.to_csv(
    SOURCE_MAPPING_CSV,
    index=False
)

print()
print(
    "Source mapping:"
)

print(
    SOURCE_MAPPING_CSV
)


# ------------------------------------------------------------
# Food selection review
# ------------------------------------------------------------

food_selection_review_df.to_csv(
    FOOD_SELECTION_REVIEW_CSV,
    index=False
)

print()
print(
    "Duplicate food review:"
)

print(
    FOOD_SELECTION_REVIEW_CSV
)


# ------------------------------------------------------------
# Nutrient mapping
# ------------------------------------------------------------

nutrient_mapping_df.to_csv(
    NUTRIENT_MAPPING_CSV,
    index=False
)

print()
print(
    "Nutrient mapping:"
)

print(
    NUTRIENT_MAPPING_CSV
)


# ------------------------------------------------------------
# Duplicate nutrient review
# ------------------------------------------------------------
#
# Even when duplicates are safely resolved, save the original
# duplicate records for audit purposes.
# ------------------------------------------------------------

if duplicate_nutrient_rows.empty:

    duplicate_review_df = pd.DataFrame(
        columns=[
            "fdc_id",
            "nutrient_id",
            "count",
        ]
    )

else:

    duplicate_review_df = (
        foundation_food_nutrients[
            foundation_food_nutrients[
                [
                    "fdc_id",
                    "nutrient_id",
                ]
            ]
            .apply(
                tuple,
                axis=1
            )
            .isin(
                duplicate_nutrient_rows[
                    [
                        "fdc_id",
                        "nutrient_id",
                    ]
                ]
                .apply(
                    tuple,
                    axis=1
                )
            )
        ]
        .copy()
    )


duplicate_review_df.to_csv(
    DUPLICATE_NUTRIENT_REVIEW_CSV,
    index=False
)

print()
print(
    "Duplicate nutrient review:"
)

print(
    DUPLICATE_NUTRIENT_REVIEW_CSV
)


# ============================================================
# RELOAD SAVED DATABASE
# ============================================================

print_section(
    "RELOADING SAVED DATABASE"
)

reloaded_database = pd.read_csv(
    DATABASE_CSV
)


# ------------------------------------------------------------
# Reload structure
# ------------------------------------------------------------

if list(
    reloaded_database.columns
) != database_columns:

    raise RuntimeError(
        "Saved database columns do not match "
        "expected schema."
    )


# ------------------------------------------------------------
# Reload count
# ------------------------------------------------------------

if len(
    reloaded_database
) != len(
    food_database
):

    raise RuntimeError(
        "Saved database row count changed after reload."
    )


# ------------------------------------------------------------
# Reload FDC IDs
# ------------------------------------------------------------

if set(
    reloaded_database["fdc_id"]
    .astype(int)
) != set(
    food_database["fdc_id"]
    .astype(int)
):

    raise RuntimeError(
        "Saved database FDC IDs do not match original."
    )


# ------------------------------------------------------------
# Reload food names
# ------------------------------------------------------------

if set(
    reloaded_database["food_name"]
) != set(
    food_database["food_name"]
):

    raise RuntimeError(
        "Saved database food names do not match original."
    )


print(
    "Saved database reload validation: PASS"
)


# ============================================================
# NUTRITION CALCULATION FUNCTIONS
# ============================================================

print_section(
    "NUTRITION CALCULATOR"
)


def nutrition_for_grams(
    food_name,
    grams,
    database=None
):
    """
    Calculate nutrition for a specified food amount.

    Database values are stored per 100 g.

    Missing nutrients remain NaN.

    Example:

        nutrition_for_grams(
            "chicken_breast",
            150
        )
    """

    if database is None:
        database = reloaded_database

    if food_name not in set(
        database["food_name"]
    ):

        raise KeyError(
            f"Unknown food: {food_name}"
        )

    grams = float(
        grams
    )

    if grams < 0:

        raise ValueError(
            "Food weight cannot be negative."
        )

    row = (
        database[
            database["food_name"]
            == food_name
        ]
        .iloc[0]
    )

    multiplier = (
        grams / 100.0
    )

    result = {}

    for nutrient_column in nutrient_columns:

        value = pd.to_numeric(
            row[nutrient_column],
            errors="coerce"
        )

        if pd.isna(value):

            result[
                nutrient_column
            ] = np.nan

        else:

            result[
                nutrient_column
            ] = float(
                value * multiplier
            )

    return result


def print_nutrition(
    nutrition
):
    """
    Print a nutrition result while preserving
    missing-vs-zero distinction.
    """

    display_names = {

        "calories_kcal":
            "calories_kcal",

        "protein_g":
            "protein_g",

        "fat_g":
            "fat_g",

        "carbohydrate_g":
            "carbohydrate_g",

        "fiber_g":
            "fiber_g",

        "sugar_g":
            "sugar_g",

        "saturated_fat_g":
            "saturated_fat_g",
    }

    for key in nutrient_columns:

        value = nutrition.get(
            key,
            np.nan
        )

        if pd.isna(value):

            display_value = "-"

        else:

            display_value = (
                f"{value:.2f}"
            )

        print(
            f"{display_names[key]:<25}: "
            f"{display_value}"
        )


# ============================================================
# EXAMPLE NUTRITION TEST
# ============================================================

print_subsection(
    "EXAMPLE NUTRITION TEST"
)

example_food = (
    "chicken_breast"
)

example_grams = 150.0

print()
print(
    f"{example_food} — "
    f"{example_grams:g} g"
)

example_nutrition = (
    nutrition_for_grams(
        example_food,
        example_grams
    )
)

print_nutrition(
    example_nutrition
)


# ============================================================
# MEAL CALCULATOR
# ============================================================

print_section(
    "MEAL"
)


meal = [

    (
        "chicken_breast",
        150.0
    ),

    (
        "oats_rolled",
        50.0
    ),

    (
        "apple_fuji",
        150.0
    ),
]


for food_name, grams in meal:

    print(
        f"{food_name:<25} "
        f"{grams:>8.1f} g"
    )


def calculate_meal(
    meal_items,
    database=None
):
    """
    Calculate total nutrition for a meal.

    meal_items format:

        [
            ("chicken_breast", 150),
            ("oats_rolled", 50),
            ("apple_fuji", 150),
        ]

    Missing nutrients are preserved.

    Important:
        If a nutrient is missing from every food,
        result is NaN.

        If a nutrient is present for at least one food,
        available values are summed while missing values
        contribute no numerical amount.

    This means:
        NaN != 0
    """

    if database is None:
        database = reloaded_database

    totals = {
        column: 0.0
        for column in nutrient_columns
    }

    present_any = {
        column: False
        for column in nutrient_columns
    }

    for food_name, grams in meal_items:

        nutrition = (
            nutrition_for_grams(
                food_name,
                grams,
                database
            )
        )

        for column in nutrient_columns:

            value = nutrition[
                column
            ]

            if pd.isna(value):

                continue

            present_any[
                column
            ] = True

            totals[
                column
            ] += float(value)

    # Convert nutrients that were unavailable for the
    # entire meal back to NaN.
    for column in nutrient_columns:

        if not present_any[column]:

            totals[column] = np.nan

    return totals


meal_totals = calculate_meal(
    meal
)


print()
print("-" * 70)
print("TOTAL NUTRITION")
print("-" * 70)

print_nutrition(
    meal_totals
)


# ============================================================
# FINAL PIPELINE SUMMARY
# ============================================================

print_section(
    "PIPELINE COMPLETE"
)

duplicate_food_count = (
    sum(
        len(
            matches
        ) > 1
        for matches in target_matches.values()
    )
)


print(
    f"Foundation Foods available: "
    f"{len(foundation_food):,}"
)

print(
    f"Foods selected: "
    f"{len(food_database):,}"
)

print(
    f"Foods requiring duplicate review: "
    f"{duplicate_food_count}"
)

print(
    "Foods not found: 0"
)

print()
print(
    "Database validation: PASS"
)

print()
print(
    "Database:"
)

print(
    DATABASE_CSV
)

print()
print(
    "Source mapping:"
)

print(
    SOURCE_MAPPING_CSV
)

print()
print(
    "Review:"
)

print(
    FOOD_SELECTION_REVIEW_CSV
)

print()
print(
    "Nutrient mapping:"
)

print(
    NUTRIENT_MAPPING_CSV
)

print()
print(
    "Duplicate nutrient review:"
)

print(
    DUPLICATE_NUTRIENT_REVIEW_CSV
)

print()
print(
    "Done."
)



LOADING USDA FOODDATA CENTRAL FILES
Search directory:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30

----------------------------------------------------------------------
CHECKING USDA FILES
----------------------------------------------------------------------
PASS: food.csv
PASS: food_nutrient.csv
PASS: nutrient.csv

----------------------------------------------------------------------
SELECTED USDA FILES
----------------------------------------------------------------------
food.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
food_nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv
nutrient.csv:
C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nut

### sr 

In [30]:
# ============================================================
# USDA FOODDATA CENTRAL — SR LEGACY DATASET AUDITOR
# ============================================================
#
# PURPOSE
# -------
# This script performs a complete structural audit of a USDA
# FoodData Central SR Legacy dataset BEFORE we build the
# SR_LEGACY application database.
#
# IMPORTANT:
#
# This script does NOT:
#   - select foods
#   - fuzzy match foods
#   - build the final database
#   - modify source files
#   - assume a particular USDA schema
#
# It only investigates and documents the dataset.
#
# It examines:
#
#   1. Every CSV file recursively
#   2. File size
#   3. Row count
#   4. Column count
#   5. Column names
#   6. Data types
#   7. Missing values
#   8. Unique values
#   9. Duplicate rows
#  10. Candidate primary keys
#  11. Candidate foreign keys
#  12. Food identifiers
#  13. Nutrient identifiers
#  14. Portion / serving structures
#  15. Food category structures
#  16. Relationships between files
#  17. Important categorical values
#  18. Sample records
#
# OUTPUT
# ------
# Creates an audit directory containing:
#
#   sr_legacy_audit_report.txt
#   sr_legacy_file_inventory.csv
#   sr_legacy_column_inventory.csv
#   sr_legacy_missing_values.csv
#   sr_legacy_duplicate_summary.csv
#   sr_legacy_unique_values.csv
#   sr_legacy_key_analysis.csv
#   sr_legacy_relationship_analysis.csv
#
# After this script finishes, we can use the report to design
# the actual SR_LEGACY database builder correctly.
#
# ============================================================


import os
import re
import math
import json
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# CHANGE THIS PATH
# ------------------------------------------------------------

BASE_DIR = (
    r"C:\Users\AK\Downloads\zip"
    r""
    # Add your actual SR Legacy folder here.
)


# Example:
#
# BASE_DIR = (
#     r"C:\Users\AK\Downloads\zip"
#     r"\FoodData_Central_sr_legacy_food_csv_2026-04-30"
#     r"\FoodData_Central_sr_legacy_food_csv_2026-04-30"
# )


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

AUDIT_DIR = os.path.join(
    BASE_DIR,
    "sr_legacy_audit"
)


# ============================================================
# AUDIT CONFIGURATION
# ============================================================

# Number of sample rows to display/store.
SAMPLE_ROWS = 5


# Number of unique values to report for categorical columns.
MAX_UNIQUE_VALUES_TO_REPORT = 50


# Columns with very high cardinality are not dumped completely.
MAX_FULL_UNIQUE_VALUES = 100


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def normalize_text(value):
    """
    Normalize text for analysis only.

    This function is NOT used for food matching.
    """

    if pd.isna(value):

        return ""

    value = str(value)

    value = value.strip().lower()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def safe_string(value):
    """
    Convert arbitrary values to printable strings.
    """

    if pd.isna(value):

        return ""

    return str(value)


def format_bytes(size):
    """
    Convert bytes to human-readable size.
    """

    if size < 1024:

        return f"{size} B"

    if size < 1024 ** 2:

        return f"{size / 1024:.2f} KB"

    if size < 1024 ** 3:

        return f"{size / (1024 ** 2):.2f} MB"

    return f"{size / (1024 ** 3):.2f} GB"


def percentage(
    numerator,
    denominator
):
    """
    Safely calculate percentage.
    """

    if denominator == 0:

        return 0.0

    return (
        float(numerator)
        /
        float(denominator)
        *
        100.0
    )


def infer_column_role(
    column_name
):
    """
    Infer the likely semantic role of a column.

    This is ONLY an audit hint.

    It does not define the schema.
    """

    name = (
        str(column_name)
        .strip()
        .lower()
    )

    # --------------------------------------------------------
    # Identifiers
    # --------------------------------------------------------

    if name == "fdc_id":

        return "FOOD_IDENTIFIER"

    if name in {
        "id",
        "nutrient_id",
        "food_id",
        "food_nutrient_id",
        "portion_id",
        "measure_unit_id",
        "category_id"
    }:

        return "IDENTIFIER"


    # --------------------------------------------------------
    # Nutrient-related
    # --------------------------------------------------------

    if "nutrient" in name:

        return "NUTRIENT_RELATED"


    # --------------------------------------------------------
    # Food descriptions
    # --------------------------------------------------------

    if name in {
        "description",
        "food_description",
        "food_name"
    }:

        return "FOOD_DESCRIPTION"


    # --------------------------------------------------------
    # Data type
    # --------------------------------------------------------

    if "data_type" in name:

        return "DATA_TYPE"


    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    if "category" in name:

        return "CATEGORY"


    # --------------------------------------------------------
    # Portion / serving
    # --------------------------------------------------------

    if any(
        token in name
        for token in [
            "portion",
            "serving",
            "measure",
            "gram_weight",
            "household"
        ]
    ):

        return "PORTION_OR_MEASURE"


    # --------------------------------------------------------
    # Units
    # --------------------------------------------------------

    if "unit" in name:

        return "UNIT"


    # --------------------------------------------------------
    # Amount
    # --------------------------------------------------------

    if name in {
        "amount",
        "value",
        "quantity",
        "weight"
    }:

        return "NUMERIC_VALUE"


    # --------------------------------------------------------
    # Date
    # --------------------------------------------------------

    if (
        "date" in name
        or
        name.endswith("_dt")
    ):

        return "DATE"


    # --------------------------------------------------------
    # Default
    # --------------------------------------------------------

    return "OTHER"


# ============================================================
# CREATE AUDIT DIRECTORY
# ============================================================

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


# ============================================================
# START REPORT
# ============================================================

REPORT_LINES = []


def report(
    text=""
):
    """
    Add a line to the report and print it.
    """

    REPORT_LINES.append(
        str(text)
    )

    print(text)


report("")
report("=" * 80)
report("USDA FOODDATA CENTRAL SR LEGACY DATASET AUDIT")
report("=" * 80)

report("")
report(
    f"Audit started: "
    f"{datetime.now().isoformat()}"
)

report("")
report("Dataset directory:")
report(BASE_DIR)


# ============================================================
# VERIFY DIRECTORY
# ============================================================

if not os.path.isdir(BASE_DIR):

    raise FileNotFoundError(
        "\nSR Legacy directory does not exist:\n"
        f"{BASE_DIR}\n\n"
        "Please change BASE_DIR to the actual "
        "SR Legacy dataset directory."
    )


# ============================================================
# DISCOVER ALL CSV FILES
# ============================================================

report("")
report("=" * 80)
report("DISCOVERING ALL CSV FILES")
report("=" * 80)


csv_files = sorted(
    [
        path
        for path in Path(BASE_DIR).rglob("*.csv")
        if "sr_legacy_audit"
        not in str(path).lower()
    ]
)


if not csv_files:

    raise RuntimeError(
        "No CSV files were found in the SR Legacy directory."
    )


report("")
report(
    f"CSV files discovered: {len(csv_files)}"
)


for path in csv_files:

    report(
        f"  {path}"
    )


# ============================================================
# FILE INVENTORY
# ============================================================

file_inventory_rows = []


for file_path in csv_files:

    file_size = os.path.getsize(
        file_path
    )


    file_inventory_rows.append({

        "file_name":
            file_path.name,

        "relative_path":
            os.path.relpath(
                file_path,
                BASE_DIR
            ),

        "absolute_path":
            str(file_path),

        "size_bytes":
            file_size,

        "size_human":
            format_bytes(
                file_size
            )
    })


file_inventory = pd.DataFrame(
    file_inventory_rows
)


file_inventory.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_file_inventory.csv"
    ),
    index=False
)


# ============================================================
# STORAGE FOR ALL ANALYSIS
# ============================================================

loaded_tables = {}

column_inventory_rows = []

missing_value_rows = []

duplicate_summary_rows = []

unique_value_rows = []

key_analysis_rows = []

relationship_analysis_rows = []


# ============================================================
# READ EVERY FILE
# ============================================================

for file_number, file_path in enumerate(
    csv_files,
    start=1
):

    relative_path = os.path.relpath(
        file_path,
        BASE_DIR
    )


    report("")
    report("=" * 80)
    report(
        f"FILE {file_number}/{len(csv_files)}"
    )
    report("=" * 80)

    report("")
    report(
        f"File: {relative_path}"
    )


    file_size = os.path.getsize(
        file_path
    )


    report(
        f"Size: {format_bytes(file_size)}"
    )


    # --------------------------------------------------------
    # READ CSV
    # --------------------------------------------------------

    try:

        df = pd.read_csv(
            file_path,
            low_memory=False
        )

    except Exception as exc:

        report("")
        report(
            "ERROR READING FILE:"
        )

        report(
            repr(exc)
        )

        # Record failed file.

        file_inventory.loc[
            file_inventory["relative_path"]
            == relative_path,
            "read_error"
        ] = repr(exc)

        continue


    table_key = relative_path

    loaded_tables[
        table_key
    ] = df


    # --------------------------------------------------------
    # BASIC FILE INFORMATION
    # --------------------------------------------------------

    row_count = len(df)

    column_count = len(df.columns)


    report("")
    report(
        f"Rows:    {row_count:,}"
    )

    report(
        f"Columns: {column_count:,}"
    )


    report("")
    report("Columns:")
    report("-" * 80)


    for column in df.columns:

        report(
            f"  {column}"
        )


    # --------------------------------------------------------
    # DUPLICATE ROWS
    # --------------------------------------------------------

    duplicate_rows = int(
        df.duplicated()
        .sum()
    )


    duplicate_percentage = percentage(
        duplicate_rows,
        row_count
    )


    report("")
    report("Duplicate rows:")
    report(
        f"  {duplicate_rows:,} "
        f"({duplicate_percentage:.4f}%)"
    )


    duplicate_summary_rows.append({

        "file":
            relative_path,

        "rows":
            row_count,

        "duplicate_rows":
            duplicate_rows,

        "duplicate_percentage":
            duplicate_percentage
    })


    # --------------------------------------------------------
    # COLUMN ANALYSIS
    # --------------------------------------------------------

    for column in df.columns:

        series = df[column]


        dtype = str(
            series.dtype
        )


        null_count = int(
            series.isna()
            .sum()
        )


        non_null_count = (
            row_count
            -
            null_count
        )


        null_percentage = percentage(
            null_count,
            row_count
        )


        unique_count = int(
            series.nunique(
                dropna=True
            )
        )


        role = infer_column_role(
            column
        )


        column_inventory_rows.append({

            "file":
                relative_path,

            "column":
                column,

            "dtype":
                dtype,

            "rows":
                row_count,

            "null_count":
                null_count,

            "null_percentage":
                null_percentage,

            "non_null_count":
                non_null_count,

            "unique_count":
                unique_count,

            "role_hint":
                role
        })


        missing_value_rows.append({

            "file":
                relative_path,

            "column":
                column,

            "rows":
                row_count,

            "null_count":
                null_count,

            "non_null_count":
                non_null_count,

            "null_percentage":
                null_percentage
        })


        # ----------------------------------------------------
        # UNIQUE VALUES
        # ----------------------------------------------------

        # Only analyze low-cardinality columns deeply.

        if unique_count <= MAX_FULL_UNIQUE_VALUES:

            value_counts = (
                series
                .value_counts(
                    dropna=False
                )
                .head(
                    MAX_UNIQUE_VALUES_TO_REPORT
                )
            )


            for value, count in (
                value_counts.items()
            ):

                unique_value_rows.append({

                    "file":
                        relative_path,

                    "column":
                        column,

                    "unique_count":
                        unique_count,

                    "value":
                        safe_string(value),

                    "count":
                        int(count),

                    "percentage":
                        percentage(
                            count,
                            row_count
                        )
                })


        # ----------------------------------------------------
        # KEY CANDIDATE ANALYSIS
        # ----------------------------------------------------

        if row_count > 0:

            uniqueness_ratio = (
                unique_count
                /
                row_count
            )

        else:

            uniqueness_ratio = 0


        if (
            unique_count
            == row_count
            and
            row_count > 0
        ):

            key_candidate = (
                "STRONG_SINGLE_COLUMN_KEY"
            )

        elif uniqueness_ratio >= 0.95:

            key_candidate = (
                "POSSIBLE_KEY"
            )

        else:

            key_candidate = ""


        key_analysis_rows.append({

            "file":
                relative_path,

            "column":
                column,

            "rows":
                row_count,

            "unique_values":
                unique_count,

            "uniqueness_ratio":
                uniqueness_ratio,

            "key_candidate":
                key_candidate
        })


    # --------------------------------------------------------
    # PRINT IMPORTANT COLUMNS
    # --------------------------------------------------------

    report("")
    report("Column summary:")
    report("-" * 80)


    summary_df = pd.DataFrame([
        {

            "column":
                column,

            "dtype":
                str(df[column].dtype),

            "nulls":
                int(
                    df[column].isna().sum()
                ),

            "unique":
                int(
                    df[column].nunique(
                        dropna=True
                    )
                ),

            "role":
                infer_column_role(
                    column
                )
        }

        for column in df.columns
    ])


    report(
        summary_df.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # SAMPLE DATA
    # --------------------------------------------------------

    report("")
    report(
        f"First {SAMPLE_ROWS} rows:"
    )
    report("-" * 80)


    if row_count > 0:

        report(
            df.head(
                SAMPLE_ROWS
            )
            .to_string(
                index=False
            )
        )

    else:

        report(
            "FILE IS EMPTY"
        )


# ============================================================
# SAVE BASIC AUDIT TABLES
# ============================================================

column_inventory = pd.DataFrame(
    column_inventory_rows
)


missing_values = pd.DataFrame(
    missing_value_rows
)


duplicate_summary = pd.DataFrame(
    duplicate_summary_rows
)


unique_values = pd.DataFrame(
    unique_value_rows
)


key_analysis = pd.DataFrame(
    key_analysis_rows
)


column_inventory.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_column_inventory.csv"
    ),
    index=False
)


missing_values.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_missing_values.csv"
    ),
    index=False
)


duplicate_summary.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_duplicate_summary.csv"
    ),
    index=False
)


unique_values.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_unique_values.csv"
    ),
    index=False
)


key_analysis.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_key_analysis.csv"
    ),
    index=False
)


# ============================================================
# CROSS-FILE COLUMN DISCOVERY
# ============================================================

report("")
report("=" * 80)
report("CROSS-FILE COLUMN DISCOVERY")
report("=" * 80)


all_columns = {}


for file_name, df in loaded_tables.items():

    for column in df.columns:

        all_columns.setdefault(
            column,
            []
        )

        all_columns[
            column
        ].append(
            file_name
        )


for column in sorted(
    all_columns.keys()
):

    files = all_columns[
        column
    ]


    report("")
    report(
        f"{column}"
    )


    for file_name in files:

        report(
            f"    {file_name}"
        )


# ============================================================
# IDENTIFIER DISCOVERY
# ============================================================

report("")
report("=" * 80)
report("IDENTIFIER DISCOVERY")
report("=" * 80)


identifier_columns = [

    "fdc_id",
    "id",
    "nutrient_id",
    "food_id",
    "food_nutrient_id",
    "portion_id",
    "measure_unit_id",
    "category_id"
]


for file_name, df in loaded_tables.items():

    matching_columns = [

        column

        for column in df.columns

        if str(column).lower()
        in identifier_columns
    ]


    if not matching_columns:

        continue


    report("")
    report(
        f"FILE: {file_name}"
    )


    for column in matching_columns:

        series = df[column]


        numeric_series = pd.to_numeric(
            series,
            errors="coerce"
        )


        report(
            f"  {column}:"
        )

        report(
            f"    dtype: {series.dtype}"
        )

        report(
            f"    non-null: "
            f"{numeric_series.notna().sum():,}"
        )

        report(
            f"    unique: "
            f"{numeric_series.nunique():,}"
        )

        report(
            f"    min: "
            f"{numeric_series.min()}"
        )

        report(
            f"    max: "
            f"{numeric_series.max()}"
        )


# ============================================================
# POSSIBLE RELATIONSHIPS
# ============================================================

report("")
report("=" * 80)
report("CROSS-FILE RELATIONSHIP ANALYSIS")
report("=" * 80)


# ------------------------------------------------------------
# Build a map:
#
# column name -> files containing that column
# ------------------------------------------------------------

column_files = {}


for file_name, df in loaded_tables.items():

    for column in df.columns:

        normalized_column = (
            str(column)
            .strip()
            .lower()
        )


        column_files.setdefault(
            normalized_column,
            []
        )


        column_files[
            normalized_column
        ].append(
            file_name
        )


# ------------------------------------------------------------
# Analyze common columns appearing in multiple files
# ------------------------------------------------------------

for column, files in sorted(
    column_files.items()
):

    if len(files) < 2:

        continue


    report("")
    report(
        f"COMMON COLUMN: {column}"
    )


    for file_name in files:

        df = loaded_tables[
            file_name
        ]


        series = df[
            [
                col
                for col in df.columns
                if str(col).strip().lower()
                == column
            ][0]
        ]


        values = set(
            series
            .dropna()
            .astype(str)
            .head(100000)
            .tolist()
        )


        report(
            f"  {file_name}: "
            f"{len(values):,} sampled values"
        )


    # --------------------------------------------------------
    # Pairwise overlap
    # --------------------------------------------------------

    for i in range(
        len(files)
    ):

        for j in range(
            i + 1,
            len(files)
        ):

            file_a = files[i]

            file_b = files[j]


            df_a = loaded_tables[
                file_a
            ]


            df_b = loaded_tables[
                file_b
            ]


            col_a = [

                col

                for col in df_a.columns

                if str(col).strip().lower()
                == column
            ][0]


            col_b = [

                col

                for col in df_b.columns

                if str(col).strip().lower()
                == column
            ][0]


            values_a = set(
                df_a[col_a]
                .dropna()
                .astype(str)
                .tolist()
            )


            values_b = set(
                df_b[col_b]
                .dropna()
                .astype(str)
                .tolist()
            )


            overlap = (
                values_a
                &
                values_b
            )


            relationship_analysis_rows.append({

                "column":
                    column,

                "file_a":
                    file_a,

                "file_b":
                    file_b,

                "unique_a":
                    len(values_a),

                "unique_b":
                    len(values_b),

                "overlap":
                    len(overlap),

                "overlap_pct_of_a":
                    percentage(
                        len(overlap),
                        len(values_a)
                    ),

                "overlap_pct_of_b":
                    percentage(
                        len(overlap),
                        len(values_b)
                    )
            })


            report(
                f"    {file_a}"
            )

            report(
                f"       ↕️"
            )

            report(
                f"    {file_b}"
            )

            report(
                f"       overlap: "
                f"{len(overlap):,}"
            )


# ============================================================
# SAVE RELATIONSHIP REPORT
# ============================================================

relationship_analysis = pd.DataFrame(
    relationship_analysis_rows
)


relationship_analysis.to_csv(
    os.path.join(
        AUDIT_DIR,
        "sr_legacy_relationship_analysis.csv"
    ),
    index=False
)


# ============================================================
# SPECIAL FOODDATA CENTRAL ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("FOODDATA CENTRAL STRUCTURAL ANALYSIS")
report("=" * 80)


# ============================================================
# FOOD TABLE DETECTION
# ============================================================

food_tables = []


for file_name, df in loaded_tables.items():

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }


    if (
        "fdc_id" in columns_lower
        and
        "description" in columns_lower
    ):

        food_tables.append(
            file_name
        )


report("")
report("Possible food tables:")


if food_tables:

    for file_name in food_tables:

        report(
            f"  {file_name}"
        )

else:

    report(
        "  NONE DETECTED"
    )


# ============================================================
# NUTRIENT TABLE DETECTION
# ============================================================

nutrient_tables = []


for file_name, df in loaded_tables.items():

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }


    if (
        "id" in columns_lower
        and
        "name" in columns_lower
        and
        (
            "unit_name"
            in columns_lower
            or
            "unit" in columns_lower
        )
    ):

        nutrient_tables.append(
            file_name
        )


report("")
report("Possible nutrient definition tables:")


if nutrient_tables:

    for file_name in nutrient_tables:

        report(
            f"  {file_name}"
        )

else:

    report(
        "  NONE DETECTED"
    )


# ============================================================
# FOOD-NUTRIENT TABLE DETECTION
# ============================================================

food_nutrient_tables = []


for file_name, df in loaded_tables.items():

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }


    if (
        "fdc_id" in columns_lower
        and
        "nutrient_id" in columns_lower
        and
        "amount" in columns_lower
    ):

        food_nutrient_tables.append(
            file_name
        )


report("")
report("Possible food-nutrient tables:")


if food_nutrient_tables:

    for file_name in food_nutrient_tables:

        report(
            f"  {file_name}"
        )

else:

    report(
        "  NONE DETECTED"
    )


# ============================================================
# PORTION / MEASURE TABLE DETECTION
# ============================================================

portion_tables = []


for file_name, df in loaded_tables.items():

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }


    portion_score = 0


    for column in columns_lower:

        if "portion" in column:

            portion_score += 1

        if "gram_weight" in column:

            portion_score += 1

        if "measure" in column:

            portion_score += 1

        if "household" in column:

            portion_score += 1

        if "serving" in column:

            portion_score += 1


    if portion_score >= 2:

        portion_tables.append(
            (
                file_name,
                portion_score
            )
        )


report("")
report("Possible portion/measure tables:")


if portion_tables:

    for file_name, score in (
        portion_tables
    ):

        report(
            f"  {file_name} "
            f"(score={score})"
        )

else:

    report(
        "  NONE DETECTED"
    )


# ============================================================
# CATEGORY TABLE DETECTION
# ============================================================

category_tables = []


for file_name, df in loaded_tables.items():

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }


    if any(
        "category" in column
        for column in columns_lower
    ):

        category_tables.append(
            file_name
        )


report("")
report("Possible category tables:")


if category_tables:

    for file_name in category_tables:

        report(
            f"  {file_name}"
        )

else:

    report(
        "  NONE DETECTED"
    )


# ============================================================
# DATA TYPE ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("DATA TYPE ANALYSIS")
report("=" * 80)


for file_name, df in loaded_tables.items():

    if "data_type" not in df.columns:

        continue


    report("")
    report(
        f"FILE: {file_name}"
    )


    counts = (
        df["data_type"]
        .astype(str)
        .value_counts(
            dropna=False
        )
    )


    report(
        counts.to_string()
    )


# ============================================================
# DESCRIPTION ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("FOOD DESCRIPTION ANALYSIS")
report("=" * 80)


for file_name in food_tables:

    df = loaded_tables[
        file_name
    ]


    report("")
    report(
        f"FILE: {file_name}"
    )


    description_column = [

        column

        for column in df.columns

        if str(column).lower()
        == "description"
    ]


    if not description_column:

        continue


    description_column = (
        description_column[0]
    )


    descriptions = (
        df[description_column]
        .dropna()
        .astype(str)
    )


    report(
        f"Descriptions: "
        f"{len(descriptions):,}"
    )


    report(
        f"Unique descriptions: "
        f"{descriptions.nunique():,}"
    )


    duplicate_descriptions = (
        descriptions
        .value_counts()
        .loc[
            lambda x: x > 1
        ]
    )


    report(
        f"Duplicate descriptions: "
        f"{len(duplicate_descriptions):,}"
    )


    if not duplicate_descriptions.empty:

        report("")
        report(
            "Top duplicate descriptions:"
        )


        report(
            duplicate_descriptions
            .head(20)
            .to_string()
        )


# ============================================================
# FOOD ID ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("FDC_ID ANALYSIS")
report("=" * 80)


for file_name, df in loaded_tables.items():

    if "fdc_id" not in df.columns:

        continue


    fdc = pd.to_numeric(
        df["fdc_id"],
        errors="coerce"
    )


    report("")
    report(
        f"FILE: {file_name}"
    )


    report(
        f"Rows: "
        f"{len(df):,}"
    )


    report(
        f"Valid FDC IDs: "
        f"{fdc.notna().sum():,}"
    )


    report(
        f"Unique FDC IDs: "
        f"{fdc.nunique():,}"
    )


    report(
        f"Duplicated FDC IDs: "
        f"{fdc.duplicated().sum():,}"
    )


# ============================================================
# NUTRIENT ID ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("NUTRIENT_ID ANALYSIS")
report("=" * 80)


for file_name, df in loaded_tables.items():

    if "nutrient_id" not in df.columns:

        continue


    nutrient_ids = pd.to_numeric(
        df["nutrient_id"],
        errors="coerce"
    )


    report("")
    report(
        f"FILE: {file_name}"
    )


    report(
        f"Rows: "
        f"{len(df):,}"
    )


    report(
        f"Valid nutrient IDs: "
        f"{nutrient_ids.notna().sum():,}"
    )


    report(
        f"Unique nutrient IDs: "
        f"{nutrient_ids.nunique():,}"
    )


    report(
        f"Duplicated nutrient IDs: "
        f"{nutrient_ids.duplicated().sum():,}"
    )


# ============================================================
# FOOD + NUTRIENT COMBINATION ANALYSIS
# ============================================================

report("")
report("=" * 80)
report("FOOD/NUTRIENT COMBINATION ANALYSIS")
report("=" * 80)


for file_name, df in loaded_tables.items():

    if not {
        "fdc_id",
        "nutrient_id"
    }.issubset(
        df.columns
    ):

        continue


    combination_counts = (
        df.groupby(
            [
                "fdc_id",
                "nutrient_id"
            ],
            dropna=False
        )
        .size()
    )


    duplicate_combinations = (
        combination_counts[
            combination_counts > 1
        ]
    )


    report("")
    report(
        f"FILE: {file_name}"
    )


    report(
        f"Unique FDC + nutrient combinations: "
        f"{len(combination_counts):,}"
    )


    report(
        f"Duplicate combinations: "
        f"{len(duplicate_combinations):,}"
    )


    if not duplicate_combinations.empty:

        report("")
        report(
            "Duplicate combinations:"
        )


        report(
            duplicate_combinations
            .head(50)
            .to_string()
        )


# ============================================================
# NUTRIENT DEFINITION SUMMARY
# ============================================================

report("")
report("=" * 80)
report("NUTRIENT DEFINITION SUMMARY")
report("=" * 80)


for file_name in nutrient_tables:

    df = loaded_tables[
        file_name
    ]


    report("")
    report(
        f"FILE: {file_name}"
    )


    display_columns = [

        column

        for column in [
            "id",
            "name",
            "unit_name",
            "nutrient_nbr",
            "rank"
        ]

        if column in df.columns
    ]


    if display_columns:

        report(
            df[
                display_columns
            ]
            .head(100)
            .to_string(
                index=False
            )
        )

    else:

        report(
            "No standard nutrient columns detected."
        )


# ============================================================
# PORTION / MEASURE SUMMARY
# ============================================================

report("")
report("=" * 80)
report("PORTION / MEASURE SUMMARY")
report("=" * 80)


for file_name, score in portion_tables:

    df = loaded_tables[
        file_name
    ]


    report("")
    report(
        f"FILE: {file_name}"
    )


    report(
        "Columns:"
    )


    for column in df.columns:

        report(
            f"  {column}"
        )


    report("")
    report(
        f"Sample rows:"
    )


    report(
        df.head(
            SAMPLE_ROWS
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# FOOD CATEGORY SUMMARY
# ============================================================

report("")
report("=" * 80)
report("FOOD CATEGORY SUMMARY")
report("=" * 80)


for file_name in category_tables:

    df = loaded_tables[
        file_name
    ]


    report("")
    report(
        f"FILE: {file_name}"
    )


    for column in df.columns:

        if "category" not in (
            str(column)
            .lower()
        ):

            continue


        report("")
        report(
            f"Column: {column}"
        )


        counts = (
            df[column]
            .astype(str)
            .value_counts()
            .head(50)
        )


        report(
            counts.to_string()
        )


# ============================================================
# SEARCH FOR IMPORTANT FOOD TYPES
# ============================================================
#
# This is exploratory only.
#
# It does NOT define SR_LEGACY membership.
# ============================================================

report("")
report("=" * 80)
report("EXPLORATORY INGREDIENT KEYWORD ANALYSIS")
report("=" * 80)


ingredient_keywords = {

    "spices_herbs": [

        "spice",
        "spices",
        "cinnamon",
        "turmeric",
        "basil",
        "oregano",
        "thyme",
        "rosemary",
        "parsley",
        "pepper",
        "paprika"
    ],

    "baking": [

        "baking powder",
        "baking soda",
        "yeast",
        "leavening",
        "cream of tartar"
    ],

    "niche_produce": [

        "jackfruit",
        "dragonfruit",
        "dragon fruit",
        "kelp",
        "seaweed"
    ],

    "oils_fats": [

        "olive oil",
        "coconut oil",
        "sesame oil",
        "canola oil",
        "vegetable oil",
        "lard",
        "shortening"
    ],

    "seasonings": [

        "salt",
        "sodium chloride",
        "seasoning"
    ],

    "meat_cuts": [

        "shoulder",
        "loin",
        "sirloin",
        "tenderloin",
        "rib",
        "chop",
        "roast",
        "shank",
        "brisket"
    ]
}


for file_name in food_tables:

    df = loaded_tables[
        file_name
    ]


    description_columns = [

        column

        for column in df.columns

        if str(column).lower()
        == "description"
    ]


    if not description_columns:

        continue


    description_column = (
        description_columns[0]
    )


    descriptions = (
        df[description_column]
        .fillna("")
        .astype(str)
    )


    normalized_descriptions = (
        descriptions
        .str.lower()
    )


    report("")
    report(
        f"FILE: {file_name}"
    )


    for category, keywords in (
        ingredient_keywords.items()
    ):

        mask = pd.Series(
            False,
            index=df.index
        )


        for keyword in keywords:

            mask = (
                mask
                |
                normalized_descriptions
                .str.contains(
                    re.escape(keyword),
                    regex=True,
                    na=False
                )
            )


        matches = df.loc[
            mask
        ]


        report("")
        report(
            f"{category}: "
            f"{len(matches):,} matches"
        )


        if not matches.empty:

            report(
                matches[
                    [
                        column
                        for column
                        in [
                            "fdc_id",
                            "description",
                            "data_type",
                            "food_category_id"
                        ]
                        if column
                        in matches.columns
                    ]
                ]
                .head(30)
                .to_string(
                    index=False
                )
            )


# ============================================================
# GENERATE MASTER REPORT
# ============================================================

report("")
report("=" * 80)
report("AUDIT SUMMARY")
report("=" * 80)


report("")
report(
    f"CSV files discovered: "
    f"{len(csv_files)}"
)


report(
    f"CSV files successfully loaded: "
    f"{len(loaded_tables)}"
)


report(
    f"Total column definitions analyzed: "
    f"{len(column_inventory)}"
)


report(
    f"Potential key columns analyzed: "
    f"{len(key_analysis)}"
)


report(
    f"Cross-file relationships analyzed: "
    f"{len(relationship_analysis)}"
)


report("")
report(
    "IMPORTANT:"
)


report(
    "This audit does NOT define the SR_LEGACY food-selection rules."
)


report(
    "The results must be reviewed before the database builder "
    "is written."
)


# ============================================================
# WRITE TEXT REPORT
# ============================================================

report_path = os.path.join(
    AUDIT_DIR,
    "sr_legacy_audit_report.txt"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "\n".join(
            REPORT_LINES
        )
    )


# ============================================================
# FINAL FILE LIST
# ============================================================

print("")
print("=" * 80)
print("SR LEGACY AUDIT COMPLETE")
print("=" * 80)

print("")
print("Audit directory:")
print(AUDIT_DIR)

print("")
print("Files created:")

for filename in sorted(
    os.listdir(AUDIT_DIR)
):

    print(
        "  ",
        filename
    )


print("")
print("IMPORTANT:")
print(
    "Do NOT build the SR_LEGACY database yet."
)

print(
    "Review the audit output first."
)

print("")
print("Done.")


USDA FOODDATA CENTRAL SR LEGACY DATASET AUDIT

Audit started: 2026-08-16T12:01:02.236507

Dataset directory:
C:\Users\AK\Downloads\zip

DISCOVERING ALL CSV FILES

CSV files discovered: 65
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\acquisition_samples.csv
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\agricultural_samples.csv
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\duplicate_food_nutrient_review.csv
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_attribute.csv
  C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\Foo

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from datetime import datetime


# =============================================================================
# CONFIGURATION
# =============================================================================

ROOT_DIR = Path(r"C:\Users\AK\Downloads\zip")

OUTPUT_DIR = ROOT_DIR / "sr_legacy_database"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASIS_G = 100.0


# =============================================================================
# DISPLAY HELPERS
# =============================================================================

def header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def section(title):
    print("\n" + "-" * 80)
    print(title)
    print("-" * 80)


def safe_read_csv(path):
    """
    Read CSV with a few fallbacks because USDA files can contain
    mixed data types.
    """
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        return pd.read_csv(path, low_memory=False, encoding="latin1")


def normalize_columns(df):
    df = df.copy()
    df.columns = [
        str(c).strip().lower().replace("\ufeff", "")
        for c in df.columns
    ]
    return df


def find_sr_legacy_directory(root):
    candidates = list(
        root.rglob("FoodData_Central_sr_legacy_food_csv_2018-04")
    )

    if candidates:
        # Prefer directory containing sr_legacy_food.csv
        for candidate in candidates:
            if (candidate / "sr_legacy_food.csv").exists():
                return candidate

    # Fallback: search directly for sr_legacy_food.csv
    files = list(root.rglob("sr_legacy_food.csv"))

    if files:
        return files[0].parent

    raise FileNotFoundError(
        "Could not locate sr_legacy_food.csv under:\n"
        f"{root}"
    )


def require_file(directory, filename):
    path = directory / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Required SR Legacy file not found:\n{path}"
        )

    return path


# =============================================================================
# FIND DATASET
# =============================================================================

header("USDA FOODDATA CENTRAL SR LEGACY DATABASE BUILDER")

print(
    "Build started:",
    datetime.now().isoformat()
)

print("Search directory:")
print(ROOT_DIR)

sr_dir = find_sr_legacy_directory(ROOT_DIR)

print("\nSR Legacy directory:")
print(sr_dir)


# =============================================================================
# REQUIRED FILES
# =============================================================================

header("CHECKING REQUIRED SR LEGACY FILES")

required_files = {
    "food": "food.csv",
    "sr_legacy_food": "sr_legacy_food.csv",
    "food_nutrient": "food_nutrient.csv",
    "nutrient": "nutrient.csv",
    "food_portion": "food_portion.csv",
    "measure_unit": "measure_unit.csv",
    "food_category": "food_category.csv",
}

paths = {}

for key, filename in required_files.items():
    path = require_file(sr_dir, filename)
    paths[key] = path

    size_mb = path.stat().st_size / (1024 * 1024)

    print(
        f"PASS: {filename:<45} "
        f"{size_mb:,.2f} MB"
    )


# =============================================================================
# OPTIONAL FILES
# =============================================================================

section("CHECKING OPTIONAL SUPPORT FILES")

optional_files = {
    "food_attribute": "food_attribute.csv",
    "food_attribute_type": "food_attribute_type.csv",
    "food_calorie_conversion_factor":
        "food_calorie_conversion_factor.csv",
    "food_nutrient_conversion_factor":
        "food_nutrient_conversion_factor.csv",
    "food_nutrient_derivation":
        "food_nutrient_derivation.csv",
    "food_nutrient_source":
        "food_nutrient_source.csv",
    "food_protein_conversion_factor":
        "food_protein_conversion_factor.csv",
    "food_update_log_entry":
        "food_update_log_entry.csv",
    "retention_factor":
        "retention_factor.csv",
}

optional_paths = {}

for key, filename in optional_files.items():
    path = sr_dir / filename

    if path.exists():
        optional_paths[key] = path
        print(f"FOUND: {filename}")
    else:
        print(f"NOT FOUND: {filename}")


# =============================================================================
# LOAD CORE FILES
# =============================================================================

header("READING SR LEGACY CSV FILES")

food = normalize_columns(
    safe_read_csv(paths["food"])
)

sr_legacy_food = normalize_columns(
    safe_read_csv(paths["sr_legacy_food"])
)

food_nutrient = normalize_columns(
    safe_read_csv(paths["food_nutrient"])
)

nutrient = normalize_columns(
    safe_read_csv(paths["nutrient"])
)

food_portion = normalize_columns(
    safe_read_csv(paths["food_portion"])
)

measure_unit = normalize_columns(
    safe_read_csv(paths["measure_unit"])
)

food_category = normalize_columns(
    safe_read_csv(paths["food_category"])
)

print(f"food.csv rows:              {len(food):,}")
print(f"sr_legacy_food.csv rows:    {len(sr_legacy_food):,}")
print(f"food_nutrient.csv rows:     {len(food_nutrient):,}")
print(f"nutrient.csv rows:          {len(nutrient):,}")
print(f"food_portion.csv rows:      {len(food_portion):,}")
print(f"measure_unit.csv rows:      {len(measure_unit):,}")
print(f"food_category.csv rows:     {len(food_category):,}")


# =============================================================================
# COLUMN VALIDATION
# =============================================================================

header("CORE COLUMN VALIDATION")

required_columns = {
    "food": {
        "fdc_id",
        "data_type",
        "description",
        "food_category_id",
    },

    "sr_legacy_food": {
        "fdc_id",
    },

    "food_nutrient": {
        "fdc_id",
        "nutrient_id",
        "amount",
    },

    "nutrient": {
        "id",
        "name",
        "unit_name",
    },

    "food_portion": {
        "fdc_id",
    },

    "measure_unit": {
        "id",
        "name",
    },
}

for name, columns in required_columns.items():

    df = locals()[name]

    missing = columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )

    print(f"PASS: {name}")


# =============================================================================
# NORMALIZE FDC IDs
# =============================================================================

section("NORMALIZING IDENTIFIERS")

for df_name in [
    "food",
    "sr_legacy_food",
    "food_nutrient",
    "food_portion",
]:
    df = locals()[df_name]

    if "fdc_id" in df.columns:
        df["fdc_id"] = pd.to_numeric(
            df["fdc_id"],
            errors="coerce"
        ).astype("Int64")

print("FDC ID normalization: PASS")


# =============================================================================
# SR LEGACY MEMBERSHIP
# =============================================================================

header("IDENTIFYING SR LEGACY FOODS")

sr_ids = (
    sr_legacy_food["fdc_id"]
    .dropna()
    .astype("int64")
    .drop_duplicates()
)

print(
    f"Unique SR Legacy FDC IDs: {len(sr_ids):,}"
)

food_sr = food[
    food["fdc_id"].isin(sr_ids)
].copy()

print(
    f"Foods matched in food.csv: {len(food_sr):,}"
)

missing_food_records = set(sr_ids) - set(
    food_sr["fdc_id"].astype("int64")
)

if missing_food_records:
    print(
        "WARNING:",
        len(missing_food_records),
        "SR Legacy IDs not found in food.csv"
    )
else:
    print("SR Legacy membership validation: PASS")


# =============================================================================
# DUPLICATE FOOD IDs
# =============================================================================

section("CHECKING SR LEGACY FOOD DUPLICATES")

duplicate_food_ids = (
    food_sr["fdc_id"]
    .value_counts()
)

duplicate_food_ids = duplicate_food_ids[
    duplicate_food_ids > 1
]

print(
    f"Duplicate FDC IDs in food table: "
    f"{len(duplicate_food_ids):,}"
)

if len(duplicate_food_ids):
    print(
        duplicate_food_ids.head(20)
    )


# =============================================================================
# DATA TYPE VALIDATION
# =============================================================================

header("SR LEGACY DATA TYPE VALIDATION")

print(
    food_sr["data_type"]
    .value_counts(dropna=False)
    .to_string()
)


# =============================================================================
# FOOD CATEGORY
# =============================================================================

section("BUILDING FOOD CATEGORY MAP")

category_map = food_category.copy()

if "id" in category_map.columns:
    category_map["id"] = pd.to_numeric(
        category_map["id"],
        errors="coerce"
    ).astype("Int64")

category_name_column = None

for candidate in [
    "description",
    "name",
    "category",
]:
    if candidate in category_map.columns:
        category_name_column = candidate
        break

if category_name_column:
    category_map = category_map[
        ["id", category_name_column]
    ].drop_duplicates(
        "id"
    )

    category_map = category_map.rename(
        columns={
            "id": "food_category_id",
            category_name_column: "category_name",
        }
    )

    food_sr = food_sr.merge(
        category_map,
        on="food_category_id",
        how="left"
    )
else:
    food_sr["category_name"] = pd.NA

print(
    "Food category mapping completed."
)


# =============================================================================
# NUTRIENT TABLE
# =============================================================================

header("BUILDING NUTRIENT REFERENCE TABLE")

nutrient = nutrient.copy()

nutrient["id"] = pd.to_numeric(
    nutrient["id"],
    errors="coerce"
).astype("Int64")

nutrient["name"] = nutrient["name"].astype("string")

nutrient["unit_name"] = nutrient[
    "unit_name"
].astype("string")

print(
    f"Unique nutrient definitions: "
    f"{nutrient['id'].nunique():,}"
)


# =============================================================================
# SR LEGACY NUTRIENT RECORDS
# =============================================================================

header("FILTERING SR LEGACY NUTRIENT RECORDS")

sr_nutrients = food_nutrient[
    food_nutrient["fdc_id"].isin(sr_ids)
].copy()

print(
    f"SR Legacy nutrient rows: "
    f"{len(sr_nutrients):,}"
)

print(
    f"Unique foods with nutrient records: "
    f"{sr_nutrients['fdc_id'].nunique():,}"
)

print(
    f"Unique nutrient IDs used: "
    f"{sr_nutrients['nutrient_id'].nunique():,}"
)


# =============================================================================
# JOIN NUTRIENT DEFINITIONS
# =============================================================================

section("JOINING NUTRIENT DEFINITIONS")

sr_nutrients = sr_nutrients.merge(
    nutrient[
        ["id", "name", "unit_name"]
    ],
    left_on="nutrient_id",
    right_on="id",
    how="left",
    suffixes=("", "_reference")
)

missing_nutrient_definitions = sr_nutrients[
    sr_nutrients["name"].isna()
]["nutrient_id"].nunique()

print(
    "Missing nutrient definitions:",
    missing_nutrient_definitions
)

if missing_nutrient_definitions == 0:
    print("Nutrient definition validation: PASS")


# =============================================================================
# DUPLICATE FOOD/NUTRIENT RECORDS
# =============================================================================

header("CHECKING FOOD/NUTRIENT DUPLICATES")

duplicate_counts = (
    sr_nutrients
    .groupby(
        ["fdc_id", "nutrient_id"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_counts[
    duplicate_counts["count"] > 1
]

print(
    f"Duplicate food/nutrient combinations: "
    f"{len(duplicates):,}"
)


# =============================================================================
# CANONICAL NUTRIENT RECORD SELECTION
# =============================================================================

section("RESOLVING NUTRIENT RECORDS")

# Prefer rows with numeric amount.
sr_nutrients["_amount_numeric"] = pd.to_numeric(
    sr_nutrients["amount"],
    errors="coerce"
)

sr_nutrients["_has_amount"] = (
    sr_nutrients["_amount_numeric"].notna()
)

# Stable deterministic ordering.
sort_columns = [
    "fdc_id",
    "nutrient_id",
    "_has_amount",
]

sr_nutrients = sr_nutrients.sort_values(
    sort_columns,
    ascending=[True, True, False]
)

canonical_nutrients = (
    sr_nutrients
    .drop_duplicates(
        ["fdc_id", "nutrient_id"],
        keep="first"
    )
    .copy()
)

print(
    f"Canonical nutrient rows: "
    f"{len(canonical_nutrients):,}"
)

unique_pairs = (
    canonical_nutrients[
        ["fdc_id", "nutrient_id"]
    ]
    .drop_duplicates()
)

if len(unique_pairs) == len(canonical_nutrients):
    print(
        "Unique FDC ID + nutrient ID: PASS"
    )


# =============================================================================
# NUTRIENT SEARCH HELPERS
# =============================================================================

def find_nutrients(pattern):
    regex = re.compile(
        pattern,
        flags=re.IGNORECASE
    )

    result = nutrient[
        nutrient["name"]
        .fillna("")
        .str.contains(regex)
    ].copy()

    return result[
        ["id", "name", "unit_name"]
    ]


# =============================================================================
# SHOW ENERGY NUTRIENTS
# =============================================================================

header("ENERGY NUTRIENTS AVAILABLE IN SR LEGACY")

energy_candidates = nutrient[
    nutrient["name"]
    .fillna("")
    .str.contains(
        "energy",
        case=False,
        regex=False
    )
].copy()

print(
    energy_candidates[
        ["id", "name", "unit_name"]
    ].to_string(index=False)
)


# =============================================================================
# NUTRIENT MAPPING
# =============================================================================

header("BUILDING SR LEGACY NUTRIENT MAP")


def choose_nutrient(
    exact_names=None,
    contains_patterns=None,
    allowed_units=None,
):
    exact_names = exact_names or []
    contains_patterns = contains_patterns or []

    candidates = nutrient.copy()

    candidates["name_lower"] = (
        candidates["name"]
        .fillna("")
        .str.lower()
        .str.strip()
    )

    if allowed_units:
        candidates = candidates[
            candidates["unit_name"]
            .fillna("")
            .str.upper()
            .isin(
                [u.upper() for u in allowed_units]
            )
        ]

    # Exact match first
    for name in exact_names:
        match = candidates[
            candidates["name_lower"] == name.lower()
        ]

        if not match.empty:
            return match.iloc[0]

    # Contains match second
    for pattern in contains_patterns:

        match = candidates[
            candidates["name"]
            .fillna("")
            .str.contains(
                pattern,
                case=False,
                regex=True
            )
        ]

        if not match.empty:
            return match.iloc[0]

    return None


nutrient_map = {}

# Energy
energy = choose_nutrient(
    contains_patterns=[
        r"^Energy$",
        r"Energy.*Atwater",
    ],
    allowed_units=["KCAL"]
)

if energy is not None:
    nutrient_map["calories_kcal"] = energy
else:
    nutrient_map["calories_kcal"] = None


# Protein
protein = choose_nutrient(
    exact_names=[
        "Protein"
    ],
    allowed_units=["G"]
)

nutrient_map["protein_g"] = protein


# Fat
fat = choose_nutrient(
    exact_names=[
        "Total lipid (fat)"
    ],
    allowed_units=["G"]
)

nutrient_map["fat_g"] = fat


# Carbohydrate
carb = choose_nutrient(
    exact_names=[
        "Carbohydrate, by difference"
    ],
    allowed_units=["G"]
)

nutrient_map["carbohydrate_g"] = carb


# Fiber
fiber = choose_nutrient(
    exact_names=[
        "Fiber, total dietary"
    ],
    allowed_units=["G"]
)

nutrient_map["fiber_g"] = fiber


# Sugar
sugar = choose_nutrient(
    exact_names=[
        "Sugars, Total"
    ],
    contains_patterns=[
        r"^Sugars"
    ],
    allowed_units=["G"]
)

nutrient_map["sugar_g"] = sugar


# Saturated fat
satfat = choose_nutrient(
    exact_names=[
        "Fatty acids, total saturated"
    ],
    allowed_units=["G"]
)

nutrient_map["saturated_fat_g"] = satfat


# Sodium
sodium = choose_nutrient(
    exact_names=[
        "Sodium, Na"
    ],
    contains_patterns=[
        r"^Sodium"
    ],
    allowed_units=["MG"]
)

nutrient_map["sodium_mg"] = sodium


# Cholesterol
cholesterol = choose_nutrient(
    exact_names=[
        "Cholesterol"
    ],
    allowed_units=["MG"]
)

nutrient_map["cholesterol_mg"] = cholesterol


# =============================================================================
# PRINT NUTRIENT MAP
# =============================================================================

for field, record in nutrient_map.items():

    if record is None:
        print(
            f"{field:<25} -> NOT FOUND"
        )
    else:
        print(
            f"{field:<25} -> "
            f"ID={int(record['id'])} "
            f"UNIT={record['unit_name']} "
            f"NAME={record['name']}"
        )


# =============================================================================
# CREATE NUTRIENT PIVOT
# =============================================================================

header("CREATING CANONICAL SR LEGACY NUTRIENT DATABASE")

selected_nutrients = []

for field, record in nutrient_map.items():

    if record is not None:
        selected_nutrients.append(
            (
                field,
                int(record["id"]),
                str(record["unit_name"])
            )
        )


nutrient_id_to_field = {
    nutrient_id: field
    for field, nutrient_id, unit in selected_nutrients
}


canonical_nutrients["amount"] = (
    canonical_nutrients["_amount_numeric"]
)

canonical_nutrients["mapped_field"] = (
    canonical_nutrients["nutrient_id"]
    .map(nutrient_id_to_field)
)


mapped = canonical_nutrients[
    canonical_nutrients["mapped_field"].notna()
].copy()

print(
    f"Mapped nutrient rows: "
    f"{len(mapped):,}"
)


# =============================================================================
# PIVOT
# =============================================================================

nutrient_values = mapped[
    [
        "fdc_id",
        "mapped_field",
        "amount",
    ]
].copy()

nutrient_values = (
    nutrient_values
    .drop_duplicates(
        ["fdc_id", "mapped_field"]
    )
)

nutrient_pivot = (
    nutrient_values
    .pivot(
        index="fdc_id",
        columns="mapped_field",
        values="amount"
    )
    .reset_index()
)

nutrient_pivot.columns.name = None


# =============================================================================
# BUILD FOOD DATABASE
# =============================================================================

header("BUILDING FINAL SR LEGACY FOOD DATABASE")

food_db = food_sr[
    [
        "fdc_id",
        "description",
        "data_type",
        "food_category_id",
        "category_name",
    ]
].copy()

food_db = food_db.merge(
    nutrient_pivot,
    on="fdc_id",
    how="left"
)

food_db["basis_g"] = BASIS_G

# Ensure expected columns exist.
expected_nutrient_columns = [
    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g",
    "sodium_mg",
    "cholesterol_mg",
]

for col in expected_nutrient_columns:

    if col not in food_db.columns:
        food_db[col] = np.nan


# Reorder columns
food_db = food_db[
    [
        "fdc_id",
        "description",
        "data_type",
        "food_category_id",
        "category_name",
        "basis_g",
        "calories_kcal",
        "protein_g",
        "fat_g",
        "carbohydrate_g",
        "fiber_g",
        "sugar_g",
        "saturated_fat_g",
        "sodium_mg",
        "cholesterol_mg",
    ]
]


# =============================================================================
# NUTRIENT SANITY
# =============================================================================

header("NUTRIENT DATABASE SANITY CHECK")

for col in expected_nutrient_columns:

    coverage = food_db[col].notna().sum()

    print(
        f"{col:<25}: "
        f"{coverage:,}/{len(food_db):,}"
    )


negative_columns = []

for col in expected_nutrient_columns:

    if (
        pd.to_numeric(
            food_db[col],
            errors="coerce"
        )
        .lt(0)
        .any()
    ):
        negative_columns.append(col)

if negative_columns:
    print(
        "WARNING: negative nutrient values:",
        negative_columns
    )
else:
    print(
        "Negative nutrient validation: PASS"
    )


# =============================================================================
# FOOD PORTIONS
# =============================================================================

header("BUILDING SR LEGACY PORTION DATABASE")

sr_portions = food_portion[
    food_portion["fdc_id"].isin(sr_ids)
].copy()

print(
    f"SR Legacy portion rows: "
    f"{len(sr_portions):,}"
)

print(
    f"Unique foods with portions: "
    f"{sr_portions['fdc_id'].nunique():,}"
)


# =============================================================================
# INSPECT PORTION COLUMNS
# =============================================================================

print("\nAvailable food_portion columns:")

for col in sr_portions.columns:
    print(
        f"  {col}"
    )


# =============================================================================
# PORTION COLUMN DETECTION
# =============================================================================

def first_existing(df, candidates):

    for col in candidates:
        if col in df.columns:
            return col

    return None


portion_weight_col = first_existing(
    sr_portions,
    [
        "gram_weight",
        "gram_weight_value",
    ]
)

portion_amount_col = first_existing(
    sr_portions,
    [
        "amount",
    ]
)

portion_modifier_col = first_existing(
    sr_portions,
    [
        "modifier",
        "portion_description",
    ]
)

portion_unit_col = first_existing(
    sr_portions,
    [
        "measure_unit_id",
        "unit_id",
    ]
)


print(
    "\nDetected portion columns:"
)

print(
    "Gram weight:",
    portion_weight_col
)

print(
    "Amount:",
    portion_amount_col
)

print(
    "Modifier:",
    portion_modifier_col
)

print(
    "Unit ID:",
    portion_unit_col
)


# =============================================================================
# MEASURE UNIT MAP
# =============================================================================

unit_map = measure_unit.copy()

unit_map["id"] = pd.to_numeric(
    unit_map["id"],
    errors="coerce"
).astype("Int64")

unit_name_column = first_existing(
    unit_map,
    [
        "name",
        "unit_name",
        "description",
    ]
)

if unit_name_column is None:
    unit_map["unit_name"] = pd.NA
else:
    unit_map["unit_name"] = (
        unit_map[unit_name_column]
        .astype("string")
    )

unit_map = unit_map[
    [
        "id",
        "unit_name",
    ]
].drop_duplicates("id")


# =============================================================================
# JOIN PORTION UNITS
# =============================================================================

if portion_unit_col == "measure_unit_id":

    sr_portions = sr_portions.merge(
        unit_map,
        left_on="measure_unit_id",
        right_on="id",
        how="left"
    )

elif portion_unit_col == "unit_id":

    sr_portions = sr_portions.merge(
        unit_map,
        left_on="unit_id",
        right_on="id",
        how="left"
    )

else:

    sr_portions["unit_name"] = pd.NA


# =============================================================================
# STANDARDIZE PORTION DATABASE
# =============================================================================

portion_db = pd.DataFrame()

portion_db["fdc_id"] = sr_portions["fdc_id"]

if portion_amount_col:
    portion_db["amount"] = pd.to_numeric(
        sr_portions[portion_amount_col],
        errors="coerce"
    )
else:
    portion_db["amount"] = np.nan


if portion_unit_col:
    portion_db["measure_unit_id"] = (
        sr_portions[portion_unit_col]
    )
else:
    portion_db["measure_unit_id"] = np.nan


portion_db["unit"] = sr_portions.get(
    "unit_name",
    pd.Series(
        pd.NA,
        index=sr_portions.index
    )
)


if portion_weight_col:
    portion_db["gram_weight"] = pd.to_numeric(
        sr_portions[portion_weight_col],
        errors="coerce"
    )
else:
    portion_db["gram_weight"] = np.nan


if portion_modifier_col:
    portion_db["modifier"] = (
        sr_portions[portion_modifier_col]
        .astype("string")
    )
else:
    portion_db["modifier"] = pd.NA


# Add description
portion_db = portion_db.merge(
    food_sr[
        [
            "fdc_id",
            "description",
        ]
    ].drop_duplicates("fdc_id"),
    on="fdc_id",
    how="left"
)

portion_db = portion_db[
    [
        "fdc_id",
        "description",
        "amount",
        "unit",
        "measure_unit_id",
        "gram_weight",
        "modifier",
    ]
]


# =============================================================================
# PORTION VALIDATION
# =============================================================================

section("PORTION VALIDATION")

valid_gram_weights = portion_db[
    portion_db["gram_weight"].notna()
]

print(
    "Portions with gram weight:",
    len(valid_gram_weights),
    "/",
    len(portion_db)
)

print(
    "Foods with gram-weight portions:",
    valid_gram_weights["fdc_id"].nunique()
)


# =============================================================================
# FOOD DATABASE VALIDATION
# =============================================================================

header("FINAL SR LEGACY DATABASE VALIDATION")

print(
    f"Foods: {len(food_db):,}"
)

print(
    f"Unique FDC IDs: "
    f"{food_db['fdc_id'].nunique():,}"
)

print(
    "Basis values:",
    sorted(
        food_db["basis_g"]
        .dropna()
        .unique()
        .tolist()
    )
)

print(
    "Data types:"
)

print(
    food_db["data_type"]
    .value_counts()
    .to_string()
)


# Duplicate FDC validation
duplicate_final_ids = (
    food_db["fdc_id"]
    .duplicated()
    .sum()
)

if duplicate_final_ids == 0:
    print(
        "Unique FDC ID validation: PASS"
    )
else:
    print(
        "WARNING:",
        duplicate_final_ids,
        "duplicate final FDC IDs"
    )


# Basis validation
if (
    food_db["basis_g"]
    .eq(BASIS_G)
    .all()
):
    print(
        "100 g basis validation: PASS"
    )


# Description coverage
description_missing = (
    food_db["description"]
    .isna()
    .sum()
)

print(
    "Missing descriptions:",
    description_missing
)


# =============================================================================
# NUTRIENT COVERAGE TABLE
# =============================================================================

section("FINAL NUTRIENT COVERAGE")

coverage_rows = []

for col in expected_nutrient_columns:

    count = food_db[col].notna().sum()

    coverage_rows.append(
        {
            "nutrient": col,
            "foods_with_value": count,
            "total_foods": len(food_db),
            "coverage_percent":
                round(
                    count / len(food_db) * 100,
                    2
                )
            if len(food_db)
            else 0,
        }
    )

coverage_df = pd.DataFrame(
    coverage_rows
)

print(
    coverage_df.to_string(
        index=False
    )
)


# =============================================================================
# PORTION COVERAGE TABLE
# =============================================================================

section("PORTION COVERAGE")

foods_with_portions = (
    portion_db[
        portion_db["gram_weight"].notna()
    ]["fdc_id"]
    .nunique()
)

portion_coverage = (
    foods_with_portions /
    len(food_db) *
    100
    if len(food_db)
    else 0
)

print(
    f"Foods with gram-weight portions: "
    f"{foods_with_portions:,}"
)

print(
    f"Portion food coverage: "
    f"{portion_coverage:.2f}%"
)


# =============================================================================
# SEARCHABLE FOOD NAME INDEX
# =============================================================================

header("BUILDING FOOD SEARCH INDEX")

search_index = food_db[
    [
        "fdc_id",
        "description",
        "category_name",
    ]
].copy()

search_index["search_text"] = (
    search_index["description"]
    .fillna("")
    .str.lower()
    .str.strip()
)

search_index["description_normalized"] = (
    search_index["description"]
    .fillna("")
    .str.lower()
    .str.replace(
        r"[^a-z0-9\s]",
        " ",
        regex=True
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# =============================================================================
# DUPLICATE DESCRIPTION REVIEW
# =============================================================================

header("CHECKING DUPLICATE FOOD DESCRIPTIONS")

description_counts = (
    food_db[
        "description"
    ]
    .fillna("")
    .value_counts()
)

duplicate_descriptions = (
    description_counts[
        description_counts > 1
    ]
)

print(
    "Duplicate descriptions:",
    len(duplicate_descriptions)
)

if len(duplicate_descriptions):

    review_rows = []

    for description, count in (
        duplicate_descriptions.items()
    ):

        rows = food_db[
            food_db["description"]
            .fillna("")
            .eq(description)
        ]

        for _, row in rows.iterrows():

            review_rows.append(
                {
                    "fdc_id": row["fdc_id"],
                    "description": description,
                    "duplicate_count": count,
                    "category":
                        row["category_name"],
                }
            )

    duplicate_review = pd.DataFrame(
        review_rows
    )

else:

    duplicate_review = pd.DataFrame(
        columns=[
            "fdc_id",
            "description",
            "duplicate_count",
            "category",
        ]
    )


# =============================================================================
# NUTRIENT MAP OUTPUT
# =============================================================================

nutrient_mapping_output = []

for field, record in nutrient_map.items():

    if record is None:

        nutrient_mapping_output.append(
            {
                "field": field,
                "nutrient_id": pd.NA,
                "nutrient_name": pd.NA,
                "unit_name": pd.NA,
            }
        )

    else:

        nutrient_mapping_output.append(
            {
                "field": field,
                "nutrient_id": int(record["id"]),
                "nutrient_name": record["name"],
                "unit_name": record["unit_name"],
            }
        )

nutrient_mapping_df = pd.DataFrame(
    nutrient_mapping_output
)


# =============================================================================
# SAVE DATABASE
# =============================================================================

header("SAVING SR LEGACY DATABASE FILES")

food_database_path = (
    OUTPUT_DIR /
    "sr_legacy_food_database.csv"
)

portion_database_path = (
    OUTPUT_DIR /
    "sr_legacy_food_portions.csv"
)

search_index_path = (
    OUTPUT_DIR /
    "sr_legacy_food_search_index.csv"
)

nutrient_mapping_path = (
    OUTPUT_DIR /
    "sr_legacy_nutrient_mapping.csv"
)

coverage_path = (
    OUTPUT_DIR /
    "sr_legacy_nutrient_coverage.csv"
)

duplicate_review_path = (
    OUTPUT_DIR /
    "sr_legacy_duplicate_description_review.csv"
)

food_source_path = (
    OUTPUT_DIR /
    "sr_legacy_food_source_mapping.csv"
)


food_database = food_db.copy()

food_database.to_csv(
    food_database_path,
    index=False
)

portion_db.to_csv(
    portion_database_path,
    index=False
)

search_index.to_csv(
    search_index_path,
    index=False
)

nutrient_mapping_df.to_csv(
    nutrient_mapping_path,
    index=False
)

coverage_df.to_csv(
    coverage_path,
    index=False
)

duplicate_review.to_csv(
    duplicate_review_path,
    index=False
)


# =============================================================================
# SOURCE MAPPING
# =============================================================================

food_source_mapping = food_db[
    [
        "fdc_id",
        "description",
        "data_type",
        "food_category_id",
        "category_name",
    ]
].copy()

food_source_mapping["source_dataset"] = (
    "SR Legacy"
)

food_source_mapping["source_release"] = (
    "FoodData Central SR Legacy 2018-04"
)

food_source_mapping["source_food_file"] = (
    str(paths["food"])
)

food_source_mapping["source_sr_legacy_file"] = (
    str(paths["sr_legacy_food"])
)

food_source_mapping.to_csv(
    food_source_path,
    index=False
)


# =============================================================================
# RELOAD VALIDATION
# =============================================================================

header("RELOADING SAVED DATABASE")

reloaded_food = pd.read_csv(
    food_database_path,
    low_memory=False
)

reloaded_portions = pd.read_csv(
    portion_database_path,
    low_memory=False
)

reloaded_mapping = pd.read_csv(
    nutrient_mapping_path,
    low_memory=False
)

print(
    f"Reloaded food rows: "
    f"{len(reloaded_food):,}"
)

print(
    f"Reloaded portion rows: "
    f"{len(reloaded_portions):,}"
)

print(
    f"Reloaded nutrient mappings: "
    f"{len(reloaded_mapping):,}"
)

if (
    len(reloaded_food)
    == len(food_database)
):
    print(
        "Food database reload validation: PASS"
    )
else:
    print(
        "Food database reload validation: FAIL"
    )


# =============================================================================
# SAMPLE FOOD SEARCH
# =============================================================================

header("SAMPLE SR LEGACY FOOD SEARCH")

sample_terms = [
    "cinnamon",
    "turmeric",
    "olive oil",
    "baking powder",
    "jackfruit",
    "kelp",
    "pork shoulder",
]

for term in sample_terms:

    matches = food_db[
        food_db["description"]
        .fillna("")
        .str.contains(
            term,
            case=False,
            regex=False
        )
    ]

    print(
        f"\n{term.upper()}: "
        f"{len(matches)} match(es)"
    )

    if not matches.empty:

        print(
            matches[
                [
                    "fdc_id",
                    "description",
                    "category_name",
                ]
            ]
            .head(5)
            .to_string(index=False)
        )


# =============================================================================
# FINAL STATUS
# =============================================================================

header("SR LEGACY PIPELINE COMPLETE")

print(
    f"SR Legacy foods available: "
    f"{len(food_db):,}"
)

print(
    f"Foods with portions: "
    f"{foods_with_portions:,}"
)

print(
    f"Portion records: "
    f"{len(portion_db):,}"
)

print(
    f"Duplicate descriptions: "
    f"{len(duplicate_descriptions):,}"
)

print(
    "\nDatabase:"
)

print(
    food_database_path
)

print(
    "\nPortions:"
)

print(
    portion_database_path
)

print(
    "\nSearch index:"
)

print(
    search_index_path
)

print(
    "\nNutrient mapping:"
)

print(
    nutrient_mapping_path
)

print(
    "\nNutrient coverage:"
)

print(
    coverage_path
)

print(
    "\nDuplicate review:"
)

print(
    duplicate_review_path
)

print(
    "\nSource mapping:"
)

print(
    food_source_path
)

print(
    "\nSTATUS: PASS"
)


USDA FOODDATA CENTRAL SR LEGACY DATABASE BUILDER
Build started: 2026-08-16T11:11:36.411247
Search directory:
C:\Users\AK\Downloads\zip

SR Legacy directory:
C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04

CHECKING REQUIRED SR LEGACY FILES
PASS: food.csv                                      0.76 MB
PASS: sr_legacy_food.csv                            0.12 MB
PASS: food_nutrient.csv                             34.68 MB
PASS: nutrient.csv                                  0.02 MB
PASS: food_portion.csv                              0.88 MB
PASS: measure_unit.csv                              0.00 MB
PASS: food_category.csv                             0.00 MB

--------------------------------------------------------------------------------
CHECKING OPTIONAL SUPPORT FILES
--------------------------------------------------------------------------------
FOUND: food_attribute.csv
FOUND: food_attribute_type.csv
FOUND: food_calorie_

In [31]:
# =============================================================================
# USDA FOODDATA CENTRAL — SR LEGACY APPLICATION INGREDIENT LAYER
# =============================================================================
#
# PURPOSE
# -------
# Build the application-facing ingredient/search/portion layer on top of the
# validated SR Legacy database created by the previous Step 2 builder.
#
# THIS SCRIPT DOES NOT:
#   - modify USDA source files
#   - modify Step 2 output files
#   - automatically choose a food from a fuzzy match
#   - invent gram weights
#   - invent USDA portions
#
# IT CREATES:
#
#   1. Searchable ingredient records
#   2. Normalized food descriptions
#   3. Search tokens
#   4. Search aliases
#   5. Portion lookup records
#   6. Normalized household-unit names
#   7. Search test results
#   8. Application-layer validation report
#
# ARCHITECTURE
# ------------
#
# USER SEARCH
#      |
#      v
# ingredient search layer
#      |
#      v
# ranked candidates
#      |
#      v
# user/application selects FDC ID
#      |
#      v
# exact food record
#      |
#      +----> portion resolver
#      |
#      +----> nutrient calculator
#
# IMPORTANT
# ---------
# Search ranking is for candidate retrieval only.
#
# A search result MUST NOT be treated as an automatic food selection.
#
# =============================================================================


from pathlib import Path
from datetime import datetime
from collections import Counter
import math
import re

import pandas as pd
import numpy as np


# =============================================================================
# CONFIGURATION
# =============================================================================

ROOT_DIR = Path(
    r"C:\Users\AK\Downloads\zip"
)

STEP2_DIR = (
    ROOT_DIR /
    "sr_legacy_database"
)

OUTPUT_DIR = (
    ROOT_DIR /
    "sr_legacy_application"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Number of search results to return in tests.
SEARCH_RESULT_LIMIT = 10


# =============================================================================
# EXPECTED STEP 2 FILES
# =============================================================================

FOOD_DATABASE_FILE = (
    STEP2_DIR /
    "sr_legacy_food_database.csv"
)

PORTION_DATABASE_FILE = (
    STEP2_DIR /
    "sr_legacy_food_portions.csv"
)

SEARCH_INDEX_FILE = (
    STEP2_DIR /
    "sr_legacy_food_search_index.csv"
)


# =============================================================================
# OUTPUT FILES
# =============================================================================

INGREDIENT_SEARCH_FILE = (
    OUTPUT_DIR /
    "sr_legacy_ingredient_search.csv"
)

INGREDIENT_ALIAS_FILE = (
    OUTPUT_DIR /
    "sr_legacy_ingredient_aliases.csv"
)

PORTION_LOOKUP_FILE = (
    OUTPUT_DIR /
    "sr_legacy_portion_lookup.csv"
)

SEARCH_TEST_FILE = (
    OUTPUT_DIR /
    "sr_legacy_search_test_results.csv"
)

REPORT_FILE = (
    OUTPUT_DIR /
    "sr_legacy_application_report.txt"
)


# =============================================================================
# DISPLAY HELPERS
# =============================================================================

REPORT_LINES = []


def header(title):

    line = "\n" + "=" * 80

    print(line)
    print(title)
    print("=" * 80)

    REPORT_LINES.append(line)
    REPORT_LINES.append(title)
    REPORT_LINES.append("=" * 80)


def section(title):

    line = "\n" + "-" * 80

    print(line)
    print(title)
    print("-" * 80)

    REPORT_LINES.append(line)
    REPORT_LINES.append(title)
    REPORT_LINES.append("-" * 80)


def report(text=""):

    print(text)
    REPORT_LINES.append(
        str(text)
    )


# =============================================================================
# BASIC HELPERS
# =============================================================================

def safe_read_csv(path):

    try:

        return pd.read_csv(
            path,
            low_memory=False
        )

    except Exception:

        return pd.read_csv(
            path,
            low_memory=False,
            encoding="latin1"
        )


def normalize_columns(df):

    df = df.copy()

    df.columns = [
        str(column)
        .strip()
        .lower()
        .replace("\ufeff", "")
        for column in df.columns
    ]

    return df


def normalize_text(value):

    if pd.isna(value):

        return ""

    text = str(value)

    text = text.lower()

    # Replace punctuation with spaces.
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Normalize whitespace.
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def tokenize(value):

    text = normalize_text(
        value
    )

    if not text:

        return []

    return text.split()


def normalize_token(token):

    token = token.lower().strip()

    # Small, conservative plural normalization.
    #
    # We deliberately avoid aggressive stemming because food terms such
    # as "grass", "molasses", etc. can be damaged by generic stemming.
    if (
        len(token) > 4
        and token.endswith("ies")
    ):

        token = (
            token[:-3]
            + "y"
        )

    elif (
        len(token) > 4
        and token.endswith("es")
        and not token.endswith("ses")
    ):

        token = token[:-2]

    elif (
        len(token) > 4
        and token.endswith("s")
    ):

        token = token[:-1]

    return token


def normalized_tokens(value):

    return [
        normalize_token(token)
        for token in tokenize(value)
        if token
    ]


def token_set(value):

    return set(
        normalized_tokens(value)
    )


# =============================================================================
# SEARCH TERM NORMALIZATION
# =============================================================================

SEARCH_SYNONYMS = {

    # Oil
    "oils": "oil",

    # Spices
    "spices": "spice",

    # Herbs
    "herbs": "herb",

    # Fruits
    "fruits": "fruit",

    # Vegetables
    "vegetables": "vegetable",

    # Meats
    "meats": "meat",

    # Eggs
    "eggs": "egg",

    # Nuts
    "nuts": "nut",

    # Seeds
    "seeds": "seed",

    # Beans
    "beans": "bean",

    # Peas
    "peas": "pea",

    # Potatoes
    "potatoes": "potato",
}


def normalize_search_token(token):

    token = normalize_token(
        token
    )

    return SEARCH_SYNONYMS.get(
        token,
        token
    )


def search_tokens(value):

    tokens = tokenize(
        value
    )

    return [
        normalize_search_token(token)
        for token in tokens
        if token
    ]


# =============================================================================
# SEARCH STOPWORDS
# =============================================================================
#
# These words commonly appear in USDA descriptions but provide little value
# when the user is searching for the ingredient identity.
#
# IMPORTANT:
# We do NOT remove them from the stored USDA description.
# They are removed only from the application search token set.
# =============================================================================

SEARCH_STOPWORDS = {

    "and",
    "or",
    "with",
    "without",
    "the",
    "of",
    "a",
    "an",
    "for",
    "by",
    "from",
    "in",
    "on",
    "to",
    "as",
    "style",
    "type",
    "includes",
    "include",
    "including",
}


def useful_search_tokens(value):

    return [
        token
        for token in search_tokens(value)
        if token not in SEARCH_STOPWORDS
    ]


# =============================================================================
# UNIT NORMALIZATION
# =============================================================================

UNIT_ALIASES = {

    # Teaspoon
    "tsp": "teaspoon",
    "tsps": "teaspoon",
    "tsp.": "teaspoon",
    "teaspoon": "teaspoon",
    "teaspoons": "teaspoon",

    # Tablespoon
    "tbsp": "tablespoon",
    "tbs": "tablespoon",
    "tb": "tablespoon",
    "tablespoon": "tablespoon",
    "tablespoons": "tablespoon",

    # Cup
    "cup": "cup",
    "cups": "cup",

    # Pint
    "pint": "pint",
    "pints": "pint",

    # Quart
    "quart": "quart",
    "quarts": "quart",

    # Gallon
    "gallon": "gallon",
    "gallons": "gallon",

    # Ounce
    "oz": "ounce",
    "ounce": "ounce",
    "ounces": "ounce",

    # Pound
    "lb": "pound",
    "lbs": "pound",
    "pound": "pound",
    "pounds": "pound",

    # Gram
    "g": "gram",
    "gram": "gram",
    "grams": "gram",

    # Kilogram
    "kg": "kilogram",
    "kilogram": "kilogram",
    "kilograms": "kilogram",

    # Milligram
    "mg": "milligram",
    "milligram": "milligram",
    "milligrams": "milligram",

    # Milliliter
    "ml": "milliliter",
    "milliliter": "milliliter",
    "milliliters": "milliliter",

    # Liter
    "l": "liter",
    "liter": "liter",
    "liters": "liter",

    # Large household units
    "quart": "quart",
    "gallon": "gallon",

    # Count/piece style units
    "piece": "piece",
    "pieces": "piece",
    "item": "item",
    "items": "item",
}


def normalize_unit(value):

    if pd.isna(value):

        return ""

    text = (
        str(value)
        .strip()
        .lower()
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return UNIT_ALIASES.get(
        text,
        text
    )


# =============================================================================
# REQUIRED FILE VALIDATION
# =============================================================================

header(
    "SR LEGACY APPLICATION INGREDIENT LAYER"
)

report(
    f"Build started: {datetime.now().isoformat()}"
)

report("")
report(
    "Step 2 directory:"
)

report(
    str(STEP2_DIR)
)

report("")
report(
    "Output directory:"
)

report(
    str(OUTPUT_DIR)
)


header(
    "CHECKING STEP 2 DATABASE FILES"
)

required_files = {

    "food":
        FOOD_DATABASE_FILE,

    "portions":
        PORTION_DATABASE_FILE,

    "search_index":
        SEARCH_INDEX_FILE,
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required Step 2 file missing:\n{path}"
        )

    size_mb = (
        path.stat().st_size
        /
        (1024 * 1024)
    )

    report(
        f"PASS: {name:<15} "
        f"{path.name:<50} "
        f"{size_mb:.2f} MB"
    )


# =============================================================================
# LOAD STEP 2 DATABASE
# =============================================================================

header(
    "LOADING STEP 2 DATABASE"
)

food_db = normalize_columns(
    safe_read_csv(
        FOOD_DATABASE_FILE
    )
)

portion_db = normalize_columns(
    safe_read_csv(
        PORTION_DATABASE_FILE
    )
)

search_index = normalize_columns(
    safe_read_csv(
        SEARCH_INDEX_FILE
    )
)

report(
    f"Food rows:       {len(food_db):,}"
)

report(
    f"Portion rows:    {len(portion_db):,}"
)

report(
    f"Search rows:     {len(search_index):,}"
)


# =============================================================================
# CORE COLUMN VALIDATION
# =============================================================================

header(
    "VALIDATING STEP 2 COLUMNS"
)

required_food_columns = {

    "fdc_id",
    "description",
    "data_type",
    "food_category_id",
    "category_name",
    "basis_g",

    "calories_kcal",
    "protein_g",
    "fat_g",
    "carbohydrate_g",
    "fiber_g",
    "sugar_g",
    "saturated_fat_g",
    "sodium_mg",
    "cholesterol_mg",
}

required_portion_columns = {

    "fdc_id",
    "description",
    "amount",
    "unit",
    "measure_unit_id",
    "gram_weight",
    "modifier",
}

required_search_columns = {

    "fdc_id",
    "description",
    "category_name",
    "search_text",
    "description_normalized",
}


missing_food = (
    required_food_columns
    -
    set(food_db.columns)
)

missing_portions = (
    required_portion_columns
    -
    set(portion_db.columns)
)

missing_search = (
    required_search_columns
    -
    set(search_index.columns)
)


if missing_food:

    raise ValueError(
        f"Food database missing columns: "
        f"{missing_food}"
    )


if missing_portions:

    raise ValueError(
        f"Portion database missing columns: "
        f"{missing_portions}"
    )


if missing_search:

    raise ValueError(
        f"Search index missing columns: "
        f"{missing_search}"
    )


report(
    "Food database columns: PASS"
)

report(
    "Portion database columns: PASS"
)

report(
    "Search index columns: PASS"
)


# =============================================================================
# NORMALIZE IDS
# =============================================================================

section(
    "NORMALIZING FDC IDS"
)

food_db["fdc_id"] = pd.to_numeric(
    food_db["fdc_id"],
    errors="coerce"
).astype("Int64")

portion_db["fdc_id"] = pd.to_numeric(
    portion_db["fdc_id"],
    errors="coerce"
).astype("Int64")

search_index["fdc_id"] = pd.to_numeric(
    search_index["fdc_id"],
    errors="coerce"
).astype("Int64")

report(
    "FDC ID normalization: PASS"
)


# =============================================================================
# BUILD APPLICATION INGREDIENT SEARCH TABLE
# =============================================================================

header(
    "BUILDING APPLICATION INGREDIENT SEARCH TABLE"
)

ingredient_search = food_db[
    [
        "fdc_id",
        "description",
        "data_type",
        "food_category_id",
        "category_name",
        "basis_g",
        "calories_kcal",
        "protein_g",
        "fat_g",
        "carbohydrate_g",
        "fiber_g",
        "sugar_g",
        "saturated_fat_g",
        "sodium_mg",
        "cholesterol_mg",
    ]
].copy()


# -------------------------------------------------------------------------
# Normalized description
# -------------------------------------------------------------------------

ingredient_search[
    "description_normalized"
] = (
    ingredient_search[
        "description"
    ]
    .fillna("")
    .map(normalize_text)
)


# -------------------------------------------------------------------------
# Useful search tokens
# -------------------------------------------------------------------------

ingredient_search[
    "search_tokens"
] = (
    ingredient_search[
        "description"
    ]
    .fillna("")
    .map(
        lambda value:
        " ".join(
            useful_search_tokens(
                value
            )
        )
    )
)


# -------------------------------------------------------------------------
# Token count
# -------------------------------------------------------------------------

ingredient_search[
    "search_token_count"
] = (
    ingredient_search[
        "search_tokens"
    ]
    .fillna("")
    .map(
        lambda value:
        len(
            value.split()
        )
        if value
        else 0
    )
)


# -------------------------------------------------------------------------
# First-token index
# -------------------------------------------------------------------------

ingredient_search[
    "first_search_token"
] = (
    ingredient_search[
        "search_tokens"
    ]
    .fillna("")
    .map(
        lambda value:
        value.split()[0]
        if value
        else ""
    )
)


# -------------------------------------------------------------------------
# Searchable combined text
# -------------------------------------------------------------------------

ingredient_search[
    "search_text"
] = (
    ingredient_search[
        [
            "description_normalized",
            "search_tokens",
            "category_name",
        ]
    ]
    .fillna("")
    .astype(str)
    .agg(
        " ".join,
        axis=1
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# =============================================================================
# PORTION COUNTS
# =============================================================================

portion_counts = (
    portion_db
    .groupby(
        "fdc_id"
    )
    .size()
    .rename(
        "portion_count"
    )
)

portion_food_counts = (
    portion_db
    .groupby(
        "fdc_id"
    )[
        "gram_weight"
    ]
    .count()
    .rename(
        "valid_gram_portion_count"
    )
)


ingredient_search = (
    ingredient_search
    .merge(
        portion_counts,
        on="fdc_id",
        how="left"
    )
    .merge(
        portion_food_counts,
        on="fdc_id",
        how="left"
    )
)


ingredient_search[
    "portion_count"
] = ingredient_search[
    "portion_count"
].fillna(0).astype(int)


ingredient_search[
    "valid_gram_portion_count"
] = ingredient_search[
    "valid_gram_portion_count"
].fillna(0).astype(int)


ingredient_search[
    "has_portions"
] = (
    ingredient_search[
        "portion_count"
    ]
    > 0
)


ingredient_search[
    "has_gram_portions"
] = (
    ingredient_search[
        "valid_gram_portion_count"
    ]
    > 0
)


# =============================================================================
# SEARCH QUALITY FLAGS
# =============================================================================

section(
    "CREATING SEARCH QUALITY FLAGS"
)


def create_search_quality(row):

    description = (
        str(
            row["description"]
        )
        if not pd.isna(
            row["description"]
        )
        else ""
    )

    tokens = (
        row["search_tokens"]
        .split()
        if row["search_tokens"]
        else []
    )

    score = 0

    # Description exists.
    if description:

        score += 1

    # Useful search tokens exist.
    if tokens:

        score += 1

    # Has nutrition.
    if pd.notna(
        row["calories_kcal"]
    ):

        score += 1

    # Has portion.
    if row["has_portions"]:

        score += 1

    # Has gram-weight portion.
    if row["has_gram_portions"]:

        score += 1

    return score


ingredient_search[
    "search_quality_score"
] = ingredient_search.apply(
    create_search_quality,
    axis=1
)


# =============================================================================
# CREATE ALIAS TABLE
# =============================================================================

header(
    "BUILDING INGREDIENT ALIAS TABLE"
)

alias_rows = []


for _, row in ingredient_search.iterrows():

    fdc_id = row[
        "fdc_id"
    ]

    description = (
        str(
            row["description"]
        )
        if not pd.isna(
            row["description"]
        )
        else ""
    )

    normalized_description = (
        row[
            "description_normalized"
        ]
    )

    tokens = (
        useful_search_tokens(
            description
        )
    )

    # -------------------------------------------------------------------------
    # Alias 1: full normalized USDA description
    # -------------------------------------------------------------------------

    if normalized_description:

        alias_rows.append({

            "fdc_id":
                fdc_id,

            "alias":
                normalized_description,

            "alias_normalized":
                normalized_description,

            "alias_type":
                "USDA_DESCRIPTION",

            "source":
                "USDA_SR_LEGACY",

            "confidence":
                1.00,
        })


    # -------------------------------------------------------------------------
    # Alias 2: useful token sequence
    # -------------------------------------------------------------------------

    if tokens:

        token_alias = " ".join(
            tokens
        )

        alias_rows.append({

            "fdc_id":
                fdc_id,

            "alias":
                token_alias,

            "alias_normalized":
                token_alias,

            "alias_type":
                "TOKEN_NORMALIZED",

            "source":
                "GENERATED",

            "confidence":
                0.95,
        })


    # -------------------------------------------------------------------------
    # Alias 3: first useful token
    #
    # Only useful as a low-confidence retrieval alias.
    # -------------------------------------------------------------------------

    if tokens:

        first_token = tokens[0]

        if len(first_token) >= 4:

            alias_rows.append({

                "fdc_id":
                    fdc_id,

                "alias":
                    first_token,

                "alias_normalized":
                    first_token,

                "alias_type":
                    "FIRST_TOKEN",

                "source":
                    "GENERATED",

                "confidence":
                    0.40,
            })


alias_df = pd.DataFrame(
    alias_rows
)


# Remove exact duplicate alias rows.
alias_df = (
    alias_df
    .drop_duplicates(
        [
            "fdc_id",
            "alias_normalized",
            "alias_type",
        ]
    )
    .reset_index(
        drop=True
    )
)


report(
    f"Alias records: {len(alias_df):,}"
)

report(
    f"Foods represented: "
    f"{alias_df['fdc_id'].nunique():,}"
)


# =============================================================================
# SEARCH ENGINE
# =============================================================================

header(
    "BUILDING SEARCH ENGINE"
)


def calculate_search_score(
    query,
    description,
    category="",
):
    """
    Calculate a deterministic candidate score.

    This is intentionally transparent rather than using a black-box fuzzy
    matcher.

    Score components:

        +100 exact normalized description
        + 80 exact token sequence
        + 50 all query tokens present
        + 30 token order match
        + 10 per matching token
        + category bonus
        - token count difference penalty
    """

    query_normalized = normalize_text(
        query
    )

    description_normalized = normalize_text(
        description
    )

    query_tokens = set(
        useful_search_tokens(
            query
        )
    )

    description_tokens = set(
        useful_search_tokens(
            description
        )
    )

    if not query_normalized:

        return 0.0, {
            "exact_description": False,
            "all_tokens_match": False,
            "matched_tokens": 0,
            "query_tokens": 0,
        }


    score = 0.0


    # -------------------------------------------------------------------------
    # Exact normalized description
    # -------------------------------------------------------------------------

    exact_description = (
        query_normalized
        ==
        description_normalized
    )

    if exact_description:

        score += 100


    # -------------------------------------------------------------------------
    # Exact token sequence
    # -------------------------------------------------------------------------

    query_token_list = useful_search_tokens(
        query
    )

    description_token_list = useful_search_tokens(
        description
    )

    token_query_string = " ".join(
        query_token_list
    )

    token_description_string = " ".join(
        description_token_list
    )

    exact_token_sequence = (
        token_query_string
        ==
        token_description_string
        and
        bool(token_query_string)
    )

    if exact_token_sequence:

        score += 80


    # -------------------------------------------------------------------------
    # Token overlap
    # -------------------------------------------------------------------------

    matched_tokens = (
        query_tokens
        &
        description_tokens
    )

    matched_count = len(
        matched_tokens
    )

    query_count = len(
        query_tokens
    )


    if (
        query_count > 0
        and
        matched_count == query_count
    ):

        score += 50

        all_tokens_match = True

    else:

        all_tokens_match = False


    score += (
        matched_count
        * 10
    )


    # -------------------------------------------------------------------------
    # Query token order
    # -------------------------------------------------------------------------

    if query_token_list:

        positions = []

        for token in query_token_list:

            if token in description_token_list:

                positions.append(
                    description_token_list.index(
                        token
                    )
                )

        if (
            len(positions)
            ==
            len(query_token_list)
        ):

            if positions == sorted(
                positions
            ):

                score += 30


    # -------------------------------------------------------------------------
    # Penalize descriptions that are much longer than query.
    #
    # We do NOT heavily penalize this because USDA descriptions are often
    # much more detailed than the user's search.
    # -------------------------------------------------------------------------

    token_difference = max(
        0,
        len(
            description_tokens
        )
        -
        len(
            query_tokens
        )
    )

    score -= min(
        token_difference * 1.5,
        15
    )


    # -------------------------------------------------------------------------
    # Category bonus for matching category words.
    # -------------------------------------------------------------------------

    category_tokens = set(
        useful_search_tokens(
            category
        )
    )

    if (
        query_tokens
        &
        category_tokens
    ):

        score += 5


    return score, {

        "exact_description":
            exact_description,

        "exact_token_sequence":
            exact_token_sequence,

        "all_tokens_match":
            all_tokens_match,

        "matched_tokens":
            matched_count,

        "query_tokens":
            query_count,
    }


def search_ingredients(
    query,
    limit=10,
    category=None,
):
    """
    Search the application ingredient layer.

    Returns ranked candidate foods.

    IMPORTANT:
    Results are candidates, NOT automatic food selections.
    """

    query = str(
        query
    ).strip()


    if not query:

        return pd.DataFrame()


    query_tokens = set(
        useful_search_tokens(
            query
        )
    )


    if not query_tokens:

        return pd.DataFrame()


    candidates = ingredient_search.copy()


    # -------------------------------------------------------------------------
    # Fast pre-filter.
    #
    # A candidate must contain at least one useful query token.
    # -------------------------------------------------------------------------

    def contains_query_token(
        token_string
    ):

        candidate_tokens = set(
            token_string.split()
        )

        return bool(
            query_tokens
            &
            candidate_tokens
        )


    candidates = candidates[
        candidates[
            "search_tokens"
        ]
        .fillna("")
        .map(
            contains_query_token
        )
    ].copy()


    # -------------------------------------------------------------------------
    # Optional category filter.
    # -------------------------------------------------------------------------

    if category:

        category_normalized = normalize_text(
            category
        )

        candidates = candidates[
            candidates[
                "category_name"
            ]
            .fillna("")
            .map(
                normalize_text
            )
            .str.contains(
                re.escape(
                    category_normalized
                ),
                regex=True,
                na=False
            )
        ]


    if candidates.empty:

        return candidates


    # -------------------------------------------------------------------------
    # Score candidates.
    # -------------------------------------------------------------------------

    scores = []

    metadata = []


    for _, row in candidates.iterrows():

        score, meta = calculate_search_score(

            query,

            row[
                "description"
            ],

            row[
                "category_name"
            ]
        )

        scores.append(
            score
        )

        metadata.append(
            meta
        )


    candidates[
        "search_score"
    ] = scores


    candidates[
        "exact_description"
    ] = [
        item[
            "exact_description"
        ]
        for item in metadata
    ]


    candidates[
        "exact_token_sequence"
    ] = [
        item[
            "exact_token_sequence"
        ]
        for item in metadata
    ]


    candidates[
        "all_query_tokens_match"
    ] = [
        item[
            "all_tokens_match"
        ]
        for item in metadata
    ]


    candidates[
        "matched_query_tokens"
    ] = [
        item[
            "matched_tokens"
        ]
        for item in metadata
    ]


    candidates[
        "query_token_count"
    ] = [
        item[
            "query_tokens"
        ]
        for item in metadata
    ]


    # -------------------------------------------------------------------------
    # Stable deterministic ranking.
    # -------------------------------------------------------------------------

    candidates = candidates.sort_values(

        [
            "search_score",
            "all_query_tokens_match",
            "has_gram_portions",
            "search_quality_score",
            "description",
            "fdc_id",
        ],

        ascending=[
            False,
            False,
            False,
            False,
            True,
            True,
        ]
    )


    return candidates.head(
        limit
    ).reset_index(
        drop=True
    )


# =============================================================================
# PORTION RESOLVER
# =============================================================================

header(
    "BUILDING PORTION RESOLVER"
)


def portion_candidates(
    fdc_id,
    unit=None,
    modifier=None,
):
    """
    Return USDA portions for an exact FDC ID.

    No conversions are invented.
    """

    numeric_fdc_id = pd.to_numeric(
        fdc_id,
        errors="coerce"
    )

    if pd.isna(
        numeric_fdc_id
    ):

        return pd.DataFrame()


    results = portion_db[
        portion_db[
            "fdc_id"
        ]
        ==
        int(
            numeric_fdc_id
        )
    ].copy()


    if results.empty:

        return results


    # -------------------------------------------------------------------------
    # Normalize unit
    # -------------------------------------------------------------------------

    results[
        "unit_normalized"
    ] = (
        results[
            "unit"
        ]
        .fillna("")
        .map(
            normalize_unit
        )
    )


    # -------------------------------------------------------------------------
    # Optional unit filter
    # -------------------------------------------------------------------------

    if unit:

        requested_unit = normalize_unit(
            unit
        )

        results = results[
            results[
                "unit_normalized"
            ]
            ==
            requested_unit
        ]


    # -------------------------------------------------------------------------
    # Optional modifier filter
    # -------------------------------------------------------------------------

    if (
        modifier
        and
        not results.empty
    ):

        modifier_normalized = normalize_text(
            modifier
        )

        results[
            "_modifier_normalized"
        ] = (
            results[
                "modifier"
            ]
            .fillna("")
            .map(
                normalize_text
            )
        )

        results = results[
            results[
                "_modifier_normalized"
            ]
            .str.contains(
                re.escape(
                    modifier_normalized
                ),
                regex=True,
                na=False
            )
        ]


    return results


def resolve_portion(
    fdc_id,
    amount,
    unit,
    modifier=None,
):
    """
    Resolve a requested USDA portion.

    Example:

        resolve_portion(
            171320,
            1,
            "teaspoon"
        )

    Returns a dictionary describing whether an exact USDA portion was found.

    IMPORTANT:
    If USDA does not provide the requested unit, this function does NOT invent
    a conversion.
    """

    try:

        amount = float(
            amount
        )

    except Exception:

        return {

            "status":
                "INVALID_AMOUNT",

            "fdc_id":
                fdc_id,

            "requested_amount":
                amount,

            "requested_unit":
                unit,
        }


    if amount <= 0:

        return {

            "status":
                "INVALID_AMOUNT",

            "fdc_id":
                fdc_id,

            "requested_amount":
                amount,

            "requested_unit":
                unit,
        }


    requested_unit = normalize_unit(
        unit
    )


    portions = portion_candidates(
        fdc_id,
        unit=requested_unit,
        modifier=modifier,
    )


    # -------------------------------------------------------------------------
    # Exact USDA unit found.
    # -------------------------------------------------------------------------

    if not portions.empty:

        # If multiple records exist, choose the first deterministic record.
        portions = portions.sort_values(
            [
                "amount",
                "gram_weight",
                "modifier",
            ],
            na_position="last"
        )

        selected = portions.iloc[0]


        source_amount = pd.to_numeric(
            selected[
                "amount"
            ],
            errors="coerce"
        )

        gram_weight = pd.to_numeric(
            selected[
                "gram_weight"
            ],
            errors="coerce"
        )


        if (
            pd.notna(
                source_amount
            )
            and
            pd.notna(
                gram_weight
            )
            and
            source_amount > 0
        ):

            grams_per_requested_unit = (
                gram_weight
                /
                source_amount
            )

            requested_grams = (
                amount
                *
                grams_per_requested_unit
            )

            return {

                "status":
                    "EXACT_USDA_PORTION",

                "fdc_id":
                    int(fdc_id),

                "requested_amount":
                    amount,

                "requested_unit":
                    requested_unit,

                "usda_amount":
                    float(
                        source_amount
                    ),

                "usda_unit":
                    selected[
                        "unit"
                    ],

                "gram_weight":
                    float(
                        gram_weight
                    ),

                "requested_grams":
                    float(
                        requested_grams
                    ),

                "modifier":
                    selected[
                        "modifier"
                    ],
            }


    # -------------------------------------------------------------------------
    # No exact unit.
    #
    # We deliberately return an unresolved result rather than making an
    # unsupported conversion.
    # -------------------------------------------------------------------------

    return {

        "status":
            "NO_EXACT_USDA_PORTION",

        "fdc_id":
            int(fdc_id),

        "requested_amount":
            amount,

        "requested_unit":
            requested_unit,

        "requested_grams":
            None,

        "message":
            (
                "USDA SR Legacy does not provide an exact "
                "portion for this food/unit combination."
            ),
    }


# =============================================================================
# NUTRIENT CALCULATION
# =============================================================================

header(
    "BUILDING NUTRIENT CALCULATOR"
)


NUTRIENT_COLUMNS = {

    "calories_kcal":
        "calories_kcal",

    "protein_g":
        "protein_g",

    "fat_g":
        "fat_g",

    "carbohydrate_g":
        "carbohydrate_g",

    "fiber_g":
        "fiber_g",

    "sugar_g":
        "sugar_g",

    "saturated_fat_g":
        "saturated_fat_g",

    "sodium_mg":
        "sodium_mg",

    "cholesterol_mg":
        "cholesterol_mg",
}


def calculate_nutrients_from_grams(
    fdc_id,
    grams,
):
    """
    Calculate nutrients for an exact gram quantity using the canonical
    100 g SR Legacy values.

    Formula:

        nutrient_for_quantity =
            nutrient_per_100g * grams / 100
    """

    try:

        grams = float(
            grams
        )

    except Exception:

        return {

            "status":
                "INVALID_GRAMS",

            "fdc_id":
                fdc_id,
        }


    if grams < 0:

        return {

            "status":
                "INVALID_GRAMS",

            "fdc_id":
                fdc_id,
        }


    matching = food_db[
        food_db[
            "fdc_id"
        ]
        ==
        int(fdc_id)
    ]


    if matching.empty:

        return {

            "status":
                "FOOD_NOT_FOUND",

            "fdc_id":
                fdc_id,
        }


    food = matching.iloc[0]


    result = {

        "status":
            "PASS",

        "fdc_id":
            int(fdc_id),

        "grams":
            grams,
    }


    multiplier = (
        grams
        /
        100.0
    )


    for output_name, source_column in (
        NUTRIENT_COLUMNS.items()
    ):

        value = pd.to_numeric(
            food[
                source_column
            ],
            errors="coerce"
        )


        if pd.notna(
            value
        ):

            result[
                output_name
            ] = float(
                value
                *
                multiplier
            )

        else:

            result[
                output_name
            ] = None


    return result


def calculate_nutrients_from_portion(
    fdc_id,
    amount,
    unit,
    modifier=None,
):
    """
    Resolve a USDA portion and calculate nutrients.

    No result is produced if the USDA portion cannot be resolved exactly.
    """

    portion = resolve_portion(

        fdc_id,

        amount,

        unit,

        modifier
    )


    if portion[
        "status"
    ] != "EXACT_USDA_PORTION":

        return {

            **portion,

            "nutrition":
                None,
        }


    nutrition = calculate_nutrients_from_grams(

        fdc_id,

        portion[
            "requested_grams"
        ]
    )


    return {

        **portion,

        "nutrition":
            nutrition,
    }


# =============================================================================
# SEARCH TESTS
# =============================================================================

header(
    "RUNNING APPLICATION SEARCH TESTS"
)


TEST_QUERIES = [

    "cinnamon",

    "ground cinnamon",

    "turmeric",

    "basil",

    "fresh basil",

    "olive oil",

    "oil olive",

    "baking powder",

    "jackfruit",

    "raw jackfruit",

    "kelp",

    "seaweed kelp",

    "pork shoulder",

    "shoulder pork",
]


search_test_rows = []


for query in TEST_QUERIES:

    results = search_ingredients(

        query,

        limit=SEARCH_RESULT_LIMIT
    )


    report("")
    report(
        f"SEARCH: {query}"
    )


    if results.empty:

        report(
            "  NO MATCHES"
        )

        search_test_rows.append({

            "query":
                query,

            "rank":
                None,

            "fdc_id":
                None,

            "description":
                None,

            "category_name":
                None,

            "search_score":
                None,

            "all_query_tokens_match":
                False,

            "has_gram_portions":
                False,
        })

        continue


    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):

        report(
            f"  {rank}. "
            f"[FDC {row['fdc_id']}] "
            f"{row['description']} "
            f"(score={row['search_score']:.1f})"
        )


        search_test_rows.append({

            "query":
                query,

            "rank":
                rank,

            "fdc_id":
                row["fdc_id"],

            "description":
                row["description"],

            "category_name":
                row["category_name"],

            "search_score":
                row["search_score"],

            "exact_description":
                row["exact_description"],

            "exact_token_sequence":
                row["exact_token_sequence"],

            "all_query_tokens_match":
                row[
                    "all_query_tokens_match"
                ],

            "matched_query_tokens":
                row[
                    "matched_query_tokens"
                ],

            "query_token_count":
                row[
                    "query_token_count"
                ],

            "has_portions":
                row[
                    "has_portions"
                ],

            "has_gram_portions":
                row[
                    "has_gram_portions"
                ],
        })


search_test_df = pd.DataFrame(
    search_test_rows
)


# =============================================================================
# SPECIFIC IMPORTANT SEARCH VALIDATION
# =============================================================================

header(
    "IMPORTANT INGREDIENT SEARCH VALIDATION"
)


IMPORTANT_SEARCHES = {

    "cinnamon":
        [
            "cinnamon"
        ],

    "turmeric":
        [
            "turmeric"
        ],

    "olive_oil":
        [
            "olive oil",
            "oil olive"
        ],

    "baking_powder":
        [
            "baking powder"
        ],

    "jackfruit":
        [
            "jackfruit"
        ],

    "kelp":
        [
            "kelp"
        ],

    "pork_shoulder":
        [
            "pork shoulder",
            "shoulder pork"
        ],
}


important_results = {}


for name, queries in (
    IMPORTANT_SEARCHES.items()
):

    report("")
    report(
        f"{name.upper()}"
    )


    combined = []


    for query in queries:

        results = search_ingredients(

            query,

            limit=5
        )


        report(
            f"  Query: {query}"
        )


        if results.empty:

            report(
                "    NO MATCHES"
            )

            continue


        for _, row in results.iterrows():

            combined.append(
                int(
                    row["fdc_id"]
                )
            )

            report(
                f"    FDC {row['fdc_id']} "
                f"| {row['description']} "
                f"| score={row['search_score']:.1f}"
            )


    important_results[
        name
    ] = sorted(
        set(combined)
    )


# =============================================================================
# PORTION TESTING
# =============================================================================

header(
    "RUNNING PORTION RESOLUTION TESTS"
)


portion_test_cases = []


# -------------------------------------------------------------------------
# Test using top search result for cinnamon.
# -------------------------------------------------------------------------

cinnamon_results = search_ingredients(
    "cinnamon",
    limit=5
)


if not cinnamon_results.empty:

    cinnamon_fdc = int(
        cinnamon_results.iloc[0][
            "fdc_id"
        ]
    )


    test_units = [
        "teaspoon",
        "tablespoon",
        "cup",
    ]


    for unit in test_units:

        result = resolve_portion(

            cinnamon_fdc,

            1,

            unit
        )


        report("")
        report(
            f"CINNAMON: 1 {unit}"
        )

        report(
            str(result)
        )


        portion_test_cases.append({

            "food":
                "cinnamon",

            "fdc_id":
                cinnamon_fdc,

            "amount":
                1,

            "unit":
                unit,

            "status":
                result[
                    "status"
                ],

            "requested_grams":
                result.get(
                    "requested_grams"
                ),
        })


# =============================================================================
# BUILD APPLICATION PORTION LOOKUP
# =============================================================================

header(
    "BUILDING APPLICATION PORTION LOOKUP"
)


portion_lookup = portion_db.copy()


portion_lookup[
    "unit_normalized"
] = (
    portion_lookup[
        "unit"
    ]
    .fillna("")
    .map(
        normalize_unit
    )
)


portion_lookup[
    "modifier_normalized"
] = (
    portion_lookup[
        "modifier"
    ]
    .fillna("")
    .map(
        normalize_text
    )
)


portion_lookup[
    "amount_numeric"
] = pd.to_numeric(
    portion_lookup[
        "amount"
    ],
    errors="coerce"
)


portion_lookup[
    "gram_weight_numeric"
] = pd.to_numeric(
    portion_lookup[
        "gram_weight"
    ],
    errors="coerce"
)


portion_lookup[
    "grams_per_unit"
] = np.where(

    (
        portion_lookup[
            "amount_numeric"
        ]
        > 0
    )
    &
    (
        portion_lookup[
            "gram_weight_numeric"
        ].notna()
    ),

    portion_lookup[
        "gram_weight_numeric"
    ]
    /
    portion_lookup[
        "amount_numeric"
    ],

    np.nan
)


portion_lookup[
    "has_valid_gram_conversion"
] = (
    portion_lookup[
        "grams_per_unit"
    ].notna()
    &
    (
        portion_lookup[
            "grams_per_unit"
        ]
        >= 0
    )
)


# -------------------------------------------------------------------------
# Add useful food metadata.
# -------------------------------------------------------------------------

portion_lookup = portion_lookup.merge(

    food_db[
        [
            "fdc_id",
            "data_type",
            "food_category_id",
            "category_name",
        ]
    ].drop_duplicates(
        "fdc_id"
    ),

    on="fdc_id",

    how="left"
)


# -------------------------------------------------------------------------
# Final portion columns.
# -------------------------------------------------------------------------

portion_lookup = portion_lookup[
    [
        "fdc_id",
        "description",
        "amount",
        "amount_numeric",
        "unit",
        "unit_normalized",
        "measure_unit_id",
        "gram_weight",
        "gram_weight_numeric",
        "grams_per_unit",
        "modifier",
        "modifier_normalized",
        "has_valid_gram_conversion",
        "data_type",
        "food_category_id",
        "category_name",
    ]
]


# =============================================================================
# BUILD UNIT SUMMARY
# =============================================================================

section(
    "PORTION UNIT SUMMARY"
)


unit_summary = (

    portion_lookup[
        [
            "unit",
            "unit_normalized",
        ]
    ]

    .fillna("")

    .drop_duplicates()

    .sort_values(
        [
            "unit_normalized",
            "unit",
        ]
    )
)


report(
    unit_summary.to_string(
        index=False
    )
)


# =============================================================================
# VALIDATION
# =============================================================================

header(
    "APPLICATION LAYER VALIDATION"
)


# -------------------------------------------------------------------------
# Food count
# -------------------------------------------------------------------------

food_count = len(
    ingredient_search
)

unique_food_count = (
    ingredient_search[
        "fdc_id"
    ]
    .nunique()
)


report(
    f"Application ingredient rows: "
    f"{food_count:,}"
)

report(
    f"Unique FDC IDs: "
    f"{unique_food_count:,}"
)


if (
    food_count
    ==
    unique_food_count
):

    report(
        "Unique ingredient FDC IDs: PASS"
    )

else:

    report(
        "Unique ingredient FDC IDs: FAIL"
    )


# -------------------------------------------------------------------------
# Description coverage
# -------------------------------------------------------------------------

description_coverage = (
    ingredient_search[
        "description"
    ]
    .notna()
    .sum()
)


report(
    f"Descriptions present: "
    f"{description_coverage:,}/{food_count:,}"
)


# -------------------------------------------------------------------------
# Search token coverage
# -------------------------------------------------------------------------

token_coverage = (
    ingredient_search[
        "search_tokens"
    ]
    .fillna("")
    .ne("")
    .sum()
)


report(
    f"Foods with search tokens: "
    f"{token_coverage:,}/{food_count:,}"
)


# -------------------------------------------------------------------------
# Portion coverage
# -------------------------------------------------------------------------

foods_with_portions = (
    ingredient_search[
        ingredient_search[
            "has_portions"
        ]
    ][
        "fdc_id"
    ]
    .nunique()
)


foods_with_gram_portions = (
    ingredient_search[
        ingredient_search[
            "has_gram_portions"
        ]
    ][
        "fdc_id"
    ]
    .nunique()
)


report(
    f"Foods with portions: "
    f"{foods_with_portions:,}"
)

report(
    f"Foods with gram portions: "
    f"{foods_with_gram_portions:,}"
)


# -------------------------------------------------------------------------
# Valid grams per unit
# -------------------------------------------------------------------------

valid_gram_portions = int(
    portion_lookup[
        "has_valid_gram_conversion"
    ].sum()
)


report(
    f"Portions with valid gram conversion: "
    f"{valid_gram_portions:,}/"
    f"{len(portion_lookup):,}"
)


# -------------------------------------------------------------------------
# Invalid gram weights
# -------------------------------------------------------------------------

negative_grams = int(
    (
        pd.to_numeric(
            portion_lookup[
                "gram_weight_numeric"
            ],
            errors="coerce"
        )
        < 0
    )
    .sum()
)


if negative_grams == 0:

    report(
        "Negative portion gram weights: PASS"
    )

else:

    report(
        f"WARNING: negative portion gram weights: "
        f"{negative_grams}"
    )


# -------------------------------------------------------------------------
# Search test coverage
# -------------------------------------------------------------------------

search_queries_tested = (
    search_test_df[
        "query"
    ]
    .nunique()
)


search_queries_with_results = (
    search_test_df[
        search_test_df[
            "fdc_id"
        ].notna()
    ][
        "query"
    ]
    .nunique()
)


report(
    f"Search queries tested: "
    f"{search_queries_tested}"
)

report(
    f"Search queries with results: "
    f"{search_queries_with_results}"
)


# =============================================================================
# SEARCH QUALITY REPORT
# =============================================================================

header(
    "SEARCH QUALITY SUMMARY"
)


for name, ids in important_results.items():

    if ids:

        report(
            f"{name:<20} "
            f"PASS — {len(ids)} candidate FDC IDs"
        )

    else:

        report(
            f"{name:<20} "
            f"WARNING — no candidates"
        )


# =============================================================================
# SAVE OUTPUTS
# =============================================================================

header(
    "SAVING APPLICATION LAYER"
)


ingredient_search.to_csv(
    INGREDIENT_SEARCH_FILE,
    index=False
)

report(
    f"Saved: {INGREDIENT_SEARCH_FILE}"
)


alias_df.to_csv(
    INGREDIENT_ALIAS_FILE,
    index=False
)

report(
    f"Saved: {INGREDIENT_ALIAS_FILE}"
)


portion_lookup.to_csv(
    PORTION_LOOKUP_FILE,
    index=False
)

report(
    f"Saved: {PORTION_LOOKUP_FILE}"
)


search_test_df.to_csv(
    SEARCH_TEST_FILE,
    index=False
)

report(
    f"Saved: {SEARCH_TEST_FILE}"
)


# =============================================================================
# WRITE REPORT
# =============================================================================

header(
    "FINAL APPLICATION LAYER REPORT"
)


report(
    f"Foods indexed: {food_count:,}"
)

report(
    f"Unique foods: {unique_food_count:,}"
)

report(
    f"Foods with portions: {foods_with_portions:,}"
)

report(
    f"Foods with gram portions: "
    f"{foods_with_gram_portions:,}"
)

report(
    f"Portion records: "
    f"{len(portion_lookup):,}"
)

report(
    f"Valid gram conversions: "
    f"{valid_gram_portions:,}"
)

report(
    f"Alias records: "
    f"{len(alias_df):,}"
)


# =============================================================================
# APPLICATION USAGE EXAMPLES
# =============================================================================

header(
    "APPLICATION USAGE EXAMPLES"
)

report(
    "The following functions are available when this script is imported:"
)

report("")
report(
    "search_ingredients(query, limit=10)"
)

report(
    "portion_candidates(fdc_id, unit=None, modifier=None)"
)

report(
    "resolve_portion(fdc_id, amount, unit, modifier=None)"
)

report(
    "calculate_nutrients_from_grams(fdc_id, grams)"
)

report(
    "calculate_nutrients_from_portion("
)

report(
    "    fdc_id,"
)

report(
    "    amount,"
)

report(
    "    unit,"
)

report(
    "    modifier=None"
)

report(
    ")"
)


# =============================================================================
# WRITE REPORT FILE
# =============================================================================

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "\n".join(
            REPORT_LINES
        )
    )


# =============================================================================
# FINAL STATUS
# =============================================================================

header(
    "SR LEGACY APPLICATION INGREDIENT LAYER COMPLETE"
)


report(
    f"Ingredient records: "
    f"{len(ingredient_search):,}"
)

report(
    f"Alias records: "
    f"{len(alias_df):,}"
)

report(
    f"Portion records: "
    f"{len(portion_lookup):,}"
)

report(
    f"Search test records: "
    f"{len(search_test_df):,}"
)

report("")
report(
    "Output directory:"
)

report(
    str(OUTPUT_DIR)
)

report("")
report(
    "STATUS: PASS"
)

print("")
print("=" * 80)
print("DONE")
print("=" * 80)
print("")
print(
    f"Application layer: {OUTPUT_DIR}"
)



SR LEGACY APPLICATION INGREDIENT LAYER
Build started: 2026-08-16T12:03:45.038391

Step 2 directory:
C:\Users\AK\Downloads\zip\sr_legacy_database

Output directory:
C:\Users\AK\Downloads\zip\sr_legacy_application

CHECKING STEP 2 DATABASE FILES
PASS: food            sr_legacy_food_database.csv                        1.14 MB
PASS: portions        sr_legacy_food_portions.csv                        1.44 MB
PASS: search_index    sr_legacy_food_search_index.csv                    1.46 MB

LOADING STEP 2 DATABASE
Food rows:       7,793
Portion rows:    14,449
Search rows:     7,793

VALIDATING STEP 2 COLUMNS
Food database columns: PASS
Portion database columns: PASS
Search index columns: PASS

--------------------------------------------------------------------------------
NORMALIZING FDC IDS
--------------------------------------------------------------------------------
FDC ID normalization: PASS

BUILDING APPLICATION INGREDIENT SEARCH TABLE

-----------------------------------------------

In [32]:
# ============================================================
# USDA FOODDATA CENTRAL SR LEGACY
# STEP 3 — INGREDIENT RESOLVER + SEARCH RANKING HARDENING
# ============================================================
#
# PURPOSE
# -------
# Harden the Step 2 application ingredient layer.
#
# This script:
#
#   1. Loads the Step 2 application database
#   2. Preserves every USDA food
#   3. Improves ingredient-oriented search ranking
#   4. Distinguishes exact ingredient matches from containing foods
#   5. Handles state/modifier words such as:
#         fresh
#         dried
#         raw
#         ground
#         cooked
#         canned
#         frozen
#   6. Penalizes prepared foods when the query is ingredient-like
#   7. Preserves broader search results
#   8. Builds an ingredient resolver
#   9. Builds ingredient search validation
#  10. Tests known problematic searches
#
# IMPORTANT
# ---------
# This script DOES NOT:
#
#   - delete USDA foods
#   - alter the Step 2 source files
#   - invent nutrient values
#   - invent USDA portion weights
#   - perform fuzzy food selection automatically
#
# It only improves application-level search interpretation.
#
# OUTPUT
# ------
# Creates:
#
#   sr_legacy_step3/
#
# containing:
#
#   sr_legacy_ingredient_resolver.csv
#   sr_legacy_ingredient_search_results.csv
#   sr_legacy_ingredient_test_results.csv
#   sr_legacy_ingredient_quality_report.txt
#
# ============================================================


import os
import re
import math
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

STEP2_DIR = (
    r"C:\Users\AK\Downloads\zip"
    r"\sr_legacy_database"
)

STEP3_DIR = (
    r"C:\Users\AK\Downloads\zip"
    r"\sr_legacy_step3"
)


FOOD_FILE = os.path.join(
    STEP2_DIR,
    "sr_legacy_food_database.csv"
)

PORTION_FILE = os.path.join(
    STEP2_DIR,
    "sr_legacy_food_portions.csv"
)

SEARCH_FILE = os.path.join(
    STEP2_DIR,
    "sr_legacy_food_search_index.csv"
)


# ============================================================
# SEARCH CONFIGURATION
# ============================================================

DEFAULT_LIMIT = 10

MAX_INTERNAL_CANDIDATES = 500

MAX_TEST_RESULTS = 10


# ============================================================
# QUERY / INGREDIENT VOCABULARY
# ============================================================

STATE_WORDS = {
    "raw",
    "fresh",
    "dried",
    "dry",
    "ground",
    "powdered",
    "powder",
    "frozen",
    "canned",
    "jarred",
    "cooked",
    "roasted",
    "baked",
    "boiled",
    "steamed",
    "fried",
    "grilled",
    "broiled",
    "unheated",
    "heated",
    "unsalted",
    "salted",
    "sweetened",
    "unsweetened",
    "reduced",
    "low",
    "fat",
    "whole",
    "chopped",
    "sliced",
    "diced",
    "minced",
    "crushed"
}


PREPARED_FOOD_TERMS = {
    "bread",
    "bagel",
    "buns",
    "bun",
    "pastry",
    "danish",
    "muffin",
    "cookie",
    "cake",
    "pie",
    "pizza",
    "sandwich",
    "burger",
    "hamburger",
    "lasagna",
    "casserole",
    "pancake",
    "waffle",
    "cereal",
    "cracker",
    "crackers",
    "chips",
    "fries",
    "fries",
    "sauce",
    "dressing",
    "mayonnaise",
    "ketchup",
    "mustard",
    "soup",
    "stew",
    "meal",
    "entree",
    "entrée",
    "prepared",
    "restaurant",
    "fast food",
    "frozen meal",
    "snack"
}


INGREDIENT_CLASS_TERMS = {
    "spices",
    "spice",
    "herbs",
    "herb",
    "oil",
    "oils",
    "flour",
    "sugar",
    "salt",
    "vinegar",
    "leavening",
    "baking",
    "seasoning",
    "seasonings",
    "extract",
    "extracts",
    "nuts",
    "nut",
    "seeds",
    "seed",
    "beans",
    "bean",
    "rice",
    "grain",
    "grains",
    "vegetable",
    "vegetables",
    "fruit",
    "fruits",
    "meat",
    "pork",
    "beef",
    "chicken",
    "lamb",
    "fish",
    "seafood",
    "seaweed",
    "kelp"
}


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(value):
    """
    Normalize text for search analysis.
    """

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = value.replace("&", " and ")

    value = re.sub(
        r"[^a-z0-9\s\-]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def tokenize(value):
    """
    Convert text to normalized tokens.
    """

    text = normalize_text(value)

    if not text:
        return []

    return text.split()


def token_set(value):
    return set(
        tokenize(value)
    )


def safe_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default

        return float(value)

    except Exception:
        return default


# ============================================================
# LOAD STEP 2
# ============================================================

print("")
print("=" * 80)
print("SR LEGACY STEP 3 — INGREDIENT RESOLVER")
print("=" * 80)

print("")
print(
    f"Build started: {datetime.now().isoformat()}"
)

print("")
print("Step 2 directory:")
print(STEP2_DIR)

print("")
print("Step 3 output directory:")
print(STEP3_DIR)


os.makedirs(
    STEP3_DIR,
    exist_ok=True
)


# ============================================================
# VERIFY FILES
# ============================================================

print("")
print("=" * 80)
print("CHECKING STEP 2 FILES")
print("=" * 80)


required_files = {
    "food": FOOD_FILE,
    "portion": PORTION_FILE,
    "search": SEARCH_FILE
}


for label, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\nMissing Step 2 {label} file:\n{path}"
        )

    size = os.path.getsize(path)

    print(
        f"PASS: {label:<10} "
        f"{os.path.basename(path):<45} "
        f"{size / (1024 * 1024):.2f} MB"
    )


# ============================================================
# LOAD DATA
# ============================================================

print("")
print("=" * 80)
print("LOADING STEP 2 DATA")
print("=" * 80)


food_df = pd.read_csv(
    FOOD_FILE,
    low_memory=False
)


portion_df = pd.read_csv(
    PORTION_FILE,
    low_memory=False
)


search_df = pd.read_csv(
    SEARCH_FILE,
    low_memory=False
)


print(
    f"Food rows:       {len(food_df):,}"
)

print(
    f"Portion rows:    {len(portion_df):,}"
)

print(
    f"Search rows:     {len(search_df):,}"
)


# ============================================================
# COLUMN DISCOVERY
# ============================================================

print("")
print("=" * 80)
print("DISCOVERING STEP 2 COLUMNS")
print("=" * 80)


print("")
print("Food columns:")

for column in food_df.columns:
    print(
        f"  {column}"
    )


print("")
print("Search columns:")

for column in search_df.columns:
    print(
        f"  {column}"
    )


print("")
print("Portion columns:")

for column in portion_df.columns:
    print(
        f"  {column}"
    )


# ============================================================
# COLUMN RESOLUTION
# ============================================================

def find_column(
    dataframe,
    candidates,
    required=True
):
    """
    Find the first matching column
    case-insensitively.
    """

    normalized = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for candidate in candidates:

        key = candidate.lower()

        if key in normalized:

            return normalized[key]

    if required:

        raise RuntimeError(
            "Required column not found.\n"
            f"Candidates: {candidates}\n"
            f"Available: {list(dataframe.columns)}"
        )

    return None


food_fdc_col = find_column(
    food_df,
    [
        "fdc_id",
        "food_id"
    ]
)


food_description_col = find_column(
    food_df,
    [
        "description",
        "food_description",
        "food_name"
    ]
)


search_fdc_col = find_column(
    search_df,
    [
        "fdc_id",
        "food_id"
    ]
)


search_description_col = find_column(
    search_df,
    [
        "description",
        "food_description",
        "food_name"
    ],
    required=False
)


# ============================================================
# NORMALIZE FDC IDS
# ============================================================

print("")
print("=" * 80)
print("NORMALIZING FDC IDS")
print("=" * 80)


food_df["_fdc_id"] = pd.to_numeric(
    food_df[food_fdc_col],
    errors="coerce"
)


search_df["_fdc_id"] = pd.to_numeric(
    search_df[search_fdc_col],
    errors="coerce"
)


if food_df["_fdc_id"].isna().any():

    raise RuntimeError(
        "Food database contains invalid FDC IDs."
    )


if search_df["_fdc_id"].isna().any():

    raise RuntimeError(
        "Search index contains invalid FDC IDs."
    )


food_df["_fdc_id"] = (
    food_df["_fdc_id"]
    .astype("int64")
)


search_df["_fdc_id"] = (
    search_df["_fdc_id"]
    .astype("int64")
)


print("FDC ID normalization: PASS")


# ============================================================
# BUILD MASTER SEARCH TABLE
# ============================================================

print("")
print("=" * 80)
print("BUILDING STEP 3 SEARCH TABLE")
print("=" * 80)


# Start from food database.
#
# This guarantees every USDA food remains represented.

resolver_df = food_df.copy()


resolver_df["fdc_id"] = (
    resolver_df["_fdc_id"]
)


resolver_df["description"] = (
    resolver_df[
        food_description_col
    ]
    .fillna("")
    .astype(str)
)


resolver_df["description_normalized"] = (
    resolver_df["description"]
    .map(normalize_text)
)


resolver_df["tokens"] = (
    resolver_df["description_normalized"]
    .map(tokenize)
)


resolver_df["token_set"] = (
    resolver_df["description_normalized"]
    .map(token_set)
)


# ============================================================
# FOOD TYPE CLASSIFICATION
# ============================================================

def contains_any_term(
    tokens,
    terms
):
    """
    Check whether tokenized text contains
    any vocabulary term.
    """

    token_values = set(tokens)

    for term in terms:

        normalized = normalize_text(term)

        if " " in normalized:

            if normalized in " ".join(tokens):
                return True

        elif normalized in token_values:

            return True

    return False


def classify_food(
    description
):
    """
    Classify food for search ranking.

    This is an application search hint,
    NOT USDA food taxonomy.
    """

    tokens = tokenize(
        description
    )

    text = " ".join(tokens)

    prepared = contains_any_term(
        tokens,
        PREPARED_FOOD_TERMS
    )

    ingredient_class = contains_any_term(
        tokens,
        INGREDIENT_CLASS_TERMS
    )

    state_present = contains_any_term(
        tokens,
        STATE_WORDS
    )

    # Strong ingredient-like patterns.

    if text.startswith("spices "):
        return "INGREDIENT"

    if text.startswith("oil "):
        return "INGREDIENT"

    if text.startswith("basil "):
        return "INGREDIENT"

    if text.startswith("salt "):
        return "INGREDIENT"

    if text.startswith("vinegar "):
        return "INGREDIENT"

    if text.startswith("seaweed "):
        return "INGREDIENT"

    if prepared and ingredient_class:
        return "PREPARED_CONTAINING_INGREDIENT"

    if prepared:
        return "PREPARED_FOOD"

    if ingredient_class:
        return "INGREDIENT"

    if state_present:
        return "FOOD_ITEM"

    return "OTHER"


resolver_df["food_class"] = (
    resolver_df["description"]
    .map(classify_food)
)


# ============================================================
# SEARCH FEATURE ENGINEERING
# ============================================================

print("")
print("Creating search quality features...")


def first_tokens(
    tokens,
    count
):
    return tokens[:count]


resolver_df["first_token"] = (
    resolver_df["tokens"]
    .map(
        lambda x:
        x[0]
        if x
        else ""
    )
)


resolver_df["second_token"] = (
    resolver_df["tokens"]
    .map(
        lambda x:
        x[1]
        if len(x) > 1
        else ""
    )
)


resolver_df["description_token_count"] = (
    resolver_df["tokens"]
    .map(len)
)


resolver_df["is_prepared_food"] = (
    resolver_df["food_class"]
    .isin(
        [
            "PREPARED_FOOD",
            "PREPARED_CONTAINING_INGREDIENT"
        ]
    )
)


resolver_df["is_ingredient_like"] = (
    resolver_df["food_class"]
    .isin(
        [
            "INGREDIENT"
        ]
    )
)


# ============================================================
# QUERY NORMALIZATION
# ============================================================

def normalize_query(
    query
):
    return normalize_text(query)


def query_tokens(
    query
):
    return tokenize(query)


# ============================================================
# QUERY INTENT
# ============================================================

def classify_query_intent(
    query
):
    """
    Determine whether a query looks like
    an ingredient search.

    Examples:
        cinnamon       -> ingredient
        ground cinnamon -> ingredient
        olive oil      -> ingredient
        fresh basil    -> ingredient
        pork shoulder  -> ingredient
    """

    tokens = token_set(query)

    if not tokens:
        return "EMPTY"

    if (
        tokens & INGREDIENT_CLASS_TERMS
    ):

        return "INGREDIENT"

    # Very short food-name queries are treated
    # as ingredient-like because users commonly
    # enter raw ingredient names.

    if len(tokens) <= 3:

        return "INGREDIENT"

    return "GENERAL"


# ============================================================
# MODIFIER EXTRACTION
# ============================================================

def extract_modifiers(
    query
):
    """
    Extract state/modifier terms from query.
    """

    tokens = query_tokens(query)

    return [
        token
        for token in tokens
        if token in STATE_WORDS
    ]


# ============================================================
# CORE SEARCH SCORER
# ============================================================

def score_food(
    query,
    description,
    food_class
):
    """
    Ingredient-aware ranking score.

    Higher is better.
    """

    normalized_query = normalize_query(
        query
    )

    q_tokens = query_tokens(
        normalized_query
    )

    q_set = set(q_tokens)

    normalized_description = normalize_text(
        description
    )

    d_tokens = tokenize(
        normalized_description
    )

    d_set = set(d_tokens)

    if not q_tokens:
        return 0.0

    score = 0.0


    # --------------------------------------------------------
    # Exact description match
    # --------------------------------------------------------

    if normalized_description == normalized_query:

        score += 150.0


    # --------------------------------------------------------
    # Exact phrase match
    # --------------------------------------------------------

    if normalized_query in normalized_description:

        score += 45.0


    # --------------------------------------------------------
    # Token overlap
    # --------------------------------------------------------

    matched_tokens = (
        q_set & d_set
    )

    if matched_tokens:

        score += (
            15.0
            *
            len(matched_tokens)
        )


    # --------------------------------------------------------
    # All query tokens present
    # --------------------------------------------------------

    if q_set.issubset(d_set):

        score += 25.0


    # --------------------------------------------------------
    # Token order
    # --------------------------------------------------------

    if (
        len(q_tokens) > 1
        and
        all(
            token in d_tokens
            for token in q_tokens
        )
    ):

        positions = []

        for token in q_tokens:

            try:
                positions.append(
                    d_tokens.index(token)
                )

            except ValueError:
                pass

        if positions == sorted(
            positions
        ):

            score += 12.0


    # --------------------------------------------------------
    # Starts with query
    # --------------------------------------------------------

    if normalized_description.startswith(
        normalized_query
    ):

        score += 25.0


    # --------------------------------------------------------
    # Ingredient intent
    # --------------------------------------------------------

    intent = classify_query_intent(
        query
    )


    if intent == "INGREDIENT":

        if food_class == "INGREDIENT":

            score += 35.0

        elif (
            food_class
            ==
            "PREPARED_CONTAINING_INGREDIENT"
        ):

            score -= 18.0

        elif food_class == "PREPARED_FOOD":

            score -= 28.0


    # --------------------------------------------------------
    # Exact ingredient token placement
    #
    # Example:
    #
    # cinnamon
    #
    # Spices, cinnamon, ground
    #
    # should beat:
    #
    # Bread, cinnamon
    # --------------------------------------------------------

    if len(q_tokens) == 1:

        query_token = q_tokens[0]

        if query_token in d_set:

            if food_class == "INGREDIENT":

                score += 25.0

            elif food_class == "PREPARED_FOOD":

                score -= 10.0


    # --------------------------------------------------------
    # Modifier matching
    # --------------------------------------------------------

    modifiers = extract_modifiers(
        query
    )


    if modifiers:

        for modifier in modifiers:

            if modifier in d_set:

                score += 15.0

            else:

                score -= 8.0


    # --------------------------------------------------------
    # Penalize missing modifiers only modestly
    #
    # We don't want:
    #
    # "fresh basil"
    #
    # to eliminate dried basil.
    # --------------------------------------------------------

    if modifiers:

        for modifier in modifiers:

            if modifier not in d_set:

                score -= 4.0


    # --------------------------------------------------------
    # Prefer concise descriptions for direct ingredient
    # queries.
    # --------------------------------------------------------

    if intent == "INGREDIENT":

        if len(d_tokens) <= len(q_tokens) + 4:

            score += 8.0


    # --------------------------------------------------------
    # Prepared-food penalty
    #
    # Only apply strongly when the search itself
    # looks like an ingredient request.
    # --------------------------------------------------------

    if intent == "INGREDIENT":

        if food_class == "PREPARED_FOOD":

            score -= 12.0

        elif (
            food_class
            ==
            "PREPARED_CONTAINING_INGREDIENT"
        ):

            score -= 8.0


    return round(
        score,
        3
    )


# ============================================================
# BUILD INGREDIENT SEARCH INDEX
# ============================================================

print("")
print("=" * 80)
print("BUILDING INGREDIENT SEARCH INDEX")
print("=" * 80)


search_rows = []


for _, row in resolver_df.iterrows():

    search_rows.append({

        "fdc_id":
            int(row["fdc_id"]),

        "description":
            row["description"],

        "description_normalized":
            row["description_normalized"],

        "food_class":
            row["food_class"],

        "is_ingredient_like":
            bool(row["is_ingredient_like"]),

        "is_prepared_food":
            bool(row["is_prepared_food"]),

        "description_token_count":
            int(row["description_token_count"]),

        "first_token":
            row["first_token"],

        "second_token":
            row["second_token"]
    })


ingredient_search_df = pd.DataFrame(
    search_rows
)


print(
    f"Ingredient search records: "
    f"{len(ingredient_search_df):,}"
)


# ============================================================
# SEARCH FUNCTION
# ============================================================

def search_ingredients(
    query,
    limit=10
):
    """
    Search the Step 3 ingredient index.

    Returns a DataFrame.
    """

    query = normalize_query(
        query
    )

    if not query:

        return pd.DataFrame(
            columns=[
                "fdc_id",
                "description",
                "food_class",
                "score"
            ]
        )


    q_tokens = token_set(
        query
    )


    # --------------------------------------------------------
    # Candidate filtering
    #
    # Search by at least one query token.
    # --------------------------------------------------------

    candidate_mask = (
        ingredient_search_df[
            "description_normalized"
        ]
        .apply(
            lambda text:
            any(
                token in text.split()
                for token in q_tokens
            )
        )
    )


    candidates = (
        ingredient_search_df[
            candidate_mask
        ]
        .copy()
    )


    if candidates.empty:

        return pd.DataFrame(
            columns=[
                "fdc_id",
                "description",
                "food_class",
                "score"
            ]
        )


    # --------------------------------------------------------
    # Score candidates
    # --------------------------------------------------------

    candidates["score"] = candidates.apply(

        lambda row:
        score_food(
            query,
            row["description"],
            row["food_class"]
        ),

        axis=1
    )


    # --------------------------------------------------------
    # Secondary ranking
    #
    # Prefer ingredient-like records where scores
    # are close.
    # --------------------------------------------------------

    candidates["ingredient_priority"] = (
        candidates["food_class"]
        .map({

            "INGREDIENT": 3,

            "FOOD_ITEM": 2,

            "OTHER": 1,

            "PREPARED_CONTAINING_INGREDIENT": 0,

            "PREPARED_FOOD": -1

        })
        .fillna(0)
    )


    candidates = candidates.sort_values(
        by=[
            "score",
            "ingredient_priority",
            "description_token_count",
            "fdc_id"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )


    return (
        candidates[
            [
                "fdc_id",
                "description",
                "food_class",
                "score"
            ]
        ]
        .head(limit)
        .reset_index(drop=True)
    )


# ============================================================
# INGREDIENT RESOLVER
# ============================================================

def resolve_ingredient(
    query,
    limit=10,
    minimum_score=20.0
):
    """
    Resolve a natural-language ingredient query.

    Returns ranked candidates and a recommended
    candidate when confidence is sufficient.
    """

    results = search_ingredients(
        query,
        limit=limit
    )


    if results.empty:

        return {

            "status":
                "NO_RESULTS",

            "query":
                query,

            "resolved_fdc_id":
                None,

            "resolved_description":
                None,

            "confidence":
                0.0,

            "results":
                results
        }


    top = results.iloc[0]

    top_score = safe_float(
        top["score"]
    )


    if top_score < minimum_score:

        status = "LOW_CONFIDENCE"

    else:

        status = "RESOLVED"


    return {

        "status":
            status,

        "query":
            query,

        "resolved_fdc_id":
            int(top["fdc_id"]),

        "resolved_description":
            top["description"],

        "confidence":
            top_score,

        "results":
            results
    }


# ============================================================
# IMPORTANT TEST QUERIES
# ============================================================

TEST_QUERIES = [

    "cinnamon",
    "ground cinnamon",
    "turmeric",
    "basil",
    "fresh basil",
    "dried basil",
    "olive oil",
    "oil olive",
    "baking powder",
    "jackfruit",
    "raw jackfruit",
    "kelp",
    "seaweed kelp",
    "pork shoulder",
    "shoulder pork",
    "salt",
    "black pepper",
    "garlic",
    "ginger",
    "rice",
    "flour",
    "sugar"
]


# ============================================================
# RUN SEARCH TESTS
# ============================================================

print("")
print("=" * 80)
print("RUNNING STEP 3 SEARCH TESTS")
print("=" * 80)


test_rows = []


for query in TEST_QUERIES:

    print("")
    print(
        f"SEARCH: {query}"
    )

    results = search_ingredients(
        query,
        limit=MAX_TEST_RESULTS
    )


    if results.empty:

        print(
            "  NO RESULTS"
        )

        test_rows.append({

            "query":
                query,

            "status":
                "FAIL",

            "result_count":
                0,

            "top_fdc_id":
                None,

            "top_description":
                None,

            "top_score":
                0.0
        })

        continue


    for index, row in results.iterrows():

        print(
            f"  {index + 1}. "
            f"[FDC {int(row['fdc_id'])}] "
            f"{row['description']} "
            f"(class={row['food_class']}, "
            f"score={row['score']})"
        )


    top = results.iloc[0]


    test_rows.append({

        "query":
            query,

        "status":
            "PASS",

        "result_count":
            len(results),

        "top_fdc_id":
            int(top["fdc_id"]),

        "top_description":
            top["description"],

        "top_score":
            safe_float(
                top["score"]
            )
    })


# ============================================================
# EXPECTED RESULT VALIDATION
# ============================================================

print("")
print("=" * 80)
print("IMPORTANT INGREDIENT RANKING VALIDATION")
print("=" * 80)


EXPECTED_TOP_PATTERNS = {

    "cinnamon": [
        "cinnamon"
    ],

    "ground cinnamon": [
        "cinnamon",
        "ground"
    ],

    "turmeric": [
        "turmeric"
    ],

    "fresh basil": [
        "basil",
        "fresh"
    ],

    "olive oil": [
        "oil",
        "olive"
    ],

    "oil olive": [
        "oil",
        "olive"
    ],

    "baking powder": [
        "baking powder"
    ],

    "jackfruit": [
        "jackfruit"
    ],

    "raw jackfruit": [
        "jackfruit",
        "raw"
    ],

    "kelp": [
        "kelp"
    ],

    "seaweed kelp": [
        "seaweed",
        "kelp"
    ],

    "pork shoulder": [
        "pork",
        "shoulder"
    ],

    "shoulder pork": [
        "pork",
        "shoulder"
    ]
}


ranking_validation_rows = []


for query, expected_tokens in (
    EXPECTED_TOP_PATTERNS.items()
):

    results = search_ingredients(
        query,
        limit=5
    )


    if results.empty:

        print("")
        print(
            f"{query}: FAIL — no results"
        )

        ranking_validation_rows.append({

            "query":
                query,

            "status":
                "FAIL",

            "top_description":
                "",

            "expected_tokens":
                " ".join(expected_tokens)
        })

        continue


    top_description = normalize_text(
        results.iloc[0]["description"]
    )


    top_tokens = set(
        tokenize(
            top_description
        )
    )


    token_match = all(
        token in top_tokens
        for token in expected_tokens
    )


    if token_match:

        status = "PASS"

    else:

        status = "REVIEW"


    print("")
    print(
        f"{query}: {status}"
    )

    print(
        f"  Top: "
        f"{results.iloc[0]['description']}"
    )

    print(
        f"  Expected concepts: "
        f"{', '.join(expected_tokens)}"
    )


    ranking_validation_rows.append({

        "query":
            query,

        "status":
            status,

        "top_description":
            results.iloc[0]["description"],

        "expected_tokens":
            " ".join(expected_tokens)
    })


# ============================================================
# SPECIFIC PREPARED-FOOD PROTECTION TESTS
# ============================================================

print("")
print("=" * 80)
print("PREPARED-FOOD RANKING TESTS")
print("=" * 80)


PREPARED_FOOD_TESTS = {

    "cinnamon": [
        "Bread, cinnamon"
    ],

    "olive oil": [
        "Mayonnaise, reduced fat, with olive oil"
    ]
}


prepared_validation_rows = []


for query, known_prepared in (
    PREPARED_FOOD_TESTS.items()
):

    results = search_ingredients(
        query,
        limit=10
    )


    if results.empty:

        status = "FAIL"

        top_description = ""

    else:

        top_description = (
            results.iloc[0]["description"]
        )


        top_class = (
            results.iloc[0]["food_class"]
        )


        if (
            top_class
            ==
            "PREPARED_FOOD"
        ):

            status = "REVIEW"

        else:

            status = "PASS"


    print("")
    print(
        f"{query}: {status}"
    )

    print(
        f"  Top: {top_description}"
    )


    prepared_validation_rows.append({

        "query":
            query,

        "status":
            status,

        "top_description":
            top_description,

        "known_prepared_examples":
            " | ".join(
                known_prepared
            )
    })


# ============================================================
# BUILD FINAL RESOLVER TABLE
# ============================================================

print("")
print("=" * 80)
print("BUILDING FINAL INGREDIENT RESOLVER TABLE")
print("=" * 80)


resolver_output = resolver_df[
    [
        "fdc_id",
        "description",
        "description_normalized",
        "food_class",
        "is_ingredient_like",
        "is_prepared_food",
        "description_token_count",
        "first_token",
        "second_token"
    ]
].copy()


# ============================================================
# SAVE RESOLVER
# ============================================================

resolver_output_path = os.path.join(
    STEP3_DIR,
    "sr_legacy_ingredient_resolver.csv"
)


resolver_output.to_csv(
    resolver_output_path,
    index=False
)


print(
    f"Saved: {resolver_output_path}"
)


# ============================================================
# SAVE SEARCH RESULTS
# ============================================================

search_result_rows = []


for query in TEST_QUERIES:

    results = search_ingredients(
        query,
        limit=MAX_TEST_RESULTS
    )


    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):

        search_result_rows.append({

            "query":
                query,

            "rank":
                rank,

            "fdc_id":
                int(row["fdc_id"]),

            "description":
                row["description"],

            "food_class":
                row["food_class"],

            "score":
                safe_float(
                    row["score"]
                )
        })


search_results_output = pd.DataFrame(
    search_result_rows
)


search_results_path = os.path.join(
    STEP3_DIR,
    "sr_legacy_ingredient_search_results.csv"
)


search_results_output.to_csv(
    search_results_path,
    index=False
)


print(
    f"Saved: {search_results_path}"
)


# ============================================================
# SAVE TEST RESULTS
# ============================================================

test_results_df = pd.DataFrame(
    test_rows
)


ranking_validation_df = pd.DataFrame(
    ranking_validation_rows
)


prepared_validation_df = pd.DataFrame(
    prepared_validation_rows
)


test_results_path = os.path.join(
    STEP3_DIR,
    "sr_legacy_ingredient_test_results.csv"
)


with open(
    test_results_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "SEARCH TEST RESULTS\n"
    )

    file.write(
        test_results_df.to_csv(
            index=False
        )
    )

    file.write(
        "\n\nRANKING VALIDATION\n"
    )

    file.write(
        ranking_validation_df.to_csv(
            index=False
        )
    )

    file.write(
        "\n\nPREPARED FOOD VALIDATION\n"
    )

    file.write(
        prepared_validation_df.to_csv(
            index=False
        )
    )


print(
    f"Saved: {test_results_path}"
)


# ============================================================
# FINAL QUALITY COUNTS
# ============================================================

total_foods = len(
    resolver_output
)


ingredient_like_count = int(
    resolver_output[
        "is_ingredient_like"
    ].sum()
)


prepared_count = int(
    resolver_output[
        "is_prepared_food"
    ].sum()
)


search_pass_count = int(
    (
        test_results_df["status"]
        == "PASS"
    ).sum()
)


ranking_pass_count = int(
    (
        ranking_validation_df["status"]
        == "PASS"
    ).sum()
)


ranking_review_count = int(
    (
        ranking_validation_df["status"]
        == "REVIEW"
    ).sum()
)


ranking_fail_count = int(
    (
        ranking_validation_df["status"]
        == "FAIL"
    ).sum()
)


# ============================================================
# WRITE REPORT
# ============================================================

report_path = os.path.join(
    STEP3_DIR,
    "sr_legacy_ingredient_quality_report.txt"
)


report_lines = []


def add_report(
    text=""
):
    report_lines.append(
        str(text)
    )


add_report("=" * 80)
add_report(
    "SR LEGACY STEP 3 — INGREDIENT RESOLVER QUALITY REPORT"
)
add_report("=" * 80)


add_report("")
add_report(
    f"Build completed: "
    f"{datetime.now().isoformat()}"
)


add_report("")
add_report(
    f"Step 2 food records: "
    f"{total_foods:,}"
)


add_report(
    f"Ingredient-like foods: "
    f"{ingredient_like_count:,}"
)


add_report(
    f"Prepared foods: "
    f"{prepared_count:,}"
)


add_report("")
add_report(
    "SEARCH VALIDATION"
)


add_report(
    f"Queries tested: "
    f"{len(test_results_df):,}"
)


add_report(
    f"Queries with results: "
    f"{search_pass_count:,}"
)


add_report("")
add_report(
    "RANKING VALIDATION"
)


add_report(
    f"Ranking tests passed: "
    f"{ranking_pass_count:,}"
)


add_report(
    f"Ranking tests requiring review: "
    f"{ranking_review_count:,}"
)


add_report(
    f"Ranking tests failed: "
    f"{ranking_fail_count:,}"
)


add_report("")
add_report(
    "IMPORTANT DESIGN RULES"
)


add_report(
    "1. Every Step 2 USDA food remains represented."
)


add_report(
    "2. Search ranking does not delete foods."
)


add_report(
    "3. Ingredient-like foods receive ranking preference "
    "for ingredient queries."
)


add_report(
    "4. Prepared foods containing an ingredient remain searchable."
)


add_report(
    "5. Modifier words such as fresh, dried, raw and ground "
    "affect ranking."
)


add_report(
    "6. This layer does not invent USDA portion weights."
)


add_report(
    "7. This layer does not invent nutrient values."
)


add_report("")
add_report(
    "TOP RANKING VALIDATION"
)


for _, row in ranking_validation_df.iterrows():

    add_report(
        f"{row['query']}: "
        f"{row['status']} — "
        f"{row['top_description']}"
    )


add_report("")
add_report(
    "PREPARED FOOD VALIDATION"
)


for _, row in prepared_validation_df.iterrows():

    add_report(
        f"{row['query']}: "
        f"{row['status']} — "
        f"{row['top_description']}"
    )


add_report("")
add_report("=" * 80)
add_report(
    "STEP 3 COMPLETE"
)
add_report("=" * 80)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "\n".join(
            report_lines
        )
    )


print(
    f"Saved: {report_path}"
)


# ============================================================
# FINAL CONSOLE SUMMARY
# ============================================================

print("")
print("=" * 80)
print("STEP 3 FINAL VALIDATION")
print("=" * 80)


print(
    f"Foods preserved:             "
    f"{total_foods:,}"
)


print(
    f"Ingredient-like foods:       "
    f"{ingredient_like_count:,}"
)


print(
    f"Prepared foods:              "
    f"{prepared_count:,}"
)


print(
    f"Search queries tested:       "
    f"{len(test_results_df):,}"
)


print(
    f"Queries with results:        "
    f"{search_pass_count:,}"
)


print(
    f"Ranking tests PASS:          "
    f"{ranking_pass_count:,}"
)


print(
    f"Ranking tests REVIEW:        "
    f"{ranking_review_count:,}"
)


print(
    f"Ranking tests FAIL:          "
    f"{ranking_fail_count:,}"
)


print("")
print(
    "Output directory:"
)


print(
    STEP3_DIR
)


print("")
print(
    "Files created:"
)


for filename in sorted(
    os.listdir(STEP3_DIR)
):

    print(
        f"  {filename}"
    )


print("")
print("=" * 80)
print("STEP 3 INGREDIENT RESOLVER COMPLETE")
print("=" * 80)



SR LEGACY STEP 3 — INGREDIENT RESOLVER

Build started: 2026-08-16T12:59:54.302894

Step 2 directory:
C:\Users\AK\Downloads\zip\sr_legacy_database

Step 3 output directory:
C:\Users\AK\Downloads\zip\sr_legacy_step3

CHECKING STEP 2 FILES
PASS: food       sr_legacy_food_database.csv                   1.14 MB
PASS: portion    sr_legacy_food_portions.csv                   1.44 MB
PASS: search     sr_legacy_food_search_index.csv               1.46 MB

LOADING STEP 2 DATA
Food rows:       7,793
Portion rows:    14,449
Search rows:     7,793

DISCOVERING STEP 2 COLUMNS

Food columns:
  fdc_id
  description
  data_type
  food_category_id
  category_name
  basis_g
  calories_kcal
  protein_g
  fat_g
  carbohydrate_g
  fiber_g
  sugar_g
  saturated_fat_g
  sodium_mg
  cholesterol_mg

Search columns:
  fdc_id
  description
  category_name
  search_text
  description_normalized

Portion columns:
  fdc_id
  description
  amount
  unit
  measure_unit_id
  gram_weight
  modifier

NORMALIZING FDC IDS

In [33]:
from __future__ import annotations

import csv
import re
import sys
from pathlib import Path
from datetime import datetime


# =============================================================================
# SR LEGACY STEP 4 — INGREDIENT CANONICAL APPLICATION LAYER
# =============================================================================

STEP3_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step3")
OUTPUT_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step4")

RESOLVER_FILE = STEP3_DIR / "sr_legacy_ingredient_resolver.csv"
SEARCH_FILE = STEP3_DIR / "sr_legacy_ingredient_search_results.csv"
TEST_FILE = STEP3_DIR / "sr_legacy_ingredient_test_results.csv"


# =============================================================================
# HELPERS
# =============================================================================

def log(msg=""):
    print(msg)


def fail(msg):
    print(f"FAIL: {msg}")
    sys.exit(1)


def normalize_text(value):
    if value is None:
        return ""

    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def read_csv(path: Path):
    if not path.exists():
        fail(f"Missing required file: {path}")

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames or []

    return columns, rows


def write_csv(path: Path, columns, rows):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=columns,
            extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


def first_value(row, *names, default=""):
    for name in names:
        if name in row and row[name] not in (None, ""):
            return row[name]
    return default


def normalize_fdc(value):
    value = str(value).strip()

    # Preserve numeric FDC IDs as integer-looking strings.
    if value.endswith(".0"):
        value = value[:-2]

    return value


# =============================================================================
# START
# =============================================================================

started = datetime.now()

log("=" * 80)
log("SR LEGACY STEP 4 — INGREDIENT CANONICAL APPLICATION LAYER")
log("=" * 80)
log()
log(f"Build started: {started.isoformat()}")
log()
log(f"Step 3 directory:")
log(f"{STEP3_DIR}")
log()
log(f"Step 4 output directory:")
log(f"{OUTPUT_DIR}")
log()


# =============================================================================
# CHECK INPUTS
# =============================================================================

log("=" * 80)
log("CHECKING STEP 3 FILES")
log("=" * 80)

required_files = [
    ("resolver", RESOLVER_FILE),
    ("search", SEARCH_FILE),
    ("tests", TEST_FILE),
]

for label, path in required_files:
    if not path.exists():
        fail(f"{label}: {path}")

    size_mb = path.stat().st_size / (1024 * 1024)
    log(f"PASS: {label:<10} {path.name:<50} {size_mb:.2f} MB")

log()


# =============================================================================
# LOAD STEP 3
# =============================================================================

log("=" * 80)
log("LOADING STEP 3 DATA")
log("=" * 80)

resolver_columns, resolver_rows = read_csv(RESOLVER_FILE)
search_columns, search_rows = read_csv(SEARCH_FILE)
test_columns, test_rows = read_csv(TEST_FILE)

log(f"Resolver rows: {len(resolver_rows):,}")
log(f"Search rows:   {len(search_rows):,}")
log(f"Test rows:     {len(test_rows):,}")
log()


# =============================================================================
# DISCOVER COLUMNS
# =============================================================================

log("=" * 80)
log("DISCOVERING STEP 3 COLUMNS")
log("=" * 80)

log()
log("Resolver columns:")
for col in resolver_columns:
    log(f"  {col}")

log()
log("Search columns:")
for col in search_columns:
    log(f"  {col}")

log()


# =============================================================================
# REQUIRED COLUMN VALIDATION
# =============================================================================

log("=" * 80)
log("VALIDATING RESOLVER COLUMNS")
log("=" * 80)

fdc_column_candidates = [
    "fdc_id",
    "FDC_ID",
    "Fdc_ID",
]

description_candidates = [
    "description",
    "food_description",
]

required_found = {}

for logical_name, candidates in [
    ("fdc_id", fdc_column_candidates),
    ("description", description_candidates),
]:
    found = next(
        (c for c in candidates if c in resolver_columns),
        None
    )

    if found is None:
        fail(
            f"Could not find required {logical_name} column. "
            f"Available columns: {resolver_columns}"
        )

    required_found[logical_name] = found
    log(f"PASS: {logical_name:<15} -> {found}")

log()


# =============================================================================
# NORMALIZE FDC IDS
# =============================================================================

log("-" * 80)
log("NORMALIZING FDC IDS")
log("-" * 80)

resolver_fdc = required_found["fdc_id"]

for row in resolver_rows:
    row[resolver_fdc] = normalize_fdc(row[resolver_fdc])

fdc_ids = [
    row[resolver_fdc]
    for row in resolver_rows
    if row[resolver_fdc]
]

if len(fdc_ids) != len(resolver_rows):
    fail("One or more resolver rows have missing FDC IDs.")

if len(set(fdc_ids)) != len(fdc_ids):
    fail("Duplicate FDC IDs found in resolver table.")

log(f"FDC ID normalization: PASS")
log(f"Unique FDC IDs:       {len(set(fdc_ids)):,}")
log()


# =============================================================================
# BUILD CANONICAL APPLICATION TABLE
# =============================================================================

log("=" * 80)
log("BUILDING STEP 4 CANONICAL INGREDIENT TABLE")
log("=" * 80)

canonical_columns = [
    "fdc_id",
    "description",
    "description_normalized",
    "ingredient_class",
    "search_score",
    "searchable",
    "is_ingredient",
    "is_prepared_food",
    "is_food_item",
    "resolver_status",
]

canonical_rows = []

for row in resolver_rows:

    fdc_id = normalize_fdc(
        first_value(row, "fdc_id", "FDC_ID", "Fdc_ID")
    )

    description = first_value(
        row,
        "description",
        "food_description"
    )

    normalized_description = normalize_text(description)

    ingredient_class = first_value(
        row,
        "ingredient_class",
        "class",
        "food_class",
        "resolver_class",
        default=""
    )

    search_score = first_value(
        row,
        "search_score",
        "score",
        "base_score",
        default=""
    )

    ingredient_class_upper = ingredient_class.upper()

    is_ingredient = (
        ingredient_class_upper == "INGREDIENT"
    )

    is_prepared_food = (
        ingredient_class_upper in {
            "PREPARED_FOOD",
            "PREPARED_CONTAINING_INGREDIENT",
        }
    )

    is_food_item = (
        ingredient_class_upper == "FOOD_ITEM"
    )

    searchable = bool(normalized_description)

    resolver_status = "RESOLVED" if searchable else "REVIEW"

    canonical_rows.append({
        "fdc_id": fdc_id,
        "description": description,
        "description_normalized": normalized_description,
        "ingredient_class": ingredient_class,
        "search_score": search_score,
        "searchable": "1" if searchable else "0",
        "is_ingredient": "1" if is_ingredient else "0",
        "is_prepared_food": "1" if is_prepared_food else "0",
        "is_food_item": "1" if is_food_item else "0",
        "resolver_status": resolver_status,
    })

log(f"Canonical records: {len(canonical_rows):,}")
log()


# =============================================================================
# QUALITY VALIDATION
# =============================================================================

log("=" * 80)
log("RUNNING STEP 4 QUALITY VALIDATION")
log("=" * 80)

missing_description = [
    row for row in canonical_rows
    if not row["description"]
]

missing_normalized = [
    row for row in canonical_rows
    if not row["description_normalized"]
]

duplicate_ids = {}

for row in canonical_rows:
    fdc = row["fdc_id"]
    duplicate_ids[fdc] = duplicate_ids.get(fdc, 0) + 1

duplicates = {
    fdc: count
    for fdc, count in duplicate_ids.items()
    if count > 1
}

if missing_description:
    fail(
        f"{len(missing_description)} canonical rows have "
        "missing descriptions."
    )

if missing_normalized:
    fail(
        f"{len(missing_normalized)} canonical rows have "
        "missing normalized descriptions."
    )

if duplicates:
    fail(
        f"Duplicate FDC IDs remain: {list(duplicates.items())[:10]}"
    )

log(f"Descriptions present:          {len(canonical_rows):,}/{len(canonical_rows):,}")
log(f"Normalized descriptions:       {len(canonical_rows) - len(missing_normalized):,}/{len(canonical_rows):,}")
log(f"Unique FDC IDs:                {len(duplicate_ids):,}")
log(f"Duplicate FDC IDs:             {len(duplicates)}")
log()


# =============================================================================
# CLASS SUMMARY
# =============================================================================

log("-" * 80)
log("INGREDIENT CLASS SUMMARY")
log("-" * 80)

class_counts = {}

for row in canonical_rows:
    cls = row["ingredient_class"] or "UNCLASSIFIED"
    class_counts[cls] = class_counts.get(cls, 0) + 1

for cls, count in sorted(
    class_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    log(f"{cls:<35} {count:,}")

log()


# =============================================================================
# SEARCH QUALITY CROSS-CHECK
# =============================================================================

log("=" * 80)
log("CROSS-CHECKING SEARCH RESULTS")
log("=" * 80)

search_fdc_column = next(
    (
        c for c in
        ["fdc_id", "FDC_ID", "Fdc_ID"]
        if c in search_columns
    ),
    None
)

if search_fdc_column:
    search_ids = {
        normalize_fdc(row[search_fdc_column])
        for row in search_rows
        if row.get(search_fdc_column)
    }

    canonical_id_set = {
        row["fdc_id"]
        for row in canonical_rows
    }

    missing_from_canonical = search_ids - canonical_id_set

    if missing_from_canonical:
        log(
            f"REVIEW: {len(missing_from_canonical):,} search FDC IDs "
            "are not present in canonical table."
        )
    else:
        log("PASS: All search FDC IDs represented in canonical table.")

else:
    log("REVIEW: Search file has no recognizable FDC ID column.")

log()


# =============================================================================
# BUILD LOOKUP TABLE
# =============================================================================

log("=" * 80)
log("BUILDING FDC LOOKUP TABLE")
log("=" * 80)

lookup_columns = [
    "fdc_id",
    "description",
    "description_normalized",
    "ingredient_class",
    "search_score",
    "searchable",
    "is_ingredient",
    "is_prepared_food",
    "is_food_item",
    "resolver_status",
]

lookup_rows = sorted(
    canonical_rows,
    key=lambda r: (
        r["description_normalized"],
        int(r["fdc_id"]) if r["fdc_id"].isdigit() else r["fdc_id"]
    )
)

log(f"Lookup records: {len(lookup_rows):,}")
log()


# =============================================================================
# BUILD QUALITY REPORT
# =============================================================================

log("=" * 80)
log("BUILDING QUALITY REPORT")
log("=" * 80)

ingredient_count = sum(
    row["is_ingredient"] == "1"
    for row in canonical_rows
)

prepared_count = sum(
    row["is_prepared_food"] == "1"
    for row in canonical_rows
)

food_item_count = sum(
    row["is_food_item"] == "1"
    for row in canonical_rows
)

quality_lines = [
    "SR LEGACY STEP 4 — INGREDIENT CANONICAL APPLICATION LAYER",
    "=" * 70,
    "",
    f"Build started:              {started.isoformat()}",
    f"Build finished:             {datetime.now().isoformat()}",
    "",
    f"Step 3 resolver rows:       {len(resolver_rows):,}",
    f"Canonical rows:             {len(canonical_rows):,}",
    f"Unique FDC IDs:             {len(duplicate_ids):,}",
    "",
    f"Ingredient-like rows:       {ingredient_count:,}",
    f"Prepared-food rows:         {prepared_count:,}",
    f"Food-item rows:             {food_item_count:,}",
    "",
    f"Search rows:                {len(search_rows):,}",
    f"Test rows:                  {len(test_rows):,}",
    "",
    "VALIDATION",
    "-" * 70,
    "FDC IDs:                    PASS",
    "Descriptions:               PASS",
    "Normalized descriptions:    PASS",
    "Duplicate FDC IDs:          PASS",
    "",
    "STATUS: PASS",
]

report_path = OUTPUT_DIR / "sr_legacy_step4_quality_report.txt"


# =============================================================================
# SAVE OUTPUTS
# =============================================================================

log("=" * 80)
log("SAVING STEP 4 OUTPUT")
log("=" * 80)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

resolver_output = (
    OUTPUT_DIR /
    "sr_legacy_ingredient_canonical.csv"
)

lookup_output = (
    OUTPUT_DIR /
    "sr_legacy_ingredient_lookup.csv"
)

write_csv(
    resolver_output,
    canonical_columns,
    canonical_rows
)

write_csv(
    lookup_output,
    lookup_columns,
    lookup_rows
)

report_path.write_text(
    "\n".join(quality_lines) + "\n",
    encoding="utf-8"
)

log(f"Saved: {resolver_output}")
log(f"Saved: {lookup_output}")
log(f"Saved: {report_path}")
log()


# =============================================================================
# FINAL REPORT
# =============================================================================

log("=" * 80)
log("STEP 4 FINAL VALIDATION")
log("=" * 80)

log(f"Canonical ingredient records: {len(canonical_rows):,}")
log(f"Unique FDC IDs:               {len(duplicate_ids):,}")
log(f"Ingredient-like records:      {ingredient_count:,}")
log(f"Prepared-food records:        {prepared_count:,}")
log(f"Food-item records:             {food_item_count:,}")
log(f"Search records:               {len(search_rows):,}")
log()

log("Output directory:")
log(str(OUTPUT_DIR))
log()

log("Files created:")
log("  sr_legacy_ingredient_canonical.csv")
log("  sr_legacy_ingredient_lookup.csv")
log("  sr_legacy_step4_quality_report.txt")
log()

log("=" * 80)
log("SR LEGACY STEP 4 COMPLETE")
log("=" * 80)
log("STATUS: PASS")
log("=" * 80)


SR LEGACY STEP 4 — INGREDIENT CANONICAL APPLICATION LAYER

Build started: 2026-08-16T13:12:39.311464

Step 3 directory:
C:\Users\AK\Downloads\zip\sr_legacy_step3

Step 4 output directory:
C:\Users\AK\Downloads\zip\sr_legacy_step4

CHECKING STEP 3 FILES
PASS: resolver   sr_legacy_ingredient_resolver.csv                  1.16 MB
PASS: search     sr_legacy_ingredient_search_results.csv            0.01 MB
PASS: tests      sr_legacy_ingredient_test_results.csv              0.00 MB

LOADING STEP 3 DATA
Resolver rows: 7,793
Search rows:   180
Test rows:     42

DISCOVERING STEP 3 COLUMNS

Resolver columns:
  fdc_id
  description
  description_normalized
  food_class
  is_ingredient_like
  is_prepared_food
  description_token_count
  first_token
  second_token

Search columns:
  query
  rank
  fdc_id
  description
  food_class
  score

VALIDATING RESOLVER COLUMNS
PASS: fdc_id          -> fdc_id
PASS: description     -> description

--------------------------------------------------------------

In [34]:
from __future__ import annotations

import csv
import re
import sys
from pathlib import Path
from collections import defaultdict
from datetime import datetime


# =============================================================================
# SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER
# =============================================================================

STEP4_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step4")
OUTPUT_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step5")

CANONICAL_FILE = STEP4_DIR / "sr_legacy_ingredient_canonical.csv"
LOOKUP_FILE = STEP4_DIR / "sr_legacy_ingredient_lookup.csv"

OUTPUT_RESOLUTIONS = OUTPUT_DIR / "sr_legacy_ingredient_resolutions.csv"
OUTPUT_TESTS = OUTPUT_DIR / "sr_legacy_step5_test_results.csv"
OUTPUT_REPORT = OUTPUT_DIR / "sr_legacy_step5_quality_report.txt"


# =============================================================================
# CONFIGURATION
# =============================================================================

SEARCH_LIMIT = 10

FOOD_CLASSES = {
    "INGREDIENT",
    "FOOD_ITEM",
    "OTHER",
    "PREPARED_FOOD",
    "PREPARED_CONTAINING_INGREDIENT",
}

# Terms which strongly indicate that the user is asking for a basic ingredient.
INGREDIENT_HINTS = {
    "fresh",
    "raw",
    "dried",
    "ground",
    "whole",
    "powder",
    "root",
    "seed",
    "seeds",
    "leaf",
    "leaves",
    "herb",
    "spice",
    "oil",
    "flour",
    "sugar",
    "salt",
    "rice",
    "beans",
    "bean",
    "meat",
    "fish",
    "pork",
    "beef",
    "chicken",
}

PREPARED_HINTS = {
    "sauce",
    "bread",
    "cake",
    "cookie",
    "cookies",
    "pastry",
    "pizza",
    "cereal",
    "roll",
    "rolls",
    "mayonnaise",
    "pesto",
    "dressing",
    "sandwich",
    "prepared",
    "frozen meal",
}


# =============================================================================
# TEXT NORMALIZATION
# =============================================================================

def normalize_text(value: str) -> str:
    """
    Normalize user input and USDA descriptions.

    Keeps alphanumeric tokens while making matching deterministic.
    """
    if value is None:
        return ""

    value = str(value).lower().strip()

    # Replace punctuation/separators with spaces.
    value = re.sub(r"[/,_;:()\[\]{}\-]+", " ", value)

    # Keep apostrophes from creating strange token boundaries.
    value = value.replace("'", "")

    # Remove everything else except letters/numbers/spaces.
    value = re.sub(r"[^a-z0-9\s]", " ", value)

    # Collapse whitespace.
    value = re.sub(r"\s+", " ", value).strip()

    return value


def tokenize(value: str) -> list[str]:
    normalized = normalize_text(value)
    return normalized.split() if normalized else []


# =============================================================================
# CSV HELPERS
# =============================================================================

def read_csv(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(rows)


def require_columns(rows: list[dict], required: list[str], label: str) -> None:
    if not rows:
        raise ValueError(f"{label} contains no records.")

    actual = set(rows[0].keys())

    missing = [c for c in required if c not in actual]

    if missing:
        raise ValueError(
            f"{label} missing required columns: {', '.join(missing)}"
        )


# =============================================================================
# FDC ID NORMALIZATION
# =============================================================================

def normalize_fdc_id(value: str) -> str:
    """
    Keep FDC IDs deterministic as strings.

    Handles values such as:
      171320
      171320.0
    """
    if value is None:
        return ""

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    return value


# =============================================================================
# CANONICAL RECORD
# =============================================================================

class IngredientRecord:
    def __init__(self, row: dict):
        self.fdc_id = normalize_fdc_id(row.get("fdc_id", ""))

        self.description = (
            row.get("description", "").strip()
        )

        normalized = row.get("description_normalized", "")

        self.description_normalized = (
            normalize_text(normalized)
            if normalized
            else normalize_text(self.description)
        )

        self.food_class = (
            row.get("food_class", "OTHER").strip().upper()
        )

        self.is_ingredient_like = (
            str(row.get("is_ingredient_like", "")).lower()
            in {"1", "true", "yes"}
        )

        self.is_prepared_food = (
            str(row.get("is_prepared_food", "")).lower()
            in {"1", "true", "yes"}
        )

        self.tokens = tokenize(self.description_normalized)
        self.token_set = set(self.tokens)

        self.first_token = self.tokens[0] if self.tokens else ""
        self.second_token = self.tokens[1] if len(self.tokens) > 1 else ""

    def as_dict(self) -> dict:
        return {
            "fdc_id": self.fdc_id,
            "description": self.description,
            "description_normalized": self.description_normalized,
            "food_class": self.food_class,
            "is_ingredient_like": str(
                self.is_ingredient_like
            ).lower(),
            "is_prepared_food": str(
                self.is_prepared_food
            ).lower(),
            "token_count": len(self.tokens),
        }


# =============================================================================
# SEARCH INDEX
# =============================================================================

class IngredientResolver:

    def __init__(self, records: list[IngredientRecord]):
        self.records = records

        self.by_fdc_id: dict[str, IngredientRecord] = {
            r.fdc_id: r
            for r in records
            if r.fdc_id
        }

        self.exact_index: dict[str, list[str]] = defaultdict(list)

        self.token_index: dict[str, set[str]] = defaultdict(set)

        self.prefix_index: dict[str, set[str]] = defaultdict(set)

        self._build_indexes()

    # -------------------------------------------------------------------------
    # INDEX BUILD
    # -------------------------------------------------------------------------

    def _build_indexes(self) -> None:

        for record in self.records:

            if not record.fdc_id:
                continue

            normalized = record.description_normalized

            if normalized:
                self.exact_index[normalized].append(
                    record.fdc_id
                )

            for token in record.token_set:
                self.token_index[token].add(record.fdc_id)

            for token in record.tokens:
                if len(token) >= 3:
                    for length in range(3, min(len(token), 8) + 1):
                        self.prefix_index[token[:length]].add(
                            record.fdc_id
                        )

    # -------------------------------------------------------------------------
    # CLASS PRIORITY
    # -------------------------------------------------------------------------

    @staticmethod
    def class_bonus(
        record: IngredientRecord,
        query_tokens: list[str],
    ) -> float:

        query_set = set(query_tokens)

        bonus = 0.0

        if record.food_class == "INGREDIENT":
            bonus += 20.0

        elif record.food_class == "FOOD_ITEM":
            bonus += 10.0

        elif record.food_class == "PREPARED_FOOD":
            bonus -= 12.0

        elif record.food_class == "PREPARED_CONTAINING_INGREDIENT":
            bonus -= 8.0

        elif record.food_class == "OTHER":
            bonus -= 5.0

        # Explicit raw/fresh/ground/dried queries should prefer
        # matching ingredient forms.
        if query_set & INGREDIENT_HINTS:
            if record.food_class == "INGREDIENT":
                bonus += 12.0

        if query_set & PREPARED_HINTS:
            if record.food_class in {
                "PREPARED_FOOD",
                "PREPARED_CONTAINING_INGREDIENT",
            }:
                bonus += 12.0

        return bonus

    # -------------------------------------------------------------------------
    # QUERY/RECORD SCORE
    # -------------------------------------------------------------------------

    def score_record(
        self,
        query: str,
        record: IngredientRecord,
    ) -> tuple[float, list[str]]:

        q = normalize_text(query)
        q_tokens = tokenize(q)
        q_set = set(q_tokens)

        if not q_tokens:
            return 0.0, []

        description = record.description_normalized
        description_tokens = record.tokens
        description_set = record.token_set

        score = 0.0
        reasons = []

        # ---------------------------------------------------------------------
        # Exact description
        # ---------------------------------------------------------------------

        if q == description:
            score += 200.0
            reasons.append("EXACT_DESCRIPTION")

        # ---------------------------------------------------------------------
        # Exact token sequence contained in description
        # ---------------------------------------------------------------------

        if q in description:
            score += 40.0
            reasons.append("PHRASE_MATCH")

        # ---------------------------------------------------------------------
        # Token matching
        # ---------------------------------------------------------------------

        matched_tokens = q_set & description_set

        if matched_tokens:
            score += 40.0 * len(matched_tokens)
            reasons.append(
                f"TOKEN_MATCH:{len(matched_tokens)}"
            )

        # All query tokens matched.
        if q_set and q_set.issubset(description_set):
            score += 35.0
            reasons.append("ALL_QUERY_TOKENS")

        # ---------------------------------------------------------------------
        # Ordered token match
        # ---------------------------------------------------------------------

        if len(q_tokens) > 1:

            position = 0

            for token in description_tokens:
                if position < len(q_tokens) and token == q_tokens[position]:
                    position += 1

            if position == len(q_tokens):
                score += 25.0
                reasons.append("ORDERED_TOKEN_MATCH")

        # ---------------------------------------------------------------------
        # Token prefix match
        # ---------------------------------------------------------------------

        prefix_matches = 0

        for qt in q_tokens:
            if len(qt) < 3:
                continue

            if any(
                dt.startswith(qt)
                for dt in description_tokens
            ):
                prefix_matches += 1

        if prefix_matches:
            score += 10.0 * prefix_matches
            reasons.append(
                f"PREFIX_MATCH:{prefix_matches}"
            )

        # ---------------------------------------------------------------------
        # Description starts with query
        # ---------------------------------------------------------------------

        if description.startswith(q):
            score += 25.0
            reasons.append("DESCRIPTION_STARTS_WITH_QUERY")

        # ---------------------------------------------------------------------
        # Class preference
        # ---------------------------------------------------------------------

        class_bonus = self.class_bonus(record, q_tokens)

        score += class_bonus

        if class_bonus:
            reasons.append(
                f"CLASS_BONUS:{class_bonus:g}"
            )

        # ---------------------------------------------------------------------
        # Penalize unrelated extra tokens when query is specific.
        # ---------------------------------------------------------------------

        extra_tokens = description_set - q_set

        if len(q_tokens) >= 2 and len(extra_tokens) > 5:
            score -= min(
                20.0,
                (len(extra_tokens) - 5) * 2.0
            )
            reasons.append("EXTRA_TOKEN_PENALTY")

        return max(score, 0.0), reasons

    # -------------------------------------------------------------------------
    # CANDIDATE GENERATION
    # -------------------------------------------------------------------------

    def candidate_ids(self, query: str) -> set[str]:

        normalized = normalize_text(query)
        tokens = tokenize(normalized)

        candidates: set[str] = set()

        # Exact.
        candidates.update(
            self.exact_index.get(normalized, [])
        )

        # Token index.
        for token in tokens:
            candidates.update(
                self.token_index.get(token, set())
            )

        # Prefix index.
        for token in tokens:
            if len(token) >= 3:
                candidates.update(
                    self.prefix_index.get(
                        token[: min(8, len(token))],
                        set(),
                    )
                )

        # Safety fallback for very short/unusual queries.
        if not candidates:
            candidates = set(self.by_fdc_id.keys())

        return candidates

    # -------------------------------------------------------------------------
    # SEARCH
    # -------------------------------------------------------------------------

    def search(
        self,
        query: str,
        limit: int = SEARCH_LIMIT,
    ) -> list[dict]:

        normalized_query = normalize_text(query)

        if not normalized_query:
            return []

        candidates = self.candidate_ids(normalized_query)

        results = []

        for fdc_id in candidates:

            record = self.by_fdc_id.get(fdc_id)

            if record is None:
                continue

            score, reasons = self.score_record(
                normalized_query,
                record,
            )

            if score <= 0:
                continue

            results.append(
                {
                    "fdc_id": record.fdc_id,
                    "description": record.description,
                    "food_class": record.food_class,
                    "score": round(score, 3),
                    "match_reasons": "|".join(reasons),
                }
            )

        results.sort(
            key=lambda x: (
                -x["score"],
                x["food_class"] != "INGREDIENT",
                x["description"].lower(),
                x["fdc_id"],
            )
        )

        return results[:limit]

    # -------------------------------------------------------------------------
    # RESOLUTION
    # -------------------------------------------------------------------------

    def resolve(
        self,
        query: str,
        limit: int = SEARCH_LIMIT,
        min_score: float = 30.0,
        ambiguity_delta: float = 10.0,
    ) -> dict:

        normalized_query = normalize_text(query)

        if not normalized_query:
            return {
                "status": "EMPTY_QUERY",
                "query": query,
                "normalized_query": "",
                "fdc_id": "",
                "description": "",
                "food_class": "",
                "score": 0.0,
                "confidence": 0.0,
                "reason": "EMPTY_QUERY",
            }

        results = self.search(
            normalized_query,
            limit=limit,
        )

        if not results:
            return {
                "status": "NO_MATCH",
                "query": query,
                "normalized_query": normalized_query,
                "fdc_id": "",
                "description": "",
                "food_class": "",
                "score": 0.0,
                "confidence": 0.0,
                "reason": "NO_MATCH",
            }

        best = results[0]

        second_score = (
            results[1]["score"]
            if len(results) > 1
            else 0.0
        )

        # Confidence is intentionally bounded.
        #
        # A high score alone isn't enough:
        # the gap between the first and second candidates matters.
        score_confidence = min(
            1.0,
            best["score"] / 180.0
        )

        if len(results) == 1:
            margin_confidence = 1.0
        else:
            margin = best["score"] - second_score
            margin_confidence = min(
                1.0,
                max(0.0, margin / 50.0)
            )

        confidence = (
            0.65 * score_confidence
            + 0.35 * margin_confidence
        )

        # Exact normalized description is deterministic.
        if (
            best["description"].strip().lower()
            == normalized_query
        ):
            status = "EXACT_MATCH"
            reason = "EXACT_NORMALIZED_DESCRIPTION"
            confidence = max(confidence, 0.98)

        elif best["score"] < min_score:
            status = "LOW_CONFIDENCE"
            reason = "BEST_SCORE_BELOW_THRESHOLD"

        elif (
            len(results) > 1
            and (best["score"] - second_score)
            < ambiguity_delta
        ):
            status = "AMBIGUOUS"
            reason = "TOP_RESULTS_TOO_CLOSE"

        else:
            status = "RESOLVED"
            reason = "BEST_RANKED_MATCH"

        return {
            "status": status,
            "query": query,
            "normalized_query": normalized_query,
            "fdc_id": best["fdc_id"],
            "description": best["description"],
            "food_class": best["food_class"],
            "score": best["score"],
            "confidence": round(confidence, 4),
            "reason": reason,
            "candidate_count": len(results),
            "second_fdc_id": (
                results[1]["fdc_id"]
                if len(results) > 1
                else ""
            ),
            "second_description": (
                results[1]["description"]
                if len(results) > 1
                else ""
            ),
            "second_score": second_score,
        }


# =============================================================================
# TEST SUITE
# =============================================================================

TEST_CASES = [
    {
        "query": "cinnamon",
        "expected_terms": ["cinnamon"],
        "expected_fdc": "171320",
        "label": "cinnamon",
    },
    {
        "query": "ground cinnamon",
        "expected_terms": ["cinnamon", "ground"],
        "expected_fdc": "171320",
        "label": "ground cinnamon",
    },
    {
        "query": "turmeric",
        "expected_terms": ["turmeric"],
        "expected_fdc": "172231",
        "label": "turmeric",
    },
    {
        "query": "fresh basil",
        "expected_terms": ["basil", "fresh"],
        "expected_fdc": "172232",
        "label": "fresh basil",
    },
    {
        "query": "dried basil",
        "expected_terms": ["basil", "dried"],
        "expected_fdc": "171317",
        "label": "dried basil",
    },
    {
        "query": "olive oil",
        "expected_terms": ["oil", "olive"],
        "expected_fdc": "171413",
        "label": "olive oil",
    },
    {
        "query": "oil olive",
        "expected_terms": ["oil", "olive"],
        "expected_fdc": "171413",
        "label": "oil olive",
    },
    {
        "query": "baking powder",
        "expected_terms": ["baking", "powder"],
        "expected_fdc": "",
        "label": "baking powder",
    },
    {
        "query": "jackfruit",
        "expected_terms": ["jackfruit"],
        "expected_fdc": "174687",
        "label": "jackfruit",
    },
    {
        "query": "raw jackfruit",
        "expected_terms": ["jackfruit", "raw"],
        "expected_fdc": "174687",
        "label": "raw jackfruit",
    },
    {
        "query": "kelp",
        "expected_terms": ["kelp"],
        "expected_fdc": "168457",
        "label": "kelp",
    },
    {
        "query": "seaweed kelp",
        "expected_terms": ["seaweed", "kelp"],
        "expected_fdc": "168457",
        "label": "seaweed kelp",
    },
    {
        "query": "pork shoulder",
        "expected_terms": ["pork", "shoulder"],
        "expected_fdc": "169187",
        "label": "pork shoulder",
    },
    {
        "query": "shoulder pork",
        "expected_terms": ["pork", "shoulder"],
        "expected_fdc": "167843",
        "label": "shoulder pork",
    },
    {
        "query": "black pepper",
        "expected_terms": ["pepper", "black"],
        "expected_fdc": "170931",
        "label": "black pepper",
    },
    {
        "query": "garlic",
        "expected_terms": ["garlic"],
        "expected_fdc": "171325",
        "label": "garlic",
    },
    {
        "query": "ginger",
        "expected_terms": ["ginger"],
        "expected_fdc": "170926",
        "label": "ginger",
    },
    {
        "query": "rice",
        "expected_terms": ["rice"],
        "expected_fdc": "",
        "label": "rice",
    },
    {
        "query": "flour",
        "expected_terms": ["flour"],
        "expected_fdc": "",
        "label": "flour",
    },
    {
        "query": "sugar",
        "expected_terms": ["sugar"],
        "expected_fdc": "",
        "label": "sugar",
    },
]


def run_tests(
    resolver: IngredientResolver,
) -> list[dict]:

    output = []

    for test in TEST_CASES:

        result = resolver.resolve(
            test["query"],
            limit=10,
        )

        top_description = result["description"]

        normalized_top = normalize_text(
            top_description
        )

        terms_pass = all(
            term in normalized_top
            for term in test["expected_terms"]
        )

        fdc_pass = True

        if test["expected_fdc"]:
            fdc_pass = (
                result["fdc_id"]
                == test["expected_fdc"]
            )

        passed = terms_pass and fdc_pass

        output.append(
            {
                "query": test["query"],
                "expected_terms": "|".join(
                    test["expected_terms"]
                ),
                "expected_fdc_id": test["expected_fdc"],
                "status": result["status"],
                "resolved_fdc_id": result["fdc_id"],
                "resolved_description": result["description"],
                "food_class": result["food_class"],
                "score": result["score"],
                "confidence": result["confidence"],
                "reason": result["reason"],
                "terms_pass": str(terms_pass).lower(),
                "fdc_pass": str(fdc_pass).lower(),
                "test_pass": str(passed).lower(),
            }
        )

    return output


# =============================================================================
# BUILD RESOLUTION TABLE
# =============================================================================

def build_resolution_table(
    resolver: IngredientResolver,
) -> list[dict]:

    rows = []

    for record in resolver.records:

        result = resolver.resolve(
            record.description,
            limit=10,
        )

        # Canonical description should resolve to itself.
        self_resolve_pass = (
            result["fdc_id"] == record.fdc_id
        )

        rows.append(
            {
                "fdc_id": record.fdc_id,
                "description": record.description,
                "description_normalized": (
                    record.description_normalized
                ),
                "food_class": record.food_class,
                "is_ingredient_like": str(
                    record.is_ingredient_like
                ).lower(),
                "is_prepared_food": str(
                    record.is_prepared_food
                ).lower(),
                "self_resolution_fdc_id": (
                    result["fdc_id"]
                ),
                "self_resolution_status": (
                    result["status"]
                ),
                "self_resolution_score": (
                    result["score"]
                ),
                "self_resolution_confidence": (
                    result["confidence"]
                ),
                "self_resolution_pass": str(
                    self_resolve_pass
                ).lower(),
            }
        )

    return rows


# =============================================================================
# QUALITY REPORT
# =============================================================================

def build_report(
    records: list[IngredientRecord],
    resolver: IngredientResolver,
    tests: list[dict],
    resolutions: list[dict],
) -> str:

    total = len(records)

    unique_fdc = len({
        r.fdc_id
        for r in records
        if r.fdc_id
    })

    ingredient_like = sum(
        r.is_ingredient_like
        for r in records
    )

    prepared = sum(
        r.is_prepared_food
        for r in records
    )

    test_pass = sum(
        row["test_pass"] == "true"
        for row in tests
    )

    test_fail = len(tests) - test_pass

    self_pass = sum(
        row["self_resolution_pass"] == "true"
        for row in resolutions
    )

    status_counts = defaultdict(int)

    for row in tests:
        status_counts[row["status"]] += 1

    class_counts = defaultdict(int)

    for record in records:
        class_counts[record.food_class] += 1

    lines = []

    lines.append("=" * 80)
    lines.append("SR LEGACY STEP 5 — INGREDIENT RESOLUTION QUALITY REPORT")
    lines.append("=" * 80)
    lines.append("")
    lines.append(
        f"Build timestamp: {datetime.now().isoformat()}"
    )
    lines.append("")
    lines.append("INPUT")
    lines.append("-" * 80)
    lines.append(f"Step 4 directory: {STEP4_DIR}")
    lines.append(f"Canonical records: {total:,}")
    lines.append("")
    lines.append("VALIDATION")
    lines.append("-" * 80)
    lines.append(
        f"Unique FDC IDs:             {unique_fdc:,}"
    )
    lines.append(
        f"Duplicate FDC IDs:          {total - unique_fdc:,}"
    )
    lines.append(
        f"Ingredient-like records:    {ingredient_like:,}"
    )
    lines.append(
        f"Prepared-food records:      {prepared:,}"
    )
    lines.append(
        f"Self-resolution PASS:       {self_pass:,}/{total:,}"
    )
    lines.append("")
    lines.append("FOOD CLASS SUMMARY")
    lines.append("-" * 80)

    for food_class in sorted(class_counts):
        lines.append(
            f"{food_class:<35} {class_counts[food_class]:>8,}"
        )

    lines.append("")
    lines.append("TEST SUMMARY")
    lines.append("-" * 80)
    lines.append(
        f"Tests executed:             {len(tests):,}"
    )
    lines.append(
        f"Tests PASS:                 {test_pass:,}"
    )
    lines.append(
        f"Tests FAIL:                 {test_fail:,}"
    )

    lines.append("")
    lines.append("TEST STATUS COUNTS")
    lines.append("-" * 80)

    for status in sorted(status_counts):
        lines.append(
            f"{status:<30} {status_counts[status]:>8,}"
        )

    lines.append("")
    lines.append("TEST DETAILS")
    lines.append("-" * 80)

    for row in tests:
        state = "PASS" if row["test_pass"] == "true" else "FAIL"

        lines.append(
            f"{state:<5} "
            f"{row['query']:<25} "
            f"FDC={row['resolved_fdc_id']:<8} "
            f"score={row['score']:<8} "
            f"confidence={row['confidence']}"
        )

        lines.append(
            f"      {row['resolved_description']}"
        )

    lines.append("")
    lines.append("=" * 80)

    if (
        unique_fdc == total
        and self_pass == total
        and test_fail == 0
    ):
        lines.append("STEP 5 STATUS: PASS")
    else:
        lines.append("STEP 5 STATUS: REVIEW")

    lines.append("=" * 80)

    return "\n".join(lines)


# =============================================================================
# MAIN BUILD
# =============================================================================

def main() -> int:

    started = datetime.now()

    print("=" * 80)
    print("SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER")
    print("=" * 80)
    print()
    print(f"Build started: {started.isoformat()}")
    print()
    print(f"Step 4 directory:    {STEP4_DIR}")
    print(f"Step 5 output:       {OUTPUT_DIR}")
    print()

    # -------------------------------------------------------------------------
    # CHECK FILES
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("CHECKING STEP 4 FILES")
    print("=" * 80)

    required_files = [
        CANONICAL_FILE,
        LOOKUP_FILE,
    ]

    for path in required_files:

        if not path.exists():
            print(f"FAIL: {path.name}")
            print()
            print("Build aborted.")
            return 1

        size_mb = path.stat().st_size / (1024 * 1024)

        print(
            f"PASS: {path.name:<50} "
            f"{size_mb:.2f} MB"
        )

    # -------------------------------------------------------------------------
    # LOAD
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("LOADING STEP 4 DATA")
    print("=" * 80)

    canonical_rows = read_csv(CANONICAL_FILE)
    lookup_rows = read_csv(LOOKUP_FILE)

    print(
        f"Canonical rows: {len(canonical_rows):,}"
    )
    print(
        f"Lookup rows:    {len(lookup_rows):,}"
    )

    # -------------------------------------------------------------------------
    # VALIDATE COLUMNS
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("VALIDATING STEP 4 COLUMNS")
    print("=" * 80)

    require_columns(
        canonical_rows,
        [
            "fdc_id",
            "description",
        ],
        "Canonical table",
    )

    require_columns(
        lookup_rows,
        [
            "fdc_id",
            "description",
        ],
        "Lookup table",
    )

    print("PASS: Canonical columns")
    print("PASS: Lookup columns")

    # -------------------------------------------------------------------------
    # NORMALIZE
    # -------------------------------------------------------------------------

    print()
    print("-" * 80)
    print("NORMALIZING FDC IDS")
    print("-" * 80)

    records = [
        IngredientRecord(row)
        for row in canonical_rows
    ]

    fdc_ids = [
        r.fdc_id
        for r in records
    ]

    if any(not x for x in fdc_ids):
        print("FAIL: Blank FDC IDs detected.")
        return 1

    if len(fdc_ids) != len(set(fdc_ids)):
        print("FAIL: Duplicate FDC IDs detected.")
        return 1

    print("FDC ID normalization: PASS")
    print(
        f"Unique FDC IDs:       {len(set(fdc_ids)):,}"
    )

    # -------------------------------------------------------------------------
    # BUILD RESOLVER
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("BUILDING INGREDIENT RESOLUTION ENGINE")
    print("=" * 80)

    resolver = IngredientResolver(records)

    print(
        f"Indexed records: {len(resolver.records):,}"
    )
    print(
        f"Exact descriptions: {len(resolver.exact_index):,}"
    )
    print(
        f"Token index entries: {len(resolver.token_index):,}"
    )
    print(
        f"Prefix index entries: {len(resolver.prefix_index):,}"
    )

    # -------------------------------------------------------------------------
    # RUN SEARCH TESTS
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("RUNNING STEP 5 RESOLUTION TESTS")
    print("=" * 80)

    tests = run_tests(resolver)

    for row in tests:

        state = (
            "PASS"
            if row["test_pass"] == "true"
            else "FAIL"
        )

        print()
        print(
            f"{row['query']}: {state}"
        )
        print(
            f"  Status:       {row['status']}"
        )
        print(
            f"  FDC ID:       {row['resolved_fdc_id']}"
        )
        print(
            f"  Description:  {row['resolved_description']}"
        )
        print(
            f"  Class:        {row['food_class']}"
        )
        print(
            f"  Score:        {row['score']}"
        )
        print(
            f"  Confidence:   {row['confidence']}"
        )
        print(
            f"  Reason:       {row['reason']}"
        )

    # -------------------------------------------------------------------------
    # BUILD SELF-RESOLUTION TABLE
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("BUILDING CANONICAL SELF-RESOLUTION TABLE")
    print("=" * 80)

    resolutions = build_resolution_table(
        resolver
    )

    self_pass = sum(
        row["self_resolution_pass"] == "true"
        for row in resolutions
    )

    print(
        f"Self-resolution PASS: "
        f"{self_pass:,}/{len(resolutions):,}"
    )

    # -------------------------------------------------------------------------
    # BUILD REPORT
    # -------------------------------------------------------------------------

    report = build_report(
        records,
        resolver,
        tests,
        resolutions,
    )

    # -------------------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("SAVING STEP 5 OUTPUT")
    print("=" * 80)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolution_fields = [
        "fdc_id",
        "description",
        "description_normalized",
        "food_class",
        "is_ingredient_like",
        "is_prepared_food",
        "self_resolution_fdc_id",
        "self_resolution_status",
        "self_resolution_score",
        "self_resolution_confidence",
        "self_resolution_pass",
    ]

    test_fields = [
        "query",
        "expected_terms",
        "expected_fdc_id",
        "status",
        "resolved_fdc_id",
        "resolved_description",
        "food_class",
        "score",
        "confidence",
        "reason",
        "terms_pass",
        "fdc_pass",
        "test_pass",
    ]

    write_csv(
        OUTPUT_RESOLUTIONS,
        resolutions,
        resolution_fields,
    )

    write_csv(
        OUTPUT_TESTS,
        tests,
        test_fields,
    )

    OUTPUT_REPORT.write_text(
        report,
        encoding="utf-8",
    )

    print(
        f"Saved: {OUTPUT_RESOLUTIONS}"
    )
    print(
        f"Saved: {OUTPUT_TESTS}"
    )
    print(
        f"Saved: {OUTPUT_REPORT}"
    )

    # -------------------------------------------------------------------------
    # FINAL VALIDATION
    # -------------------------------------------------------------------------

    test_fail = sum(
        row["test_pass"] != "true"
        for row in tests
    )

    duplicate_count = (
        len(records) - len(set(fdc_ids))
    )

    print()
    print("=" * 80)
    print("STEP 5 FINAL VALIDATION")
    print("=" * 80)

    print(
        f"Canonical records:          {len(records):,}"
    )
    print(
        f"Unique FDC IDs:             {len(set(fdc_ids)):,}"
    )
    print(
        f"Duplicate FDC IDs:          {duplicate_count:,}"
    )
    print(
        f"Self-resolution PASS:       "
        f"{self_pass:,}/{len(records):,}"
    )
    print(
        f"Resolution tests:           {len(tests):,}"
    )
    print(
        f"Resolution tests PASS:      {len(tests) - test_fail:,}"
    )
    print(
        f"Resolution tests FAIL:      {test_fail:,}"
    )

    print()
    print(
        f"Output directory:\n{OUTPUT_DIR}"
    )

    print()

    if (
        duplicate_count == 0
        and self_pass == len(records)
        and test_fail == 0
    ):
        print("=" * 80)
        print("SR LEGACY STEP 5 COMPLETE")
        print("STATUS: PASS")
        print("=" * 80)
        return 0

    print("=" * 80)
    print("SR LEGACY STEP 5 COMPLETE")
    print("STATUS: REVIEW")
    print("=" * 80)

    return 0


# =============================================================================
# APPLICATION API
# =============================================================================

# These functions are available when this file is imported and the Step 5
# resolver has been initialized through load_application_resolver().


_APPLICATION_RESOLVER: IngredientResolver | None = None


def load_application_resolver() -> IngredientResolver:
    """
    Load Step 5 resolver directly from the Step 4 canonical table.
    """

    global _APPLICATION_RESOLVER

    if _APPLICATION_RESOLVER is not None:
        return _APPLICATION_RESOLVER

    rows = read_csv(CANONICAL_FILE)

    records = [
        IngredientRecord(row)
        for row in rows
    ]

    _APPLICATION_RESOLVER = IngredientResolver(
        records
    )

    return _APPLICATION_RESOLVER


def resolve_ingredient(
    query: str,
    limit: int = 10,
) -> dict:
    """
    Resolve one user-entered ingredient to a canonical FDC record.

    Example:

        resolve_ingredient("fresh basil")
    """

    resolver = load_application_resolver()

    return resolver.resolve(
        query,
        limit=limit,
    )


def search_ingredients(
    query: str,
    limit: int = 10,
) -> list[dict]:
    """
    Search ingredient records.

    Example:

        search_ingredients("olive oil", 10)
    """

    resolver = load_application_resolver()

    return resolver.search(
        query,
        limit=limit,
    )


# =============================================================================
# SCRIPT ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    sys.exit(main())


SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER

Build started: 2026-08-16T13:17:49.747142

Step 4 directory:    C:\Users\AK\Downloads\zip\sr_legacy_step4
Step 5 output:       C:\Users\AK\Downloads\zip\sr_legacy_step5

CHECKING STEP 4 FILES
PASS: sr_legacy_ingredient_canonical.csv                 1.09 MB
PASS: sr_legacy_ingredient_lookup.csv                    1.09 MB

LOADING STEP 4 DATA
Canonical rows: 7,793
Lookup rows:    7,793

VALIDATING STEP 4 COLUMNS
PASS: Canonical columns
PASS: Lookup columns

--------------------------------------------------------------------------------
NORMALIZING FDC IDS
--------------------------------------------------------------------------------
FDC ID normalization: PASS
Unique FDC IDs:       7,793

BUILDING INGREDIENT RESOLUTION ENGINE
Indexed records: 7,793
Exact descriptions: 7,792
Token index entries: 2,822
Prefix index entries: 7,885

RUNNING STEP 5 RESOLUTION TESTS

cinnamon: FAIL
  Status:       RESOLVED
  FDC ID:       167940
  D

SystemExit: 0

In [ ]:
print("Hello world!")

In [ ]:
from pathlib import Path
import pandas as pd
import re
import math
from collections import defaultdict

# =============================================================================
# SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER
# Corrected resolver:
#   - preserves Step 3 food classification
#   - strongly prefers ingredient-like foods
#   - penalizes prepared foods containing an ingredient
#   - handles exact/phrase/token matches
#   - handles reversed word order
#   - handles raw/fresh/dried modifiers
#   - avoids substring-only matches such as "garlic bread" for "garlic"
# =============================================================================

STEP4_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step4")
OUT_DIR = Path(r"C:\Users\AK\Downloads\zip\sr_legacy_step5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_FILE = STEP4_DIR / "sr_legacy_ingredient_canonical.csv"
LOOKUP_FILE = STEP4_DIR / "sr_legacy_ingredient_lookup.csv"

OUT_RESOLUTION = OUT_DIR / "sr_legacy_ingredient_resolution.csv"
OUT_TEST_RESULTS = OUT_DIR / "sr_legacy_ingredient_resolution_test_results.csv"
OUT_SEARCH_RESULTS = OUT_DIR / "sr_legacy_ingredient_resolution_search_results.csv"
OUT_REPORT = OUT_DIR / "sr_legacy_step5_quality_report.txt"


# =============================================================================
# NORMALIZATION
# =============================================================================

def normalize_text(value):
    if pd.isna(value):
        return ""

    s = str(value).lower().strip()

    # punctuation -> spaces
    s = re.sub(r"[^a-z0-9]+", " ", s)

    # normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def tokens(value):
    s = normalize_text(value)
    return s.split() if s else []


def token_set(value):
    return set(tokens(value))


# =============================================================================
# CLASSIFICATION
# =============================================================================

INGREDIENT_CLASSES = {
    "INGREDIENT",
}

FOOD_ITEM_CLASSES = {
    "FOOD_ITEM",
}

PREPARED_CLASSES = {
    "PREPARED_FOOD",
    "PREPARED_CONTAINING_INGREDIENT",
}

OTHER_CLASSES = {
    "OTHER",
}


def normalize_class(value):
    if pd.isna(value):
        return "OTHER"

    value = str(value).strip().upper()

    if value in INGREDIENT_CLASSES:
        return "INGREDIENT"

    if value in FOOD_ITEM_CLASSES:
        return "FOOD_ITEM"

    if value in PREPARED_CLASSES:
        return value

    return "OTHER"


# =============================================================================
# INGREDIENT / FOOD SIGNALS
# =============================================================================

RAW_MODIFIERS = {
    "raw",
    "fresh",
    "uncooked",
}

DRIED_MODIFIERS = {
    "dried",
    "dry",
    "powder",
}

COOKED_MODIFIERS = {
    "cooked",
    "roasted",
    "baked",
    "boiled",
    "fried",
    "broiled",
    "braised",
}

NEGATIVE_PREPARED_WORDS = {
    "bread",
    "bun",
    "buns",
    "roll",
    "rolls",
    "cookie",
    "cookies",
    "cake",
    "cakes",
    "pastry",
    "pastries",
    "muffin",
    "muffins",
    "cereal",
    "cereals",
    "pizza",
    "sandwich",
    "sandwiches",
    "mayonnaise",
    "sauce",
    "sauces",
    "dressing",
    "dressings",
    "pudding",
    "dessert",
    "desserts",
    "beverage",
    "beverages",
    "drink",
    "drinks",
    "snack",
    "snacks",
    "bar",
    "bars",
    "mix",
    "mixture",
}

# Terms that indicate the food itself is likely the ingredient,
# rather than merely containing the ingredient.
STRONG_INGREDIENT_FIRST_WORDS = {
    "oil",
    "spices",
    "spice",
    "salt",
    "sugar",
    "flour",
    "rice",
    "garlic",
    "ginger",
    "basil",
    "turmeric",
    "jackfruit",
    "seaweed",
    "pork",
    "chicken",
    "beef",
    "lamb",
    "fish",
    "pepper",
    "butter",
}


# =============================================================================
# DATA LOADING
# =============================================================================

print("=" * 80)
print("SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER")
print("=" * 80)

print()
print("STEP 4 DIRECTORY:")
print(STEP4_DIR)

print()
print("OUTPUT DIRECTORY:")
print(OUT_DIR)

print()
print("=" * 80)
print("CHECKING STEP 4 FILES")
print("=" * 80)

for path in [CANONICAL_FILE, LOOKUP_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    print(f"PASS: {path.name}")

print()
print("=" * 80)
print("LOADING STEP 4 DATA")
print("=" * 80)

canonical = pd.read_csv(CANONICAL_FILE, low_memory=False)
lookup = pd.read_csv(LOOKUP_FILE, low_memory=False)

print(f"Canonical rows: {len(canonical):,}")
print(f"Lookup rows:    {len(lookup):,}")


# =============================================================================
# COLUMN DISCOVERY
# =============================================================================

def find_column(df, candidates, required=True):
    lowered = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]

    if required:
        raise ValueError(
            f"Could not find required column. Candidates={candidates}; "
            f"available={list(df.columns)}"
        )

    return None


fdc_col = find_column(
    canonical,
    ["fdc_id"]
)

desc_col = find_column(
    canonical,
    ["description"]
)

norm_desc_col = find_column(
    canonical,
    ["description_normalized"],
    required=False
)

class_col = find_column(
    canonical,
    ["food_class", "class"],
    required=False
)

ingredient_like_col = find_column(
    canonical,
    ["is_ingredient_like"],
    required=False
)

prepared_col = find_column(
    canonical,
    ["is_prepared_food"],
    required=False
)


# =============================================================================
# BUILD INTERNAL RECORDS
# =============================================================================

records = []

for _, row in canonical.iterrows():

    fdc_id = str(row[fdc_col]).strip()

    description = str(row[desc_col]).strip()

    normalized = (
        str(row[norm_desc_col]).strip()
        if norm_desc_col and not pd.isna(row[norm_desc_col])
        else normalize_text(description)
    )

    food_class = (
        normalize_class(row[class_col])
        if class_col
        else "OTHER"
    )

    if ingredient_like_col:
        value = row[ingredient_like_col]
        is_ingredient_like = (
            bool(value)
            if not pd.isna(value)
            else food_class == "INGREDIENT"
        )
    else:
        is_ingredient_like = food_class == "INGREDIENT"

    if prepared_col:
        value = row[prepared_col]
        is_prepared = (
            bool(value)
            if not pd.isna(value)
            else food_class in PREPARED_CLASSES
        )
    else:
        is_prepared = food_class in PREPARED_CLASSES

    tok = tokens(normalized)

    records.append({
        "fdc_id": fdc_id,
        "description": description,
        "description_normalized": normalized,
        "food_class": food_class,
        "is_ingredient_like": bool(is_ingredient_like),
        "is_prepared_food": bool(is_prepared),
        "tokens": tok,
        "token_set": set(tok),
    })


print()
print("=" * 80)
print("BUILDING CORRECTED INGREDIENT RESOLUTION ENGINE")
print("=" * 80)

print(f"Indexed records: {len(records):,}")


# =============================================================================
# SEARCH SCORING
# =============================================================================

def score_candidate(query, record):
    """
    Higher score = better candidate.

    Design principles:

    1. Exact normalized description is very strong.
    2. Exact phrase / all-token match is strong.
    3. Ingredient-like foods get a major preference.
    4. Prepared foods get penalized.
    5. Query words merely appearing somewhere inside a prepared food
       are not enough.
    6. Reversed word order is supported.
    7. Modifier agreement is rewarded.
    """

    q = normalize_text(query)

    if not q:
        return -math.inf, []

    q_tokens = tokens(q)
    q_set = set(q_tokens)

    desc = record["description_normalized"]
    d_tokens = record["tokens"]
    d_set = record["token_set"]

    score = 0.0
    reasons = []

    # -------------------------------------------------------------------------
    # EXACT DESCRIPTION
    # -------------------------------------------------------------------------

    if desc == q:
        score += 300
        reasons.append("EXACT_DESCRIPTION")

    # -------------------------------------------------------------------------
    # EXACT TOKEN SET
    # -------------------------------------------------------------------------

    if q_set and q_set == d_set:
        score += 240
        reasons.append("EXACT_TOKEN_SET")

    # -------------------------------------------------------------------------
    # PHRASE MATCH
    # -------------------------------------------------------------------------

    if q in desc:
        score += 70
        reasons.append("PHRASE_MATCH")

    # -------------------------------------------------------------------------
    # TOKEN MATCH
    # -------------------------------------------------------------------------

    overlap = q_set.intersection(d_set)

    if overlap:
        score += len(overlap) * 35

    missing = q_set - d_set

    if not missing:
        score += 45
        reasons.append("ALL_QUERY_TOKENS_PRESENT")

    # -------------------------------------------------------------------------
    # WORD ORDER
    # -------------------------------------------------------------------------

    if len(q_tokens) > 1:

        if q_tokens == d_tokens[:len(q_tokens)]:
            score += 50
            reasons.append("QUERY_PREFIX")

        elif d_tokens[:len(q_tokens)] == list(reversed(q_tokens)):
            score += 35
            reasons.append("REVERSED_ORDER")

        elif all(t in d_set for t in q_tokens):
            score += 20
            reasons.append("TOKEN_ORDER_FLEXIBLE")

    # -------------------------------------------------------------------------
    # INGREDIENT-LIKE PREFERENCE
    # -------------------------------------------------------------------------

    if record["is_ingredient_like"]:
        score += 85
        reasons.append("INGREDIENT_LIKE")

    # Food items are still valid, but slightly below true ingredient records.
    elif record["food_class"] == "FOOD_ITEM":
        score += 30
        reasons.append("FOOD_ITEM")

    # -------------------------------------------------------------------------
    # PREPARED FOOD PENALTY
    # -------------------------------------------------------------------------

    if record["is_prepared_food"]:
        score -= 90
        reasons.append("PREPARED_FOOD_PENALTY")

    # -------------------------------------------------------------------------
    # PREPARED / COMPOSITE DESCRIPTION PENALTY
    # -------------------------------------------------------------------------

    prepared_words = d_set.intersection(NEGATIVE_PREPARED_WORDS)

    if prepared_words:
        score -= min(60, len(prepared_words) * 20)
        reasons.append("COMPOSITE_FOOD_PENALTY")

    # -------------------------------------------------------------------------
    # MODIFIER MATCHING
    # -------------------------------------------------------------------------

    query_raw = q_set.intersection(RAW_MODIFIERS)
    query_dried = q_set.intersection(DRIED_MODIFIERS)
    query_cooked = q_set.intersection(COOKED_MODIFIERS)

    desc_raw = d_set.intersection(RAW_MODIFIERS)
    desc_dried = d_set.intersection(DRIED_MODIFIERS)
    desc_cooked = d_set.intersection(COOKED_MODIFIERS)

    if query_raw:
        if query_raw.intersection(desc_raw):
            score += 45
            reasons.append("RAW_MODIFIER_MATCH")
        elif desc_cooked or desc_dried:
            score -= 35
            reasons.append("RAW_MODIFIER_MISMATCH")

    if query_dried:
        if query_dried.intersection(desc_dried):
            score += 45
            reasons.append("DRIED_MODIFIER_MATCH")
        elif desc_raw:
            score -= 20
            reasons.append("DRIED_MODIFIER_MISMATCH")

    if query_cooked:
        if query_cooked.intersection(desc_cooked):
            score += 35
            reasons.append("COOKED_MODIFIER_MATCH")

    # -------------------------------------------------------------------------
    # GENERIC SINGLE-WORD QUERY HANDLING
    # -------------------------------------------------------------------------

    # For queries like "garlic", "ginger", "cinnamon", "kelp":
    # an ingredient whose description starts with the concept should dominate
    # a prepared food that merely contains it.

    if len(q_tokens) == 1:

        term = q_tokens[0]

        if term in d_set:

            if d_tokens and d_tokens[0] == term:
                score += 70
                reasons.append("TERM_IS_PRIMARY_FOOD")

            elif term in STRONG_INGREDIENT_FIRST_WORDS:
                score += 30
                reasons.append("PRIMARY_INGREDIENT_SIGNAL")

            else:
                score -= 20
                reasons.append("CONTAINED_TERM")

    # -------------------------------------------------------------------------
    # SINGLE-WORD PREPARED FOOD SAFETY
    # -------------------------------------------------------------------------

    if len(q_tokens) == 1 and q_tokens[0] in d_set:

        if record["is_prepared_food"]:
            score -= 80
            reasons.append("SINGLE_TERM_PREPARED_PENALTY")

    # -------------------------------------------------------------------------
    # QUERY CONTAINS "OIL"
    # -------------------------------------------------------------------------

    if "oil" in q_set:

        if "oil" in d_set and record["is_ingredient_like"]:
            score += 65
            reasons.append("OIL_INGREDIENT_PREFERENCE")

        if "mayonnaise" in d_set:
            score -= 100
            reasons.append("MAYONNAISE_PENALTY")

    # -------------------------------------------------------------------------
    # QUERY CONTAINS SPECIFIC INGREDIENT CONCEPT
    # -------------------------------------------------------------------------

    if "garlic" in q_set and "garlic" in d_set:
        if "bread" in d_set:
            score -= 100
            reasons.append("GARLIC_BREAD_PENALTY")

    if "ginger" in q_set and "ginger" in d_set:
        if "pickled" in d_set or "canned" in d_set:
            score -= 60
            reasons.append("PICKLED_GINGER_PENALTY")

    if "kelp" in q_set and "kelp" in d_set:
        if "fish" in d_set or "herring" in d_set:
            score -= 100
            reasons.append("FISH_WITH_KELP_PENALTY")

    if "cinnamon" in q_set and "cinnamon" in d_set:
        if "bun" in d_set or "buns" in d_set:
            score -= 100
            reasons.append("CINNAMON_BUN_PENALTY")

    # -------------------------------------------------------------------------
    # RAW JACKFRUIT
    # -------------------------------------------------------------------------

    if "jackfruit" in q_set and "jackfruit" in d_set:

        if "raw" in q_set and "raw" in d_set:
            score += 50
            reasons.append("RAW_JACKFRUIT_MATCH")

        if "canned" in d_set:
            score -= 50
            reasons.append("CANNED_JACKFRUIT_PENALTY")

        if "syrup" in d_set:
            score -= 40
            reasons.append("SYRUP_JACKFRUIT_PENALTY")

    # -------------------------------------------------------------------------
    # PORK SHOULDER
    # -------------------------------------------------------------------------

    if "pork" in q_set and "shoulder" in q_set:

        if "pork" in d_set and "shoulder" in d_set:
            score += 50
            reasons.append("PORK_SHOULDER_MATCH")

        # Prefer fresh raw whole shoulder over specialty shoulder products
        if "fresh" in d_set and "raw" in d_set:
            score += 25
            reasons.append("FRESH_RAW_PORK_PREFERENCE")

        if "cured" in d_set:
            score -= 25
            reasons.append("CURED_PORK_PENALTY")

    # -------------------------------------------------------------------------
    # RETURN
    # -------------------------------------------------------------------------

    return score, reasons


# =============================================================================
# SEARCH
# =============================================================================

def search_candidates(query, limit=10):

    scored = []

    for record in records:

        score, reasons = score_candidate(query, record)

        if score == -math.inf:
            continue

        if score <= 0:
            continue

        scored.append({
            "query": query,
            "fdc_id": record["fdc_id"],
            "description": record["description"],
            "food_class": record["food_class"],
            "is_ingredient_like": record["is_ingredient_like"],
            "is_prepared_food": record["is_prepared_food"],
            "score": round(score, 4),
            "reasons": "|".join(reasons),
        })

    scored.sort(
        key=lambda x: (
            -x["score"],
            not x["is_ingredient_like"],
            x["description"]
        )
    )

    return scored[:limit]


# =============================================================================
# CONFIDENCE
# =============================================================================

def calculate_confidence(results):

    if not results:
        return 0.0

    top = results[0]["score"]

    if len(results) == 1:
        return 1.0

    second = results[1]["score"]

    if top <= 0:
        return 0.0

    # Margin-based confidence
    margin = max(0.0, top - second)

    # Saturating function
    margin_component = 1.0 - math.exp(-margin / 30.0)

    # Absolute-score component
    absolute_component = min(1.0, top / 200.0)

    confidence = (
        0.65 * margin_component +
        0.35 * absolute_component
    )

    return round(max(0.0, min(1.0, confidence)), 4)


# =============================================================================
# RESOLUTION
# =============================================================================

def resolve_ingredient(query):

    results = search_candidates(query, limit=10)

    if not results:
        return {
            "query": query,
            "status": "NOT_FOUND",
            "fdc_id": "",
            "description": "",
            "food_class": "",
            "score": 0,
            "confidence": 0,
            "reason": "NO_MATCHES",
        }

    top = results[0]

    confidence = calculate_confidence(results)

    if len(results) == 1:
        status = "RESOLVED"
        reason = "ONLY_MATCH"

    else:
        second = results[1]

        # Require a meaningful margin.
        margin = top["score"] - second["score"]

        if confidence >= 0.72 and margin >= 18:
            status = "RESOLVED"
            reason = "STRONG_BEST_MATCH"

        elif confidence >= 0.55 and margin >= 10:
            status = "RESOLVED"
            reason = "GOOD_BEST_MATCH"

        else:
            status = "AMBIGUOUS"
            reason = "TOP_RESULTS_TOO_CLOSE"

    return {
        "query": query,
        "status": status,
        "fdc_id": top["fdc_id"],
        "description": top["description"],
        "food_class": top["food_class"],
        "score": top["score"],
        "confidence": confidence,
        "reason": reason,
    }


# =============================================================================
# TEST SUITE
# =============================================================================

TEST_QUERIES = [
    ("cinnamon", "Spices, cinnamon, ground"),
    ("ground cinnamon", "Spices, cinnamon, ground"),
    ("turmeric", "Spices, turmeric, ground"),
    ("fresh basil", "Basil, fresh"),
    ("dried basil", "Spices, basil, dried"),
    ("olive oil", "Oil, olive, salad or cooking"),
    ("oil olive", "Oil, olive, salad or cooking"),
    ("baking powder", "Leavening agents, baking powder, low-sodium"),
    ("jackfruit", "Jackfruit, raw"),
    ("raw jackfruit", "Jackfruit, raw"),
    ("kelp", "Seaweed, kelp, raw"),
    ("seaweed kelp", "Seaweed, kelp, raw"),
    ("pork shoulder", "Pork, fresh, shoulder, whole, separable lean and fat, raw"),
    ("shoulder pork", "Pork, fresh, shoulder, whole, separable lean and fat, raw"),
    ("black pepper", "Spices, pepper, black"),
    ("garlic", "Garlic, raw"),
    ("ginger", "Ginger root, raw"),
    ("rice", "Rice, white, medium-grain, raw, enriched"),
    ("flour", "Potato flour"),
    ("sugar", "Sugar, turbinado"),
]


print()
print("=" * 80)
print("RUNNING CORRECTED STEP 5 RESOLUTION TESTS")
print("=" * 80)

test_rows = []

for query, expected in TEST_QUERIES:

    result = resolve_ingredient(query)

    passed = normalize_text(result["description"]) == normalize_text(expected)

    print()
    print(f"{query}: {'PASS' if passed else 'FAIL'}")
    print(f"  Status:       {result['status']}")
    print(f"  FDC ID:       {result['fdc_id']}")
    print(f"  Description:  {result['description']}")
    print(f"  Class:        {result['food_class']}")
    print(f"  Score:        {result['score']}")
    print(f"  Confidence:   {result['confidence']}")
    print(f"  Reason:       {result['reason']}")

    test_rows.append({
        **result,
        "expected_description": expected,
        "passed": passed,
    })


# =============================================================================
# SEARCH RESULTS
# =============================================================================

print()
print("=" * 80)
print("BUILDING SEARCH RESULT TABLE")
print("=" * 80)

search_rows = []

for query, _ in TEST_QUERIES:

    results = search_candidates(query, limit=10)

    for rank, result in enumerate(results, start=1):

        search_rows.append({
            **result,
            "rank": rank,
        })

search_df = pd.DataFrame(search_rows)


# =============================================================================
# CANONICAL SELF-RESOLUTION TABLE
# =============================================================================

print()
print("=" * 80)
print("BUILDING CANONICAL SELF-RESOLUTION TABLE")
print("=" * 80)

resolution_rows = []

for record in records:

    # Resolve the normalized description itself.
    results = search_candidates(record["description_normalized"], limit=5)

    if results:

        best = results[0]

        self_match = (
            str(best["fdc_id"]) == str(record["fdc_id"])
        )

        resolution_rows.append({
            "fdc_id": record["fdc_id"],
            "description": record["description"],
            "description_normalized": record["description_normalized"],
            "food_class": record["food_class"],
            "is_ingredient_like": record["is_ingredient_like"],
            "is_prepared_food": record["is_prepared_food"],
            "self_resolved_fdc_id": best["fdc_id"],
            "self_resolved_description": best["description"],
            "self_resolution_score": best["score"],
            "self_resolution_confidence": calculate_confidence(results),
            "self_resolution_pass": self_match,
        })

    else:

        resolution_rows.append({
            "fdc_id": record["fdc_id"],
            "description": record["description"],
            "description_normalized": record["description_normalized"],
            "food_class": record["food_class"],
            "is_ingredient_like": record["is_ingredient_like"],
            "is_prepared_food": record["is_prepared_food"],
            "self_resolved_fdc_id": "",
            "self_resolved_description": "",
            "self_resolution_score": 0,
            "self_resolution_confidence": 0,
            "self_resolution_pass": False,
        })


resolution_df = pd.DataFrame(resolution_rows)
test_df = pd.DataFrame(test_rows)


# =============================================================================
# VALIDATION
# =============================================================================

print()
print("=" * 80)
print("STEP 5 FINAL VALIDATION")
print("=" * 80)

total_tests = len(test_df)
passed_tests = int(test_df["passed"].sum())
failed_tests = total_tests - passed_tests

print(f"Test queries:       {total_tests}")
print(f"Tests PASS:         {passed_tests}")
print(f"Tests FAIL:         {failed_tests}")

class_counts = canonical.copy()

if class_col:
    class_counts["_normalized_class"] = (
        class_counts[class_col]
        .apply(normalize_class)
    )

    print()
    print("FOOD CLASS SUMMARY")

    for cls, count in (
        class_counts["_normalized_class"]
        .value_counts()
        .items()
    ):
        print(f"{cls:<35} {count:,}")

self_pass_count = int(
    resolution_df["self_resolution_pass"].sum()
)

self_total = len(resolution_df)

print()
print(f"Canonical self-resolution: "
      f"{self_pass_count:,}/{self_total:,}")

# =============================================================================
# SAVE
# =============================================================================

print()
print("=" * 80)
print("SAVING STEP 5 OUTPUT")
print("=" * 80)

resolution_df.to_csv(
    OUT_RESOLUTION,
    index=False,
    encoding="utf-8-sig"
)

search_df.to_csv(
    OUT_SEARCH_RESULTS,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    OUT_TEST_RESULTS,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# QUALITY REPORT
# =============================================================================

report_lines = []

report_lines.append(
    "SR LEGACY STEP 5 — INGREDIENT RESOLUTION QUALITY REPORT"
)

report_lines.append("=" * 80)

report_lines.append(
    f"Canonical records: {len(canonical):,}"
)

report_lines.append(
    f"Resolution records: {len(resolution_df):,}"
)

report_lines.append(
    f"Search records: {len(search_df):,}"
)

report_lines.append(
    f"Test queries: {total_tests:,}"
)

report_lines.append(
    f"Tests PASS: {passed_tests:,}"
)

report_lines.append(
    f"Tests FAIL: {failed_tests:,}"
)

report_lines.append(
    f"Self-resolution PASS: {self_pass_count:,}/{self_total:,}"
)

report_lines.append("")
report_lines.append("TEST RESULTS")
report_lines.append("-" * 80)

for _, row in test_df.iterrows():

    report_lines.append(
        f"{row['query']}: "
        f"{'PASS' if row['passed'] else 'FAIL'} | "
        f"{row['fdc_id']} | "
        f"{row['description']} | "
        f"score={row['score']} | "
        f"confidence={row['confidence']}"
    )

OUT_REPORT.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)
print("STEP 5 OUTPUT FILES")
print("=" * 80)

print(f"Saved: {OUT_RESOLUTION}")
print(f"Saved: {OUT_SEARCH_RESULTS}")
print(f"Saved: {OUT_TEST_RESULTS}")
print(f"Saved: {OUT_REPORT}")

print()
print("=" * 80)
print("SR LEGACY STEP 5 COMPLETE")
print("=" * 80)

if failed_tests == 0:
    print("STATUS: PASS")
else:
    print(
        f"STATUS: REVIEW — {failed_tests} test(s) failed"
    )

print("=" * 80)


SR LEGACY STEP 5 — INGREDIENT RESOLUTION APPLICATION LAYER

STEP 4 DIRECTORY:
C:\Users\AK\Downloads\zip\sr_legacy_step4

OUTPUT DIRECTORY:
C:\Users\AK\Downloads\zip\sr_legacy_step5

CHECKING STEP 4 FILES
PASS: sr_legacy_ingredient_canonical.csv
PASS: sr_legacy_ingredient_lookup.csv

LOADING STEP 4 DATA
Canonical rows: 7,793
Lookup rows:    7,793

BUILDING CORRECTED INGREDIENT RESOLUTION ENGINE
Indexed records: 7,793

RUNNING CORRECTED STEP 5 RESOLUTION TESTS

cinnamon: FAIL
  Status:       AMBIGUOUS
  FDC ID:       172664
  Description:  Bagels, cinnamon-raisin
  Class:        OTHER
  Score:        130.0
  Confidence:   0.2275
  Reason:       TOP_RESULTS_TOO_CLOSE

ground cinnamon: PASS
  Status:       RESOLVED
  FDC ID:       171320
  Description:  Spices, cinnamon, ground
  Class:        OTHER
  Score:        135.0
  Confidence:   0.8631
  Reason:       STRONG_BEST_MATCH

turmeric: PASS
  Status:       RESOLVED
  FDC ID:       172231
  Description:  Spices, turmeric, ground
  Class: 